# Self-Supervised Hybrid CNN+ViT PCB Anomaly Detection -- Thesis Evaluation Notebook
### VisA PCB1-PCB4 | leak-free baseline comparison | ablations | robustness benchmark | statistics | Excel | demo

**What this notebook establishes, and how:**

| Question | Experiment | Where |
|---|---|---|
| How accurate is the model on the standard benchmark? | Held-out AUROC + thresholded metrics | Cell 10, Cell 18 |
| Is the CNN+ViT fusion better than either backbone alone? How should it be weighted? | Branch + fusion-rule ablation (no refit) | Cell 15c |
| What does rotation/lighting invariance cost on aligned images? | Same model refit without invariance augmentation | Cell 15c |
| How does it compare to established methods, fairly? | PatchCore, PatchCore at matched bank size, PaDiM, EfficientAD -- identical data roles | Cells 16-18 |
| Does invariance pay off where it should? | Rotated / relit / cropped copies of held-out images, all models | Cells 15d, 20 |
| Can alignment to a golden reference replace learned invariance? | Same shifted images after canonicalization, all models | Cells 7c, 15d, 20 |
| Does our model work when trained on aligned boards too? | Canonical pipeline: fit, calibrate and test on canonicalized images | Cell 15e |
| Which complete system should be deployed? | Systems ranked by robustness, conclusions generated from the tests | Cell 20b, `results_summary.md` |
| Can our model be improved without touching held-out data? | Pre-registered variants, selected on calibration only | Cell 15c2 |
| Board vs background surface: is only the board inspected? | Automatic board mask per class, shown for checking | Cell 15d2 |
| Angle-independent at EVERY angle? Defects found in cropped photos? | Every 30 deg; crops 90-60% per side | Cell 20d |
| Do the top-system comparisons survive re-training with other seeds? | Top 3 systems re-trained with extra seeds | Cell 20c |
| Are the test sets confounded by how images were captured? | Orientation-only and lighting-only "detectors" | Cell 15e |
| Which TYPE of defect is each highlighted region? | Per-class linear classifier on the frozen CNN+ViT features pooled inside each region; calibration -> eval | Cells 12b, 12c |
| Inspect a photo with the recommended system | Registration -> canonicalization -> detector -> regions & priorities | Cells 22a, 22 |
| Are differences real or noise? | Stratified paired bootstrap 95% CIs; verdicts only when the CI excludes 0 | every table |


## Table of contents (run order)

| step | cell | what it does |
|---|---|---|
| 1 | 1 | Install (BEFORE any import) & imports |
| 2 | 2 | Config |
| 3 | 2b | THESIS DOCUMENTATION TOOLKIT: run log, section timer, every figure saved & numbered, tables, fail-safe steps |
| 4 | 3 | Locate ZIP file, extract dataset, find DATA_ROOT |
| 5 | 4 | Robust image discovery & per-class file lists |
| 6 | 5 | Preprocessing (letterbox + illumination normalization) & invariance augmentation |
| 7 | 6 | Hybrid CNN + ViT frozen feature extractor |
| 8 | 6b | Exact GPU k-NN: drop-in replacement for sklearn's NearestNeighbors |
| 9 | 7 | Greedy coreset subsampling (memory bank) + threshold-search utilities |
| 10 | 7b | Evaluation utilities: paired bootstrap CIs, robustness perturbations, |
| 11 | 7c | Golden-reference CANONICALIZATION: undo rotation / scale / crop / lighting |
| 12 | 8 | Per-class hybrid PatchCore-style detector (our model) |
| 13 | 9 | Train + calibrate OUR model, one detector per PCB class |
| 14 | D1 | DATASET OVERVIEW: what the model learns from and what it is tested on |
| 15 | D2 | PREPROCESSING & ALIGNMENT WALKTHROUGH: every step a photo goes through, shown on a hard example |
| 16 | D3 | MODEL ARCHITECTURE & WHAT THE TWO BRANCHES SEE |
| 17 | 10 | Evaluate OUR model on the full held-out eval split |
| 18 | 11 | Region localization, defect-type shape hints, priority, visualize_result |
| 19 | 12 | Save all trained detectors |
| 19a | 12b | DEFECT-TYPE LABELS + REGION DATASET: ground-truth defect blobs and detector-highlighted regions |
| 19b | 12c | REGION DEFECT-TYPE CLASSIFIER: trained on CALIBRATION regions, reported on held-out EVAL regions |
| 20 | 13 | test_single_image(): ad-hoc single-image testing |
| 21 | 14 | Secondary contribution: orientation-aware canonical-alignment pathway (opt-in) |
| 22 | 15 | inspect_pcb(): practical quick-screen + severity + report wrapper |
| 23 | 15b | Hyperparameter sweep: KNN_K and TOPK_SCORE_FRAC (calibration-set signal) |
| 24 | 15c | ABLATIONS: (a) which branch carries the signal? (b) what does invariance cost? |
| 25 | 15c2 | MODEL IMPROVEMENT STUDY: close the same-backbone gap, selected on the CALIBRATION split only |
| 26 | 15d | Build the robustness test sets (the experiment the thesis claim depends on) |
| 27 | 15d2 | BOARD vs BACKGROUND SURFACE: only the board is inspected |
| 28 | 15e | CANONICAL PIPELINE for our model: fit, calibrate AND evaluate on canonicalized images |
| 29 | 15e2 | CANONICAL-PIPELINE STUDY (final, PRE-REGISTERED): letterboxing x CLAHE, selected on calibration only |
| 30 | 15e3 | PIPELINE-CONSISTENT AUGMENTATION STUDY (pre-registered, selected on CALIBRATION shift sets) |
| 31 | D4 | OUR FINAL MODEL: settings, memory banks, thresholds and score distributions |
| 32 | D5 | ABLATION OF THE FINAL PIPELINE: what each component contributes (documentation only) |
| 33 | 16 | Baseline setup: install Anomalib, build LEAK-FREE fit / predict folders |
| 34 | 17 | Train baselines on the fit folder, predict on the held-out predict folder |
| 35 | 18 | Main comparison on the SAME held-out images for every model: |
| 36 | D6 | ROC & PRECISION-RECALL CURVES, OPERATING POINTS, CONFUSION MATRICES |
| 37 | 19a | Ensemble (ours + a baseline): blend weight tuned on the CALIBRATION split, |
| 38 | 19 | Per-image inventory: EVERY test image, every model (for the Excel workbook) |
| 39 | 20 | ROBUSTNESS BENCHMARK: raw shifts vs. shifts + canonicalization, every model |
| 40 | 20b | FINAL RESULTS: complete inspection SYSTEMS, conclusions, deployment recommendation |
| 41 | 20c | SEED ROBUSTNESS: do the top-system comparisons survive RE-TRAINING with other random seeds? |
| 42 | 20d | ANGLE, CROP & LIGHTING SWEEPS: independent of angle and light? defects found in cropped photos? |
| 43 | 20e | PIXEL-LEVEL INSPECTION ENGINE & LOCALIZATION METRICS: WHERE exactly is the defect? |
| 43a | 20e2 | DEFECT TYPES ON THE FINAL PIPELINE'S REGIONS: what a user of the deployed system actually gets |
| 44 | 20f | ROTATION & LIGHTING GALLERIES: the same defective board at every angle and under every light |
| 45 | 21 | Export everything to one multi-sheet Excel workbook |
| 46 | 22a | DEPLOYABLE INSPECTION: run ANY evaluated system, including the one Cell 20b recommends |
| 47 | 22 | DEMO: honest qualitative results with the RECOMMENDED system (or any system / your own photos) |
| 48 | 22b | DEMO: 2 GOOD + 2 DEFECTIVE BOARDS PER PCB -- verdict, heatmap, highlighted defect, pixel details |
| 49 | 22c | TEST ANY IMAGE: point MY_IMAGE_PATH at a photo of a PCB |
| 50 | 99 | THESIS OUTPUT INDEX: every figure, table and log of this run, organised for writing |

In [ ]:
# ============================================================
# CELL 1 -- Install (BEFORE any import) & imports
# ============================================================
# Every pip install happens here, before numpy / torch are imported. In an earlier run Anomalib
# was installed later (Cell 16) and upgraded numpy to 2.5.3 underneath a kernel that had already
# imported the older numpy -- one process, two numpy versions. It happened to work; it is not
# reproducible. Kaggle's core stack is pinned with a constraints file so Anomalib cannot replace it
# (a new Anomalib release could otherwise swap out the CUDA build of torch). Only if Anomalib truly
# needs newer versions do we fall back to an unconstrained install -- still before any import --
# and say so. Installed versions are recorded in the Excel 'Protocol' sheet.
INSTALL_ANOMALIB = True     # False if you will not run the baseline comparison (CFG.RUN_BASELINES)

import subprocess, sys
import importlib.metadata as _md

_CORE = ["numpy", "pandas", "torch", "torchvision", "scikit-learn", "scipy",
         "opencv-python", "opencv-python-headless", "opencv-contrib-python"]
with open("/tmp/core_constraints.txt", "w") as _f:
    for _p in _CORE:
        try:
            _f.write(_p + "==" + _md.version(_p) + "\n")
        except _md.PackageNotFoundError:
            pass


def _pip(*args):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)


_r = _pip("timm>=0.9.16", "openpyxl", "-c", "/tmp/core_constraints.txt")
if _r.returncode != 0:
    print("timm/openpyxl install problem:", _r.stderr.strip()[-400:])

ANOMALIB_INSTALL_MODE = "skipped (INSTALL_ANOMALIB=False)"
if INSTALL_ANOMALIB:
    _r = _pip("anomalib", "-c", "/tmp/core_constraints.txt")
    ANOMALIB_INSTALL_MODE = "constrained -- Kaggle core stack unchanged"
    if _r.returncode != 0:
        print("Anomalib needs versions outside Kaggle's core stack; installing unconstrained (before any import).")
        print("   pip said:", _r.stderr.strip()[-300:])
        _r = _pip("anomalib")
        ANOMALIB_INSTALL_MODE = ("unconstrained -- core stack may have changed" if _r.returncode == 0
                                 else "FAILED: " + _r.stderr.strip()[-200:])

import os, glob, json, math, time, random, zipfile, pickle, shutil, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import os as _os
_os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # deterministic cuBLAS -- must precede the first CUDA call
import torch
import torch.nn as nn
import torch.nn.functional as F

import timm
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

VERSIONS = {}
for _p in ["numpy", "pandas", "torch", "torchvision", "timm", "scikit-learn", "scipy", "opencv-python", "anomalib"]:
    try:
        VERSIONS[_p] = _md.version(_p)
    except _md.PackageNotFoundError:
        pass
print("Versions:", VERSIONS)
print("Anomalib install:", ANOMALIB_INSTALL_MODE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected -- enable GPU in Kaggle notebook settings for reasonable runtime.")

In [ ]:
# ============================================================
# CELL 2 -- Config
# ============================================================
class CFG:
    CLASSES = ["pcb1", "pcb2", "pcb3", "pcb4"]

    # IMG_SIZE must be a multiple of 14 (DINOv2 ViT-S/14 patch size) AND reasonably
    # divisible by the CNN's stride-8 layer2 output for clean alignment. 336 satisfies both.
    IMG_SIZE = 336

    CNN_BACKBONE = "wide_resnet50_2"
    VIT_BACKBONE = "vit_small_patch14_dinov2.lvd142m"   # DINOv2 ViT-S/14, self-supervised pretrained
    NEIGHBORHOOD_POOL = 3                    # local-neighborhood aggregation kernel for the CNN branch (PatchCore trick)

    MAX_TRAIN_IMAGES_PER_CLASS = 150         # raised from 100: richer candidate pool for the coreset
    AUGS_PER_IMAGE = 5                       # invariance views per training image (0 = the Cell 15c ablation)

    CORESET_RATIO = 0.02                     # fraction of patch vectors kept in memory bank
    CORESET_MAX_POINTS_FOR_GREEDY = 20000    # cap for the greedy coreset pass (keeps it fast)
    MAX_BANK_SIZE = 5000                     # per-branch memory-bank cap. HONEST RECORD of this value: it was
                                              # raised from 2000 citing a 'PatchCore 159K-point vs 2000-point' comparison that
                                              # was WRONG -- Anomalib's default PatchCore bank is 15,359 points and the earlier
                                              # 'matched' variant really had ~193-482 points. Held-out AUROC did not measurably
                                              # change (0.841 -> 0.836), and a 482-point PatchCore beat this 5,000-point model,
                                              # so bank size is NOT this model's bottleneck. Kept at 5000 because GPU k-NN
                                              # (Cell 6b) makes it cheap. Hard cap on final memory-bank size, regardless of
                                              # CORESET_RATIO -- bank size drives THREE separate
                                              # costs that compound across this notebook's several
                                              # memory banks (main CNN/ViT + canonical CNN/ViT, x4
                                              # classes): the coreset greedy-selection loop (linear
                                              # in bank size), compute_bank_density's self-kNN
                                              # (QUADRATIC in bank size), and every single k-NN
                                              # query for the rest of the notebook's life (fit,
                                              # calibration, evaluation, every test_single_image
                                              # call).
                                              #
                                              # CUT from 8000 to 2000 after a real Kaggle run's
                                              # papermill cell timings showed Cell 9 (fit + calibrate)
                                              # alone taking 124 minutes -- with the quadratic density
                                              # step almost certainly the dominant cost at bank size
                                              # 8000 (an 8000x8000 self-kNN is 16x more expensive than
                                              # a 2000x2000 one). This is a real accuracy/speed
                                              # trade-off, not a free win -- a smaller coreset is a
                                              # less exhaustive reference set. Raise this back up if
                                              # you have compute budget to spare and want to test
                                              # whether it measurably improves AUROC; 2000 diverse
                                              # (greedy farthest-point-selected) points per bank is
                                              # still well within the range PatchCore's own ablations
                                              # show working reasonably well.

    KNN_K = 5                                # raised from 3 after the Cell 15b sweep on the CALIBRATION split
                                              # favored larger K for 3/4 classes (pcb3 preferred K=1). Held-out AUROC did not
                                              # measurably change -- report as a sensitivity result, not an improvement.
    TOPK_SCORE_FRAC = 0.03                   # raised from 0.01 on the same calibration-sweep evidence. Image
                                              # score = mean of the top X% most anomalous patches, NOT a single max (Cell 8)

    USE_DENSITY_REWEIGHT = True              # PatchCore-style local-density re-weighting (Cell 7/8):
                                              # a test patch matched to a memory-bank point sitting in a
                                              # DENSE/typical region gets its distance AMPLIFIED (more
                                              # suspicious); a match to a SPARSE/rare-but-legitimate
                                              # region gets DAMPENED -- reduces false positives from
                                              # naturally rare normal patterns (text labels, mounting
                                              # holes) without losing sensitivity to real defects
    DENSITY_K = 3                            # neighbors used to estimate each bank point's local density
    DENSITY_MAX_AMPLIFY = 8.0                # safety cap on how much a single sparse/dense match can
                                              # amplify a raw distance, in case of near-duplicate bank points
    THRESHOLD_PERCENTILE = 99.0              # fallback percentile if a calib split has no anomaly examples
    CALIBRATION_FRAC = 0.5                   # fraction of each class's labeled test images used to PICK
                                              # the threshold; the rest is held out for Cell 10's AUROC.
                                              #
                                              # Tried raising this to 0.65 to close the gap between the
                                              # calibration-set FPR target and actual held-out FPR (see
                                              # MAX_FALSE_POSITIVE_RATE below) -- REVERTED after a real
                                              # A/B comparison at matched settings showed no real payoff:
                                              # held-out FPR barely moved (15.9% -> 15.5%, noise-level)
                                              # while recall came out slightly WORSE (51.5% -> 46.4%) and
                                              # the smaller eval set gave noisier Cell 10 metrics for no
                                              # benefit. Combined with MAX_FALSE_POSITIVE_RATE's revert
                                              # below, three real runs now consistently show this
                                              # generalization gap doesn't respond to threshold-selection
                                              # hyperparameters -- it reflects genuine class separability
                                              # limits (AUROC ~0.68-0.90) and calibration-sample noise at
                                              # this scale (~100-130 images/class), not a methodology bug
                                              # fixable by tuning CFG constants further.

    TTA_CALIBRATION_MAX_IMAGES = None        # calibrate the TTA threshold on the FULL calibration split (was
                                              # capped at 50 for runtime; affordable now with GPU k-NN)

    REGION_PERCENTILE = 94.0                 # per-image percentile used to binarize the anomaly map
                                              # for region localization (Cell 11) -- lower than before
                                              # (was 97) to capture a fuller halo around each defect so
                                              # the merge step below has enough to consolidate into one
                                              # big region instead of several small hot-pixel fragments
    REGION_MERGE_FACTOR = 1.6                # circles within this many x (r1+r2) of each other get
                                              # merged into one bigger region -- see merge_close_regions
    MAX_REGIONS = 2                          # report at most 1-2 big regions per image, not many small ones

    MAX_FALSE_POSITIVE_RATE = 0.10           # the FINAL decision threshold (single-pass, TTA, and
                                              # canonical) is chosen to maximize recall SUBJECT TO this
                                              # false-positive-rate cap on the calibration set, instead
                                              # of pure F1 optimization. F1 treats false positives and
                                              # false negatives as equally costly and is tuned on an
                                              # artificially balanced 50/50 calibration set -- verified
                                              # in a real run that this can silently produce a threshold
                                              # with an unacceptable ~50% false-positive rate against a
                                              # realistic, mostly-normal deployment stream (rejecting
                                              # half of all genuinely good boards).
                                              #
                                              # REVERTED from 0.07 back to 0.10 -- a real run at 0.07
                                              # showed this lever backfiring: held-out FPR barely moved
                                              # (15.9%->12.7%) while held-out recall COLLAPSED
                                              # (51.5%->37.9%, and one class dropped to 25.7% recall --
                                              # missing 3 of every 4 real defects). The target-overshoot
                                              # ratio (held-out FPR / calibration target) also got WORSE,
                                              # not better (1.59x -> 1.81x): tightening this value is not
                                              # actually closing the calibration/held-out generalization
                                              # gap, it's just paying an ever-steeper recall cost while
                                              # still missing the target. That pattern -- a steep
                                              # recall-vs-FPR trade-off that doesn't improve with a
                                              # stricter target -- is a sign of a genuine separability
                                              # ceiling (AUROC ~0.68-0.90 here), not a threshold-tuning
                                              # problem. If you need a materially lower real-world FPR,
                                              # the effective lever is improving the underlying score
                                              # quality (CFG.MAX_TRAIN_IMAGES_PER_CLASS, CFG.AUGS_PER_IMAGE,
                                              # CFG.CORESET_RATIO), not squeezing this value further.

    # Orientation-aware canonical-alignment pathway (Cell 14) -- an OPTIONAL, additive
    # second pathway that estimates a board's true rotation and screens for defects that
    # are DEFINED by orientation (wrong polarity, tombstoning), which the main
    # rotation-INVARIANT pathway above is structurally blind to by design.
    BUILD_CANONICAL_PATHWAY = False          # Cell 14 only builds/calibrates this pathway if True.
                                              # DEFAULT OFF after a real Kaggle run's papermill timings
                                              # showed Cell 14 alone taking 47 minutes (20% of a 3h45m
                                              # total run) for a feature that is already off by default
                                              # at inference (test_single_image/inspect_pcb only use it
                                              # when you explicitly pass check_orientation=True). Most
                                              # deployments don't need wrong-polarity/tombstoning
                                              # detection specifically and can skip this cost entirely.
                                              # Set True (then re-run Cell 14) if you do want it --
                                              # check_orientation=True will otherwise just print a note
                                              # and skip gracefully, same as if the pathway were never
                                              # built for a class.
    CANONICAL_ANGLE_COARSE_STEP = 45         # degrees, initial coarse orientation search
    CANONICAL_ANGLE_FINE_STEP = 10           # degrees, refinement step size
    CANONICAL_ANGLE_FINE_RANGE = 30          # +/- degrees around the coarse best angle to refine

    EVAL_MAX_IMAGES_PER_CLASS = None         # FULL held-out eval split. Was 60, which silently compared our
                                              # model on 60 images/class against baselines on 100 -- not the same data. GPU
                                              # k-NN makes the full set cheap. Must stay None for the paired statistics.

    TTA_ANGLES = [0, 90, 180, 270]           # rotation test-time augmentation angles

    KAGGLE_INPUT = "/kaggle/input"
    WORK_DIR = "/kaggle/working"
    EXTRACT_DIR = "/kaggle/working"
    ARTIFACT_DIR = "/kaggle/working/pcb_anomaly_artifacts"

    # ---- Efficiency ----
    USE_GPU_KNN = True                       # exact GPU k-NN (Cell 6b), self-checked against sklearn; ~80% of
                                              # scoring time was CPU k-NN in a profiled run

    # ---- Statistics ----
    BOOTSTRAP_N = 2000                       # stratified paired-bootstrap replicates for every CI
    PAC_CONFIDENCE = 0.90                    # Cell 18's second operating point: held-out false alarms <= target
                                              # MAX_FALSE_POSITIVE_RATE with this probability (pac_threshold, Cell 7b)

    # ---- Ablations (Cell 15c) ----
    RUN_ABLATION = True                      # branch ablation (free) + no-augmentation refit (~10 min)

    # ---- Baselines (Cells 16-18) ----
    ANOMALIB_ROOT = "/kaggle/working/anomalib_data"
    RUN_BASELINES = True
    BASELINE_MODEL_NAMES = ["PatchCore", "PatchCore (matched bank)", "PaDiM", "EfficientAD",
                            "PatchCore (canonical pipeline)", "EfficientAD (canonical pipeline)",
                            "STFPM (canonical pipeline)", "Reverse Distillation (canonical pipeline)"]
                                              # "(canonical pipeline)" = trained, calibrated AND tested on canonicalized
                                              # boards, exactly like our canonical pipeline (Cell 15e) -- so a comparison of
                                              # DETECTORS is not confounded with a comparison of PIPELINES. A real run showed
                                              # test-only canonicalization left EfficientAD sensitive to reference-filled
                                              # border strips (scores rose with fill area in 23/24 groups). Drop the
                                              # EfficientAD variant to save ~40 min.
                                              # keep PatchCore FIRST: it is fast, so if raw-score mode is not
                                              # supported by the installed Anomalib this is discovered before
                                              # EfficientAD's long training (see Cell 17)
    EFFICIENTAD_EPOCHS = 20                  # already far below the paper's schedule; lowering it further would
                                              # handicap the strongest baseline -- remove EfficientAD instead
    ANOMALIB_PATCHES_PER_IMAGE = 1024        # Anomalib PatchCore patch grid per image (32x32 at its 256 px
                                              # input). Verified from a real log: ratio 0.1 on 150 images
                                              # selected 15,359 points = 1,024/image. Used to size the
                                              # matched-bank variant to exactly MAX_BANK_SIZE.
    BASELINE_RAW_SCORES = True               # disable Anomalib's min-max+clamp post-processing (clamping
                                              # creates ties that distort AUROC under distribution shift);
                                              # falls back automatically if unsupported

    # ---- Robustness benchmark (Cells 15d, 20) ----
    RUN_ROBUSTNESS = True
    ROBUSTNESS_CONDITIONS = ["identity", "rot90", "rot180", "rot15", "dark", "uneven_light", "crop"]
    ROBUSTNESS_MAX_IMAGES_PER_CLASS = 60     # class-balanced subset of the HELD-OUT eval split
    ROBUSTNESS_LONG_SIDE = 512               # every condition (incl. the identity control) re-saved at this size
    ROBUSTNESS_CROP_FRAC = 0.8               # centered crop keeps 80% of each side
    ROBUSTNESS_CROP_MIN_DEFECT_FRAC = 0.9    # anomalous images kept for 'crop' only if >=90% of the defect
                                              # (ground-truth mask) survives -- keeps labels truthful
    ROBUSTNESS_DIR = "/kaggle/working/robustness_sets"
    ROBUSTNESS_BASELINES = ["PatchCore", "EfficientAD", "PatchCore (canonical pipeline)", "EfficientAD (canonical pipeline)",
                            "STFPM (canonical pipeline)", "Reverse Distillation (canonical pipeline)"]
    BASELINE_EPOCHS = {"STFPM": 50, "Reverse Distillation": 50}   # trainable baselines besides EfficientAD (EFFICIENTAD_EPOCHS)   # strongest two; others skip the extra predictions
    ROBUSTNESS_INCLUDE_TTA = True            # TTA is our mode built for arbitrary angles -- test it under rotation
    ROBUSTNESS_CANONICALIZE = True           # ALSO score every condition after golden-reference canonicalization
                                              # (Cell 7c) -- the "+canon" columns of Part 3, for EVERY model
    CANON_ORB_FEATURES = 3000                # ~40% faster than 5000 with equal accuracy on synthetic boards
    CANON_MIN_INLIERS = 20                   # fewer RANSAC inliers -> registration declared failed (and counted)
    CANON_RATIO_TEST = 0.8                   # Lowe ratio test for ORB matches
    CANON_ILLUM_SIGMA_FRAC = 0.08            # illumination field = Gaussian low-pass at 8% of image size; defects
                                              # much smaller than that survive (verified on synthetic boards)
    CANON_ECC_SIZE = 256                     # resolution of the orientation check + sub-pixel ECC refinement (7c v2)
    CANON_PHOTOMETRIC_GATE = 0.15            # correct lighting ONLY if the median |log brightness ratio| to the
    CANON_GATE_PERCENTILE = 90               # FINAL BUILD bug fix: the gate tests this percentile of |log gain| over the board. It
                                             # used the MEDIAN, so shadows / glare covering < half the board never triggered a correction
    SATURATION_LEVEL = 250                   # FINAL BUILD over-exposure / glare gate: a pixel is clipped when ALL channels are >= this
    SATURATION_GATE_FRAC = 1.0               # DISABLED after run 30: the gate fired on 44.6% of NORMAL photos (VisA boards have
                                             # naturally clipped white areas) and cut clean AUROC 0.950 -> 0.872. 1.0 = never fires.
    SATURATION_DILATE_FRAC = 0.004           # margin around clipped pixels (fraction of the long side) also treated as unseen
    GLOBAL_BETA_GRID = (0.0, 0.5, 1.0)       # FINAL BUILD v4: global-evidence correction -- image score minus beta x the board's
                                             # MEDIAN patch score; beta chosen per class on calibration data only
    GLOBAL_BETA_MIN_GAIN = 0.005             # beta > 0 is adopted only if it raises calibration AUROC by at least this much
                                              # reference exceeds this (~16%); correctly-lit boards stay untouched
                                              # (ungated correction altered every image and hurt PatchCore under
                                              # uneven light in a real run)
    CANON_FEATHER_PX = 8                     # blend width where unseen regions are filled in from the reference

    # ---- Canonical pipeline for OUR model (Cell 15e) ----
    RUN_CANON_PIPELINE = True                # fit + calibrate our model ON canonicalized images (train/test
                                              # consistency). A real run showed canonicalizing only at TEST time
                                              # cut our model's clean AUROC 0.848 -> 0.694 while the baselines were
                                              # unaffected. No invariance augmentation: canonicalization replaces it.
    CANON_PIPELINE_DIR = "/kaggle/working/canonical_pipeline"
    CANON_REFERENCE_CANDIDATES = 30          # golden reference = the most TYPICAL of the first N training boards
                                              # (median brightness); the first image alone was atypically lit for
                                              # pcb4 in a real run (the lighting gate fired on 71% of its boards)
    DETERMINISTIC_BASELINES = True           # deterministic cuDNN kernels while training baselines. Seeds alone did not
                                              # make EfficientAD reproducible (identical seeds: pcb3 0.913 vs 0.842)
    SEED_STUDY_EXTRA_SEEDS = 2               # Cell 20c: re-train the 3 top systems with this many EXTRA seeds (~65 min
                                              # each, mostly EfficientAD); 0 skips. Claims then hold across training seeds.
    BOARD_ONLY = True                        # aligned systems score ONLY the board (not the background surface) and only
                                              # the part of the board that was in the photo
    BOARD_DEFECT_MAX_DIST = 0.15             # board mask grown LOCALLY to calibration defects within this fraction of the
                                              # long side of the board (convex hull) -- never a uniform margin
    CANON_RETRY_UPSCALE = 1.5                # failed alignment (e.g. a half-board photo): retry once on an upscaled copy
    BOARD_MASK_MODE = "auto"                 # "auto" = find the board on each class's reference (Cell 15d2 shows it);
                                              # "full" = treat the whole frame as board
    SEEN_DIR = "/kaggle/working/seen_masks"  # per-image "what the photo showed" masks of aligned images
    RUN_SWEEPS = True                        # Cell 20d: AUROC at every SWEEP_ANGLES and SWEEP_CROPS (~30 min)
    SWEEP_ANGLES = [0, 15, 30, 45, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360]   # degrees, counter-clockwise
    SWEEP_LIGHTS = ["bright x1.6", "dim x0.5", "very dim x0.3", "gamma 0.6", "gamma 1.6", "gradient left-right",
                    "gradient top-bottom", "vignette", "warm light", "cool light", "low contrast", "shadow band", "glare spot"]
    DISK_WARN_GB = 3.0                       # every section logs free disk space; below this it lists the largest folders
    DOC_FIG_DPI = 150                        # resolution of every saved thesis figure (thesis_outputs/figures)
    RUN_FINAL_ABLATION = True                # Cell D5 (~20 min): what each component of the final pipeline contributes
    PIXEL_NORMAL_QUANTILE = 0.90             # pixel highlight threshold = strongest pixel of 90% of CALIBRATION normal boards
    PIXEL_CALIB_IMAGES = 40                  # calibration normal boards per class used for that threshold
    PIXEL_NEG_SAMPLES = 4000                 # normal pixels sampled per board for the pixel-level AUROC
    DEMO_GOOD_PER_CLASS = 2                  # Cell 22b: random held-out NORMAL boards shown per class
    DEMO_BAD_PER_CLASS = 2                   # Cell 22b: random held-out DEFECTIVE boards shown per class
    SWEEP_CROPS = [0.9, 0.8, 0.7, 0.6, 0.5]  # fraction of each side kept in the photo (centred)
    SWEEP_HALVES = ["left", "right", "top", "bottom"]   # photos showing only HALF of the board
    SWEEP_IMAGES_PER_CLASS = 40              # held-out boards per class (half normal, half defective)
    RUN_AUG_STUDY = True                     # Cell 15e3: one repaired training copy per board, selected on calibration
                                              # shift sets; if selected, used by EVERY canonical-pipeline detector
    RUN_CANON_STUDY = True                   # Cell 15e2: final pre-registered letterbox x CLAHE study of the canonical
                                              # pipeline (~30 min); selection on calibration only
    RUN_IMPROVEMENT_STUDY = True             # Cell 15c2: pre-registered, calibration-selected model improvements
    DEPLOY_BASELINES = ["EfficientAD", "PatchCore", "PatchCore (canonical pipeline)", "EfficientAD (canonical pipeline)",
                        "STFPM (canonical pipeline)", "Reverse Distillation (canonical pipeline)"]   # keep these trained Anomalib models (instead of freeing them)
                                              # so the inspection tool (Cell 22a) can run whichever system Cell 20b
                                              # recommends -- ~0.5 GB of GPU memory in total

    # ---- Defect type of every highlighted region (Part 1.4, Cells 12b-12c) ----
    RUN_DEFECT_CLASSIFIER = True             # supervised type head on the FROZEN self-supervised features (detector untouched)
    VISA_ANNOTATION_DIR = None               # folder holding <class>/image_anno.csv, if you supply VisA's label files yourself
    FETCH_VISA_ANNOTATIONS = True            # the converted Kaggle VisA has no type labels: stream the official archive once and
                                              # keep only the four pcb*/image_anno.csv files (needs the notebook's internet)
    VISA_TAR_URL = "https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar"
    DEFECT_MIN_CALIB_PER_TYPE = 2            # a type needs this many CALIBRATION images to be its own class, else "other"
    DEFECT_C_GRID = [0.01, 0.1, 1.0, 10.0]   # logistic-regression C, chosen by grouped CV on CALIBRATION regions only
    REGION_MIN_DEFECT_PIXELS = 25            # ground-truth defect blobs smaller than this (photo pixels) are ignored
    REGION_MIN_CELLS = 3                     # every region is pooled over at least 3 x 3 feature-grid cells (local context)
    REGION_HIT_FRAC = 0.25                   # a highlighted region is "on a defect" if it covers >= 25% of a true defect blob
    DEFECT_GALLERY_PER_CLASS = 2             # held-out boards per class in the region-type gallery
    RUN_DEFECT_TYPE_FINAL_EVAL = True        # Cell 20e2 (~15-20 min): type the FINAL pipeline's regions on every held-out defect
    DEFECT_TYPE_BOX = "auto"                 # box of an inspect_board region the classifier reads: "region" (whole region box),
                                              # "highlighted" (tight box around its above-threshold pixels) or "auto" = chosen by
                                              # Cell 20e2 on CALIBRATION defects with out-of-fold classifiers

    # ---- Outputs / extras ----
    EXCEL_OUTPUT_PATH = "/kaggle/working/pcb_all_results.xlsx"
    SCORE_ALL_TEST_IMAGES = True
    KNN_K_SWEEP = [1, 3, 5, 7, 10]
    TOPK_SCORE_FRAC_SWEEP = [0.005, 0.01, 0.02, 0.05, 0.1]
    ENSEMBLE_ALPHA_GRID_STEPS = 11

assert CFG.IMG_SIZE % 14 == 0, "IMG_SIZE must be a multiple of 14 for the DINOv2 ViT-S/14 patch grid."
assert CFG.EVAL_MAX_IMAGES_PER_CLASS is None, "paired statistics need the full eval split"
os.makedirs(CFG.ARTIFACT_DIR, exist_ok=True)
os.makedirs(CFG.ANOMALIB_ROOT, exist_ok=True)

## Documentation toolkit

In [ ]:
# ============================================================
# CELL 2b -- THESIS DOCUMENTATION TOOLKIT: run log, section timer, every figure saved & numbered, tables, fail-safe steps
# ============================================================
# Everything this notebook shows is ALSO written to /kaggle/working/thesis_outputs (zipped at the end, Cell 99):
#   figures/FigNNN_<section>.png  -- EVERY figure of the notebook, numbered in order; captions in figures_index.csv / .md
#   tables/TabNNN_<name>.csv      -- every documented table, indexed in tables_index.csv (and thesis_tables.xlsx)
#   run_log.txt                   -- every log line with clock time and minutes since the start
#   section_timings.csv           -- how long every section took
# Documentation-only steps run inside `doc_step(...)`: if one fails it is logged and the run CONTINUES, so a figure can
# never cost the results of a multi-hour run. Training and evaluation cells are deliberately NOT wrapped.
import re, time, datetime, traceback, shutil
from contextlib import contextmanager

DOC_DIR = "/kaggle/working/thesis_outputs"
FIG_DIR, TAB_DIR = os.path.join(DOC_DIR, "figures"), os.path.join(DOC_DIR, "tables")
for _d in (DOC_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(_d, exist_ok=True)
RUN_LOG_PATH = os.path.join(DOC_DIR, "run_log.txt")
FIGURES, TABLES, STEPS = [], [], []
_DOC = {"section": None, "title": "", "t0": time.time(), "section_t0": time.time(), "figs": 0, "next_caption": ""}


def log(msg, level="INFO"):
    line = "[%s | +%6.1f min | %-7s] %s" % (datetime.datetime.now().strftime("%H:%M:%S"),
                                            (time.time() - _DOC["t0"]) / 60.0, level, msg)
    print(line)
    with open(RUN_LOG_PATH, "a") as fh:
        fh.write(line + "\n")


def _close_section():
    if _DOC["section"] is not None:
        STEPS.append({"section": _DOC["section"], "title": _DOC["title"],
                      "minutes": round((time.time() - _DOC["section_t0"]) / 60.0, 2), "figures": _DOC["figs"]})


def disk_free_gb(path="/kaggle/working/"):
    try:
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return float("nan")


def disk_report(top=10, root="/kaggle/working/"):
    """Largest folders under /kaggle/working (Kaggle keeps at most ~20 GB there)."""
    sizes = []
    for name in os.listdir(root):
        p = os.path.join(root, name)
        tot = 0
        for dp, _, fs in os.walk(p):
            for f in fs:
                try:
                    tot += os.lstat(os.path.join(dp, f)).st_size
                except OSError:
                    pass
        sizes.append((tot / 1e9, name) if os.path.isdir(p) else (os.lstat(p).st_size / 1e9, name))
    for gb, name in sorted(sizes, reverse=True)[:top]:
        print("   %7.2f GB  %s" % (gb, name))


def doc_section(label, title):
    """Called automatically at the top of every code cell: logs and times the section, and watches the disk."""
    _close_section()
    _DOC.update(section=label, title=title, section_t0=time.time(), figs=0)
    print("\n" + "=" * 110)
    free = disk_free_gb()
    log("%s -- %s | disk free %.1f GB" % (label, title, free), "SECTION")
    if free < CFG.DISK_WARN_GB:
        log("LOW DISK SPACE (%.1f GB free) -- largest folders:" % free, "WARN")
        disk_report()
    print("=" * 110)


def caption(text):
    """Caption for the NEXT figure shown (written to figures_index)."""
    _DOC["next_caption"] = text


def _slug(s):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_")[:60]


_plt_show_original = plt.show


def _show_and_save(*args, **kwargs):
    """Replacement for plt.show(): every open figure is saved (numbered, captioned), shown, then closed."""
    for num in plt.get_fignums():
        fig = plt.figure(num)
        n = len(FIGURES) + 1
        name = "Fig%03d_%s.png" % (n, _slug(_DOC["section"] or "setup"))
        try:
            fig.savefig(os.path.join(FIG_DIR, name), dpi=CFG.DOC_FIG_DPI, bbox_inches="tight")
            auto = (fig._suptitle.get_text() if getattr(fig, "_suptitle", None) else "") or \
                "; ".join(a.get_title() for a in fig.axes if a.get_title())[:240]
            FIGURES.append({"figure": n, "file": name, "section": _DOC["section"], "section title": _DOC["title"],
                            "caption": _DOC["next_caption"] or auto})
            _DOC["figs"] += 1
            print("[Figure %d saved: thesis_outputs/figures/%s]" % (n, name))
        except Exception as e:
            print("[figure not saved: %r]" % (e,))
    _DOC["next_caption"] = ""
    _plt_show_original(*args, **kwargs)
    plt.close("all")


plt.show = _show_and_save


def save_table(df, name, caption_text=""):
    """Writes a documented table to tables/ (CSV), indexes it and logs it."""
    if df is None or len(df) == 0:
        return
    n = len(TABLES) + 1
    fn = "Tab%03d_%s.csv" % (n, _slug(name))
    df.to_csv(os.path.join(TAB_DIR, fn), index=False)
    TABLES.append({"table": n, "file": fn, "section": _DOC["section"], "caption": caption_text or name})
    log("Table %d saved: thesis_outputs/tables/%s (%d rows) -- %s" % (n, fn, len(df), caption_text or name))


@contextmanager
def doc_step(name):
    """A documentation-only step: a failure is logged and the run continues."""
    t = time.time()
    log("step: " + name)
    try:
        yield
        log("step done: %s (%.1f s)" % (name, time.time() - t))
    except Exception as e:
        log("step FAILED (documentation only -- the run continues): %s -- %r" % (name, e), "WARN")
        traceback.print_exc()


log("Documentation toolkit ready. All figures, tables and logs are also written to " + DOC_DIR)
log("Run configuration: classes %s | image size %d | seed %d" % (CFG.CLASSES, CFG.IMG_SIZE, SEED))

In [ ]:
# ============================================================
# CELL 3 -- Locate ZIP file, extract dataset, find DATA_ROOT
doc_section("CELL 3", "Locate ZIP file, extract dataset, find DATA_ROOT")
# (confirmed-working approach, wired to CFG paths)
# ============================================================
zip_files = glob.glob(os.path.join(CFG.KAGGLE_INPUT, "**", "*.zip"), recursive=True)
print("ZIP Files Found:\n")
for z in zip_files:
    print(z)


def find_pcb_root(search_dir):
    """Walk search_dir looking for a folder that directly contains a 'pcb1' subfolder."""
    if not os.path.isdir(search_dir):
        return None
    for root, dirs, files in os.walk(search_dir):
        if "pcb1" in dirs:
            return root
        lower_dirs = [d.lower() for d in dirs]
        if "pcb1" in lower_dirs:
            return root
    return None


# Skip re-extraction if this session (or a pre-extracted dataset) already has it.
DATA_ROOT = find_pcb_root(CFG.KAGGLE_INPUT) or find_pcb_root(CFG.EXTRACT_DIR)

if DATA_ROOT is None:
    if not zip_files:
        raise FileNotFoundError(
            "No .zip file found under " + CFG.KAGGLE_INPUT +
            " and no already-extracted 'pcb1' folder either. Add the VisA dataset via 'Add Data' and re-run."
        )
    visa_zip = zip_files[0]
    print("Extracting...")
    os.makedirs(CFG.EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(visa_zip, "r") as zip_ref:
        zip_ref.extractall(CFG.EXTRACT_DIR)
    print("Done")

    DATA_ROOT = find_pcb_root(CFG.EXTRACT_DIR)
    if DATA_ROOT is None:
        raise FileNotFoundError("Could not find a 'pcb1' folder anywhere under " + CFG.EXTRACT_DIR)
else:
    print("Found already-extracted VisA data -- skipping re-extraction.")

print("DATA_ROOT:", DATA_ROOT)
VISA_ROOT = DATA_ROOT   # kept for readability elsewhere in the notebook

In [ ]:
# ============================================================
# CELL 4 -- Robust image discovery & per-class file lists
doc_section("CELL 4", "Robust image discovery & per-class file lists")
# Handles both common VisA layouts:
#   A) 1-cls / MVTec-style:  <cat>/train/good/*, <cat>/test/good/*, <cat>/test/bad/*
#   B) raw VisA:             <cat>/Data/Images/Normal/*, <cat>/Data/Images/Anomaly/*
# Falls back to keyword search on any nested folder name otherwise.
# (confirmed-working approach; output is converted into this notebook's
#  FILE_LISTS[cls] = {"train_normal", "test_images", "test_labels"} format
#  so every downstream cell -- preprocessing, hybrid extractor, memory bank,
#  training, evaluation, test_single_image -- needs no further changes.)
# ============================================================
IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".JPG", ".PNG", ".JPEG", ".BMP")
TEST_NORMAL_FRAC = 0.15


def _list_images(folder):
    if not os.path.isdir(folder):
        return []
    out = []
    for r, _, files in os.walk(folder):
        for f in files:
            if f.endswith(IMG_EXTS):
                out.append(os.path.join(r, f))
    return sorted(out)


def gather_category_splits(cat_dir):
    """Returns dict: {"train_normal": [...], "test_normal": [...], "test_anomaly": [...]}"""
    # --- Layout A: train/good, test/good, test/bad ---
    train_good = os.path.join(cat_dir, "train", "good")
    test_good = os.path.join(cat_dir, "test", "good")
    test_bad = os.path.join(cat_dir, "test", "bad")

    if os.path.isdir(train_good):
        train_normal = _list_images(train_good)
        test_normal = _list_images(test_good)
        test_anomaly = _list_images(test_bad)
        if len(test_anomaly) == 0 and os.path.isdir(os.path.join(cat_dir, "test")):
            for sub in os.listdir(os.path.join(cat_dir, "test")):
                if sub.lower() not in ("good",):
                    test_anomaly += _list_images(os.path.join(cat_dir, "test", sub))
        return {"train_normal": train_normal, "test_normal": test_normal, "test_anomaly": test_anomaly}

    # --- Layout B: Data/Images/Normal, Data/Images/Anomaly ---
    normal_dir = os.path.join(cat_dir, "Data", "Images", "Normal")
    anomaly_dir = os.path.join(cat_dir, "Data", "Images", "Anomaly")

    if os.path.isdir(normal_dir):
        normal_imgs = _list_images(normal_dir)
        anomaly_imgs = _list_images(anomaly_dir)
        random.Random(SEED).shuffle(normal_imgs)
        n_test = max(1, int(len(normal_imgs) * TEST_NORMAL_FRAC))
        test_normal = normal_imgs[:n_test]
        train_normal = normal_imgs[n_test:]
        return {"train_normal": train_normal, "test_normal": test_normal, "test_anomaly": anomaly_imgs}

    # --- Fallback: keyword search anywhere under the category folder ---
    all_imgs = _list_images(cat_dir)
    normal_kw = ("normal", "good")
    anomaly_kw = ("anomaly", "bad", "defect", "abnormal")
    normal_imgs, anomaly_imgs = [], []
    for p in all_imgs:
        lp = p.lower()
        if any(k in lp for k in anomaly_kw):
            anomaly_imgs.append(p)
        elif any(k in lp for k in normal_kw):
            normal_imgs.append(p)
    random.Random(SEED).shuffle(normal_imgs)
    n_test = max(1, int(len(normal_imgs) * TEST_NORMAL_FRAC))
    test_normal = normal_imgs[:n_test]
    train_normal = normal_imgs[n_test:]
    return {"train_normal": train_normal, "test_normal": test_normal, "test_anomaly": anomaly_imgs}


FILE_LISTS = {}
for cls in CFG.CLASSES:
    cat_dir = os.path.join(VISA_ROOT, cls)
    splits = gather_category_splits(cat_dir)

    test_images = splits["test_normal"] + splits["test_anomaly"]
    test_labels = [0] * len(splits["test_normal"]) + [1] * len(splits["test_anomaly"])

    FILE_LISTS[cls] = {
        "train_normal": splits["train_normal"],
        "test_images": test_images,
        "test_labels": test_labels,
    }

    print(cls + ": train_normal=" + str(len(splits["train_normal"])) +
          "  test_normal=" + str(len(splits["test_normal"])) +
          "  test_anomaly=" + str(len(splits["test_anomaly"])))

# Fail fast with a clear message here rather than a cryptic np.concatenate crash in Cell 9
# if a category's folder layout didn't match any of the patterns gather_category_splits knows.
_missing_train = [c for c in CFG.CLASSES if len(FILE_LISTS[c]["train_normal"]) == 0]
for c in CFG.CLASSES:
    if len(FILE_LISTS[c]["test_images"]) == 0:
        print("WARNING: no test images found for '" + c + "' -- its AUROC will be unavailable "
              "and its threshold will fall back to reusing a few train images.")

if _missing_train:
    raise RuntimeError(
        "No normal training images found for: " + ", ".join(_missing_train) +
        ". Check that " + VISA_ROOT + "/<class> matches one of the layouts gather_category_splits "
        "expects (train/good + test/good + test/bad, or Data/Images/Normal + Data/Images/Anomaly)."
    )

In [ ]:
# ============================================================
# CELL 5 -- Preprocessing (letterbox + illumination normalization) & invariance augmentation
# ============================================================
doc_section("CELL 5", "Preprocessing (letterbox + illumination normalization) & invariance augmentation")
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def letterbox_square(img_bgr, fill_mode=cv2.BORDER_REFLECT_101):
    """Pad to a square canvas (no aspect-ratio distortion) so that cropped / partial /
    non-square input photos are never stretched before resizing to the model's input size."""
    h, w = img_bgr.shape[:2]
    side = max(h, w)
    top = (side - h) // 2
    bottom = side - h - top
    left = (side - w) // 2
    right = side - w - left
    return cv2.copyMakeBorder(img_bgr, top, bottom, left, right, borderType=fill_mode)


def illumination_normalize(img_bgr):
    """CLAHE on the L channel (Lab space) -- suppresses shadows / uneven lighting / specular
    highlights before either backbone ever sees the image."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge([l, a, b])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def preprocess_bgr(img_bgr, size=CFG.IMG_SIZE, normalize_light=True, letterbox=True):
    img = letterbox_square(img_bgr) if letterbox else img_bgr   # letterbox=False: full frame, stretched like PatchCore
    if normalize_light:
        img = illumination_normalize(img)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img_rgb = (img_rgb - IMAGENET_MEAN) / IMAGENET_STD
    tensor = torch.from_numpy(img_rgb.transpose(2, 0, 1)).float()
    return tensor


def load_image(path, size=CFG.IMG_SIZE, normalize_light=True, letterbox=True):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return preprocess_bgr(img, size=size, normalize_light=normalize_light, letterbox=letterbox)


def load_image_from_array(img_rgb_uint8, size=CFG.IMG_SIZE, normalize_light=True, letterbox=True):
    img_bgr = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2BGR)
    return preprocess_bgr(img_bgr, size=size, normalize_light=normalize_light, letterbox=letterbox)


def random_photometric_geometric_augment(img_bgr):
    """Used only to build a rotation/lighting-diverse MEMORY BANK from normal images.
    Random full-range rotation + perspective jitter + brightness/contrast/gamma jitter +
    synthetic shadow -- the memory bank ends up already containing the appearance
    variation the detector must be robust to at test time."""
    h, w = img_bgr.shape[:2]

    angle = random.uniform(0, 360)
    scale = random.uniform(0.92, 1.08)
    center = ((w - 1) / 2.0, (h - 1) / 2.0)   # mathematically correct pixel center (w/2,h/2 is off by half a pixel)
    M = cv2.getRotationMatrix2D(center, angle, scale)
    img = cv2.warpAffine(img_bgr, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)

    if random.random() < 0.5:
        d = 0.06
        src = np.float32([[0, 0], [w, 0], [0, h], [w, h]])
        dst = np.float32([
            [random.uniform(0, d) * w, random.uniform(0, d) * h],
            [w - random.uniform(0, d) * w, random.uniform(0, d) * h],
            [random.uniform(0, d) * w, h - random.uniform(0, d) * h],
            [w - random.uniform(0, d) * w, h - random.uniform(0, d) * h],
        ])
        Mp = cv2.getPerspectiveTransform(src, dst)
        img = cv2.warpPerspective(img, Mp, (w, h), borderMode=cv2.BORDER_REFLECT_101)

    alpha = random.uniform(0.7, 1.3)
    beta = random.uniform(-35, 35)
    img = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    gamma = random.uniform(0.7, 1.4)
    inv_gamma = 1.0 / gamma
    table = (np.linspace(0, 1, 256) ** inv_gamma * 255).astype(np.uint8)
    img = cv2.LUT(img, table)

    if random.random() < 0.4:
        overlay = img.copy()
        x1, y1 = random.randint(0, w), random.randint(0, h)
        x2, y2 = random.randint(0, w), random.randint(0, h)
        cv2.line(overlay, (x1, y1), (x2, y2), (0, 0, 0), thickness=random.randint(w // 4, w // 2))
        img = cv2.addWeighted(overlay, 0.25, img, 0.75, 0)

    return img


def random_crop_view(img_bgr, min_frac=0.5, max_frac=0.95):
    """Simulates a partially-visible / zoomed-in board region so the memory bank also
    contains sub-region appearance, not just full-board views."""
    h, w = img_bgr.shape[:2]
    frac = random.uniform(min_frac, max_frac)
    ch, cw = int(h * frac), int(w * frac)
    y0 = random.randint(0, h - ch)
    x0 = random.randint(0, w - cw)
    return img_bgr[y0:y0 + ch, x0:x0 + cw]


def rotate_image_np(img_np, angle_deg, border_mode=cv2.BORDER_REFLECT_101):
    """Rotates a HxWxC array by angle_deg (clockwise, degrees). For EXACT 90/180/270,
    uses cv2.rotate -- a true lossless pixel permutation, zero interpolation -- instead of
    the general affine warp. This matters: verified that cv2.getRotationMatrix2D +
    warpAffine, even for a 'clean' 90/180/270 degree rotation, differs from the true
    rotation by up to 64% of pixels on a fine-structure test pattern (PCB traces/text are
    exactly this kind of fine structure), because (a) no single rotation-center choice
    makes the general affine path exactly reproduce a lossless permutation for all three
    angles simultaneously, and (b) it still applies bilinear interpolation regardless.
    This directly matters for TTA (Cell 8's score_image_tensor_tta, which rotates at
    exactly 0/90/180/270) and the canonical-alignment pathway's coarse search (Cell 14,
    which includes exact 90/180/270 among its coarse candidates by default) -- both were
    silently injecting rotation-artifact noise into otherwise-normal images at those
    angles. Falls back to a standard affine warp for any other angle, using the
    mathematically correct (w-1)/2,(h-1)/2 pixel-center convention (a plain w/2,h/2
    center is off by half a pixel for 0-indexed pixel grids)."""
    angle_deg = angle_deg % 360
    if angle_deg == 0:
        return img_np
    if angle_deg == 90:
        return cv2.rotate(img_np, cv2.ROTATE_90_CLOCKWISE)
    if angle_deg == 180:
        return cv2.rotate(img_np, cv2.ROTATE_180)
    if angle_deg == 270:
        return cv2.rotate(img_np, cv2.ROTATE_90_COUNTERCLOCKWISE)
    h, w = img_np.shape[:2]
    center = ((w - 1) / 2.0, (h - 1) / 2.0)
    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    return cv2.warpAffine(img_np, M, (w, h), borderMode=border_mode)

In [ ]:
# ============================================================
# CELL 6 -- Hybrid CNN + ViT frozen feature extractor
# ============================================================
doc_section("CELL 6", "Hybrid CNN + ViT frozen feature extractor")
class HybridFeatureExtractor(nn.Module):
    """Two frozen, pretrained branches fused per patch (no gradient training):

    - CNN branch (Wide-ResNet-50-2, ImageNet-supervised): out_indices=(2,3) pulls
      'layer2' (stride 8) and 'layer3' (stride 16) -- strong LOCAL texture features,
      good for solder joints, scratches, hairline cracks.
    - ViT branch (DINOv2 ViT-S/14, self-supervised): patch tokens reshaped back into a
      spatial grid -- strong GLOBAL context features, good for component layout /
      missing / misaligned parts relative to the rest of the board.

    Fusion: both branches are aligned to the CNN's finer (layer2) spatial resolution and
    L2-NORMALIZED PER BRANCH (their raw feature scales differ substantially -- without
    this, one branch would dominate any distance computed on the raw features).
    forward_branches() returns the two normalized branches SEPARATELY -- PCBAnomalyDetector
    (Cell 8) scores each branch against its OWN memory bank and fuses the two SCORES with
    a calibrated weight, which is more robust than fusing raw features into one combined
    k-NN, where one branch's noise can silently dominate the distance metric. forward()
    keeps the old concatenated-feature behavior only as a smoke test below.
    """

    def __init__(self, cnn_name=CFG.CNN_BACKBONE, vit_name=CFG.VIT_BACKBONE, pool_k=CFG.NEIGHBORHOOD_POOL):
        super().__init__()

        # ---- CNN branch ----
        self.cnn = timm.create_model(cnn_name, pretrained=True, features_only=True, out_indices=(2, 3))
        self.cnn.eval()
        for p in self.cnn.parameters():
            p.requires_grad = False
        self.pool = nn.AvgPool2d(kernel_size=pool_k, stride=1, padding=pool_k // 2)

        # ---- ViT branch (DINOv2) ----
        # timm's DINOv2 weights default to a fixed 518x518 input size and will raise an
        # AssertionError ("Input height (...) doesn't match model (518)") for any other
        # resolution unless dynamic_img_size=True, which makes timm interpolate DINOv2's
        # pretrained position embeddings on the fly for whatever CFG.IMG_SIZE we pass in.
        self.vit = timm.create_model(vit_name, pretrained=True, num_classes=0, dynamic_img_size=True)
        self.vit.eval()
        for p in self.vit.parameters():
            p.requires_grad = False
        patch_size = self.vit.patch_embed.patch_size
        self.vit_patch_size = patch_size[0] if isinstance(patch_size, (tuple, list)) else patch_size
        self.vit_num_prefix = getattr(self.vit, "num_prefix_tokens", 1)

    @torch.no_grad()
    def _cnn_grid(self, x):
        feats = self.cnn(x)                     # [layer2 (B,C1,H1,W1), layer3 (B,C2,H2,W2)]
        feats = [self.pool(f) for f in feats]
        target_hw = feats[0].shape[-2:]          # align to the finer (layer2) resolution
        aligned = [feats[0]]
        for f in feats[1:]:
            aligned.append(F.interpolate(f, size=target_hw, mode="bilinear", align_corners=False))
        return torch.cat(aligned, dim=1), target_hw

    @torch.no_grad()
    def _vit_grid(self, x, target_hw):
        B, C, H, W = x.shape
        tokens = self.vit.forward_features(x)    # (B, N, D)
        if isinstance(tokens, dict):              # some timm versions return a dict
            tokens = tokens.get("x", tokens.get("last_hidden_state"))
        tokens = tokens[:, self.vit_num_prefix:, :]   # drop CLS (+ register) tokens
        h = H // self.vit_patch_size
        w = W // self.vit_patch_size
        D = tokens.shape[-1]
        grid = tokens.reshape(B, h, w, D).permute(0, 3, 1, 2)      # (B, D, h, w)
        grid = F.interpolate(grid, size=target_hw, mode="bilinear", align_corners=False)
        return grid

    @torch.no_grad()
    def forward_branches(self, x):
        """Returns (cnn_feat, vit_feat, target_hw), each L2-normalized per branch but
        NOT concatenated -- this is what PCBAnomalyDetector actually uses."""
        cnn_feat, target_hw = self._cnn_grid(x)
        vit_feat = self._vit_grid(x, target_hw)
        cnn_feat = F.normalize(cnn_feat, dim=1) if getattr(self, "cnn_l2", True) else cnn_feat   # per detector (Cell 15c2)
        vit_feat = F.normalize(vit_feat, dim=1)
        return cnn_feat, vit_feat, target_hw

    @torch.no_grad()
    def forward(self, x):
        """Concatenated-feature view, kept only for the Cell 6 smoke test below --
        PCBAnomalyDetector uses forward_branches() instead."""
        cnn_feat, vit_feat, target_hw = self.forward_branches(x)
        return torch.cat([cnn_feat, vit_feat], dim=1)   # (B, C1+C2+D_vit, h, w)


extractor = HybridFeatureExtractor().to(DEVICE)
extractor.eval()

with torch.no_grad():
    dummy = torch.zeros(1, 3, CFG.IMG_SIZE, CFG.IMG_SIZE).to(DEVICE)
    cnn_feat, vit_feat, target_hw = extractor.forward_branches(dummy)
    print("Hybrid CNN+ViT extractor ready. Patch-grid:", tuple(target_hw),
          "| CNN branch:", tuple(cnn_feat.shape), "| ViT branch:", tuple(vit_feat.shape))

In [ ]:
# ============================================================
# CELL 6b -- Exact GPU k-NN: drop-in replacement for sklearn's NearestNeighbors
# ============================================================
doc_section("CELL 6b", "Exact GPU k-NN: drop-in replacement for sklearn's NearestNeighbors")
# EFFICIENCY FIX grounded in real per-cell timings from a full Kaggle run: feature
# extraction costs ~0.1 s per view (Cell 9 extracts ~3,600 views), yet EVERY scoring cell
# cost ~0.55 s PER IMAGE -- so ~80% of scoring time was scikit-learn's CPU brute-force
# k-NN (1,764 query patches x 5,000 bank points x 1,536 dims, per branch, per image).
#
# TorchExactKNN is EXACT brute-force Euclidean k-NN (not approximate) on the GPU, with the
# same fit()/kneighbors() interface and return conventions as sklearn (ascending float64
# distances, int64 indices). Rebinding the global name `NearestNeighbors` below makes every
# existing call site (Cells 7, 8, 14, 15b) use it unchanged. Before rebinding, it is
# checked against sklearn on random unit-norm data; on any disagreement it is NOT used and
# the notebook silently keeps sklearn (slower, identical results).
from sklearn.neighbors import NearestNeighbors as _SklearnNearestNeighbors


class TorchExactKNN:
    def __init__(self, n_neighbors=5, algorithm=None, metric="euclidean", query_chunk=4096, **unused):
        if metric != "euclidean":
            raise ValueError("TorchExactKNN only supports metric='euclidean'")
        self.n_neighbors = n_neighbors
        self.query_chunk = query_chunk
        self._bank = None
        self._bank_sq = None

    def fit(self, X):
        self._bank = torch.as_tensor(np.ascontiguousarray(X), dtype=torch.float32, device=DEVICE)
        self._bank_sq = (self._bank * self._bank).sum(dim=1)
        return self

    @torch.no_grad()
    def kneighbors(self, X, n_neighbors=None, return_distance=True):
        k = int(n_neighbors or self.n_neighbors)
        n_bank = int(self._bank.shape[0])
        if k > n_bank:
            raise ValueError("Expected n_neighbors <= n_samples_fit, but n_neighbors = %d, "
                             "n_samples_fit = %d" % (k, n_bank))
        X = np.ascontiguousarray(X)
        dist_chunks, idx_chunks = [], []
        for start in range(0, X.shape[0], self.query_chunk):
            q = torch.as_tensor(X[start:start + self.query_chunk], dtype=torch.float32, device=DEVICE)
            d2 = (q * q).sum(dim=1, keepdim=True) + self._bank_sq.unsqueeze(0) - 2.0 * (q @ self._bank.T)
            d2 = d2.clamp(min=0.0)
            vals, idx = torch.topk(d2, k, dim=1, largest=False, sorted=True)
            dist_chunks.append(vals.sqrt().cpu())
            idx_chunks.append(idx.cpu())
        dist = torch.cat(dist_chunks).numpy().astype(np.float64)
        ind = torch.cat(idx_chunks).numpy().astype(np.int64)
        return (dist, ind) if return_distance else ind


def _torch_knn_matches_sklearn():
    rng = np.random.default_rng(0)
    bank = rng.normal(size=(1500, 256)).astype(np.float32)
    bank /= np.linalg.norm(bank, axis=1, keepdims=True)        # unit-norm, like our branch features
    query = rng.normal(size=(700, 256)).astype(np.float32)
    query /= np.linalg.norm(query, axis=1, keepdims=True)
    d_ref, i_ref = _SklearnNearestNeighbors(n_neighbors=5).fit(bank).kneighbors(query, n_neighbors=5)
    d_new, i_new = TorchExactKNN(n_neighbors=5).fit(bank).kneighbors(query, n_neighbors=5)
    idx_agree = float((i_ref == i_new).mean())
    max_err = float(np.abs(d_ref - d_new).max())
    # self-query convention (used by compute_bank_density): each point's own index first
    _, i_self = TorchExactKNN(n_neighbors=2).fit(bank).kneighbors(bank, n_neighbors=2)
    self_first = float((i_self[:, 0] == np.arange(len(bank))).mean())
    return idx_agree, max_err, self_first


if CFG.USE_GPU_KNN:
    _agree, _err, _self_first = _torch_knn_matches_sklearn()
    if _agree >= 0.999 and _err < 1e-3 and _self_first == 1.0:
        NearestNeighbors = TorchExactKNN
        print("GPU exact k-NN ENABLED on", DEVICE, "| verified vs sklearn: index agreement",
              round(_agree, 5), "| max distance error", "%.2e" % _err, "| self-match first:", _self_first)
    else:
        NearestNeighbors = _SklearnNearestNeighbors
        print("GPU k-NN self-check did NOT match sklearn closely enough (agreement", _agree,
              ", max err", _err, ", self-first", _self_first, ") -- keeping sklearn for correctness.")
else:
    NearestNeighbors = _SklearnNearestNeighbors
    print("CFG.USE_GPU_KNN is False -- using sklearn NearestNeighbors (CPU).")

In [ ]:
# ============================================================
# CELL 7 -- Greedy coreset subsampling (memory bank) + threshold-search utilities
# ============================================================
doc_section("CELL 7", "Greedy coreset subsampling (memory bank) + threshold-search utilities")
def greedy_coreset(features, ratio=CFG.CORESET_RATIO, max_points_for_greedy=CFG.CORESET_MAX_POINTS_FOR_GREEDY,
                    max_bank_size=CFG.MAX_BANK_SIZE):
    """features: (N, D) numpy array of hybrid patch descriptors.
    Returns a (M, D) subset chosen by greedy k-center (farthest-point) sampling, which
    covers the feature space far more evenly than random sampling. For speed on Kaggle,
    greedy selection runs on a random sub-population when N is very large, and the final
    bank size is hard-capped at max_bank_size regardless of ratio (see CFG.MAX_BANK_SIZE
    -- this bounds the compute cost of everything downstream that scales with bank size:
    compute_bank_density's O(bank^2) self-kNN especially, plus every future query)."""
    n = features.shape[0]
    n_select = min(max(50, int(n * ratio)), max_bank_size)

    if n <= max_points_for_greedy:
        pool = features
        pool_idx = np.arange(n)
    else:
        pool_idx = np.random.choice(n, max_points_for_greedy, replace=False)
        pool = features[pool_idx]

    feats_t = torch.from_numpy(pool).float().to(DEVICE)
    n_pool = feats_t.shape[0]
    n_select_pool = min(n_select, n_pool)

    selected = [random.randrange(n_pool)]
    min_dists = torch.cdist(feats_t, feats_t[selected[-1]:selected[-1] + 1]).squeeze(1)

    for _ in tqdm(range(n_select_pool - 1), desc="coreset selection", leave=False):
        next_idx = int(torch.argmax(min_dists).item())
        selected.append(next_idx)
        new_dists = torch.cdist(feats_t, feats_t[next_idx:next_idx + 1]).squeeze(1)
        min_dists = torch.minimum(min_dists, new_dists)

    chosen_global_idx = pool_idx[np.array(selected)]
    return features[chosen_global_idx]


def compute_bank_density(bank, k=CFG.DENSITY_K):
    """For each point in a memory bank, returns the mean distance to its own k nearest
    OTHER points in the bank -- a simple local-density estimate used by
    PCBAnomalyDetector (Cell 8) for PatchCore-style score re-weighting: a test patch
    matched to a memory-bank point sitting in a DENSE (small value, well-supported /
    typical) region has its raw distance AMPLIFIED, while a match to a SPARSE (large
    value, rare-but-legitimate) region has its raw distance DAMPENED. Same intuition as
    PatchCore's official softmax re-weighting scheme, in a simpler, directly verifiable
    ratio form (raw_distance / this region's typical intra-bank spacing)."""
    n_neighbors = min(k + 1, len(bank))
    nn = NearestNeighbors(n_neighbors=n_neighbors, algorithm="auto", metric="euclidean")
    nn.fit(bank)
    dists, _ = nn.kneighbors(bank, n_neighbors=n_neighbors)
    return dists[:, 1:].mean(axis=1)   # column 0 is distance to self (always 0) -- drop it


def find_best_threshold_and_f1(scores, labels, fallback_percentile=CFG.THRESHOLD_PERCENTILE):
    """F1-optimal decision threshold (and the F1 it achieves) over (scores, labels).

    This is the fix for 'shows normal for anomaly images too': a pure
    99th-percentile-of-normal-scores threshold sits far in the tail of the NORMAL score
    distribution, and whenever normal/anomaly scores overlap substantially (which they
    do here -- see the AUROC numbers in Cell 10, well under 1.0), that tail-threshold
    ends up above most real anomaly scores too, so almost everything gets called NORMAL
    regardless of the actual image. Searching every candidate threshold for the one that
    maximizes F1 on a small LABELED calibration split (normal AND anomaly images, see
    Cell 9) finds a threshold that actually separates the two classes. The returned F1 is
    also what Cell 8's CNN/ViT blend-weight search (alpha) uses to pick how much to trust
    each branch for a given PCB class.

    Falls back to the old percentile-of-normals rule (F1=0.0) if the calibration split
    happens to contain only one class (so F1 can't be computed at all).
    """
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)

    if len(np.unique(labels)) < 2:
        normal_scores = scores[labels == 0]
        if len(normal_scores) > 0:
            return float(np.percentile(normal_scores, fallback_percentile)), 0.0
        return (float(np.median(scores)) if len(scores) > 0 else 0.0), 0.0

    candidates = np.unique(scores)
    best_thr, best_f1 = float(candidates[0]), -1.0
    for thr in candidates:
        preds = (scores > thr).astype(int)
        tp = int(((preds == 1) & (labels == 1)).sum())
        fp = int(((preds == 1) & (labels == 0)).sum())
        fn = int(((preds == 0) & (labels == 1)).sum())
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, best_f1


def find_best_threshold(scores, labels, fallback_percentile=CFG.THRESHOLD_PERCENTILE):
    """Thin wrapper over find_best_threshold_and_f1 for call sites that only need the
    threshold (kept so existing calls don't need to change). Used for the alpha
    blend-weight search (Cell 8) -- NOT for the final decision threshold; see
    find_best_threshold_constrained_fpr for that."""
    thr, _ = find_best_threshold_and_f1(scores, labels, fallback_percentile)
    return thr


def find_best_threshold_constrained_fpr(scores, labels, max_fpr=CFG.MAX_FALSE_POSITIVE_RATE,
                                         fallback_percentile=CFG.THRESHOLD_PERCENTILE):
    """Picks the threshold that MAXIMIZES RECALL (catches the most true anomalies)
    SUBJECT TO a false-positive-rate cap on the calibration set, instead of pure F1
    optimization. This is what CFG.MAX_FALSE_POSITIVE_RATE actually controls, and is
    used for every FINAL decision threshold in this notebook (self.threshold,
    self.threshold_tta, self.canonical_threshold) -- see CFG.MAX_FALSE_POSITIVE_RATE's
    comment for why plain F1 wasn't good enough (verified in a real run: an F1-optimal
    threshold produced a ~50% false-positive rate against a realistic, mostly-normal
    deployment folder).

    Note this caps FPR ON THE CALIBRATION SET, not a hard guarantee on fresh data --
    verified with synthetic data that generalization drift means real held-out FPR can
    still exceed the cap somewhat, but is meaningfully lower than a plain F1-optimal
    threshold's real-world FPR in the same test.

    Falls back to the percentile-of-normals rule if there's only one class present
    (same as find_best_threshold_and_f1), and to the single strictest threshold that
    achieves zero false positives on the calibration set if NO threshold satisfies the
    FPR cap (can happen with very few calibration images)."""
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)

    if len(np.unique(labels)) < 2:
        normal_scores = scores[labels == 0]
        if len(normal_scores) > 0:
            return float(np.percentile(normal_scores, fallback_percentile)), 0.0
        return (float(np.median(scores)) if len(scores) > 0 else 0.0), 0.0

    n_normal = int((labels == 0).sum())
    candidates = np.unique(scores)
    best_thr, best_recall = None, -1.0
    for thr in candidates:
        preds = (scores > thr).astype(int)
        fp = int(((preds == 1) & (labels == 0)).sum())
        fpr = fp / n_normal if n_normal > 0 else 0.0
        if fpr > max_fpr:
            continue
        tp = int(((preds == 1) & (labels == 1)).sum())
        fn = int(((preds == 0) & (labels == 1)).sum())
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        if recall > best_recall:
            best_recall = recall
            best_thr = float(thr)

    if best_thr is None:
        best_thr = float(candidates.max())
        best_recall = 0.0
    return best_thr, best_recall

In [ ]:
# ============================================================
# CELL 7b -- Evaluation utilities: paired bootstrap CIs, robustness perturbations,
doc_section("CELL 7b", "Evaluation utilities: paired bootstrap CIs, robustness perturbations,")
#            defect-mask checks, prediction-filename parsing, kwarg-fallback construction
# ============================================================
import re
from collections import defaultdict


class PairedBootstrap:
    """Stratified, PAIRED bootstrap for image-level AUROC.

    Resamples positives and negatives separately (so every replicate contains both
    classes) and reuses the SAME resamples for every model scored on the same image set
    -- so differences between two models are compared on identical resampled test sets
    (a paired comparison), which is far more sensitive and honest than comparing two
    independent confidence intervals.

    Vectorized exactly: AUROC = mean over (positive, negative) pairs of
    1[s_pos > s_neg] + 0.5 * 1[s_pos == s_neg]. A bootstrap replicate is just a
    reweighting of that pair matrix by how many times each positive / negative was drawn,
    so all replicates are one matrix product -- identical to sklearn's roc_auc_score on the
    resampled set (verified against it before this notebook was built), ~1000x faster.
    """

    def __init__(self, labels, n_boot=None, seed=SEED):
        n_boot = n_boot or CFG.BOOTSTRAP_N
        self.labels = np.asarray(labels).astype(int)
        self.pos = np.where(self.labels == 1)[0]
        self.neg = np.where(self.labels == 0)[0]
        self.valid = len(self.pos) > 0 and len(self.neg) > 0
        if self.valid:
            rng = np.random.default_rng(seed)
            self.w_pos = rng.multinomial(len(self.pos), np.full(len(self.pos), 1.0 / len(self.pos)), size=n_boot)
            self.w_neg = rng.multinomial(len(self.neg), np.full(len(self.neg), 1.0 / len(self.neg)), size=n_boot)

    def auroc(self, scores):
        """Returns (point_auroc, replicate_aurocs). NaN / empty if not computable."""
        s = np.asarray(scores, dtype=np.float64)
        if not self.valid or s.shape[0] != self.labels.shape[0] or np.isnan(s).any():
            return float("nan"), np.array([])
        sp = s[self.pos][:, None]
        sn = s[self.neg][None, :]
        pair = (sp > sn).astype(np.float64) + 0.5 * (sp == sn)
        point = float(pair.mean())
        reps = ((self.w_pos @ pair) * self.w_neg).sum(axis=1) / (len(self.pos) * len(self.neg))
        return point, reps


def ci95(reps):
    reps = np.asarray(reps)
    if reps.size == 0:
        return float("nan"), float("nan")
    lo, hi = np.percentile(reps, [2.5, 97.5])
    return float(lo), float(hi)


def fmt_ci(point, reps, nd=3):
    if point is None or (isinstance(point, float) and math.isnan(point)):
        return "n/a"
    lo, hi = ci95(reps)
    if math.isnan(lo):
        return str(round(point, nd))
    return str(round(point, nd)) + " [" + str(round(lo, nd)) + ", " + str(round(hi, nd)) + "]"


def diff_verdict(reps_a, reps_b):
    """Paired difference a - b over identical bootstrap replicates. Returns
    (mean_diff, lo, hi, verdict_string). Verdict only claims a difference when the 95%
    CI of the paired difference excludes zero -- otherwise 'no significant difference',
    which is the honest reading at this sample size, however the point estimates look."""
    reps_a, reps_b = np.asarray(reps_a), np.asarray(reps_b)
    if reps_a.size == 0 or reps_b.size == 0 or reps_a.size != reps_b.size:
        return float("nan"), float("nan"), float("nan"), "n/a"
    d = reps_a - reps_b
    lo, hi = ci95(d)
    if lo > 0:
        verdict = "significantly HIGHER"
    elif hi < 0:
        verdict = "significantly LOWER"
    else:
        verdict = "no significant difference"
    return float(d.mean()), lo, hi, verdict


def pac_threshold(scores, labels, max_fpr=None, confidence=None):
    """Threshold with a DISTRIBUTION-FREE false-alarm guarantee from a small calibration set.

    If a threshold leaves k of the n calibration normals above it, the false-alarm rate on new,
    exchangeable normal images is Beta(k+1, n-k)-distributed over calibration draws. This picks the
    LARGEST k (i.e. the best recall) whose `confidence` quantile is still <= max_fpr.
    Example, n=50, target 10%, 90% confidence -> k=1 (at most 1 calibration normal above the
    threshold), whereas the empirical '10% on calibration' rule allows k=5, which only promises ~12%
    ON AVERAGE and exceeded 20% for every model in a real run. The guarantee assumes exchangeable
    normals; clustered captures (scores correlated in capture order) can still violate it.
    """
    from scipy.stats import beta
    max_fpr = CFG.MAX_FALSE_POSITIVE_RATE if max_fpr is None else max_fpr
    confidence = CFG.PAC_CONFIDENCE if confidence is None else confidence
    s = np.asarray(scores, dtype=np.float64)
    y = np.asarray(labels, dtype=int)
    neg = np.sort(s[y == 0])[::-1]
    n = len(neg)
    if n == 0:
        return float("nan")
    ok = [k for k in range(n) if beta.ppf(confidence, k + 1, n - k) <= max_fpr]
    k = max(ok) if ok else 0          # too few normals for the guarantee -> strictest threshold seen
    return float(neg[k])              # decision rule is score > threshold: exactly <= k normals above


# ---------------- robustness perturbations (deterministic -- identical for every model) ----------------
def downscale_long_side(img, long_side):
    h, w = img.shape[:2]
    scale = long_side / float(max(h, w))
    if scale >= 1.0:
        return img
    return cv2.resize(img, (int(round(w * scale)), int(round(h * scale))), interpolation=cv2.INTER_AREA)


def rotate_bound(img, angle_deg, border=cv2.BORDER_REPLICATE, interpolation=cv2.INTER_LINEAR):
    """Rotate by an arbitrary angle on an EXPANDED canvas, so no board content is cut off
    (a defect near a corner stays in the image, so the image's label stays valid)."""
    h, w = img.shape[:2]
    c = ((w - 1) / 2.0, (h - 1) / 2.0)
    M = cv2.getRotationMatrix2D(c, angle_deg, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    nw = int(round(h * sin + w * cos))
    nh = int(round(h * cos + w * sin))
    M[0, 2] += (nw - 1) / 2.0 - c[0]
    M[1, 2] += (nh - 1) / 2.0 - c[1]
    return cv2.warpAffine(img, M, (nw, nh), flags=interpolation, borderMode=border)


def center_crop_box(h, w, frac):
    ch, cw = int(round(h * frac)), int(round(w * frac))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    return y0, x0, ch, cw


def apply_perturbation(img_bgr, cond):
    """The robustness conditions. Every one either preserves ALL board content (so labels
    stay valid) or, for 'crop', is only applied to anomalous images whose defect is
    verified (via the ground-truth mask) to survive the crop -- see defect_survives_crop."""
    if cond == "identity":
        return img_bgr
    if cond == "rot90":
        return cv2.rotate(img_bgr, cv2.ROTATE_90_CLOCKWISE)
    if cond == "rot180":
        return cv2.rotate(img_bgr, cv2.ROTATE_180)
    if cond == "rot15":
        return rotate_bound(img_bgr, 15.0)
    if cond == "dark":
        x = img_bgr.astype(np.float32) / 255.0
        return np.clip(255.0 * 0.6 * np.power(x, 1.6), 0, 255).astype(np.uint8)
    if cond == "uneven_light":
        w = img_bgr.shape[1]
        gain = np.linspace(0.35, 1.25, w, dtype=np.float32)[None, :, None]
        return np.clip(img_bgr.astype(np.float32) * gain, 0, 255).astype(np.uint8)
    if cond == "crop":
        h, w = img_bgr.shape[:2]
        y0, x0, ch, cw = center_crop_box(h, w, CFG.ROBUSTNESS_CROP_FRAC)
        return img_bgr[y0:y0 + ch, x0:x0 + cw]
    raise ValueError("Unknown robustness condition: " + cond)


def find_defect_mask(img_path):
    """Ground-truth defect mask for an anomalous VisA image, for either common layout:
    1cls/MVTec-style  <cls>/test/bad/X.JPG        -> <cls>/ground_truth/bad/X.png
    raw VisA          <cls>/Data/Images/Anomaly/X -> <cls>/Data/Masks/Anomaly/X.png"""
    stem = os.path.splitext(os.path.basename(img_path))[0]
    d = os.path.dirname(img_path)
    candidates = []
    sep = os.sep
    if sep + "test" + sep in d + sep:
        gt_dir = (d + sep).replace(sep + "test" + sep, sep + "ground_truth" + sep).rstrip(sep)
        candidates += [os.path.join(gt_dir, stem + ext) for ext in (".png", ".PNG", "_mask.png")]
    if sep + "Images" + sep in d + sep:
        m_dir = (d + sep).replace(sep + "Images" + sep, sep + "Masks" + sep).rstrip(sep)
        candidates += [os.path.join(m_dir, stem + ext) for ext in (".png", ".PNG")]
    for c in candidates:
        if os.path.exists(c):
            return c
    return None


def defect_survives_crop(img_path, frac, min_inside):
    """True only if >= min_inside of the ground-truth defect pixels lie inside the centered
    crop -- otherwise cropping could cut the defect out and silently turn an 'anomalous'
    label into a lie. Returns (ok, reason)."""
    mpath = find_defect_mask(img_path)
    if mpath is None:
        return False, "no mask found"
    mask = cv2.imread(mpath, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return False, "mask unreadable"
    defect = mask > 0
    total = int(defect.sum())
    if total == 0:
        return False, "empty mask"
    h, w = mask.shape[:2]
    y0, x0, ch, cw = center_crop_box(h, w, frac)
    inside = int(defect[y0:y0 + ch, x0:x0 + cw].sum())
    frac_inside = inside / total
    return frac_inside >= min_inside, "defect fraction inside crop = " + str(round(frac_inside, 3))


# ---------------- Anomalib prediction-filename protocol ----------------
def parse_prefixed_name(fname):
    """'<split>__<original name>' -> (split, stem). split is 'calib', 'eval', or a
    robustness condition name. Returns (None, None) for anything without the prefix."""
    base = os.path.basename(str(fname))
    if "__" not in base:
        return None, None
    split, rest = base.split("__", 1)
    return split, os.path.splitext(rest)[0]


def align_scores(score_dict, keys):
    """score_dict: {(label, stem): score}. Returns an array in `keys` order, NaN where missing."""
    return np.array([score_dict.get(k, np.nan) for k in keys], dtype=np.float64)


# ---------------- version-tolerant construction (Anomalib API varies by version) ----------------
_UNEXPECTED_KW = re.compile(r"unexpected keyword argument '([^']+)'")


def construct_with_fallback(ctor, base_kwargs, optional_kwargs, what="object"):
    """Calls ctor(**base_kwargs, **optional_kwargs). On a TypeError naming ONE unexpected
    keyword that is in optional_kwargs, drops exactly that keyword and retries. Returns
    (obj, dropped_list). Any other error is re-raised untouched -- this never hides a
    real failure, it only tolerates optional kwargs a given installed version lacks."""
    kwargs = dict(base_kwargs)
    kwargs.update(optional_kwargs)
    dropped = []
    while True:
        try:
            obj = ctor(**kwargs)
            if dropped:
                print("  (" + what + ": this installed version doesn't accept: " + ", ".join(dropped) + ")")
            return obj, dropped
        except TypeError as te:
            m = _UNEXPECTED_KW.search(str(te))
            name = m.group(1) if m else None
            if name is not None and name in optional_kwargs and name in kwargs:
                del kwargs[name]
                dropped.append(name)
                continue
            raise


print("Evaluation utilities ready (paired bootstrap, perturbations:",
      ", ".join(CFG.ROBUSTNESS_CONDITIONS) + ")")

In [ ]:
# ============================================================
# CELL 7c -- Golden-reference CANONICALIZATION: undo rotation / scale / crop / lighting
doc_section("CELL 7c", "Golden-reference CANONICALIZATION: undo rotation / scale / crop / lighting")
#            BEFORE detection (a model-agnostic front-end, evaluated for every model in Part 3)
# ============================================================
# Why this exists (evidence from a real run of Part 3): learned invariance -- rotation /
# photometric augmentation in the memory bank + CLAHE -- held up only for exact 90/180-degree
# turns, and only via TTA (whose 4-view median is mathematically invariant to them). At 15
# degrees every model, ours included, fell to chance with ~100% false alarms, and under dim or
# uneven light ours degraded while PatchCore did not. Industrial AOI solves this by aligning
# each board to a golden reference first. This cell does that, in three steps:
#   1. geometry: ORB keypoints (rotation-invariant) + ratio-test matching + RANSAC similarity
#      transform (rotation, uniform scale, translation) -> warp into the reference frame;
#   2. lighting: divide out the low-frequency illumination field relative to the reference
#      (fixes gradients), then match the tone curve by histogram matching (fixes dim/gamma);
#   3. regions the camera did not see (outside a crop, rotated-canvas corners) are filled from
#      the normal reference, so they score as normal instead of as fake anomalies -- the model
#      can only judge what it actually saw.
# If registration fails (too few inliers / implausible scale) the image is passed through with
# tone-curve correction only, and the failure is counted and reported -- never hidden.
_ORB_DETECTOR = None


def _orb():
    global _ORB_DETECTOR
    if _ORB_DETECTOR is None:
        _ORB_DETECTOR = cv2.ORB_create(nfeatures=CFG.CANON_ORB_FEATURES, scaleFactor=1.2, nlevels=8, fastThreshold=7)
    return _ORB_DETECTOR


def _keypoint_gray(img_bgr):
    # contrast-normalized grayscale used ONLY for finding keypoints (dim images otherwise yield few)
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(g)


def build_canon_reference(img_bgr):
    kp, des = _orb().detectAndCompute(_keypoint_gray(img_bgr), None)
    ecc_small = _small_gray(img_bgr, CFG.CANON_ECC_SIZE / float(max(img_bgr.shape[:2])))
    return {"img": img_bgr, "kp": kp, "des": des, "ecc_small": ecc_small, "ecc_small_hp": _highpass(ecc_small)}


def _gray(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)


def _lowpass_masked(x, mask, sigma):
    num = cv2.GaussianBlur(x * mask, (0, 0), sigma)
    den = cv2.GaussianBlur(mask, (0, 0), sigma)
    return num / np.maximum(den, 1e-3)


def _hist_match(src, ref, valid):
    """Per-channel CDF matching of src's VALID pixels to the reference image's pixels."""
    out = src.copy()
    m = valid.astype(bool)
    for ch in range(3):
        s = src[..., ch][m]
        r = ref[..., ch][m] if m.any() else ref[..., ch].ravel()
        if s.size == 0:
            continue
        s_vals, s_counts = np.unique(s, return_counts=True)
        r_vals, r_counts = np.unique(r, return_counts=True)
        mapped = np.interp(np.cumsum(s_counts) / s.size, np.cumsum(r_counts) / r.size, r_vals)
        lut = np.interp(np.arange(256), s_vals, mapped)
        out[..., ch] = np.clip(lut[src[..., ch]], 0, 255).astype(np.uint8)
    return out


def _hist_match_whole(img_bgr, ref_bgr):
    """Tone-curve correction when the frames are NOT aligned (whole-image CDFs, any sizes)."""
    out = img_bgr.copy()
    for ch in range(3):
        s, r = img_bgr[..., ch].ravel(), ref_bgr[..., ch].ravel()
        s_vals, s_counts = np.unique(s, return_counts=True)
        r_vals, r_counts = np.unique(r, return_counts=True)
        mapped = np.interp(np.cumsum(s_counts) / s.size, np.cumsum(r_counts) / r.size, r_vals)
        out[..., ch] = np.clip(np.interp(np.arange(256), s_vals, mapped)[img_bgr[..., ch]], 0, 255).astype(np.uint8)
    return out


def _affine3(M):
    return np.vstack([np.asarray(M, np.float64), [0.0, 0.0, 1.0]])


def _small_gray(img_bgr, f):
    g = cv2.resize(_gray(img_bgr), None, fx=f, fy=f, interpolation=cv2.INTER_AREA)
    return cv2.GaussianBlur(g, (0, 0), 1.0).astype(np.float32)


def _to_small(M, f):
    return (np.diag([f, f, 1.0]) @ _affine3(M) @ np.diag([1.0 / f, 1.0 / f, 1.0]))[:2]


def _to_full(M_s, f):
    return (np.diag([1.0 / f, 1.0 / f, 1.0]) @ _affine3(M_s) @ np.diag([f, f, 1.0]))[:2]


def _highpass(x):
    """Structure without illumination: high-pass, then LOCAL contrast normalization, so every region votes
    equally however brightly it is lit. Without the normalization, an uneven-light gradient crushed the
    contrast of a near-symmetric board's few asymmetric details (on the dark side) and flipped the chosen
    orientation; without the high-pass, the gradient itself dominated the correlation."""
    hp = x - cv2.GaussianBlur(x, (0, 0), 6.0)
    local_std = np.sqrt(cv2.GaussianBlur(hp * hp, (0, 0), 6.0))
    return hp / (local_std + 0.05 * float(local_std.mean()) + 1e-6)


def _alignment_score(img_hp, ref_hp, M_s):
    """ONE common yardstick for every hypothesis: correlation of high-passed structure between the warped
    image and the reference over their overlap; -1 if they overlap on less than half the reference frame
    (a warp that slides the board out of frame must not win on a sliver of pixels)."""
    size = (ref_hp.shape[1], ref_hp.shape[0])
    warped = cv2.warpAffine(img_hp, M_s, size, flags=cv2.INTER_LINEAR)
    valid = cv2.warpAffine(np.ones_like(img_hp), M_s, size, flags=cv2.INTER_NEAREST) > 0.5
    valid = cv2.erode(valid.astype(np.uint8), np.ones((7, 7), np.uint8)) > 0     # skip warp-border ringing
    if valid.mean() < 0.5:
        return -1.0
    a = warped[valid] - warped[valid].mean()
    b = ref_hp[valid] - ref_hp[valid].mean()
    return float((a * b).sum() / (np.sqrt((a * a).sum() * (b * b).sum()) + 1e-9))


def _ecc_refine(img_s, ref_s, M_s):
    """Sub-pixel ECC (Euclidean) refinement of an img->ref hypothesis at low resolution. Returns the refined
    transform, or the input unchanged if ECC fails or slips more than 3 deg / 5% of the frame away."""
    try:
        warp = cv2.invertAffineTransform(M_s).astype(np.float32)        # ECC wants template(ref) -> input(img)
        criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 60, 1e-5)
        _, warp = cv2.findTransformECC(ref_s, img_s, warp, cv2.MOTION_EUCLIDEAN, criteria, None, 5)
        M_r = cv2.invertAffineTransform(warp).astype(np.float64)
    except cv2.error:
        return np.asarray(M_s, np.float64)
    d_ang = abs((np.degrees(np.arctan2(M_r[1, 0], M_r[0, 0]) - np.arctan2(M_s[1, 0], M_s[0, 0])) + 180) % 360 - 180)
    d_tr = float(np.hypot(*(M_r[:, 2] - np.asarray(M_s)[:, 2])))
    return M_r if (d_ang < 3.0 and d_tr < 0.05 * max(ref_s.shape)) else np.asarray(M_s, np.float64)


def _phase_align(img_s, ref_s, M_s):
    """Fix the TRANSLATION of a coarse hypothesis by phase correlation (robust to offsets of tens of
    pixels, where ECC alone does not converge). Sign verified empirically: phaseCorrelate(ref, warped)
    returns warped's shift relative to ref, so it is subtracted."""
    warped = cv2.warpAffine(img_s, M_s, (ref_s.shape[1], ref_s.shape[0]))
    win = cv2.createHanningWindow((ref_s.shape[1], ref_s.shape[0]), cv2.CV_32F)
    (dx, dy), _ = cv2.phaseCorrelate(ref_s, warped, win)
    M_s = np.array(M_s, np.float64)
    M_s[0, 2] -= dx
    M_s[1, 2] -= dy
    return M_s


def _canonicalize_once(img_bgr, ref, input_valid=None):
    """Returns (canonical_bgr in the reference frame, info dict).

    v2 changes, each from evidence in a real Kaggle run:
      * orientation check -- 37/60 pcb1 images registered at +/-180 deg (near-symmetric board, half
        the inliers of correct matches). Both the RANSAC hypothesis and its 180-degree turn about the
        matched board centre are ECC-refined; the better pixel correlation wins.
      * lighting gate -- photometric correction ran on every image, altering correctly-lit boards and
        HURTING PatchCore under uneven light. It now runs only when the low-frequency brightness
        differs from the reference by more than CFG.CANON_PHOTOMETRIC_GATE (median |log gain|).
      * feathered fill -- unseen regions are blended in from the reference over CFG.CANON_FEATHER_PX
        pixels instead of a hard seam.
    """
    h, w = ref["img"].shape[:2]
    info = {"registered": False, "inliers": 0, "angle_deg": float("nan"), "scale": float("nan"),
            "hypothesis": "", "ecc_cc": float("nan"), "lighting_dev": float("nan"), "photometric": False}
    kp, des = _orb().detectAndCompute(_keypoint_gray(img_bgr), None)
    M = None
    if des is not None and ref["des"] is not None and len(kp) >= CFG.CANON_MIN_INLIERS:
        pairs = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(des, ref["des"], k=2)
        good = [p[0] for p in pairs if len(p) == 2 and p[0].distance < CFG.CANON_RATIO_TEST * p[1].distance]
        if len(good) >= CFG.CANON_MIN_INLIERS:
            src_pts = np.float32([kp[g.queryIdx].pt for g in good])
            dst_pts = np.float32([ref["kp"][g.trainIdx].pt for g in good])
            M_est, inl = cv2.estimateAffinePartial2D(src_pts, dst_pts, method=cv2.RANSAC,
                                                     ransacReprojThreshold=3.0, maxIters=5000, confidence=0.995)
            if M_est is not None and inl is not None:
                n_in = int(inl.sum())
                scale = float(np.hypot(M_est[0, 0], M_est[1, 0]))
                info.update(inliers=n_in, scale=scale)
                if n_in >= CFG.CANON_MIN_INLIERS and 0.5 <= scale <= 2.0:
                    # Two orientation hypotheses, scored on ONE common yardstick (_alignment_score):
                    #  - the keypoint (RANSAC) transform, ECC-refined
                    #  - its 180-degree turn about the reference centre, offset found by phase
                    #    correlation, then ECC-refined (turning about the matched-keypoint centroid instead
                    #    left ~25 px of error on a near-symmetric test board -- beyond ECC's reach)
                    f = CFG.CANON_ECC_SIZE / float(max(h, w))
                    img_s, ref_s = _small_gray(img_bgr, f), ref["ecc_small"]
                    M0 = _to_small(M_est, f)
                    c_s = ((ref_s.shape[1] - 1) / 2.0, (ref_s.shape[0] - 1) / 2.0)
                    flip = np.array([[-1.0, 0.0, 2.0 * c_s[0]], [0.0, -1.0, 2.0 * c_s[1]]])
                    M180 = _phase_align(img_s, ref_s, (_affine3(flip) @ _affine3(M0))[:2])
                    img_hp, ref_hp = _highpass(img_s), ref["ecc_small_hp"]
                    scored = []
                    for name, Mc in [("keypoints", M0), ("keypoints+180", M180)]:
                        # ECC runs on illumination-normalized structure (on raw intensities an uneven-light
                        # gradient made it drift ~7 px off a CORRECT hypothesis), and a refinement is kept
                        # only if it actually improves the common alignment score
                        s0 = _alignment_score(img_hp, ref_hp, Mc)
                        Mr = _ecc_refine(img_hp, ref_hp, Mc)
                        s1 = _alignment_score(img_hp, ref_hp, Mr)
                        scored.append((s1, name, Mr) if s1 >= s0 else (s0, name, np.asarray(Mc, np.float64)))
                    cc, name, M_best = max(scored, key=lambda t: t[0])
                    M = _to_full(M_best, f)
                    info.update(hypothesis=name, ecc_cc=cc)

    if M is None:
        # registration failed: no geometric change; tone correction only if lighting is clearly off
        dev = abs(float(np.log((ref["img"].mean() + 1.0) / (img_bgr.mean() + 1.0))))
        info.update(lighting_dev=dev, photometric=dev > CFG.CANON_PHOTOMETRIC_GATE)
        return (_hist_match_whole(img_bgr, ref["img"]) if info["photometric"] else img_bgr.copy()), info

    info.update(registered=True, scale=float(np.hypot(M[0, 0], M[1, 0])),
                angle_deg=float(np.degrees(np.arctan2(M[1, 0], M[0, 0]))))
    # CUBIC: at arbitrary angles every pixel is resampled at fractional positions; bilinear blurred the board (on
    # synthetic boards, sharpness after aligning a 15-degree photo: 0.49x of the aligned identity with bilinear vs
    # 0.75x with cubic), and memory-bank detectors read that blur as anomaly everywhere.
    warped = cv2.warpAffine(img_bgr, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    info["M"] = np.asarray(M, np.float64)  # input -> reference frame (used to align ground-truth masks and check placement)
    # input_valid marks which INPUT pixels are real. Real photos: all of them (None). Synthetic rotations on an expanded
    # canvas: NOT the padding -- in a real run, padding leaked into 2-7% of the aligned frame of offset boards and
    # memory-bank detectors flagged every board (ours rot15+canon AUROC 0.51 vs 0.90 at rot90).
    src_valid = (np.full(img_bgr.shape[:2], 255, np.uint8) if input_valid is None
                 else (np.asarray(input_valid) > 0).astype(np.uint8) * 255)
    valid = cv2.warpAffine(src_valid, M, (w, h), flags=cv2.INTER_NEAREST, borderValue=0)
    valid = cv2.erode(valid, np.ones((5, 5), np.uint8)) > 0
    info["valid_frac"] = float(valid.mean())
    info["valid_mask"] = valid             # pixels that came from the photo (the rest is filled from the reference)

    # lighting gate: correct only when the board is lit clearly differently from the reference
    m = valid.astype(np.float32)
    sigma = CFG.CANON_ILLUM_SIGMA_FRAC * max(h, w)
    ls = _lowpass_masked(_gray(warped).astype(np.float32), m, sigma)
    lr = _lowpass_masked(_gray(ref["img"]).astype(np.float32), m, sigma)
    log_gain = np.log(np.maximum(lr, 1.0) / np.maximum(ls, 1.0))
    # FINAL BUILD bug fix: the MEDIAN |log gain| ignored lighting changes covering less than half the board (shadow
    # bands, glare spots) -- the gate never fired and every aligned system collapsed under them in the real run
    # (thesis Sec. 7.7). A high percentile responds to such LOCAL deviations; global ones trigger it as before.
    dev = float(np.percentile(np.abs(log_gain[valid]), CFG.CANON_GATE_PERCENTILE)) if valid.any() else 0.0
    out = warped
    if dev > CFG.CANON_PHOTOMETRIC_GATE:
        gain = np.clip(np.exp(log_gain), 0.25, 4.0)
        out = np.clip(warped.astype(np.float32) * gain[..., None], 0, 255).astype(np.uint8)
        out = _hist_match(out, ref["img"], valid)
    info.update(lighting_dev=dev, photometric=dev > CFG.CANON_PHOTOMETRIC_GATE)

    # feathered fill of regions the camera did not see
    dist = cv2.distanceTransform(valid.astype(np.uint8), cv2.DIST_L2, 3)
    alpha = np.clip(dist / float(CFG.CANON_FEATHER_PX), 0.0, 1.0)[..., None]
    out = (alpha * out.astype(np.float32) + (1.0 - alpha) * ref["img"].astype(np.float32)).round().astype(np.uint8)
    return out, info


def build_board_mask(img_bgr, min_frac=0.15, max_frac=0.995, margin_frac=0.015):
    """Board-vs-background mask of an (aligned) board photo. Two cues: TEXTURE (a PCB is dense with traces, pads and
    components) and COLOUR distance from the frame border (the border is mostly background when any is visible).
    Union -> closing -> holes filled -> largest region -> convex hull -> slight dilation. Falls back to the whole frame
    when the board touches every edge or the result is implausible. Returns (mask bool HxW, note)."""
    h, w = img_bgr.shape[:2]
    s = max(h, w)
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    gx, gy = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    tex = cv2.GaussianBlur(np.sqrt(gx * gx + gy * gy), (0, 0), 0.02 * s)
    b = max(2, int(0.03 * s))
    border = np.zeros((h, w), bool); border[:b] = border[-b:] = True; border[:, :b] = border[:, -b:] = True
    inner = np.zeros((h, w), bool); inner[h // 4:3 * h // 4, w // 4:3 * w // 4] = True
    if tex[border].mean() > 0.6 * tex[inner].mean():                      # textured all the way to the edges
        return np.ones((h, w), bool), "board fills the frame (border is as textured as the centre)"
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    bg = np.median(lab[border], axis=0)
    col = cv2.GaussianBlur(np.linalg.norm(lab - bg, axis=2), (0, 0), 0.01 * s)
    otsu = lambda x: cv2.threshold(cv2.normalize(x, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8), 0, 255,
                                   cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1] > 0
    m = (otsu(tex) | otsu(col)).astype(np.uint8)
    k = int(0.06 * s) | 1
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
    ff = m.copy(); cv2.floodFill(ff, np.zeros((h + 2, w + 2), np.uint8), (0, 0), 2)          # fill holes
    m = ((m > 0) | (ff == 0)).astype(np.uint8)
    n, lbl, stats, _ = cv2.connectedComponentsWithStats(m)
    if n < 2:
        return np.ones((h, w), bool), "no board region found -- whole frame used"
    big = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    hull = cv2.convexHull(np.column_stack(np.where(lbl == big))[:, ::-1].astype(np.int32))
    out = np.zeros((h, w), np.uint8); cv2.fillConvexPoly(out, hull, 1)
    d = int(margin_frac * s) | 1
    out = cv2.dilate(out, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (d, d))) > 0
    frac = float(out.mean())
    if not (min_frac <= frac <= max_frac):
        return np.ones((h, w), bool), "implausible board area (%.0f%%) -- whole frame used" % (100 * frac)
    return out, "board %.0f%% of the frame" % (100 * frac)


def board_with_calibration_defects(board, defect_masks, max_dist_frac=0.15):
    """Board mask grown LOCALLY to cover every calibration defect lying near the board: the convex hull of the board and
    those defect pixels. (Growing a uniform margin instead adds background on EVERY side -- in a real run pcb1's mask grew
    from 35% to 64% of the frame to reach its last 1.6% of defect pixels, and accuracy fell.) Defect pixels farther than
    max_dist_frac of the long side from the board are ignored as misalignment. Returns (mask, n_defects_used, note)."""
    s = max(board.shape)
    dist = cv2.distanceTransform((~board).astype(np.uint8), cv2.DIST_L2, 5)
    union, used = board.copy(), 0
    for m in defect_masks:
        if m is None or not m.any():
            continue
        near = m & (dist <= max_dist_frac * s)
        if (near & ~board).any():
            used += 1
        union |= near
    if used == 0:
        return board, 0, "no calibration defect outside the board"
    hull = cv2.convexHull(np.column_stack(np.where(union))[:, ::-1].astype(np.int32))
    out = np.zeros(board.shape, np.uint8)
    cv2.fillConvexPoly(out, hull, 1)
    return (out > 0) | board, used, "grown to %d calibration defect(s) outside the board" % used


def board_mask_for(cls):
    """The class's board mask in the aligned (reference) frame."""
    return load_canon_reference(cls, CFG.ROBUSTNESS_LONG_SIDE)["board_mask"]


def seen_path(png_path):
    return os.path.join(CFG.SEEN_DIR, os.path.abspath(png_path).lstrip(os.sep) + ".png")


def save_seen(png_path, info):
    """Stores which pixels of an aligned image came from the photo (the rest was filled from the reference)."""
    m = info.get("valid_mask")
    if m is not None:
        q = seen_path(png_path)
        os.makedirs(os.path.dirname(q), exist_ok=True)
        cv2.imwrite(q, np.asarray(m, bool).astype(np.uint8) * 255)


def read_seen(png_path):
    q = seen_path(png_path)
    m = cv2.imread(q, cv2.IMREAD_GRAYSCALE) if os.path.exists(q) else None
    return None if m is None else m > 127


def _exclude_saturated(img_bgr, input_valid):
    """FINAL BUILD -- over-exposure / glare gate. A clipped pixel carries no information about the board, so no method can
    find a defect under it. If more than CFG.SATURATION_GATE_FRAC of the photo's real pixels are clipped white (every
    channel >= CFG.SATURATION_LEVEL), those pixels -- plus a small margin -- are treated as UNSEEN, exactly like the
    missing parts of a cropped photo: filled from the reference, never scored, reported as not inspected. Correctly
    exposed photos (a few specular glints) are left untouched. Returns (input_valid, clipped fraction, gate fired)."""
    real = np.ones(img_bgr.shape[:2], bool) if input_valid is None else (np.asarray(input_valid) > 0)
    low = img_bgr if img_bgr.ndim == 2 else img_bgr.min(axis=2)
    sat = (low >= CFG.SATURATION_LEVEL) & real
    frac = float(sat.sum()) / max(1, int(real.sum()))
    if frac <= CFG.SATURATION_GATE_FRAC:
        return input_valid, frac, False
    k = 2 * max(1, int(round(CFG.SATURATION_DILATE_FRAC * max(sat.shape)))) + 1
    sat = cv2.dilate(sat.astype(np.uint8), np.ones((k, k), np.uint8)) > 0
    return real & ~sat, frac, True


def canonicalize(img_bgr, ref, input_valid=None):
    """Aligns a photo to the class's golden reference (_canonicalize_once). If that fails -- typically a photo showing
    only part of the board, with too few keypoints -- it retries ONCE on a CFG.CANON_RETRY_UPSCALE x upscaled copy (more
    keypoints on small views). A successful first attempt is never changed."""
    input_valid, sat_frac, sat_on = _exclude_saturated(img_bgr, input_valid)
    out, info = _canonicalize_once(img_bgr, ref, input_valid)
    info.update(saturated_frac=sat_frac, saturation_excluded=sat_on)
    up = float(getattr(CFG, "CANON_RETRY_UPSCALE", 0) or 0)
    if info["registered"] or up <= 1.0:
        return out, info
    big = cv2.resize(img_bgr, None, fx=up, fy=up, interpolation=cv2.INTER_CUBIC)
    bv = None if input_valid is None else cv2.resize(np.asarray(input_valid, np.uint8), (big.shape[1], big.shape[0]),
                                                     interpolation=cv2.INTER_NEAREST) > 0
    out2, info2 = _canonicalize_once(big, ref, bv)
    info2.update(saturated_frac=sat_frac, saturation_excluded=sat_on)
    if not info2["registered"]:
        return out, info
    M = np.asarray(info2["M"], np.float64).copy()
    M[:, :2] *= up                                  # the transform for the ORIGINAL input
    info2.update(M=M, scale=info2["scale"] * up, retry="upscaled x%.1f" % up)
    return out2, info2


_CANON_REF_CACHE = {}


def load_canon_reference(cls, long_side):
    """The class's golden reference: its most TYPICAL normal TRAINING board -- median overall brightness among the
    first CFG.CANON_REFERENCE_CANDIDATES training boards (never a test image). The first training image alone was
    fragile: for pcb4 it was lit ~20% differently from most pcb4 boards, so the lighting gate fired on 71% of them.
    Cached, so every cell and the inspection tool use the identical reference."""
    key = (cls, long_side)
    if key not in _CANON_REF_CACHE:
        cands = []
        for p in FILE_LISTS[cls]["train_normal"][:CFG.CANON_REFERENCE_CANDIDATES]:
            im = cv2.imread(p, cv2.IMREAD_COLOR)
            if im is not None:
                cands.append((p, downscale_long_side(im, long_side)))
        if not cands:
            raise FileNotFoundError("no readable training image for " + cls)
        lum = np.array([float(np.median(_gray(im))) for _, im in cands])
        path, im = cands[int(np.argsort(lum)[len(lum) // 2])]
        ref = build_canon_reference(im)
        ref["path"] = path
        ref["board_src"] = im
        if CFG.BOARD_MASK_MODE == "auto":
            ref["board_mask"], ref["board_note"] = build_board_mask(im)
        else:
            ref["board_mask"], ref["board_note"] = np.ones(im.shape[:2], bool), "whole frame (CFG.BOARD_MASK_MODE)"
        _CANON_REF_CACHE[key] = ref
    return _CANON_REF_CACHE[key]


print("Canonicalization v2 ready (ORB features:", CFG.CANON_ORB_FEATURES, "| min inliers:", CFG.CANON_MIN_INLIERS,
      "| orientation check: ECC at", CFG.CANON_ECC_SIZE, "px | lighting gate:", CFG.CANON_PHOTOMETRIC_GATE,
      "| feather:", CFG.CANON_FEATHER_PX, "px)")

In [ ]:
# ============================================================
# CELL 8 -- Per-class hybrid PatchCore-style detector (our model)
# ============================================================
doc_section("CELL 8", "Per-class hybrid PatchCore-style detector (our model)")
# Per-detector feature/scoring settings; defaults reproduce the original model exactly. Cell 15c2 may replace them
# (calibration-selected) and every detector constructed afterwards -- e.g. the canonical pipeline -- inherits them.
OURS_FEATURE_CFG = {"cnn_l2": True, "density_reweight": bool(CFG.USE_DENSITY_REWEIGHT)}


from contextlib import contextmanager
DEFAULT_PREP = {"normalize_light": True, "letterbox": True}


def prep_for(pcb_class=None):
    """Preprocessing of the detector that will score this image (the global `detectors`, which the inspection tool
    swaps temporarily), so the live tool always preprocesses exactly as that detector was trained."""
    d = detectors.get(pcb_class) if pcb_class is not None else next(iter(detectors.values()), None)
    return d.prep() if hasattr(d, "prep") else dict(DEFAULT_PREP)


class PCBAnomalyDetector:
    """One self-supervised anomaly detector for a single PCB class.

    Unlike concatenating CNN+ViT features into one combined k-NN (where one branch's
    raw-feature noise can silently dominate the distance metric), this detector keeps
    TWO SEPARATE memory banks -- one per branch -- and scores each branch independently.
    The two per-branch score maps are z-normalized (using each branch's own score
    statistics on normal calibration images) and blended with a weight `alpha` that is
    itself calibrated per PCB class (Cell 9) to maximize F1 on labeled calibration data --
    letting the data decide how much to trust local CNN texture vs global ViT layout
    signal for THIS board, rather than assuming a fixed 50/50 split.

    Trains only on normal images. Robust to rotation/lighting via augmented memory-bank
    construction and to crops/partial views because scoring is fully patch-local plus
    letterbox preprocessing (Cell 5).
    """

    def __init__(self, class_name):
        self.class_name = class_name
        self.feat_cfg = dict(OURS_FEATURE_CFG)
        self.cnn_bank = None
        self.vit_bank = None
        self.cnn_knn = None
        self.vit_knn = None
        self.grid_hw = None
        self.cnn_norm = (0.0, 1.0)     # (mean, std) of CNN-branch top-k scores on normal calib images
        self.vit_norm = (0.0, 1.0)     # same, for the ViT branch
        self.cnn_bank_density = None   # per-bank-point local density (Cell 7 compute_bank_density)
        self.vit_bank_density = None   # used for PatchCore-style score re-weighting -- see _score_grid_branches
        self.alpha = 0.5               # blend weight: alpha*cnn_z + (1-alpha)*vit_z (1.0=CNN-only, 0.0=ViT-only)
        self.threshold = None
        self.threshold_tta = None
        self.score_stats = {}

        # Orientation-aware canonical-alignment pathway (Cell 14) -- OPTIONAL, built only
        # if build_canonical_pathway() is called. None until then, in which case
        # test_single_image(..., check_orientation=True) skips this check gracefully.
        self.canonical_cnn_bank = None
        self.canonical_vit_bank = None
        self.canonical_cnn_knn = None
        self.canonical_vit_knn = None
        self.canonical_cnn_density = None
        self.canonical_vit_density = None
        self.canonical_threshold = None

    @torch.no_grad()
    def prep(self):
        """This detector's preprocessing (Cell 15e2 may change it); the defaults reproduce the original pipeline."""
        fc = getattr(self, "feat_cfg", {})
        return {"normalize_light": fc.get("normalize_light", True), "letterbox": fc.get("letterbox", True)}

    def load_tensor(self, path):
        """Image -> tensor with THIS detector's preprocessing. (Not `load`: that name is the classmethod that
        unpickles a saved detector -- a second `load` silently replaced it and broke a real run.)"""
        return load_image(path, **self.prep())

    def _use_density(self):
        return bool(getattr(self, "feat_cfg", {}).get("density_reweight", CFG.USE_DENSITY_REWEIGHT))

    def _extract_branch_grids(self, image_tensor):
        x = image_tensor.unsqueeze(0).to(DEVICE)
        saved_l2 = getattr(extractor, "cnn_l2", True)                  # the extractor is shared: set, then restore
        extractor.cnn_l2 = getattr(self, "feat_cfg", {}).get("cnn_l2", True)
        try:
            cnn_feat, vit_feat, target_hw = extractor.forward_branches(x)
        finally:
            extractor.cnn_l2 = saved_l2
        h, w = target_hw
        self.grid_hw = (h, w)
        cnn_flat = cnn_feat.permute(0, 2, 3, 1).reshape(-1, cnn_feat.shape[1]).cpu().numpy().astype(np.float32)
        vit_flat = vit_feat.permute(0, 2, 3, 1).reshape(-1, vit_feat.shape[1]).cpu().numpy().astype(np.float32)
        return cnn_flat, vit_flat

    def fit(self, image_paths, augs_per_image=CFG.AUGS_PER_IMAGE, use_crop_views=True, verbose=True):
        cnn_feats_all, vit_feats_all = [], []
        for path in tqdm(image_paths, desc="[" + self.class_name + "] extracting normal features"):
            img_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
            if img_bgr is None:
                continue

            clean_t = preprocess_bgr(img_bgr, **self.prep())
            c, v = self._extract_branch_grids(clean_t)
            cnn_feats_all.append(c)
            vit_feats_all.append(v)

            for i in range(augs_per_image):
                if use_crop_views and i == augs_per_image - 1:
                    view = random_crop_view(img_bgr)
                else:
                    view = random_photometric_geometric_augment(img_bgr)
                aug_t = preprocess_bgr(view, **self.prep())
                c, v = self._extract_branch_grids(aug_t)
                cnn_feats_all.append(c)
                vit_feats_all.append(v)

        cnn_feats_all = np.concatenate(cnn_feats_all, axis=0)
        vit_feats_all = np.concatenate(vit_feats_all, axis=0)
        if verbose:
            print("[" + self.class_name + "] raw patch descriptors -- CNN:", cnn_feats_all.shape,
                  "| ViT:", vit_feats_all.shape)

        self.cnn_bank = greedy_coreset(cnn_feats_all, ratio=CFG.CORESET_RATIO)
        self.vit_bank = greedy_coreset(vit_feats_all, ratio=CFG.CORESET_RATIO)
        if verbose:
            print("[" + self.class_name + "] memory banks after coreset -- CNN:", self.cnn_bank.shape,
                  "| ViT:", self.vit_bank.shape)

        self.cnn_knn = NearestNeighbors(n_neighbors=CFG.KNN_K, algorithm="auto", metric="euclidean")
        self.cnn_knn.fit(self.cnn_bank)
        self.vit_knn = NearestNeighbors(n_neighbors=CFG.KNN_K, algorithm="auto", metric="euclidean")
        self.vit_knn.fit(self.vit_bank)

        if self._use_density():
            self.cnn_bank_density = compute_bank_density(self.cnn_bank)
            self.vit_bank_density = compute_bank_density(self.vit_bank)

    def _region_grid(self, seen=None):
        """Patch-grid cells that count towards the image score: the BOARD (aligned detectors with board_only) AND what
        the photo showed (seen, eroded one cell so no patch straddles a fill seam). None = every cell."""
        board = board_mask_for(self.class_name) if getattr(self, "feat_cfg", {}).get("board_only") else None
        if board is None and seen is None:
            return None
        h, w = self.grid_hw

        def to_grid(m, thr):
            m = np.asarray(m, np.float32)
            if self.prep()["letterbox"]:                     # same symmetric padding as letterbox_square; padding = not board
                H, W = m.shape
                side = max(H, W)
                top, left = (side - H) // 2, (side - W) // 2
                m = cv2.copyMakeBorder(m, top, side - H - top, left, side - W - left, cv2.BORDER_CONSTANT, value=0)
            return cv2.resize(m, (w, h), interpolation=cv2.INTER_AREA) >= thr

        region = to_grid(board, 0.5) if board is not None else np.ones((h, w), bool)
        if seen is not None:
            region = region & (cv2.erode(to_grid(seen, 0.99).astype(np.uint8), np.ones((3, 3), np.uint8)) > 0)
        if region.sum() < max(4, int(0.02 * h * w)):         # almost nothing left: fall back to the board alone
            region = to_grid(board, 0.5) if board is not None else None
        return region

    def region_pixels(self, out_hw):
        """The last scored image's inspected region, resized to a map of shape out_hw (for region reports)."""
        r = getattr(self, "_last_region", None)
        return None if r is None else cv2.resize(r.astype(np.uint8), (out_hw[1], out_hw[0]), interpolation=cv2.INTER_NEAREST) > 0

    @contextmanager
    def seen_context(self, seen):
        """Scores inside this block use `seen` (an in-memory aligned photo's seen-area mask)."""
        prev = getattr(self, "_ctx_seen", None)
        self._ctx_seen = seen
        try:
            yield
        finally:
            self._ctx_seen = prev

    def score_path(self, path):
        """Score an aligned image file, with its stored seen-area mask (if any)."""
        return self.score_image_tensor(self.load_tensor(path), seen=read_seen(path))

    def _score_grid_branches(self, image_tensor, canonical=False, seen=None, region=True):
        """Per-branch (density-reweighted, if enabled) patch anomaly maps, each (h, w).
        canonical=True queries the orientation-aware canonical banks (Cell 14) instead
        of the main rotation-invariant banks -- every existing call site defaults to
        canonical=False, so this is fully backward compatible."""
        self._last_region = self._region_grid(seen if seen is not None else getattr(self, "_ctx_seen", None)) if region else None
        cnn_flat, vit_flat = self._extract_branch_grids(image_tensor)

        cnn_knn = self.canonical_cnn_knn if canonical else self.cnn_knn
        vit_knn = self.canonical_vit_knn if canonical else self.vit_knn
        cnn_density = self.canonical_cnn_density if canonical else self.cnn_bank_density
        vit_density = self.canonical_vit_density if canonical else self.vit_bank_density

        cnn_dists, cnn_idxs = cnn_knn.kneighbors(cnn_flat, n_neighbors=CFG.KNN_K)
        vit_dists, vit_idxs = vit_knn.kneighbors(vit_flat, n_neighbors=CFG.KNN_K)
        h, w = self.grid_hw

        cnn_raw = cnn_dists.mean(axis=1)
        vit_raw = vit_dists.mean(axis=1)

        if self._use_density() and cnn_density is not None:
            cnn_raw = self._apply_density_reweight(cnn_raw, cnn_density[cnn_idxs[:, 0]])
        if self._use_density() and vit_density is not None:
            vit_raw = self._apply_density_reweight(vit_raw, vit_density[vit_idxs[:, 0]])

        cnn_map = cnn_raw.reshape(h, w)
        vit_map = vit_raw.reshape(h, w)
        return cnn_map, vit_map

    @staticmethod
    def _apply_density_reweight(raw_dist, density, eps=1e-4):
        """raw_dist / density: how many multiples of the matched memory-bank point's own
        typical intra-bank spacing is this test patch away? A match in a DENSE/typical
        region (small density) amplifies the same raw distance into a bigger, more
        suspicious weighted score; a match in a SPARSE/rare-but-legitimate region (large
        density) dampens it. Capped at DENSITY_MAX_AMPLIFY x the raw distance so a
        near-duplicate bank point (density near 0) can't cause a pathological blow-up."""
        weighted = raw_dist / (density + eps)
        cap = raw_dist * CFG.DENSITY_MAX_AMPLIFY
        return np.minimum(weighted, cap)

    def _topk_mean(self, arr_2d, frac=CFG.TOPK_SCORE_FRAC, region=None):
        # FINAL BUILD v4 bug fix: region= scores a STORED map in its own image's inspected region. Calibration's alpha
        # search re-scored all stored maps with the region of whichever image had been scored LAST.
        r = region if region is not None else getattr(self, "_last_region", None)           # inspected region of the image just scored (None = all)
        vals = arr_2d[r] if (r is not None and r.shape == arr_2d.shape and r.any()) else arr_2d.ravel()
        flat_sorted = np.sort(vals.ravel())[::-1]
        k = max(1, int(len(flat_sorted) * frac))
        s = float(flat_sorted[:k].mean())
        # FINAL BUILD v4 -- global-evidence correction. A defect is LOCAL; a uniform rise of the whole board's anomaly map
        # (capture / lighting differences: the mechanism of every false alarm inspected in the real runs) is not. beta x
        # the inspected region's MEDIAN patch score is subtracted. beta = 0 is the documented score.
        b = float(getattr(self, "beta_global", 0.0) or 0.0)
        return s - b * float(np.median(vals)) if b else s

    def _fused_map(self, cnn_map, vit_map):
        cnn_mean, cnn_std = self.cnn_norm
        vit_mean, vit_std = self.vit_norm
        cnn_z = (cnn_map - cnn_mean) / max(cnn_std, 1e-6)
        vit_z = (vit_map - vit_mean) / max(vit_std, 1e-6)
        return self.alpha * cnn_z + (1.0 - self.alpha) * vit_z

    def _score_grid(self, image_tensor, seen=None, region=True):
        cnn_map, vit_map = self._score_grid_branches(image_tensor, seen=seen, region=region)
        anomaly_map = self._fused_map(cnn_map, vit_map)
        # Image-level score: mean of the TOP-K% most anomalous (fused) patches, not a
        # single max. A pure max is fragile -- PCBs have plenty of legitimate-but-unusual
        # local patches (text labels, mounting holes, in-spec component variation) that
        # can spike ONE patch's score even on a genuinely normal board. Real defects
        # (scratches, missing/misaligned components, solder bridges) affect a cluster of
        # neighboring patches, not a single one, so averaging the top slice is far more
        # robust to noise while staying sensitive to actual localized defects.
        image_score = self._topk_mean(anomaly_map)
        return anomaly_map, image_score

    def score_image_tensor(self, image_tensor, upsample_to=CFG.IMG_SIZE, smooth=True, seen=None, region=True):
        anomaly_map, image_score = self._score_grid(image_tensor, seen=seen, region=region)
        amap = torch.from_numpy(anomaly_map).float().unsqueeze(0).unsqueeze(0)
        amap = F.interpolate(amap, size=(upsample_to, upsample_to), mode="bilinear", align_corners=False)
        amap = amap.squeeze().numpy()
        if smooth:
            amap = cv2.GaussianBlur(amap, (5, 5), sigmaX=4)
        return amap, image_score

    def score_image_tensor_tta(self, image_tensor, angles=CFG.TTA_ANGLES, upsample_to=CFG.IMG_SIZE):
        """Runs inference at several rotations and fuses via per-pixel MEDIAN after
        rotating each anomaly map back to the original frame -- robust to the input
        itself arriving at an arbitrary orientation. Uses rotate_image_np (Cell 5) for
        both the forward rotation and the inverse map-back, so the 90/180/270-degree
        angles in CFG.TTA_ANGLES are true lossless pixel permutations rather than
        interpolated affine warps (see Cell 5's docstring for why that distinction
        matters -- an interpolated 'exact' rotation was injecting real noise here)."""
        img_np = image_tensor.numpy().transpose(1, 2, 0)
        maps, scores = [], []
        for angle in angles:
            rot = rotate_image_np(img_np, angle)
            rot_tensor = torch.from_numpy(rot.transpose(2, 0, 1)).float()
            amap, score = self.score_image_tensor(rot_tensor, upsample_to=upsample_to, smooth=False, region=False)

            if angle % 360 != 0:
                amap = rotate_image_np(amap, (-angle) % 360, border_mode=cv2.BORDER_REPLICATE)
            maps.append(amap)
            scores.append(score)

        fused_map = np.median(np.stack(maps, axis=0), axis=0)
        fused_map = cv2.GaussianBlur(fused_map, (5, 5), sigmaX=4)
        fused_score = float(np.median(scores))
        return fused_map, fused_score

    def calibrate_threshold(self, calib_images, calib_labels, alpha_grid=None):
        """Three-stage calibration on a small LABELED calibration split (both normal and
        anomaly images -- see Cell 9):

        1) Score every calibration image on EACH branch separately (single pass, cheap --
           no TTA needed just to pick normalization stats and alpha).
        2) Set cnn_norm / vit_norm to the (mean, std) of each branch's scores on the
           NORMAL-labeled calibration images, so both branches sit on a comparable scale
           before blending. Grid-search `alpha` over [0, 1] to maximize F1 -- this lets
           each PCB class decide, from its own data, how much to trust CNN local-texture
           signal vs ViT global-layout signal, instead of a fixed guess.
        3) Re-score every calibration image with the finalized (alpha, norm stats), for
           BOTH single-pass and TTA modes, and pick the FINAL decision threshold for each
           via find_best_threshold_constrained_fpr (recall-maximizing subject to
           CFG.MAX_FALSE_POSITIVE_RATE) -- NOT plain F1, which was verified to produce an
           unacceptable ~50% false-positive rate on a realistic, mostly-normal deployment
           folder in a real run. F1 is still used for the alpha search above, since
           that's about picking the best feature blend, not the final decision boundary.
        """
        if alpha_grid is None:
            alpha_grid = np.linspace(0.0, 1.0, 11)
        calib_labels_arr = np.asarray(calib_labels)

        cnn_maps, vit_maps, cnn_scores, vit_scores = [], [], [], []
        regions = []                         # FINAL BUILD v4: each calibration image's own inspected region
        self.beta_global = 0.0               # (re-)calibration starts from the documented score
        for path in calib_images:
            t = self.load_tensor(path)
            cnn_map, vit_map = self._score_grid_branches(t, seen=read_seen(path))
            regions.append(getattr(self, "_last_region", None))
            cnn_maps.append(cnn_map)
            vit_maps.append(vit_map)
            cnn_scores.append(self._topk_mean(cnn_map))
            vit_scores.append(self._topk_mean(vit_map))
        cnn_scores = np.array(cnn_scores)
        vit_scores = np.array(vit_scores)

        normal_mask = (calib_labels_arr == 0)
        if normal_mask.sum() > 0:
            cnn_mean, cnn_std = float(cnn_scores[normal_mask].mean()), float(cnn_scores[normal_mask].std() + 1e-6)
            vit_mean, vit_std = float(vit_scores[normal_mask].mean()), float(vit_scores[normal_mask].std() + 1e-6)
        else:
            cnn_mean, cnn_std = float(cnn_scores.mean()), float(cnn_scores.std() + 1e-6)
            vit_mean, vit_std = float(vit_scores.mean()), float(vit_scores.std() + 1e-6)
        self.cnn_norm = (cnn_mean, cnn_std)
        self.vit_norm = (vit_mean, vit_std)

        cnn_z = (cnn_scores - cnn_mean) / cnn_std
        vit_z = (vit_scores - vit_mean) / vit_std

        # LEGACY alpha rule -- kept ONLY so Cell 15c can report it on held-out images. It had
        # three defects found in a real run's ablation: (1) it maximized F1 at the best
        # threshold, a noisy step statistic on ~100 images, while every reported metric is
        # AUROC; (2) it scored a blend of per-branch IMAGE scores, not the deployed score (the
        # top-k mean of the blended MAP); (3) its strict '>' over a grid starting at 0.0
        # resolved every tie to alpha = 0 (ViT only).
        legacy_alpha, legacy_f1 = 0.5, -1.0
        for a in alpha_grid:
            _, f1 = find_best_threshold_and_f1(a * cnn_z + (1.0 - a) * vit_z, calib_labels_arr)
            if f1 > legacy_f1:
                legacy_f1, legacy_alpha = f1, float(a)
        self.alpha_legacy_f1 = legacy_alpha

        # REVISED alpha rule: maximize calibration AUROC of exactly the score that is deployed,
        # ties broken toward an even blend (no silent bias toward either branch).
        has_both = len(np.unique(calib_labels_arr)) > 1
        best_key, best_alpha, best_scores = None, 0.5, None
        for a in alpha_grid:
            self.alpha = float(a)
            s = np.array([self._topk_mean(self._fused_map(c, v), region=r) for c, v, r in zip(cnn_maps, vit_maps, regions)])
            auc = roc_auc_score(calib_labels_arr, s) if has_both else 0.0
            key = (round(float(auc), 6), -abs(float(a) - 0.5))
            if best_key is None or key > best_key:
                best_key, best_alpha, best_scores = key, float(a), s
        self.alpha = best_alpha
        best_f1 = find_best_threshold_and_f1(best_scores, calib_labels_arr)[1]
        self.score_stats["alpha_search_auroc"] = best_key[0]
        # FINAL BUILD v4 -- global-evidence correction, PRE-REGISTERED and selected on calibration data only: beta from
        # CFG.GLOBAL_BETA_GRID is adopted only if it raises the calibration AUROC by >= CFG.GLOBAL_BETA_MIN_GAIN over
        # beta = 0; otherwise the documented score is kept unchanged.
        beta_auc = {}
        for b in CFG.GLOBAL_BETA_GRID:
            self.beta_global = float(b)
            sb = np.array([self._topk_mean(self._fused_map(c, v), region=r) for c, v, r in zip(cnn_maps, vit_maps, regions)])
            beta_auc[float(b)] = float(roc_auc_score(calib_labels_arr, sb)) if has_both else 0.0
        best_b = max(beta_auc, key=lambda b: (round(beta_auc[b], 6), -b))
        self.beta_global = best_b if beta_auc[best_b] >= beta_auc.get(0.0, 0.0) + CFG.GLOBAL_BETA_MIN_GAIN else 0.0
        self.score_stats["beta_global"] = self.beta_global
        self.score_stats["beta_calib_auroc"] = beta_auc
        print("  global-evidence correction: beta = %.1f | calibration AUROC by beta: %s" % (
            self.beta_global, ", ".join("%.1f -> %.4f" % (b, a) for b, a in sorted(beta_auc.items()))))

        scores = []
        for path in calib_images:
            t = self.load_tensor(path)
            _, s = self.score_image_tensor(t, seen=read_seen(path))
            scores.append(s)

        # TTA scoring is ~4x the cost of single-pass (CFG.TTA_ANGLES forward passes per
        # image) -- cap it to a class-balanced subset rather than the full calibration
        # set, same cost/precision trade-off used for the canonical pathway's
        # calibration (calibrate_canonical_threshold's max_calib_images).
        tta_images, tta_labels = calib_images, list(calib_labels)
        cap = CFG.TTA_CALIBRATION_MAX_IMAGES
        if cap is not None and len(calib_images) > cap:
            idx0 = [i for i, l in enumerate(calib_labels) if l == 0]
            idx1 = [i for i, l in enumerate(calib_labels) if l == 1]
            half = cap // 2
            keep = idx0[:half] + idx1[:half]
            tta_images = [calib_images[i] for i in keep]
            tta_labels = [calib_labels[i] for i in keep]

        scores_tta = []
        for path in tta_images:
            t = self.load_tensor(path)
            _, s_tta = self.score_image_tensor_tta(t)
            scores_tta.append(s_tta)

        self.threshold, thr_recall = find_best_threshold_constrained_fpr(scores, calib_labels)
        self.threshold_tta, thr_tta_recall = find_best_threshold_constrained_fpr(scores_tta, tta_labels)
        self.score_stats["calib_scores"] = scores
        self.score_stats["calib_scores_tta"] = scores_tta
        self.score_stats["calib_labels"] = list(calib_labels)
        self.score_stats["calib_labels_tta"] = tta_labels
        self.score_stats["cnn_scores"] = cnn_scores.tolist()
        self.score_stats["vit_scores"] = vit_scores.tolist()
        self.score_stats["alpha_search_best_f1"] = best_f1
        self.score_stats["threshold_calib_recall"] = thr_recall
        self.score_stats["threshold_tta_calib_recall"] = thr_tta_recall
        return self.threshold

    # ------------------------------------------------------------------
    # Orientation-aware canonical-alignment pathway (Cell 14) -- optional, additive.
    # The main pathway above is deliberately rotation-INVARIANT: its memory banks
    # explicitly contain many rotated views of every training image, so a component
    # rotated 180 degrees is treated as just as "normal" as the correct orientation.
    # That is the right design for catching scratches/missing parts/solder defects
    # regardless of camera angle -- but it means the main pathway cannot, by
    # construction, ever flag a defect that IS a wrong orientation (wrong polarity,
    # tombstoning). These methods build a SEPARATE reference bank from only the "clean"
    # (non-rotation-augmented) view of each training image, then use it two ways: (1) to
    # estimate a test board's true rotation via self-supervised nearest-neighbor
    # alignment search (no labels, no trained regressor), and (2) to re-score the
    # orientation-corrected image against that same strict reference, which DOES notice
    # when a component is installed backwards even though the main pathway would not.
    # ------------------------------------------------------------------

    def build_canonical_pathway(self, image_paths, verbose=True):
        """Builds the canonical (non-rotation-augmented) reference banks from just the
        'clean' view of each training image -- reuses the same frozen backbone, no new
        model or training involved. Call this once per class, in addition to (not
        instead of) the normal fit()."""
        cnn_feats_all, vit_feats_all = [], []
        for path in tqdm(image_paths, desc="[" + self.class_name + "] canonical pathway features"):
            img_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
            if img_bgr is None:
                continue
            clean_t = preprocess_bgr(img_bgr, **self.prep())
            c, v = self._extract_branch_grids(clean_t)
            cnn_feats_all.append(c)
            vit_feats_all.append(v)

        cnn_feats_all = np.concatenate(cnn_feats_all, axis=0)
        vit_feats_all = np.concatenate(vit_feats_all, axis=0)

        self.canonical_cnn_bank = greedy_coreset(cnn_feats_all, ratio=CFG.CORESET_RATIO)
        self.canonical_vit_bank = greedy_coreset(vit_feats_all, ratio=CFG.CORESET_RATIO)
        if verbose:
            print("[" + self.class_name + "] canonical banks -- CNN:", self.canonical_cnn_bank.shape,
                  "| ViT:", self.canonical_vit_bank.shape)

        self.canonical_cnn_knn = NearestNeighbors(n_neighbors=CFG.KNN_K, algorithm="auto", metric="euclidean")
        self.canonical_cnn_knn.fit(self.canonical_cnn_bank)
        self.canonical_vit_knn = NearestNeighbors(n_neighbors=CFG.KNN_K, algorithm="auto", metric="euclidean")
        self.canonical_vit_knn.fit(self.canonical_vit_bank)

        if self._use_density():
            self.canonical_cnn_density = compute_bank_density(self.canonical_cnn_bank)
            self.canonical_vit_density = compute_bank_density(self.canonical_vit_bank)

    def _canonical_score_and_map_at_angle(self, image_tensor, angle):
        img_np = image_tensor.numpy().transpose(1, 2, 0)
        rot = rotate_image_np(img_np, angle)
        rot_t = torch.from_numpy(rot.transpose(2, 0, 1)).float()
        cnn_map, vit_map = self._score_grid_branches(rot_t, canonical=True)
        fused = self._fused_map(cnn_map, vit_map)
        score = self._topk_mean(fused)
        return score, fused

    def estimate_canonical_rotation(self, image_tensor,
                                     coarse_step=CFG.CANONICAL_ANGLE_COARSE_STEP,
                                     fine_step=CFG.CANONICAL_ANGLE_FINE_STEP,
                                     fine_range=CFG.CANONICAL_ANGLE_FINE_RANGE):
        """Coarse-to-fine search over rotation angles for the one that makes this image
        look MOST normal against the CANONICAL reference bank -- this is how the board's
        true orientation is estimated, entirely self-supervised, by finding the angle at
        which frozen-backbone features best match a reference built only from
        correctly-oriented normal boards. Returns (best_angle_deg, score_at_that_angle,
        fused_map_at_that_angle) -- the score/map are reused directly by callers so this
        one search covers both orientation ESTIMATION and orientation-defect SCREENING
        without a redundant extra pass."""
        coarse_angles = list(range(0, 360, coarse_step))
        coarse_results = [(a,) + self._canonical_score_and_map_at_angle(image_tensor, a) for a in coarse_angles]
        best_coarse_angle = min(coarse_results, key=lambda r: r[1])[0]

        # NOTE: offset 0 is added explicitly -- if fine_range isn't an exact multiple of
        # fine_step (e.g. range=25, step=10 -> offsets -25,-15,-5,5,15,25), the coarse-best
        # angle itself would otherwise never be re-tested in the fine stage, which can
        # occasionally send the estimate wildly off (verified: mean error jumped from ~6
        # to ~30 degrees, with cases over 300 degrees off, in exactly this misaligned
        # case). Always including 0 guarantees the fine stage can never do worse than the
        # coarse stage already did, regardless of how fine_step/fine_range are tuned.
        fine_offsets = set(range(-fine_range, fine_range + 1, fine_step))
        fine_offsets.add(0)
        fine_candidates = sorted(set((best_coarse_angle + d) % 360 for d in fine_offsets))
        fine_results = [(a,) + self._canonical_score_and_map_at_angle(image_tensor, a) for a in fine_candidates]
        best_angle, best_score, best_map = min(fine_results, key=lambda r: r[1])
        return best_angle, best_score, best_map

    def score_canonical_at_angle(self, image_tensor, angle):
        """Scores the image, rotated to a SPECIFIC given angle, against the canonical
        bank -- a standalone convenience method for checking one known orientation
        without running the full search in estimate_canonical_rotation."""
        score, fused = self._canonical_score_and_map_at_angle(image_tensor, angle)
        return fused, score

    def calibrate_canonical_threshold(self, calib_images, calib_labels, max_calib_images=24):
        """F1-optimal threshold for the canonical pathway (same methodology as the main
        calibrate_threshold), scored via orientation-aligned canonical matching instead
        of the rotation-invariant main banks. Capped to max_calib_images (class-balanced)
        by default since the coarse-to-fine angle search is ~15 forward passes per image
        -- noticeably more expensive than the main pathway's single-pass calibration.
        Cutting cost here by reducing max_calib_images (rather than by shrinking the
        search grid itself) is the safe lever: it only affects how many images the
        THRESHOLD is calibrated on, not the accuracy of any individual image's angle
        estimate -- verified separately that narrowing the search grid instead can
        actually make individual estimates substantially worse in some configurations."""
        if len(calib_images) > max_calib_images:
            idx0 = [i for i, l in enumerate(calib_labels) if l == 0]
            idx1 = [i for i, l in enumerate(calib_labels) if l == 1]
            half = max_calib_images // 2
            keep = idx0[:half] + idx1[:half]
            calib_images = [calib_images[i] for i in keep]
            calib_labels = [calib_labels[i] for i in keep]

        scores = []
        for path in tqdm(calib_images, desc="[" + self.class_name + "] canonical threshold calibration", leave=False):
            t = self.load_tensor(path)
            _, s, _ = self.estimate_canonical_rotation(t)
            scores.append(s)
        self.canonical_threshold, _ = find_best_threshold_constrained_fpr(scores, calib_labels)
        self.score_stats["canonical_calib_scores"] = scores
        self.score_stats["canonical_calib_labels"] = list(calib_labels)
        return self.canonical_threshold

    def save(self, out_dir=CFG.ARTIFACT_DIR):
        path = os.path.join(out_dir, self.class_name + "_detector.pkl")
        with open(path, "wb") as f:
            pickle.dump({
                "class_name": self.class_name,
                "cnn_bank": self.cnn_bank,
                "vit_bank": self.vit_bank,
                "grid_hw": self.grid_hw,
                "cnn_norm": self.cnn_norm,
                "vit_norm": self.vit_norm,
                "cnn_bank_density": self.cnn_bank_density,
                "vit_bank_density": self.vit_bank_density,
                "alpha": self.alpha,
                "threshold": self.threshold,
                "threshold_tta": self.threshold_tta,
                "canonical_cnn_bank": self.canonical_cnn_bank,
                "canonical_vit_bank": self.canonical_vit_bank,
                "canonical_cnn_density": self.canonical_cnn_density,
                "canonical_vit_density": self.canonical_vit_density,
                "canonical_threshold": self.canonical_threshold,
            }, f)
        return path

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            data = pickle.load(f)
        det = cls(data["class_name"])
        det.cnn_bank = data["cnn_bank"]
        det.vit_bank = data["vit_bank"]
        det.grid_hw = data["grid_hw"]
        det.cnn_norm = data.get("cnn_norm", (0.0, 1.0))
        det.vit_norm = data.get("vit_norm", (0.0, 1.0))
        det.cnn_bank_density = data.get("cnn_bank_density")   # None if loading an older save -- reweighting
        det.vit_bank_density = data.get("vit_bank_density")   # is then skipped automatically (see _score_grid_branches)
        det.alpha = data.get("alpha", 0.5)
        det.threshold = data["threshold"]
        det.threshold_tta = data.get("threshold_tta")
        det.cnn_knn = NearestNeighbors(n_neighbors=CFG.KNN_K, algorithm="auto", metric="euclidean")
        det.cnn_knn.fit(det.cnn_bank)
        det.vit_knn = NearestNeighbors(n_neighbors=CFG.KNN_K, algorithm="auto", metric="euclidean")
        det.vit_knn.fit(det.vit_bank)

        # Canonical pathway (Cell 14) -- None if this save predates it, or if
        # build_canonical_pathway() was never called; check_orientation=True then
        # skips gracefully (see test_single_image, Cell 13) rather than erroring.
        det.canonical_cnn_bank = data.get("canonical_cnn_bank")
        det.canonical_vit_bank = data.get("canonical_vit_bank")
        det.canonical_cnn_density = data.get("canonical_cnn_density")
        det.canonical_vit_density = data.get("canonical_vit_density")
        det.canonical_threshold = data.get("canonical_threshold")
        if det.canonical_cnn_bank is not None:
            det.canonical_cnn_knn = NearestNeighbors(n_neighbors=CFG.KNN_K, algorithm="auto", metric="euclidean")
            det.canonical_cnn_knn.fit(det.canonical_cnn_bank)
        if det.canonical_vit_bank is not None:
            det.canonical_vit_knn = NearestNeighbors(n_neighbors=CFG.KNN_K, algorithm="auto", metric="euclidean")
            det.canonical_vit_knn.fit(det.canonical_vit_bank)
        return det

In [ ]:
# ============================================================
# CELL 9 -- Train + calibrate OUR model, one detector per PCB class
# ============================================================
doc_section("CELL 9", "Train + calibrate OUR model, one detector per PCB class")
def stratified_split(images, labels, calib_frac=CFG.CALIBRATION_FRAC, seed=SEED):
    """Splits a class's labeled test images into a CALIBRATION subset (used to pick
    the anomaly-score threshold, find_best_threshold in Cell 7) and a separate
    EVALUATION subset (used for Cell 10's AUROC numbers), stratified by label so both
    subsets get a share of whatever normal/anomaly images are available -- and so the
    reported AUROC is never computed on images the threshold was tuned against."""
    rng = random.Random(seed)
    idx_by_label = {0: [], 1: []}
    for i, l in enumerate(labels):
        idx_by_label[l].append(i)

    calib_idx, eval_idx = [], []
    for l, idxs in idx_by_label.items():
        idxs = list(idxs)
        rng.shuffle(idxs)
        n_calib = max(1, int(len(idxs) * calib_frac)) if len(idxs) > 0 else 0
        calib_idx += idxs[:n_calib]
        eval_idx += idxs[n_calib:]

    calib_images = [images[i] for i in calib_idx]
    calib_labels = [labels[i] for i in calib_idx]
    eval_images = [images[i] for i in eval_idx]
    eval_labels = [labels[i] for i in eval_idx]
    return calib_images, calib_labels, eval_images, eval_labels


detectors = {}

for cls in CFG.CLASSES:
    fl = FILE_LISTS[cls]
    train_paths = fl["train_normal"][:CFG.MAX_TRAIN_IMAGES_PER_CLASS]

    det = PCBAnomalyDetector(cls)
    det.fit(train_paths)

    # CLASS-SPECIFIC seed. VisA reuses file numbers across PCB types, so one shared seed shuffled every
    # class identically: in a real run pcb1 and pcb2 had the SAME 100 calibration files (pcb3/pcb4 97/100),
    # making the four classes one split repeated four times -- and split luck (which hard normals land in
    # eval) hit every class at once.
    calib_images, calib_labels, eval_images, eval_labels = stratified_split(
        fl["test_images"], fl["test_labels"], seed=SEED + 7919 * (CFG.CLASSES.index(cls) + 1)
    )
    if len(calib_images) == 0:
        # no labeled test images at all for this class -- fall back to a few train images
        # (find_best_threshold will itself fall back to the percentile-of-normals rule)
        calib_images = train_paths[-10:]
        calib_labels = [0] * len(calib_images)

    FILE_LISTS[cls]["eval_images"] = eval_images
    FILE_LISTS[cls]["eval_labels"] = eval_labels
    FILE_LISTS[cls]["calib_images"] = calib_images   # kept for reuse by Cell 14's canonical-pathway
    FILE_LISTS[cls]["calib_labels"] = calib_labels   # calibration, so it doesn't need its own split

    det.calibrate_threshold(calib_images, calib_labels)
    n_anom = sum(calib_labels)
    print("[" + cls + "] calibration set:", len(calib_images), "images (",
          n_anom, "anomaly /", len(calib_labels) - n_anom, "normal )")
    print("[" + cls + "] CNN/ViT blend alpha:", round(det.alpha, 2),
          "(1.0=CNN-only, 0.0=ViT-only) | calib AUROC:", round(det.score_stats["alpha_search_auroc"], 3),
          "| legacy F1 rule would pick:", det.alpha_legacy_f1)
    print("[" + cls + "] threshold -- single-pass:", round(det.threshold, 4),
          "(recall at <=" + str(int(CFG.MAX_FALSE_POSITIVE_RATE * 100)) + "% calib FPR:",
          round(det.score_stats["threshold_calib_recall"], 3), ")")
    print("[" + cls + "] threshold -- TTA:", round(det.threshold_tta, 4),
          "(recall at <=" + str(int(CFG.MAX_FALSE_POSITIVE_RATE * 100)) + "% calib FPR:",
          round(det.score_stats["threshold_tta_calib_recall"], 3), ")")

    detectors[cls] = det

print("\nAll " + str(len(detectors)) + " detectors trained and calibrated.")


# independence check: with class-specific seeds, calibration sets should overlap ~50% by file name
# (what two independent random halves share), not ~100% as with the old shared seed
_cal_names = {c: {os.path.basename(p) for p in FILE_LISTS[c]["calib_images"]} for c in CFG.CLASSES}
_ov = [len(_cal_names[a] & _cal_names[b]) / max(1, min(len(_cal_names[a]), len(_cal_names[b])))
       for i, a in enumerate(CFG.CLASSES) for b in CFG.CLASSES[i + 1:]]
print("Calibration-set file-name overlap between classes: mean " + str(round(100 * float(np.mean(_ov)))) + "% "
      "(independent splits ~50%; the old shared seed gave up to 100%)")

## Documentation 1 -- Dataset overview

In [ ]:
# ============================================================
# CELL D1 -- DATASET OVERVIEW: what the model learns from and what it is tested on
# ============================================================
doc_section("CELL D1", "DATASET OVERVIEW: what the model learns from and what it is tested on")
# VisA PCB1-PCB4. Training uses NORMAL boards only (self-supervised anomaly detection: no defect is ever shown to the
# model). The VisA test split is divided per class into CALIBRATION (thresholds, weights, design choices) and HELD-OUT
# evaluation boards, which are only ever scored. Defective boards come with pixel-accurate ground-truth defect masks.
dataset_df = None
with doc_step("dataset statistics"):
    rows = []
    for cls in CFG.CLASSES:
        fl = FILE_LISTS[cls]
        im = cv2.imread(fl["train_normal"][0], cv2.IMREAD_COLOR)
        cal_l, ev_l = np.asarray(fl["calib_labels"]), np.asarray(fl["eval_labels"])
        rows.append({"pcb_class": cls, "train images (normal)": len(fl["train_normal"]),
                     "used for training": min(len(fl["train_normal"]), CFG.MAX_TRAIN_IMAGES_PER_CLASS),
                     "calibration normal": int((cal_l == 0).sum()), "calibration defective": int((cal_l == 1).sum()),
                     "held-out normal": int((ev_l == 0).sum()), "held-out defective": int((ev_l == 1).sum()),
                     "image size (h x w)": ("%d x %d" % im.shape[:2]) if im is not None else "?"})
    dataset_df = pd.DataFrame(rows)
    print(dataset_df.to_string(index=False))
    save_table(dataset_df, "dataset_overview", "Images per PCB class and data role")
    fig, ax = plt.subplots(figsize=(11, 4.2))
    cols = ["used for training", "calibration normal", "calibration defective", "held-out normal", "held-out defective"]
    dataset_df.set_index("pcb_class")[cols].plot.bar(ax=ax, rot=0)
    ax.set_ylabel("images"); ax.set_title("VisA PCB1-PCB4: images per class and data role")
    caption("Images per PCB class and data role. Training uses normal boards only; calibration boards set thresholds and "
            "design choices; held-out boards are only scored.")
    plt.tight_layout(); plt.show()

with doc_step("defect size statistics"):
    rows = []
    for cls in CFG.CLASSES:
        for p, l in zip(FILE_LISTS[cls]["eval_images"] + FILE_LISTS[cls]["calib_images"],
                        list(FILE_LISTS[cls]["eval_labels"]) + list(FILE_LISTS[cls]["calib_labels"])):
            if l == 1:
                mp = find_defect_mask(p)
                m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE) if mp else None
                if m is not None:
                    rows.append({"pcb_class": cls, "defect area (% of image)": 100.0 * float((m > 0).mean())})
    sizes = pd.DataFrame(rows)
    summ = sizes.groupby("pcb_class")["defect area (% of image)"].describe(percentiles=[0.5]).round(3).reset_index()
    print(summ.to_string(index=False))
    save_table(summ, "defect_size_statistics", "Ground-truth defect size as % of the image (calibration + held-out defects)")
    fig, ax = plt.subplots(figsize=(11, 4))
    for cls in CFG.CLASSES:
        v = sizes[sizes.pcb_class == cls]["defect area (% of image)"]
        ax.hist(np.log10(v.clip(lower=1e-3)), bins=25, alpha=0.5, label=cls)
    ax.set_xlabel("log10(defect area, % of image)"); ax.set_ylabel("defective images"); ax.legend()
    ax.set_title("Ground-truth defect sizes (most defects cover well under 1% of the photo)")
    caption("Distribution of ground-truth defect sizes. Most defects are tiny relative to the board, which is why the model "
            "scores small patches rather than the whole image.")
    plt.tight_layout(); plt.show()

with doc_step("sample boards per class"):
    rng = np.random.default_rng(SEED + 1)
    fig, ax = plt.subplots(len(CFG.CLASSES), 4, figsize=(17, 3.3 * len(CFG.CLASSES)))
    for i, cls in enumerate(CFG.CLASSES):
        ev = list(zip(FILE_LISTS[cls]["eval_images"], FILE_LISTS[cls]["eval_labels"]))
        good = [p for p, l in ev if l == 0]; bad = [p for p, l in ev if l == 1]
        picks = [(good[k], 0) for k in rng.choice(len(good), 2, replace=False)] + [(bad[k], 1) for k in rng.choice(len(bad), 2, replace=False)]
        for j, (p, l) in enumerate(picks):
            im = cv2.cvtColor(cv2.imread(p, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
            if l == 1:
                mp = find_defect_mask(p)
                m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE) if mp else None
                if m is not None:
                    cnts, _ = cv2.findContours((m > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    cv2.drawContours(im, cnts, -1, (255, 0, 0), max(2, im.shape[1] // 300))
            ax[i, j].imshow(im); ax[i, j].axis("off")
            ax[i, j].set_title("%s -- %s" % (cls, "DEFECTIVE (ground truth in red)" if l else "normal"), fontsize=9)
    fig.suptitle("Sample held-out boards: 2 normal and 2 defective per class (random, fixed seed)")
    caption("Sample held-out boards per class; red outlines are the ground-truth defect masks supplied with VisA.")
    plt.tight_layout(); plt.show()

## Documentation 2 -- Preprocessing & alignment walkthrough

In [ ]:
# ============================================================
# CELL D2 -- PREPROCESSING & ALIGNMENT WALKTHROUGH: every step a photo goes through, shown on a hard example
# ============================================================
doc_section("CELL D2", "PREPROCESSING & ALIGNMENT WALKTHROUGH: every step a photo goes through, shown on a hard example")
# For each class, one held-out defective board is captured "badly" on purpose -- rotated 30 degrees and lit unevenly --
# and followed through the pipeline: keypoint matching against the class's golden reference board (ORB + RANSAC),
# alignment (rotation, scale, shift undone; 180-degree ambiguity checked with ECC), gated lighting correction, CLAHE local
# contrast equalization, and the final model input.
with doc_step("preprocessing & alignment walkthrough"):
    rng = np.random.default_rng(SEED + 2)
    walk_rows = []
    for cls in CFG.CLASSES:
        ref = load_canon_reference(cls, CFG.ROBUSTNESS_LONG_SIDE)
        bad = [p for p, l in zip(FILE_LISTS[cls]["eval_images"], FILE_LISTS[cls]["eval_labels"]) if l == 1]
        path = bad[int(rng.integers(len(bad)))]
        full = cv2.imread(path, cv2.IMREAD_COLOR)
        small = downscale_long_side(full, CFG.ROBUSTNESS_LONG_SIDE)
        f = small.shape[1] / float(full.shape[1])
        rot = rotate_bound(full, 30.0, interpolation=cv2.INTER_CUBIC)                    # camera-resolution rotation
        real = rotate_bound(np.full(full.shape[:2], 255, np.uint8), 30.0, border=cv2.BORDER_CONSTANT, interpolation=cv2.INTER_NEAREST)
        rot = cv2.resize(rot, None, fx=f, fy=f, interpolation=cv2.INTER_AREA)
        real = cv2.resize(real, (rot.shape[1], rot.shape[0]), interpolation=cv2.INTER_AREA) > 250
        captured = apply_perturbation(rot, "uneven_light")
        canon, info = canonicalize(captured, ref, input_valid=real)
        clahe = illumination_normalize(canon)
        model_in = cv2.resize(clahe, (CFG.IMG_SIZE, CFG.IMG_SIZE), interpolation=cv2.INTER_AREA)
        walk_rows.append({"pcb_class": cls, "file": os.path.basename(path), "registered": info["registered"],
                          "rotation undone (deg)": round(info["angle_deg"], 2), "scale": round(info["scale"], 3),
                          "RANSAC inliers": info["inliers"], "lighting corrected": info["photometric"],
                          "frame seen in the photo (%)": round(100 * info.get("valid_frac", 1.0), 1), "retry": info.get("retry", "-")})
        log("%s: rotation undone %.2f deg, scale %.3f, %d inliers, lighting corrected: %s" % (
            cls, info["angle_deg"], info["scale"], info["inliers"], info["photometric"]))
        rgb = lambda x: cv2.cvtColor(x, cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(1, 6, figsize=(24, 4))
        for a, (im, t) in zip(ax, [(small, "1. original photo"), (captured, "2. captured: rotated 30 deg + uneven light"),
                                   (ref["board_src"], "3. golden reference (training board)"),
                                   (canon, "4. aligned + lighting corrected"), (clahe, "5. CLAHE local contrast"),
                                   (model_in, "6. model input %dx%d" % (CFG.IMG_SIZE, CFG.IMG_SIZE))]):
            a.imshow(rgb(im)); a.set_title(t, fontsize=9); a.axis("off")
        fig.suptitle("%s: preprocessing pipeline (rotation undone %.1f deg, %d keypoint inliers)" % (cls, info["angle_deg"], info["inliers"]))
        caption("%s: the preprocessing pipeline on a deliberately hard capture (30 degrees rotation, uneven light): "
                "alignment to the golden reference, lighting correction, CLAHE and the final model input." % cls)
        plt.tight_layout(); plt.show()
        # keypoint matches (ORB, ratio test) between the captured photo and the reference
        orb = cv2.ORB_create(CFG.CANON_ORB_FEATURES)
        g1, g2 = cv2.cvtColor(captured, cv2.COLOR_BGR2GRAY), cv2.cvtColor(ref["board_src"], cv2.COLOR_BGR2GRAY)
        k1, d1 = orb.detectAndCompute(g1, None); k2, d2 = orb.detectAndCompute(g2, None)
        good = []
        if d1 is not None and d2 is not None:
            for mm in cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(d1, d2, k=2):
                if len(mm) == 2 and mm[0].distance < CFG.CANON_RATIO_TEST * mm[1].distance:
                    good.append(mm[0])
        vis = cv2.drawMatches(captured, k1, ref["board_src"], k2, sorted(good, key=lambda m: m.distance)[:80], None,
                              matchColor=(0, 255, 0), flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        fig, ax = plt.subplots(figsize=(16, 5)); ax.imshow(rgb(vis)); ax.axis("off")
        ax.set_title("%s: ORB keypoints matched to the reference -- %d of %d / %d keypoints pass the ratio test (80 shown)" % (
            cls, len(good), len(k1), len(k2)))
        caption("%s: keypoint matches between the rotated, unevenly lit capture and the golden reference (ORB descriptors, "
                "ratio test); RANSAC then keeps the geometrically consistent ones (%d inliers)." % (cls, info["inliers"]))
        plt.tight_layout(); plt.show()
        # alignment quality: checkerboard of aligned vs reference, and their difference
        H, W = canon.shape[:2]
        yy, xx = np.mgrid[0:H, 0:W]
        chk = ((yy // 48 + xx // 48) % 2 == 0)[..., None]
        board = np.where(chk, canon, ref["board_src"])
        diff = cv2.absdiff(cv2.cvtColor(canon, cv2.COLOR_BGR2GRAY), cv2.cvtColor(ref["board_src"], cv2.COLOR_BGR2GRAY))
        fig, ax = plt.subplots(1, 3, figsize=(18, 4.2))
        ax[0].imshow(rgb(board)); ax[0].set_title("checkerboard: aligned photo / reference (continuous lines = good alignment)", fontsize=9)
        ax[1].imshow(diff, cmap="magma"); ax[1].set_title("|aligned - reference| (bright = differences)", fontsize=9)
        for img, lab, col in [(captured, "captured", "tab:red"), (canon, "aligned + corrected", "tab:blue"), (clahe, "after CLAHE", "tab:green")]:
            L = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)[..., 0].ravel()
            ax[2].hist(L, bins=64, histtype="step", lw=1.8, label=lab, color=col, density=True)
        ax[2].set_title("lightness histograms (Lab L channel)", fontsize=9); ax[2].legend(fontsize=8)
        for a in ax[:2]:
            a.axis("off")
        caption("%s: alignment quality (checkerboard composite and difference to the reference) and the lightness "
                "distribution before and after lighting correction and CLAHE." % cls)
        plt.tight_layout(); plt.show()
    walk_df = pd.DataFrame(walk_rows)
    print(walk_df.to_string(index=False))
    save_table(walk_df, "alignment_walkthrough", "Alignment of one deliberately hard capture per class (rotated 30 deg, uneven light)")

## Documentation 3 -- Model architecture & what the two branches see

In [ ]:
# ============================================================
# CELL D3 -- MODEL ARCHITECTURE & WHAT THE TWO BRANCHES SEE
# ============================================================
doc_section("CELL D3", "MODEL ARCHITECTURE & WHAT THE TWO BRANCHES SEE")
# The hybrid detector: a CNN branch (WideResNet-50-2, mid-level layers, 3x3 neighbourhood aggregation) and a ViT branch
# (DINOv2 ViT-S/14, self-supervised) each describe every image patch; each branch keeps a memory bank of NORMAL patch
# descriptors (greedy coreset), scores a patch by its k-NN distance to that bank, and the two z-normalized maps are fused
# with a per-class weight alpha chosen on calibration data. The image score is the mean of the top-k most anomalous patches.
arch_df = bank_df = None


def _feat_to_nc(f, hw):
    """Branch features -> (N patches, C channels) and the patch grid (h, w), whatever the tensor layout."""
    a = f.detach().float().cpu().numpy()
    h, w = int(hw[0]), int(hw[1])
    if a.ndim == 4:
        h, w = a.shape[2], a.shape[3]
        a = a[0].reshape(a.shape[1], -1).T
    elif a.ndim == 3:
        a = a[0]
    if a.shape[0] != h * w and a.shape[1] == h * w:
        a = a.T
    return a, (h, w)


def _pca_rgb(a, hw):
    a = a - a.mean(0, keepdims=True)
    _, _, vt = np.linalg.svd(a[:: max(1, len(a) // 4000)], full_matrices=False)
    p = a @ vt[:3].T
    p = (p - p.min(0)) / (p.max(0) - p.min(0) + 1e-6)
    return p.reshape(hw[0], hw[1], 3)


with doc_step("architecture summary"):
    nparams = lambda m: sum(int(p.numel()) for p in m.parameters()) if m is not None and hasattr(m, "parameters") else 0
    cnn, vit = getattr(extractor, "cnn", None), getattr(extractor, "vit", None)
    t0 = load_image(FILE_LISTS[CFG.CLASSES[0]]["train_normal"][0])
    with torch.no_grad():
        cf, vf, hw = extractor.forward_branches(t0.unsqueeze(0).to(DEVICE))
    ca, chw = _feat_to_nc(cf, hw); va, vhw = _feat_to_nc(vf, hw)
    arch_df = pd.DataFrame([
        ("input", "%d x %d RGB (letterboxed for this raw model; aligned full frame in the final pipeline), CLAHE" % (CFG.IMG_SIZE, CFG.IMG_SIZE)),
        ("CNN branch", "%s, %s parameters (frozen, ImageNet)" % (type(cnn).__name__ if cnn is not None else "?", format(nparams(cnn), ","))),
        ("ViT branch", "%s, %s parameters (frozen, self-supervised DINOv2)" % (type(vit).__name__ if vit is not None else "?", format(nparams(vit), ","))),
        ("whole extractor", "%s parameters, none trained" % format(nparams(extractor), ",")),
        ("CNN patch descriptor", "%d dims on a %d x %d patch grid" % (ca.shape[1], chw[0], chw[1])),
        ("ViT patch descriptor", "%d dims on a %d x %d patch grid" % (va.shape[1], vhw[0], vhw[1])),
        ("memory bank size cap", "%d patches per branch (greedy coreset)" % CFG.MAX_BANK_SIZE),
        ("k nearest neighbours", str(CFG.KNN_K)), ("image score", "mean of the top %.0f%% patch scores" % (100 * CFG.TOPK_SCORE_FRAC)),
        ("device", str(DEVICE))], columns=["component", "value"])
    print(arch_df.to_string(index=False))
    save_table(arch_df, "model_architecture", "Hybrid CNN+ViT detector: components and sizes")

with doc_step("memory banks of the raw-input model"):
    rows = []
    for cls, d in detectors.items():
        rows.append({"pcb_class": cls, "CNN bank (patches x dims)": str(tuple(getattr(d, "cnn_bank", np.zeros(0)).shape)),
                     "ViT bank (patches x dims)": str(tuple(getattr(d, "vit_bank", np.zeros(0)).shape)),
                     "alpha (CNN weight)": round(float(d.alpha), 2), "threshold": round(float(d.threshold), 4)})
    bank_df = pd.DataFrame(rows); print(bank_df.to_string(index=False))
    save_table(bank_df, "memory_banks_raw_model", "Memory banks and fusion weights of the raw-input model")

with doc_step("feature & branch anomaly visualizations"):
    rng = np.random.default_rng(SEED + 3)
    for cls in CFG.CLASSES:
        d = detectors[cls]
        ev = list(zip(FILE_LISTS[cls]["eval_images"], FILE_LISTS[cls]["eval_labels"]))
        picks = [p for p, l in ev if l == 0][:1] + [p for p, l in ev if l == 1][int(rng.integers(5)):][:1]
        fig, ax = plt.subplots(len(picks), 6, figsize=(24, 3.9 * len(picks)))
        ax = np.atleast_2d(ax)
        for r, p in enumerate(picks):
            t = load_image(p)
            with torch.no_grad():
                cf, vf, hw = extractor.forward_branches(t.unsqueeze(0).to(DEVICE))
            ca, chw = _feat_to_nc(cf, hw); va, vhw = _feat_to_nc(vf, hw)
            cmap_, vmap_ = d._score_grid_branches(t)
            fused = d._fused_map(cmap_, vmap_)
            img = cv2.cvtColor(cv2.imread(p, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
            panels = [(img, "photo (%s)" % ("defective" if r else "normal"), None), (_pca_rgb(ca, chw), "CNN features (PCA -> RGB)", None),
                      (_pca_rgb(va, vhw), "ViT features (PCA -> RGB)", None), (np.asarray(cmap_), "CNN anomaly (z)", "jet"),
                      (np.asarray(vmap_), "ViT anomaly (z)", "jet"), (np.asarray(fused), "fused (alpha=%.2f)" % d.alpha, "jet")]
            for a, (im, title, cm) in zip(ax[r], panels):
                a.imshow(im, cmap=cm); a.set_title(title, fontsize=9); a.axis("off")
        fig.suptitle("%s: what the two branches see -- patch features (PCA) and per-branch anomaly maps" % cls)
        caption("%s: patch features of the CNN and ViT branches (first three principal components shown as colour) and "
                "each branch's anomaly map with the fused result, for a normal and a defective board." % cls)
        plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# CELL 10 -- Evaluate OUR model on the full held-out eval split
doc_section("CELL 10", "Evaluate OUR model on the full held-out eval split")
# threshold (FPR / precision / recall), for BOTH single-pass and TTA modes.
# Uses FILE_LISTS[cls]["eval_images"/"eval_labels"] -- the held-out half of each
# class's test split that Cell 9 did NOT use for threshold calibration.
#
# WHY BOTH MODES: test_folder/inspect_pcb/test_single_image default to rotation_tta=True,
# so TTA is the mode actually used by default -- but AUROC alone (a ranking-quality
# metric, blind to where the threshold actually sits) previously only ever got checked
# for single-pass. Verified in a real run: single-pass AUROC looked healthy (~0.90)
# while the TTA-calibrated threshold, actually deployed by default, produced a ~50%
# false-positive rate against a realistic all-normal folder -- a gap that a
# threshold-agnostic metric alone cannot catch. This cell evaluates both explicitly.
# ============================================================
def evaluate_class(cls, use_tta=False, max_images=None):
    fl = FILE_LISTS[cls]
    det = detectors[cls]
    images = fl["eval_images"]
    labels = fl["eval_labels"]
    if max_images and len(images) > max_images:
        # eval_images/eval_labels are label-grouped (all normals, then all anomalies --
        # see stratified_split, Cell 9), so a naive images[:max_images] front-truncation
        # would take mostly/only normal images once max_images is smaller than the
        # normal-image count, badly skewing the sample. Balance across labels instead.
        idx0 = [i for i, l in enumerate(labels) if l == 0]
        idx1 = [i for i, l in enumerate(labels) if l == 1]
        half = max_images // 2
        keep = idx0[:half] + idx1[:half]
        images = [images[i] for i in keep]
        labels = [labels[i] for i in keep]

    scores = []
    for path in tqdm(images, desc="[" + cls + "] evaluating" + (" [TTA]" if use_tta else ""), leave=False):
        t = load_image(path)
        if use_tta:
            _, s = det.score_image_tensor_tta(t)
        else:
            _, s = det.score_image_tensor(t)
        scores.append(s)

    labels_arr = np.array(labels)
    scores_arr = np.array(scores)
    auroc = roc_auc_score(labels, scores) if len(set(labels)) > 1 else float("nan")

    thr = det.threshold_tta if use_tta else det.threshold
    metrics = {"auroc": auroc, "fpr": float("nan"), "recall": float("nan"), "precision": float("nan")}
    if thr is not None and len(labels_arr) > 0:
        preds = (scores_arr > thr).astype(int)
        tp = int(((preds == 1) & (labels_arr == 1)).sum())
        fp = int(((preds == 1) & (labels_arr == 0)).sum())
        tn = int(((preds == 0) & (labels_arr == 0)).sum())
        fn = int(((preds == 0) & (labels_arr == 1)).sum())
        metrics["fpr"] = fp / (fp + tn) if (fp + tn) > 0 else float("nan")
        metrics["recall"] = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        metrics["precision"] = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    return metrics, scores, labels


results = {}
OUR_EVAL_SCORES = {}   # per-image held-out scores in eval-split order (paired statistics, ensemble)
for cls in CFG.CLASSES:
    metrics_sp, scores_sp, labels_sp = evaluate_class(cls, use_tta=False, max_images=CFG.EVAL_MAX_IMAGES_PER_CLASS)
    metrics_tta, scores_tta, labels_tta = evaluate_class(cls, use_tta=True, max_images=CFG.EVAL_MAX_IMAGES_PER_CLASS)
    results[cls] = {"single_pass": metrics_sp, "tta": metrics_tta}
    assert list(labels_sp) == list(FILE_LISTS[cls]["eval_labels"]), "eval order changed -- keep EVAL_MAX None"
    OUR_EVAL_SCORES[cls] = {"Ours (single-pass)": np.asarray(scores_sp, dtype=np.float64),
                            "Ours (TTA)": np.asarray(scores_tta, dtype=np.float64),
                            "labels": np.asarray(labels_sp, dtype=int)}

    print("[" + cls + "] AUROC -- single-pass:", round(metrics_sp["auroc"], 4),
          "| TTA:", round(metrics_tta["auroc"], 4))
    print("        at calibrated threshold -- single-pass: FPR=" + str(round(metrics_sp["fpr"], 3)) +
          " recall=" + str(round(metrics_sp["recall"], 3)) +
          " precision=" + str(round(metrics_sp["precision"], 3)))
    print("                               --          TTA: FPR=" + str(round(metrics_tta["fpr"], 3)) +
          " recall=" + str(round(metrics_tta["recall"], 3)) +
          " precision=" + str(round(metrics_tta["precision"], 3)))

valid_aurocs_sp = [v["single_pass"]["auroc"] for v in results.values() if not math.isnan(v["single_pass"]["auroc"])]
valid_aurocs_tta = [v["tta"]["auroc"] for v in results.values() if not math.isnan(v["tta"]["auroc"])]
mean_fpr_tta = np.nanmean([v["tta"]["fpr"] for v in results.values()])
mean_recall_tta = np.nanmean([v["tta"]["recall"] for v in results.values()])

print("\nMean AUROC across PCB1-PCB4 -- single-pass:", round(sum(valid_aurocs_sp) / len(valid_aurocs_sp), 4),
      "| TTA:", round(sum(valid_aurocs_tta) / len(valid_aurocs_tta), 4))
print("Mean TTA false-positive rate on held-out NORMAL images:", round(float(mean_fpr_tta), 3),
      "(target: <=" + str(CFG.MAX_FALSE_POSITIVE_RATE) + ")")
print("Mean TTA recall on held-out ANOMALY images:", round(float(mean_recall_tta), 3))

if mean_recall_tta < 0.40:
    print("\nWARNING: mean held-out recall is low (" + str(round(float(mean_recall_tta), 2)) +
          ") -- the tool is missing most real defects, which is a more serious practical problem "
          "than an elevated false-positive rate. Do NOT respond to this by lowering "
          "CFG.MAX_FALSE_POSITIVE_RATE -- verified in a real run that tightening it (0.10 -> 0.07) "
          "cut recall by roughly a third while barely moving the held-out FPR at all. If recall is "
          "too low for practical use, either RAISE CFG.MAX_FALSE_POSITIVE_RATE and accept more false "
          "alarms, or improve the underlying score quality (CFG.MAX_TRAIN_IMAGES_PER_CLASS, "
          "CFG.AUGS_PER_IMAGE, CFG.CORESET_RATIO) -- a steep recall-vs-FPR trade-off that doesn't "
          "improve with a stricter target is a sign of a genuine separability ceiling, not something "
          "threshold tuning alone can fix.")
elif mean_fpr_tta > CFG.MAX_FALSE_POSITIVE_RATE * 1.5:
    print("\nNOTE: held-out false-alarm rate is above the calibration target. That is expected of this rule, not a"
          " bug in this model: a threshold allowing " + str(CFG.MAX_FALSE_POSITIVE_RATE) + " false alarms on ~50"
          " calibration normals promises only ~12% on new images ON AVERAGE, with wide spread, and VisA normals are"
          " not independent draws (scores cluster by capture order). Every baseline shows the same gap. Cell 18"
          " also reports a PAC operating point whose threshold keeps held-out false alarms under the target with "
          + str(int(CFG.PAC_CONFIDENCE * 100)) + "% confidence -- read its recall as the honest trade-off.")

In [ ]:
# ============================================================
# CELL 11 -- Region localization, defect-type shape hints, priority, visualize_result
# ============================================================
doc_section("CELL 11", "Region localization, defect-type shape hints, priority, visualize_result")
def merge_close_regions(regions, merge_dist_factor=CFG.REGION_MERGE_FACTOR):
    """Iteratively merges any two flagged regions whose circles are close together or
    overlapping into ONE bigger region. A real, spatially-contiguous defect commonly
    fragments into several small connected components after per-pixel binarization of a
    smoothed/upsampled anomaly map -- this consolidates those fragments back into a
    single region that reflects the actual defect extent, rather than reporting several
    small scattered circles for what is really one physical defect."""
    merged = [dict(r) for r in regions]
    changed = True
    while changed and len(merged) > 1:
        changed = False
        for i in range(len(merged)):
            for j in range(i + 1, len(merged)):
                r1, r2 = merged[i], merged[j]
                x1, y1 = r1["center"]
                x2, y2 = r2["center"]
                r1r, r2r = r1["radius"], r2["radius"]
                dist = math.hypot(x2 - x1, y2 - y1)
                if dist <= (r1r + r2r) * merge_dist_factor:
                    new_r = (dist + r1r + r2r) / 2.0
                    t = (new_r - r1r) / dist if dist > 1e-6 else 0.5
                    new_cx = x1 + t * (x2 - x1)
                    new_cy = y1 + t * (y2 - y1)

                    total_area = r1["area_px"] + r2["area_px"]
                    w1 = r1["area_px"] / total_area if total_area > 0 else 0.5
                    w2 = 1.0 - w1

                    bx = min(r1["bbox"][0], r2["bbox"][0])
                    by = min(r1["bbox"][1], r2["bbox"][1])
                    bx2 = max(r1["bbox"][0] + r1["bbox"][2], r2["bbox"][0] + r2["bbox"][2])
                    by2 = max(r1["bbox"][1] + r1["bbox"][3], r2["bbox"][1] + r2["bbox"][3])
                    bw, bh = bx2 - bx, by2 - by
                    aspect_ratio = max(bw, bh) / max(1, min(bw, bh))
                    area_frac = r1["area_frac"] + r2["area_frac"]
                    circularity = min(r1["circularity"], r2["circularity"])

                    if aspect_ratio >= 3.0 and area_frac < 0.05:
                        hint = "linear defect (possible scratch / crack / hairline gap)"
                    elif area_frac >= 0.06:
                        hint = "large / diffuse defect (possible bend, warp, discoloration, or large foreign object)"
                    elif circularity >= 0.6:
                        hint = "localized compact defect (possible hole, missing component, or foreign particle)"
                    else:
                        hint = "irregular localized defect (possible solder issue or component misalignment)"

                    new_region = {
                        "bbox": (bx, by, bw, bh),
                        "center": (new_cx, new_cy),
                        "radius": new_r,
                        "area_px": total_area,
                        "area_frac": area_frac,
                        "mean_score": w1 * r1["mean_score"] + w2 * r2["mean_score"],
                        "aspect_ratio": float(aspect_ratio),
                        "circularity": float(circularity),
                        "shape_hint": hint,
                    }
                    merged = [r for k, r in enumerate(merged) if k not in (i, j)] + [new_region]
                    changed = True
                    break
            if changed:
                break
    return merged


def classify_defect_priority(region):
    """Maps a region's shape-based hint + severity into CRITICAL / MAJOR / MINOR,
    aligned to typical PCB inspection priorities: findings whose shape is consistent
    with a missing component, solder issue, or misalignment (any of which can mean a
    short or open circuit) are CRITICAL and are never dropped by the MAX_REGIONS cap
    below; a large, low-severity diffuse region (most consistent with bend/warp or a
    lighting artifact rather than a functional defect) is MINOR and can be
    de-emphasized in reporting.

    IMPORTANT -- read this before trusting CRITICAL/MINOR labels operationally: this is
    a heuristic priority built from region shape/size/severity, exactly like shape_hint
    itself -- it CANNOT actually confirm whether a linear mark is a cosmetic scratch or
    a trace-severing crack, or whether an irregular blob is a solder bridge (a real
    short) or a harmless cold joint. It deliberately errs toward CRITICAL in ambiguous
    cases so nothing that COULD be a short/open-circuit risk gets silently dropped, at
    the cost of sometimes over-flagging a cosmetic mark. Treat CRITICAL/MAJOR as 'a
    human or AOI system should look at this', not a confirmed electrical fault -- true
    defect-TYPE classification (distinguishing a missing component from a
    misalignment from a solder bridge, specifically) needs either labeled examples (see
    DefectPrototypeClassifier, Cell 18) or a golden-reference/CAD comparison, neither of
    which this shape-only heuristic has access to."""
    hint = region["shape_hint"]
    severity_score = region["mean_score"]
    area_frac = region["area_frac"]

    if "localized compact defect" in hint or "irregular localized defect" in hint:
        # missing component / foreign particle / solder issue / misalignment --
        # exactly the categories tied to a short/open-circuit risk
        return "CRITICAL"

    if "linear defect" in hint:
        # could be a cosmetic scratch OR a trace-severing crack (open-circuit risk) --
        # only de-prioritize the small, weakly-scored case
        if area_frac < 0.01 and severity_score < 0.5:
            return "MINOR"
        return "CRITICAL"

    if "large / diffuse defect" in hint:
        # most consistent with bend/warp/lighting -- but only de-prioritize the
        # LOW-severity case; a high-severity large region could be genuine damage,
        # a burn mark, or a large foreign object, not a benign bend
        if severity_score < 0.5:
            return "MINOR"
        return "MAJOR"

    return "MAJOR"


def analyze_anomaly_regions(anomaly_map, region_percentile=CFG.REGION_PERCENTILE,
                             min_area_frac=0.006, top_k=CFG.MAX_REGIONS,
                             morph_open_kernel=5, morph_close_kernel=17):
    """Finds the most anomalous region(s) in an anomaly map and returns a rough
    SHAPE-BASED defect-type HINT for each. Reports a FEW BIG, consolidated regions
    (CFG.MAX_REGIONS, default 2) rather than many small fragmented ones: a bigger
    morphological closing kernel bridges nearby hot pixels at the mask level, and
    merge_close_regions() then merges any remaining nearby components at the region level.

    IMPORTANT: this is NOT a trained defect classifier. PatchCore's anomaly score only
    measures 'how different is this patch from anything in the normal memory bank' --
    it carries no learned notion of defect category (scratch vs missing-component vs
    solder-bridge etc.), because no labeled defect-type training data was used anywhere
    in this pipeline (by design -- self-supervised, normal-images-only). The 'shape_hint'
    below is a simple geometric heuristic computed from the flagged region's shape alone
    (elongated -> linear defect; compact/round -> a hole/missing part/foreign particle;
    large & diffuse -> a broad defect like warping or a large foreign object). Treat it
    as a starting point for a human inspector to check, not a verified diagnosis.
    """
    h, w = anomaly_map.shape
    total_px = h * w
    region_thr = float(np.percentile(anomaly_map, region_percentile))
    mask = (anomaly_map >= region_thr).astype(np.uint8)

    # small opening removes pure single-pixel noise; a MUCH bigger closing bridges gaps
    # between nearby hot patches so one real defect doesn't fragment into several blobs
    if morph_open_kernel > 1:
        open_k = np.ones((morph_open_kernel, morph_open_kernel), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, open_k)
    if morph_close_kernel > 1:
        close_k = np.ones((morph_close_kernel, morph_close_kernel), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, close_k)

    num_labels, labels_im, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)

    regions = []
    for label in range(1, num_labels):   # label 0 is background
        area = int(stats[label, cv2.CC_STAT_AREA])
        if area < max(4, min_area_frac * total_px):
            continue

        bw = int(stats[label, cv2.CC_STAT_WIDTH])
        bh = int(stats[label, cv2.CC_STAT_HEIGHT])

        comp_mask = (labels_im == label).astype(np.uint8)
        contours, _ = cv2.findContours(comp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        contour = max(contours, key=cv2.contourArea)
        (circ_x, circ_y), radius = cv2.minEnclosingCircle(contour)

        perimeter = cv2.arcLength(contour, True)
        circularity = (4 * math.pi * area / (perimeter ** 2)) if perimeter > 0 else 0.0
        aspect_ratio = max(bw, bh) / max(1, min(bw, bh))
        mean_score = float(anomaly_map[comp_mask.astype(bool)].mean())
        area_frac = area / total_px

        if aspect_ratio >= 3.0 and area_frac < 0.05:
            hint = "linear defect (possible scratch / crack / hairline gap)"
        elif area_frac >= 0.06:
            hint = "large / diffuse defect (possible bend, warp, discoloration, or large foreign object)"
        elif circularity >= 0.6:
            hint = "localized compact defect (possible hole, missing component, or foreign particle)"
        else:
            hint = "irregular localized defect (possible solder issue or component misalignment)"

        regions.append({
            "bbox": (int(stats[label, cv2.CC_STAT_LEFT]), int(stats[label, cv2.CC_STAT_TOP]), bw, bh),
            "center": (float(circ_x), float(circ_y)),
            "radius": float(radius),
            "area_px": area,
            "area_frac": float(area_frac),
            "mean_score": mean_score,
            "aspect_ratio": float(aspect_ratio),
            "circularity": float(circularity),
            "shape_hint": hint,
        })

    if not regions:
        return []

    regions = merge_close_regions(regions, merge_dist_factor=CFG.REGION_MERGE_FACTOR)

    # Drop any region whose average value sits AT OR BELOW the normal-calibration
    # baseline (anomaly_map is z-score-blended, so 0 means "typical of a normal patch")
    # -- a region that doesn't even clear that bar has no business being reported as a
    # defect, however locally it happened to rank in the raw top percentile. If merging
    # accidentally drops every candidate this way, keep the single best one anyway so an
    # image already confirmed anomalous by the caller still gets SOME region shown.
    above_baseline = [r for r in regions if r["mean_score"] > 0]
    regions = above_baseline if above_baseline else regions

    for r in regions:
        r["priority"] = classify_defect_priority(r)

    # top_k is a cap on MAJOR/MINOR clutter, not on CRITICAL findings: a genuine
    # short/open-circuit-risk region must never be dropped just because a bigger or
    # higher-scoring but less important (e.g. bend-like) region also showed up.
    critical = [r for r in regions if r["priority"] == "CRITICAL"]
    non_critical = [r for r in regions if r["priority"] != "CRITICAL"]
    non_critical.sort(key=lambda r: r["mean_score"] * r["area_px"], reverse=True)
    critical.sort(key=lambda r: r["mean_score"] * r["area_px"], reverse=True)
    kept_non_critical = non_critical[:max(0, top_k - len(critical))]
    return critical + kept_non_critical


def draw_regions_on_image(img_rgb, regions, text_color=(255, 255, 0)):
    """Draws a circle + rank number around each flagged region, padded slightly beyond
    the region's actual extent so the circle clears the defect edge. Circle color reflects
    priority (classify_defect_priority, Cell 11): red=CRITICAL, orange=MAJOR, yellow=MINOR."""
    priority_colors = {"CRITICAL": (255, 0, 0), "MAJOR": (255, 140, 0), "MINOR": (230, 200, 0)}
    out = img_rgb.copy()
    for i, r in enumerate(regions, start=1):
        cx, cy = r["center"]
        radius = max(8, int(r["radius"] * 1.15))
        color = priority_colors.get(r.get("priority"), (255, 60, 60))
        cv2.circle(out, (int(cx), int(cy)), radius, color, thickness=3)
        label = "#" + str(i) + ((" " + r["defect_type"]) if r.get("defect_type") else "")   # Part 1.4 region type
        put_label(out, label, int(cx) - radius, int(cy) - radius - 8, 0.7, text_color, 2)   # kept inside the image
    return out


# ============================================================
# Visualization helper
# ============================================================
def put_label(img, text, x, y, scale, color, thick):
    """cv2.putText kept inside the image: labels of regions near an edge no longer run off the picture."""
    (tw, th), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thick)
    x = int(min(max(0, x), max(0, img.shape[1] - tw - 1)))
    y = int(min(max(th + 1, y), img.shape[0] - base - 1))
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thick, cv2.LINE_AA)


def letterbox_content_box(h, w, size):
    """(y0, y1, x0, x1): where an h x w photo sits inside the size x size letterboxed model frame (letterbox_square)."""
    side = max(h, w)
    sc = size / float(side)
    top, left = (side - h) // 2, (side - w) // 2
    return int(round(top * sc)), int(round((top + h) * sc)), int(round(left * sc)), int(round((left + w) * sc))


def _shift_regions(regions, dx, dy):
    """Copies of region dicts moved by (dx, dy) map pixels (center and bbox)."""
    out = []
    for r in regions or []:
        r = dict(r)
        cx, cy = r["center"]
        r["center"] = (cx + dx, cy + dy)
        if "bbox" in r:
            bx, by, bw, bh = r["bbox"]
            r["bbox"] = (bx + dx, by + dy, bw, bh)
        out.append(r)
    return out


def regions_in_photo(anomaly_map, photo_hw, letterbox, region_percentile=CFG.REGION_PERCENTILE, top_k=CFG.MAX_REGIONS):
    """analyze_anomaly_regions on the PHOTO area of a square letterboxed map only (the mirror padding of letterbox_square
    can echo an edge defect). Region coordinates stay in the map frame. Used by test_single_image and Part 1.4."""
    Hm, Wm = anomaly_map.shape[:2]
    h, w = photo_hw
    y0, y1, x0, x1 = 0, Hm, 0, Wm
    if letterbox and Hm == Wm and h != w:
        y0, y1, x0, x1 = letterbox_content_box(h, w, Hm)
    regs = analyze_anomaly_regions(anomaly_map[y0:y1, x0:x1], region_percentile=region_percentile, top_k=top_k)
    return _shift_regions(regs, x0, y0) if (x0 or y0) else regs


def map_box_to_photo(bbox_xywh, map_hw, photo_hw, letterbox):
    """A region box (x, y, w, h) in the model's square map -> (x0, y0, x1, y1) in photo pixels (undoes letterbox/resize)."""
    x, y, w, h = [float(v) for v in bbox_xywh]
    Hm, Wm = map_hw
    H, W = photo_hw
    if letterbox and Hm == Wm:
        side = float(max(H, W))
        s = side / Hm
        top, left = (side - H) // 2, (side - W) // 2
        x0, y0, x1, y1 = x * s - left, y * s - top, (x + w) * s - left, (y + h) * s - top
    else:
        sx, sy = W / float(Wm), H / float(Hm)
        x0, y0, x1, y1 = x * sx, y * sy, (x + w) * sx, (y + h) * sy
    return (int(max(0, round(x0))), int(max(0, round(y0))), int(min(W, round(x1))), int(min(H, round(y1))))


def visualize_result(image_path_or_array, anomaly_map, score, threshold, title="", regions=None, save_path=None,
                     letterbox=None):
    """Input / heatmap / overlay / flagged regions (each labelled with its Part 1.4 defect type when typed).

    FIXED (display only -- scores, verdicts and regions are unchanged): the photo used to be STRETCHED to the square
    anomaly map, but the detector sees the LETTERBOXED photo (Cell 5), so for non-square photos the heatmap and the region
    circles sat away from the defect. The map is now cropped to where the photo sits in the letterboxed frame.
    letterbox=None uses the setting of the detector named by `title`."""
    if isinstance(image_path_or_array, str):
        img = cv2.cvtColor(cv2.imread(image_path_or_array, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    else:
        img = image_path_or_array
    if letterbox is None:
        try:
            letterbox = bool(prep_for(title if title in CFG.CLASSES else None)["letterbox"])
        except Exception:
            letterbox = True
    Hm, Wm = anomaly_map.shape[:2]
    h0, w0 = img.shape[:2]
    if letterbox and Hm == Wm and h0 != w0:
        y0, y1, x0, x1 = letterbox_content_box(h0, w0, Hm)
        anomaly_map = anomaly_map[y0:y1, x0:x1]
        regions = _shift_regions(regions, -x0, -y0) if regions else regions

    img_disp = cv2.resize(img, (anomaly_map.shape[1], anomaly_map.shape[0]), interpolation=cv2.INTER_AREA)
    amap_norm = anomaly_map - anomaly_map.min()
    denom = anomaly_map.max() - anomaly_map.min()
    amap_norm = amap_norm / denom if denom > 1e-8 else amap_norm
    heat = cv2.cvtColor(cv2.applyColorMap((amap_norm * 255).astype(np.uint8), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(img_disp, 0.55, heat, 0.45, 0)

    verdict = "ANOMALY" if score > threshold else "NORMAL"
    n_panels = 4 if regions else 3
    panel_h = float(np.clip(4.3 * img_disp.shape[0] / max(1, img_disp.shape[1]), 1.6, 6.0))   # follows the photo's aspect
    fig, axes = plt.subplots(1, n_panels, figsize=(4.3 * n_panels, panel_h + 0.8))
    axes[0].imshow(img_disp); axes[0].set_title("Input"); axes[0].axis("off")
    axes[1].imshow(heat); axes[1].set_title("Anomaly heatmap"); axes[1].axis("off")
    axes[2].imshow(overlay); axes[2].set_title("Overlay"); axes[2].axis("off")
    if regions:
        axes[3].imshow(draw_regions_on_image(img_disp, regions))
        axes[3].set_title("Flagged region(s)" + (" + defect type" if any(r.get("defect_type") for r in regions) else ""))
        axes[3].axis("off")
    fig.suptitle((title + " | " if title else "") + "score=" + str(round(score, 4)) + "  threshold=" + str(round(threshold, 4))
                 + "  ->  " + verdict, color="red" if verdict == "ANOMALY" else "green", fontsize=12)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()

    if regions:
        print("Flagged region(s) -- priority and shape hint are heuristics; 'type' is the trained region classifier (Part 1.4):")
        for i, r in enumerate(regions, start=1):
            cx, cy = r["center"]
            print("  [" + r.get("priority", "?") + "] #" + str(i) + ": center=(" + str(round(cx)) + "," + str(round(cy)) + " px)"
                  + "  radius~" + str(round(r["radius"])) + "px"
                  + "  area=" + str(round(r["area_frac"] * 100, 2)) + "% of image"
                  + "  mean_score=" + str(round(r["mean_score"], 3))
                  + "  -> " + r["shape_hint"]
                  + (("  | type: " + region_type_text(r, alternatives=True)) if r.get("defect_type") else ""))

In [ ]:
# ============================================================
# CELL 12 -- Save all trained detectors
# ============================================================
doc_section("CELL 12", "Save all trained detectors")
saved_paths = {}
for cls, det in detectors.items():
    saved_paths[cls] = det.save()
print("Saved detector artifacts:")
for cls, path in saved_paths.items():
    print(" ", cls, "->", path)

## Part 1.4 -- Defect type of every highlighted region (supervised head on the frozen self-supervised features)
The self-supervised detector (Cells 6-9) is **not changed**: it still decides *whether* a board is defective and *where*.
This part names the **type of every highlighted region** -- one type per region, never one per image. A region's
descriptor is the detector's own frozen CNN+ViT patch features averaged inside the region box; a small linear classifier
per PCB class maps it to a defect type.

| Question | How |
|---|---|
| How well are defects typed when the region is exactly right? | **A**: ground-truth defect blobs of held-out images |
| How well are the regions the detector highlights typed? | **B**: the regions `test_single_image()` highlights, on held-out images -- share on a real defect, and typing accuracy on those |
| End to end? | **C**: per defective board -- flagged, top region on the defect, and typed correctly |
| What does the DEPLOYED system deliver? | **Cell 20e2**: the same scoring with the final pipeline's regions (aligned, board only) |
| Is the classifier better than guessing? | Paired comparison with the majority type of each class |

**Labels:** VisA's per-class `image_anno.csv` (PCB1-3: bent, melt, scratch, missing; PCB4 adds extra, wrong place, damage,
burnt, dirt). The converted Kaggle layout (`train/good`, `test/bad`) drops these files, so Cell 12b fetches them once from the
official VisA archive (internet on), or reads `CFG.VISA_ANNOTATION_DIR`. Check the printed vocabulary table.
**Data roles:** the classifier learns only from calibration images; every number is reported on eval images only.
**For the write-up:** VisA has only about 5-20 images per defect type, so per-type numbers are few-shot results; a region
of an image with several types carries all of them, because VisA masks do not say which blob is which.
**Where the types appear:** every test output -- `test_single_image()` / `inspect_pcb()` (Cells 13, 15),
`inspect_with_system()` (Cell 22a) and its demo (Cell 22), the demo boards with true vs predicted types (Cell 22b) and
*test any image* (Cell 22c). Each region is typed on the ORIGINAL photo, where the classifier learned:
regions found on the aligned board are mapped back through the alignment first. For the final pipeline's regions
the classifier reads either the whole region box or the tight box around its highlighted pixels -- chosen in Cell 20e2 on
calibration defects with out-of-fold classifiers (`CFG.DEFECT_TYPE_BOX`).
**Run only this part:** run the notebook from the top through Cell 12, then Cells 12b-12c (about 10 minutes).

In [ ]:
# ============================================================
# CELL 12b -- DEFECT-TYPE LABELS + REGION DATASET: ground-truth defect blobs and detector-highlighted regions
# ============================================================
doc_section("CELL 12b", "DEFECT-TYPE LABELS + REGION DATASET: ground-truth defect blobs and detector-highlighted regions")
# Part 1.4 names the TYPE of every highlighted defect REGION -- never of the whole image. The self-supervised detector
# (Cells 6-9) is not changed: the type head only READS its frozen CNN+ViT patch features, pooled inside a region box.
#   labels   VisA's <class>/image_anno.csv (free text; some images have several types). The converted VisA layout used on
#            Kaggle (<class>/train/good, test/bad) drops these files, so they are fetched once from the official archive
#            (CFG.FETCH_VISA_ANNOTATIONS) or read from CFG.VISA_ANNOTATION_DIR. File names are unchanged by the
#            conversion (test/bad/067.JPG is Data/Images/Anomaly/067.JPG), so labels match by file name.
#   regions  ground truth = each connected blob of a defect mask; detector = exactly the regions test_single_image()
#            highlights (Cell 9 detector, single pass, regions_in_photo of Cell 11).
#   roles    classifiers learn ONLY from CALIBRATION images; every reported number uses EVAL images only.
#   A region of an image with several defect types carries all of them: VisA masks do not say which blob is which.
from collections import Counter
import tarfile
import urllib.request

# free-text label -> canonical type; first match wins (specific keys first). VisA PCB1-3: bent, melt, scratch, missing;
# PCB4: scratch, extra, missing, wrong place, damage, burnt, dirt.
DEFECT_KEYWORDS = [
    ("wrong place", ["wrong place", "wrong_place", "wrongplace", "misplace", "misalign", "wrong position", "shift"]),
    ("missing", ["missing", "absent", "lack"]),
    ("extra", ["extra", "additional", "redundant", "excess"]),
    ("bent", ["bent", "bend"]),
    ("melt", ["melt", "molten"]),
    ("scratch", ["scratch"]),
    ("burnt", ["burnt", "burn"]),
    ("damage", ["damage", "broken", "crack", "break"]),
    ("dirt", ["dirt", "stain", "contamina", "spot", "foreign"]),
]
DEFECT_LABEL_OVERRIDES = {}      # manual fixes, e.g. {"some raw text": "scratch"} -- raw text as printed in the table below

DEFECT_OK = False                # True once at least one class has >= 2 learnable types and labelled regions
DEFECT_DETECTORS = dict(globals().get("detectors") or {})   # FROZEN reference to the Cell 9 detectors (Cell 22a swaps
                                                             # the global `detectors`; the type head always reads these)
DEFECT_VOCAB = {}                # cls -> types the classifier can output (fixed from calibration images)
DEFECT_TRUE = {}                 # (cls, file name) -> true types of a labelled defective test image (demos, Cell 22)
DEFECT_REGIONS = {}              # cls -> list of region records (see _region_record)
defect_vocab_df = None
DEFECT_ANNO_CACHE = os.path.join(CFG.EXTRACT_DIR, "visa_annotations")


# ---------------------------------------------------------------- labels
def canon_defect_types(raw):
    """Free-form VisA label text -> sorted tuple of canonical defect types."""
    raw = str(raw).strip()
    if raw in DEFECT_LABEL_OVERRIDES:
        return (DEFECT_LABEL_OVERRIDES[raw],)
    out = set()
    for part in re.split(r"[,;/&+|]|\band\b", raw.lower()):
        part = part.strip()
        if not part:
            continue
        if part in DEFECT_LABEL_OVERRIDES:
            out.add(DEFECT_LABEL_OVERRIDES[part])
            continue
        # FINAL BUILD bug fix: keywords must start a word -- plain substrings matched "lack" inside "black", so
        # "black spot" became missing and "scratch on black pad" became missing instead of scratch
        hit = next((canon for canon, keys in DEFECT_KEYWORDS if any(re.search(r"\b" + re.escape(k), part) for k in keys)), None)
        out.add(hit or (re.sub(r"[^a-z0-9 ]+", "", part).strip() or "unspecified"))
    return tuple(sorted(out)) if out else ("unspecified",)


def find_defect_annotations(cls):
    """Path of <cls>/image_anno.csv, or None: CFG.VISA_ANNOTATION_DIR, next to the images, the fetch cache, Kaggle inputs."""
    for d in (CFG.VISA_ANNOTATION_DIR, VISA_ROOT, DEFECT_ANNO_CACHE):
        if d and os.path.exists(os.path.join(d, cls, "image_anno.csv")):
            return os.path.join(d, cls, "image_anno.csv")
    for base in (CFG.VISA_ANNOTATION_DIR, CFG.KAGGLE_INPUT, CFG.EXTRACT_DIR):
        if base:
            hits = sorted(glob.glob(os.path.join(base, "**", cls, "image_anno.csv"), recursive=True))
            if hits:
                return hits[0]
    return None


def fetch_visa_annotations(classes, out_dir=DEFECT_ANNO_CACHE):
    """Streams the official VisA archive (never saved) and keeps only <cls>/image_anno.csv for `classes`."""
    want, got, seen_bytes = set(classes), {}, [0, 500 << 20]

    class _Counting:                                            # progress in the log while the archive streams past
        def __init__(self, f):
            self.f = f

        def read(self, k=-1):
            b = self.f.read(k)
            seen_bytes[0] += len(b)
            if seen_bytes[0] >= seen_bytes[1]:
                log("  ... %.1f GB of the VisA archive read" % (seen_bytes[0] / 2.0 ** 30))
                seen_bytes[1] += 500 << 20
            return b

    log("Fetching VisA defect-type labels for " + ", ".join(sorted(want)) + " from " + CFG.VISA_TAR_URL)
    req = urllib.request.Request(CFG.VISA_TAR_URL, headers={"User-Agent": "pcb-thesis-notebook"})
    with urllib.request.urlopen(req, timeout=120) as resp, tarfile.open(fileobj=_Counting(resp), mode="r|*") as tf:
        for m in tf:
            parts = m.name.replace("\\", "/").split("/")
            if m.isfile() and parts[-1] == "image_anno.csv" and len(parts) >= 2 and parts[-2] in want:
                dst = os.path.join(out_dir, parts[-2], "image_anno.csv")
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                with open(dst, "wb") as fh:
                    fh.write(tf.extractfile(m).read())
                got[parts[-2]] = dst
                if set(got) == want:
                    break                                       # stop downloading once every class is in
    return got


def read_defect_annotations(path):
    """image_anno.csv -> {file stem (lower case): raw label} for the DEFECTIVE images only."""
    df = pd.read_csv(path)
    cols = {str(c).strip().lower(): c for c in df.columns}
    icol, lcol = cols.get("image", df.columns[0]), cols.get("label", df.columns[1])
    return {os.path.splitext(os.path.basename(i))[0].lower(): l.strip()
            for i, l in zip(df[icol].astype(str), df[lcol].astype(str))
            if l.strip().lower() not in ("normal", "good", "nan", "", "0")}


# ---------------------------------------------------------------- region descriptors (shared with the deployed API)
@torch.no_grad()
def region_features(det, img_bgr):
    """(model tensor, CNN patch features, ViT patch features) of one image on the detector's grid -- read only."""
    t = preprocess_bgr(img_bgr, **det.prep())
    cnn, vit = det._extract_branch_grids(t)
    return t, cnn, vit


def box_grid_weights(box, photo_hw, det):
    """Weight of every feature-grid cell inside a box (x0, y0, x1, y1 in photo pixels), with the detector's exact
    geometry (letterbox or resize). Boxes are widened to at least CFG.REGION_MIN_CELLS x CFG.REGION_MIN_CELLS cells so a
    tiny defect is described with some local context; the same rule applies to every region source."""
    S = CFG.IMG_SIZE
    gh, gw = det.grid_hw
    H, W = photo_hw
    x0, y0, x1, y1 = [float(v) for v in box]
    if det.prep()["letterbox"]:
        side = float(max(H, W))
        sx = sy = S / side
        ox, oy = (side - W) // 2 * sx, (side - H) // 2 * sy
    else:
        sx, sy, ox, oy = S / float(W), S / float(H), 0.0, 0.0
    x0, x1, y0, y1 = x0 * sx + ox, x1 * sx + ox, y0 * sy + oy, y1 * sy + oy
    min_w, min_h = CFG.REGION_MIN_CELLS * S / float(gw), CFG.REGION_MIN_CELLS * S / float(gh)
    cx, cy = 0.5 * (x0 + x1), 0.5 * (y0 + y1)
    hw, hh = 0.5 * max(x1 - x0, min_w), 0.5 * max(y1 - y0, min_h)
    xa, xb = int(round(max(0.0, cx - hw))), int(round(min(float(S), cx + hw)))
    ya, yb = int(round(max(0.0, cy - hh))), int(round(min(float(S), cy + hh)))
    if xb <= xa or yb <= ya:
        return None
    canvas = np.zeros((S, S), np.float32)
    canvas[ya:yb, xa:xb] = 1.0
    return cv2.resize(canvas, (gw, gh), interpolation=cv2.INTER_AREA).ravel()


def pool_region(cnn, vit, weights):
    """Region descriptor: weighted mean patch feature per branch, each L2-normalized, concatenated [CNN | ViT]."""
    if weights is None or weights.sum() <= 0:
        return None
    w = (weights / weights.sum())[:, None]
    c, v = (cnn * w).sum(0), (vit * w).sum(0)
    return np.concatenate([c / (np.linalg.norm(c) + 1e-8), v / (np.linalg.norm(v) + 1e-8)]).astype(np.float32)


def defect_blobs(path, photo_hw, max_blobs=10):
    """Ground-truth defect blobs of one photo: [{"box": (x0, y0, x1, y1), "mask": bool array}], largest first."""
    mp = find_defect_mask(path)
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE) if mp else None
    if m is None:
        return []
    if m.shape[:2] != tuple(photo_hw):
        m = cv2.resize(m, (photo_hw[1], photo_hw[0]), interpolation=cv2.INTER_NEAREST)
    n, lab, stats, _ = cv2.connectedComponentsWithStats((m > 0).astype(np.uint8), connectivity=8)
    blobs = [{"box": (int(x), int(y), int(x + w), int(y + h)), "mask": lab == k}
             for k, (x, y, w, h, area) in enumerate(stats[1:], 1) if area >= CFG.REGION_MIN_DEFECT_PIXELS]
    return sorted(blobs, key=lambda b: -int(b["mask"].sum()))[:max_blobs]


def box_hits_defect(box, blobs):
    """True if the box covers at least CFG.REGION_HIT_FRAC of some ground-truth blob's pixels."""
    x0, y0, x1, y1 = [int(round(v)) for v in box]
    return any(b["mask"][max(0, y0):y1, max(0, x0):x1].sum() >= CFG.REGION_HIT_FRAC * b["mask"].sum() for b in blobs)


def _region_records(cls, path, split, types, det):
    """Every ground-truth and detector region of one defective photo, with its descriptor."""
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        return []
    hw = img.shape[:2]
    t, cnn, vit = region_features(det, img)
    blobs = defect_blobs(path, hw)
    amap, score = det.score_image_tensor(t)                     # single pass, exactly as test_single_image scores it
    lb = det.prep()["letterbox"]
    regs = regions_in_photo(amap, hw, lb)
    base = {"cls": cls, "path": path, "file": os.path.basename(path), "split": split, "types": types,
            "flagged": bool(score > det.threshold)}
    out = []
    for k, b in enumerate(blobs):
        x = pool_region(cnn, vit, box_grid_weights(b["box"], hw, det))
        if x is not None:
            out.append(dict(base, source="ground truth", rank=k, box=b["box"], hit=True, x=x))
    for k, g in enumerate(regs):
        box = map_box_to_photo(g["bbox"], amap.shape[:2], hw, lb)
        x = pool_region(cnn, vit, box_grid_weights(box, hw, det))
        if x is not None:
            out.append(dict(base, source="detector", rank=k, box=box, hit=box_hits_defect(box, blobs) if blobs else False, x=x))
    return out


# ---------------------------------------------------------------- run
if not getattr(CFG, "RUN_DEFECT_CLASSIFIER", True):
    print("CFG.RUN_DEFECT_CLASSIFIER is False -- Part 1.4 skipped.")
elif not DEFECT_DETECTORS:
    print("No trained detectors in this session -- run Cells 1-12 first (Cell 9 trains them). Part 1.4 skipped.")
else:
    labelled = {}                                               # cls -> labelled defective images (filled below)
    with doc_step("defect-type labels (VisA image_anno.csv)"):
        missing = [c for c in CFG.CLASSES if c in DEFECT_DETECTORS and find_defect_annotations(c) is None]
        if missing and CFG.FETCH_VISA_ANNOTATIONS:
            try:
                fetch_visa_annotations(missing)
            except Exception as e:
                log("Could not fetch the VisA labels (%r). Turn the notebook's internet on, attach the ORIGINAL VisA "
                    "download as a Kaggle input, or set CFG.VISA_ANNOTATION_DIR to a folder with <class>/image_anno.csv."
                    % (e,), "WARN")
        vocab_rows = []
        for cls in CFG.CLASSES:
            apath = find_defect_annotations(cls) if cls in DEFECT_DETECTORS else None
            if apath is None:
                if cls in DEFECT_DETECTORS:
                    log("[" + cls + "] no image_anno.csv -- no defect types for this class", "WARN")
                continue
            ann, fl = read_defect_annotations(apath), FILE_LISTS[cls]
            items, unmatched = [], 0
            for split in ("calib", "eval"):
                for p, l in zip(fl[split + "_images"], fl[split + "_labels"]):
                    if l != 1:
                        continue
                    raw = ann.get(os.path.splitext(os.path.basename(p))[0].lower())
                    if raw is None:
                        unmatched += 1
                    else:
                        items.append((p, split, raw, canon_defect_types(raw)))
            if unmatched:
                log("[" + cls + "] %d defective images have no label row -- left out" % unmatched, "WARN")
            # vocabulary from CALIBRATION images only; rarer types are merged into "other"
            cal_counts = Counter(t for _, s, _, ts in items if s == "calib" for t in ts)
            keep = {t for t, n in cal_counts.items() if n >= CFG.DEFECT_MIN_CALIB_PER_TYPE}
            merged = [(p, s, raw, tuple(sorted({t if t in keep else "other" for t in ts}))) for p, s, raw, ts in items]
            DEFECT_VOCAB[cls] = sorted({t for _, s, _, ts in merged if s == "calib" for t in ts})
            labelled[cls] = merged
            for p, _, _, ts in merged:
                DEFECT_TRUE[(cls, os.path.basename(p))] = ts
            raw_by_type = {}
            for _, _, raw, ts in merged:
                for t in ts:
                    raw_by_type.setdefault(t, Counter())[raw] += 1
            for t in sorted(raw_by_type):
                vocab_rows.append({"pcb_class": cls, "defect type": t,
                                   "calibration images": sum(t in ts for _, s, _, ts in merged if s == "calib"),
                                   "eval images": sum(t in ts for _, s, _, ts in merged if s == "eval"),
                                   "learnable (has calibration images)": t in DEFECT_VOCAB[cls],
                                   "raw labels (most common)": "; ".join("%s (%d)" % rn for rn in raw_by_type[t].most_common(4))})
            log("[%s] types: %s | defective images: calib %d, eval %d | with several types: %d" % (
                cls, ", ".join(DEFECT_VOCAB[cls]), sum(s == "calib" for _, s, _, _ in merged),
                sum(s == "eval" for _, s, _, _ in merged), sum(len(ts) > 1 for _, _, _, ts in merged)))
        defect_vocab_df = pd.DataFrame(vocab_rows)
        if len(defect_vocab_df):
            print("\nDEFECT-TYPE VOCABULARY (check the raw labels column: is every raw label mapped to the right type?)")
            print(defect_vocab_df.to_string(index=False))
            save_table(defect_vocab_df, "defect_type_vocabulary",
                       "Defect types per PCB class: calibration / eval image counts and the raw VisA labels mapped to each")

    if labelled:
        with doc_step("region dataset: ground-truth blobs + detector-highlighted regions, frozen CNN+ViT descriptors"):
            for cls, items in labelled.items():
                det = DEFECT_DETECTORS[cls]
                recs = []
                for p, split, _, ts in tqdm(items, desc="[" + cls + "] regions", leave=False):
                    recs += _region_records(cls, p, split, ts, det)
                DEFECT_REGIONS[cls] = recs
                gt = [r for r in recs if r["source"] == "ground truth"]
                dr = [r for r in recs if r["source"] == "detector"]
                log("[%s] regions: ground truth %d (calib %d, eval %d) | detector %d, of which on a real defect %.0f%%" % (
                    cls, len(gt), sum(r["split"] == "calib" for r in gt), sum(r["split"] == "eval" for r in gt), len(dr),
                    100.0 * np.mean([r["hit"] for r in dr]) if dr else 0.0))
        DEFECT_OK = any(len(DEFECT_VOCAB.get(c, [])) >= 2 and any(r["split"] == "eval" for r in DEFECT_REGIONS.get(c, []))
                        for c in DEFECT_REGIONS)
    if not DEFECT_OK:
        print("Defect-type classification is not possible in this session (no class has >= 2 learnable types with "
              "labelled regions). Cell 12c will skip.")

In [ ]:
# ============================================================
# CELL 12c -- REGION DEFECT-TYPE CLASSIFIER: trained on CALIBRATION regions, reported on held-out EVAL regions
# ============================================================
doc_section("CELL 12c", "REGION DEFECT-TYPE CLASSIFIER: trained on CALIBRATION regions, reported on held-out EVAL regions")
# One linear classifier per PCB class (the pipeline already knows the class): StandardScaler + class-balanced L2 logistic
# regression on the region descriptor of Cell 12b. C is chosen by grouped cross-validation on CALIBRATION regions (the
# regions of one image never sit in two folds). Training regions: calibration ground-truth blobs, plus calibration
# detector regions that lie on a real defect -- so the head also sees the looser boxes it receives at test time.
# Reported on EVAL images only (CIs resample IMAGES within each class; paired when two predictors are compared):
#   A  ground-truth regions          typing accuracy when the region is exactly right (upper bound of the head)
#   B  detector-highlighted regions  share on a real defect, and typing accuracy on those
#   C  end to end, per defective board: flagged by the detector, top region on the defect, and typed correctly
# A region's predicted type is correct when it is one of the image's true types.
# classify_region_boxes() is the ONE entry point every test output uses (Cells 13, 15, 20e, 22a): it types any boxes in
# any image, with the frozen Cell 9 features, and never raises.
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import balanced_accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix

DEFECT_CLASSIFIERS = {}          # cls -> trained region classifier (used by classify_region_boxes)
DEFECT_MAJORITY = {}             # cls -> most frequent calibration type (baseline)
# Which box of an inspect_board region (Cells 20e/22b/22c) is typed: "region" = the whole region box, "highlighted" = the
# tight box around the region's pixels above the calibrated pixel threshold (falls back to the region box when none).
# CFG.DEFECT_TYPE_BOX = "auto" lets Cell 20e2 choose on CALIBRATION defects (out-of-fold); until then "region".
DEFECT_TYPE_BOX = CFG.DEFECT_TYPE_BOX if CFG.DEFECT_TYPE_BOX in ("region", "highlighted") else "region"
defect_results_df = defect_localization_df = defect_per_type_df = defect_paired_df = None
EVAL, GT, DR, DR_HIT = [], [], [], []   # held-out regions with predictions: all / ground truth / detector / detector on a defect


# ---------------------------------------------------------------- classifier
def _defect_lr(C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight="balanced", max_iter=5000))


def _region_xy(regions):
    """Training matrix: a region with several types appears once per type, weight 1/number of its types."""
    X, y, g, w = [], [], [], []
    for r in regions:
        for t in r["types"]:
            X.append(r["x"]); y.append(t); g.append(r["path"]); w.append(1.0 / len(r["types"]))
    return np.asarray(X, np.float32), np.asarray(y), np.asarray(g), np.asarray(w, np.float64)


def _choose_C(X, y, g, w):
    """C by grouped, stratified CV (balanced accuracy); ties -> smaller C (stronger regularization)."""
    per_type = Counter(t for t, _ in set(zip(y.tolist(), g.tolist())))
    n_splits = min(3, min(per_type.values()))
    if len(per_type) < 2 or n_splits < 2:
        return 1.0, "C=1 fixed (too few calibration images per type to cross-validate)"
    folds = list(StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED).split(X, y, g))
    scores = {}
    for C in CFG.DEFECT_C_GRID:
        yt, yp = [], []
        for tr, va in folds:
            if len(set(y[tr].tolist())) < 2:
                continue
            m = _defect_lr(C).fit(X[tr], y[tr], logisticregression__sample_weight=w[tr])
            yt += y[va].tolist()
            yp += m.predict(X[va]).tolist()
        if yt:
            scores[C] = balanced_accuracy_score(yt, yp)
    if not scores:
        return 1.0, "C=1 fixed (degenerate cross-validation folds)"
    best = max(scores, key=lambda c: (round(scores[c], 4), -c))
    return best, "calibration CV balanced accuracy " + ", ".join("C=%g: %.3f" % kv for kv in scores.items())


def fit_region_classifier(regions):
    X, y, g, w = _region_xy(regions)
    if len(y) == 0:
        return None
    if len(set(y.tolist())) < 2:
        return {"kind": "constant", "label": str(y[0]), "note": "only one type among calibration regions"}
    C, note = _choose_C(X, y, g, w)
    m = _defect_lr(C).fit(X, y, logisticregression__sample_weight=w)
    return {"kind": "lr", "model": m, "C": C, "note": note, "classes": [str(c) for c in m.classes_]}


def _predict(model, X):
    """-> list of (type, probability, {type: probability}) for the rows of X."""
    if model["kind"] == "constant":
        return [(model["label"], 1.0, {model["label"]: 1.0}) for _ in range(len(X))]
    P = model["model"].predict_proba(np.asarray(X, np.float32))
    return [(model["classes"][int(np.argmax(p))], float(p.max()), dict(zip(model["classes"], map(float, p)))) for p in P]


# ---------------------------------------------------------------- deployed API (every test output calls these)
def classify_region_boxes(img_bgr, pcb_class, boxes, top=3, model=None):
    """Defect type of every box (x0, y0, x1, y1 in img_bgr's pixels) with the Part 1.4 classifier of `pcb_class`.
    One frozen-feature pass per image; never raises. Each result: defect_type (None if unavailable), type_confidence,
    type_alternatives (text), type_note. The classifier always names a known type: it does not decide WHETHER a region
    is a defect -- the detector does. model: use this classifier instead of the deployed one (Cell 20e2's out-of-fold
    selection)."""
    def none(note):
        return [{"defect_type": None, "type_confidence": None, "type_alternatives": "", "type_note": note} for _ in boxes]
    if not boxes:
        return []
    model, det = (model or DEFECT_CLASSIFIERS.get(pcb_class)), DEFECT_DETECTORS.get(pcb_class)
    if model is None or det is None:
        return none("no defect-type classifier for %s (Part 1.4 skipped or no labels)" % pcb_class)
    try:
        img_bgr = np.ascontiguousarray(img_bgr)
        _, cnn, vit = region_features(det, img_bgr)
        xs = [pool_region(cnn, vit, box_grid_weights(b, img_bgr.shape[:2], det)) for b in boxes]
        ok = [i for i, x in enumerate(xs) if x is not None]
        preds = dict(zip(ok, _predict(model, [xs[i] for i in ok]))) if ok else {}
        out = []
        for i in range(len(boxes)):
            if i not in preds:
                out += none("region outside the image")
                continue
            lab, prob, probs = preds[i]
            alt = [(k, v) for k, v in sorted(probs.items(), key=lambda kv: -kv[1]) if k != lab][:top - 1]
            out.append({"defect_type": lab, "type_confidence": round(prob, 3),
                        "type_alternatives": ", ".join("%s %.0f%%" % (k, 100 * v) for k, v in alt),
                        "type_note": "trained region classifier (Part 1.4)"})
        return out
    except Exception as e:                                       # an add-on must never break an inspection
        return none("classifier failed: %r" % (e,))


def aligned_box_to_photo(box, info, photo_hw):
    """Box (x0, y0, x1, y1) in the ALIGNED board frame -> axis-aligned box in the ORIGINAL photo's pixels, through the
    inverse registration transform (the same convention as inspect_board). None when the photo was not registered.
    Regions are typed on the original photo because the classifier learned from original photos: the aligned frame turns
    some boards by ~180 deg (many PCB1 photos are upside down relative to the class reference), and CNN+ViT features are
    not rotation-invariant."""
    if info is None or info.get("M") is None:
        return None
    Minv = cv2.invertAffineTransform(np.asarray(info["M"], np.float64))
    H, W = photo_hw
    scale = CFG.ROBUSTNESS_LONG_SIDE / float(max(H, W))
    f = 1.0 if scale >= 1.0 else int(round(W * scale)) / float(W)          # downscale_long_side, as in inspect_board
    x0, y0, x1, y1 = [float(v) for v in box]
    pts = [(Minv[0, 0] * x + Minv[0, 1] * y + Minv[0, 2], Minv[1, 0] * x + Minv[1, 1] * y + Minv[1, 2])
           for x, y in ((x0, y0), (x1, y0), (x1, y1), (x0, y1))]
    xs, ys = [p[0] / f for p in pts], [p[1] / f for p in pts]
    return (max(0, int(min(xs))), max(0, int(min(ys))), min(W, int(round(max(xs)))), min(H, int(round(max(ys)))))


def type_regions(regions, img_bgr, pcb_class, box_of):
    """Adds defect_type / type_confidence / type_alternatives to every region dict, in place.
    box_of(region) -> (x0, y0, x1, y1) in img_bgr's pixels. Returns the regions."""
    if regions:
        for r, t in zip(regions, classify_region_boxes(img_bgr, pcb_class, [box_of(r) for r in regions])):
            r.update({k: t[k] for k in ("defect_type", "type_confidence", "type_alternatives")})
    return regions


def typing_box(r):
    """The original-photo box of an inspect_board region that the classifier reads (DEFECT_TYPE_BOX)."""
    tight = r.get("bbox highlighted in original photo (x0,y0,x1,y1)")
    return tight if (DEFECT_TYPE_BOX == "highlighted" and tight) else r["bbox in original photo (x0,y0,x1,y1)"]


def region_type_text(r, alternatives=False):
    """'bent (72%)' for a typed region; '' otherwise."""
    if not r.get("defect_type"):
        return ""
    txt = "%s (%.0f%%)" % (r["defect_type"], 100 * r["type_confidence"])
    return txt + (" | alt: " + r["type_alternatives"] if alternatives and r.get("type_alternatives") else "")


def load_defect_type_pipeline(artifact_dir=CFG.ARTIFACT_DIR):
    """New session: restores the type classifiers and the Cell 9 detectors whose frozen features they read (Cell 12
    pickles). Needs Cells 1-12b (definitions). PCBAnomalyDetector.save() does not store feat_cfg, so it is restored
    from the sidecar written below."""
    global DEFECT_DETECTORS, DEFECT_CLASSIFIERS, DEFECT_VOCAB, DEFECT_MAJORITY, DEFECT_TYPE_BOX
    with open(os.path.join(artifact_dir, "defect_type_classifiers.pkl"), "rb") as fh:
        pack = pickle.load(fh)
    side = os.path.join(artifact_dir, "detector_feature_cfg.json")
    cfgs = json.load(open(side)) if os.path.exists(side) else {}
    dets = {}
    for cls in pack["classifiers"]:
        d = PCBAnomalyDetector.load(os.path.join(artifact_dir, cls + "_detector.pkl"))
        if cls in cfgs:
            d.feat_cfg = dict(cfgs[cls])
        dets[cls] = d
    DEFECT_DETECTORS, DEFECT_CLASSIFIERS = dets, pack["classifiers"]
    DEFECT_VOCAB, DEFECT_MAJORITY = pack.get("vocab", {}), pack.get("majority", {})
    DEFECT_TYPE_BOX = pack.get("type_box", "region")
    print("Loaded the region defect-type classifiers for:", sorted(dets))
    return dets


# ---------------------------------------------------------------- evaluation helpers
def _boot_ratio(per_cls, seed):
    """per_cls: {cls: [(value, n) per image]} -> (sum value / sum n, 95% CI); images resampled within each class."""
    arrs = [np.asarray(v, np.float64) for v in per_cls.values() if len(v)]
    if not arrs or sum(a[:, 1].sum() for a in arrs) == 0:
        return float("nan"), (float("nan"), float("nan"))
    point = sum(a[:, 0].sum() for a in arrs) / sum(a[:, 1].sum() for a in arrs)
    rng, reps = np.random.default_rng(seed), []
    for _ in range(CFG.BOOTSTRAP_N):
        s = [a[rng.integers(0, len(a), len(a))] for a in arrs]
        n = sum(x[:, 1].sum() for x in s)
        if n > 0:
            reps.append(sum(x[:, 0].sum() for x in s) / n)
    lo, hi = np.percentile(reps, [2.5, 97.5])
    return float(point), (float(lo), float(hi))


def _pct(p, ci, diff=False):
    f = (lambda v: "%+.1f pp" % (100 * v)) if diff else (lambda v: "%.1f%%" % (100 * v))
    return "n/a" if not np.isfinite(p) else f(p) + " [" + f(ci[0]) + ", " + f(ci[1]) + "]"


def _per_image(regions, key):
    """{cls: [(hits, n) per image]} for a per-region 0/1 field."""
    out = {}
    for r in regions:
        out.setdefault(r["cls"], {}).setdefault(r["path"], []).append(float(r[key]))
    return {c: [(sum(v), len(v)) for v in imgs.values()] for c, imgs in out.items()}


def _paired(regions, key_a, key_b):
    out = {}
    for r in regions:
        out.setdefault(r["cls"], {}).setdefault(r["path"], []).append(float(r[key_a]) - float(r[key_b]))
    return {c: [(sum(v), len(v)) for v in imgs.values()] for c, imgs in out.items()}


# ---------------------------------------------------------------- train + evaluate
if not DEFECT_OK:
    print("No labelled regions from Cell 12b -- the region classifier is skipped.")
else:
    with doc_step("train on calibration regions, evaluate on held-out eval regions"):
        for cls, recs in DEFECT_REGIONS.items():
            if len(DEFECT_VOCAB.get(cls, [])) < 2:
                continue
            train = [r for r in recs if r["split"] == "calib" and r["hit"]]
            model = fit_region_classifier(train)
            if model is None:
                continue
            DEFECT_CLASSIFIERS[cls] = model
            wc = Counter()
            for r in {r["path"]: r for r in recs if r["split"] == "calib"}.values():
                for t in r["types"]:
                    wc[t] += 1.0 / len(r["types"])
            DEFECT_MAJORITY[cls] = max(sorted(wc), key=lambda t: wc[t])
            ev = [r for r in recs if r["split"] == "eval"]
            for r, (lab, prob, _) in zip(ev, _predict(model, [r["x"] for r in ev]) if ev else []):
                r.update(pred=lab, prob=prob, ok=lab in r["types"], ok_majority=DEFECT_MAJORITY[cls] in r["types"])
            EVAL += ev
            log("[%s] region classifier: %d training regions | %s" % (cls, len(train), model.get("note", "")))
        GT = [r for r in EVAL if r["source"] == "ground truth"]
        DR = [r for r in EVAL if r["source"] == "detector"]
        DR_HIT = [r for r in DR if r["hit"]]

        rows = []
        for name, key in [("majority type (no learning)", "ok_majority"), ("trained region classifier", "ok")]:
            a, ca = _boot_ratio(_per_image(GT, key), SEED + 31)
            b, cb = _boot_ratio(_per_image(DR_HIT, key), SEED + 31)
            single = [r for r in GT if len(r["types"]) == 1]
            f1 = np.nanmean([f1_score([r["types"][0] for r in single if r["cls"] == c],
                                      [(r["pred"] if key == "ok" else DEFECT_MAJORITY[c]) for r in single if r["cls"] == c],
                                      average="macro", zero_division=0)
                             for c in DEFECT_CLASSIFIERS if any(r["cls"] == c for r in single)])
            a1, ca1 = _boot_ratio(_per_image([r for r in GT if len(r["types"]) == 1], key), SEED + 31)
            rows.append({"predictor": name, "A: ground-truth regions, accuracy [95% CI]": _pct(a, ca),
                         "A: single-type images only (strict) [95% CI]": _pct(a1, ca1),
                         "B: Cell 9 detector regions on a defect, accuracy [95% CI]": _pct(b, cb),
                         "A: macro-F1 (single-type images)": round(float(f1), 3),
                         **{c + " (A)": round(float(np.mean([r[key] for r in GT if r["cls"] == c])), 3)
                            for c in DEFECT_CLASSIFIERS if any(r["cls"] == c for r in GT)}})
        defect_results_df = pd.DataFrame(rows)

        prow = []
        for label, regs in [("A: ground-truth regions", GT), ("B: Cell 9 detector regions on a defect", DR_HIT)]:
            d, ci = _boot_ratio(_paired(regs, "ok", "ok_majority"), SEED + 37)
            prow.append({"trained classifier minus majority type": label, "eval regions": len(regs),
                         "difference in accuracy [95% CI]": _pct(d, ci, diff=True),
                         "verdict": "trained better" if ci[0] > 0 else "trained worse" if ci[1] < 0 else "no significant difference"})
        defect_paired_df = pd.DataFrame(prow)

        loc_rows = []
        for c in list(DEFECT_CLASSIFIERS) + ["all"]:
            imgs = sorted({r["path"] for r in EVAL if c in ("all", r["cls"])})
            d_c = [r for r in DR if c in ("all", r["cls"])]
            top = {r["path"]: r for r in d_c if r["rank"] == 0}
            flagged = {r["path"] for r in EVAL if c in ("all", r["cls"]) and r["flagged"]}
            loc_rows.append({"pcb_class": c, "eval defective images": len(imgs), "highlighted regions": len(d_c),
                             "regions on a real defect": round(float(np.mean([r["hit"] for r in d_c])), 3) if d_c else None,
                             "flagged by the detector": round(len(flagged) / max(1, len(imgs)), 3),
                             "top region on the defect": round(sum(top[p]["hit"] for p in imgs if p in top) / max(1, len(imgs)), 3),
                             "END TO END: flagged, top region on the defect and correctly typed":
                                 round(sum(p in flagged and top[p]["hit"] and top[p]["ok"] for p in imgs if p in top) / max(1, len(imgs)), 3)})
        defect_localization_df = pd.DataFrame(loc_rows)

        pt_rows = []
        for c in DEFECT_CLASSIFIERS:
            single = [r for r in GT if r["cls"] == c and len(r["types"]) == 1]
            if not single:
                continue
            yt, yp = [r["types"][0] for r in single], [r["pred"] for r in single]
            labels = sorted(set(yt) | set(yp))
            for lab, p, rc, f, s in zip(labels, *precision_recall_fscore_support(yt, yp, labels=labels, zero_division=0)):
                pt_rows.append({"pcb_class": c, "defect type": lab, "eval regions (single-type images)": int(s),
                                "times predicted": int(sum(v == lab for v in yp)),
                                "precision": round(float(p), 3) if lab in yp else None,
                                "recall": round(float(rc), 3) if s else None,       # undefined without true examples
                                "F1": round(float(f), 3) if s else None,
                                "learnable": lab in DEFECT_VOCAB[c]})
        defect_per_type_df = pd.DataFrame(pt_rows)

        print("\nREGION DEFECT-TYPE CLASSIFICATION -- held-out eval regions (A: exact regions, B: detector-highlighted regions)")
        print(defect_results_df.to_string(index=False))
        print("\nPAIRED: trained classifier vs majority type (same regions; verdict only when the 95% CI excludes 0)")
        print(defect_paired_df.to_string(index=False))
        print("\nDETECTOR REGIONS AND END TO END (eval defective images)")
        print(defect_localization_df.to_string(index=False))
        print("\nPER DEFECT TYPE (ground-truth regions of single-type eval images)")
        print(defect_per_type_df.to_string(index=False))
        save_table(defect_results_df, "defect_type_region_results",
                   "Region defect-type classification on held-out regions: exact (ground-truth) and detector-highlighted regions")
        save_table(defect_paired_df, "defect_type_region_paired", "Region defect-type classifier vs the majority type (paired CIs)")
        save_table(defect_localization_df, "defect_type_region_localization",
                   "Detector-highlighted regions: share on a real defect, and end-to-end typed defects per class")
        save_table(defect_per_type_df, "defect_type_region_per_type", "Region defect-type classifier: precision / recall / F1 per type")

    if DEFECT_CLASSIFIERS:
        with doc_step("figure: accuracy and confusion matrices"):
            fig, ax = plt.subplots(figsize=(8, 3.6))
            names = ["majority type", "trained region classifier"]
            for k, (lab, regs, col) in enumerate([("A: ground-truth regions", GT, "#2a7ab9"),
                                                    ("B: Cell 9 detector regions on a defect", DR_HIT, "#d08a2c")]):
                for j, key in enumerate(["ok_majority", "ok"]):
                    p, ci = _boot_ratio(_per_image(regs, key), SEED + 31)
                    if not np.isfinite(p):
                        continue
                    ax.barh(j + 0.38 * k - 0.19, p, height=0.36, color=col, xerr=[[p - ci[0]], [ci[1] - p]], capsize=3,
                            label=lab if j == 0 else None)
            ax.set_yticks(range(len(names)))
            ax.set_yticklabels(names)
            ax.set_xlim(0, 1)
            ax.set_xlabel("defect-type accuracy on held-out regions (95% bootstrap CI)")
            ax.legend(loc="lower right", fontsize=8)
            ax.set_title("Region defect-type classification")
            caption("Region defect-type accuracy on held-out eval regions: exact ground-truth regions and detector-highlighted "
                    "regions that lie on a real defect, trained classifier vs the majority type")
            plt.tight_layout()
            plt.show()

            cl = [c for c in DEFECT_CLASSIFIERS if any(r["cls"] == c and len(r["types"]) == 1 for r in GT)]
            nc = 2 if len(cl) > 1 else 1
            nr = int(np.ceil(len(cl) / nc))
            fig, axes = plt.subplots(nr, nc, figsize=(6.2 * nc, 5.4 * nr), squeeze=False)
            for a, c in zip(axes.ravel(), cl):
                single = [r for r in GT if r["cls"] == c and len(r["types"]) == 1]
                yt, yp = [r["types"][0] for r in single], [r["pred"] for r in single]
                labels = sorted(set(yt) | set(yp))
                cm = confusion_matrix(yt, yp, labels=labels)
                a.imshow(cm, cmap="Blues")
                for i in range(len(labels)):
                    for j in range(len(labels)):
                        a.text(j, i, str(cm[i, j]), ha="center", va="center",
                               color="white" if cm[i, j] > 0.5 * max(1, cm.max()) else "black")
                a.set_xticks(range(len(labels)))
                a.set_xticklabels(labels, rotation=45, ha="right")
                a.set_yticks(range(len(labels)))
                a.set_yticklabels(labels)
                a.set_xlabel("predicted type")
                a.set_ylabel("true type")
                a.set_title("%s -- %.0f%% of %d regions" % (c, 100 * np.mean([x == y for x, y in zip(yt, yp)]), len(yt)))
            for a in axes.ravel()[len(cl):]:
                a.axis("off")
            caption("Region defect-type confusion matrices per PCB class (ground-truth regions of single-type held-out images)")
            plt.tight_layout()
            plt.show()

        with doc_step("figure: highlighted regions with their predicted type (held-out images)"):
            rng = random.Random(SEED + 41)
            picks = []
            for c in DEFECT_CLASSIFIERS:
                paths = sorted({r["path"] for r in DR if r["cls"] == c})
                rng.shuffle(paths)
                picks += [(c, p) for p in paths[:CFG.DEFECT_GALLERY_PER_CLASS]]
            if picks:
                ncol = max(1, CFG.DEFECT_GALLERY_PER_CLASS)
                nrow = int(np.ceil(len(picks) / ncol))
                fig, axes = plt.subplots(nrow, ncol, figsize=(6 * ncol, 4.6 * nrow), squeeze=False)
                for a, (c, p) in zip(axes.ravel(), picks):
                    img = cv2.cvtColor(cv2.imread(p, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
                    lw, fs = max(2, img.shape[1] // 300), max(0.4, img.shape[1] / 1200.0)
                    for b in defect_blobs(p, img.shape[:2]):                      # true defects: green outline
                        cnts, _ = cv2.findContours(b["mask"].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                        cv2.drawContours(img, cnts, -1, (0, 255, 0), lw)
                    for r in sorted((r for r in DR if r["path"] == p), key=lambda r: r["rank"]):
                        col = (160, 160, 160) if not r["hit"] else (0, 200, 0) if r["ok"] else (255, 0, 0)
                        x0, y0, x1, y1 = r["box"]
                        cv2.rectangle(img, (x0, y0), (x1, y1), col, lw)
                        put_label(img, "#%d %s %.0f%%" % (r["rank"] + 1, r["pred"], 100 * r["prob"]), x0, y0 - 6, fs, col,
                                  max(1, img.shape[1] // 500))
                    a.imshow(img)
                    a.set_title("%s %s -- true: %s" % (c, os.path.basename(p), "+".join(DEFECT_TRUE.get((c, os.path.basename(p)), ("?",)))),
                                fontsize=9)
                    a.axis("off")
                for a in axes.ravel()[len(picks):]:
                    a.axis("off")
                fig.suptitle("Detector-highlighted regions and their predicted type -- green box: on the defect and typed "
                             "correctly, red: on the defect but wrong type, grey: not on a defect; green outline: true defect",
                             fontsize=10)
                caption("Detector-highlighted regions on held-out defective boards with the predicted defect type of each region")
                plt.tight_layout(rect=(0, 0, 1, 0.96))
                plt.show()

        with doc_step("save the region classifiers"):
            with open(os.path.join(CFG.ARTIFACT_DIR, "defect_type_classifiers.pkl"), "wb") as fh:
                pickle.dump({"classifiers": DEFECT_CLASSIFIERS, "vocab": DEFECT_VOCAB, "majority": DEFECT_MAJORITY,
                             "keywords": DEFECT_KEYWORDS, "region_min_cells": CFG.REGION_MIN_CELLS,
                             "type_box": DEFECT_TYPE_BOX}, fh)
            with open(os.path.join(CFG.ARTIFACT_DIR, "detector_feature_cfg.json"), "w") as fh:
                json.dump({c: dict(getattr(d, "feat_cfg", {})) for c, d in DEFECT_DETECTORS.items()}, fh, indent=1)
            log("Region defect-type classifiers saved to " + CFG.ARTIFACT_DIR +
                " (new session: load_defect_type_pipeline())")

In [ ]:
# ============================================================
# CELL 13 -- test_single_image(): ad-hoc single-image testing
# ============================================================
doc_section("CELL 13", "test_single_image(): ad-hoc single-image testing")
def _class_match_margin(det, score, thr, use_tta=False):
    """Scale-normalized (score - threshold): how many 'class-appropriate spread units'
    above/below this class's own decision boundary the score sits. More negative means
    a better match to this class's normal appearance -- used by auto class-detection
    (pcb_class=None in test_single_image and inspect_pcb) to compare across the 4
    per-class detectors on a common, meaningful scale.

    FIXED BUG: this used to be a raw ratio (score / threshold), which silently breaks
    whenever a class's calibrated threshold is negative or near zero -- routine with the
    z-score-blended scores used here (verified: all 4 classes had negative single-pass
    thresholds in one real run). Dividing by a negative threshold flips the sign, so the
    old logic ended up preferring whichever class made an image look MOST anomalous
    rather than most normal -- verified with the real thresholds from that run: a
    genuinely normal PCB1 image was auto-assigned to PCB4 with a severity of 'severe'.
    A scale-normalized DIFFERENCE has no such sign issue regardless of threshold sign."""
    if thr is None:
        return score
    stats_key = "calib_scores_tta" if use_tta else "calib_scores"
    calib_scores = np.array(det.score_stats.get(stats_key, []))
    spread = float(calib_scores.std()) if len(calib_scores) > 1 else max(abs(thr), 1e-3)
    spread = max(spread, 1e-6)
    return (score - thr) / spread


def test_single_image(image_path=None, image_array=None, pcb_class=None,
                       rotation_tta=True, visualize=True, verbose=True,
                       localize=True, region_percentile=CFG.REGION_PERCENTILE, max_regions=CFG.MAX_REGIONS,
                       check_orientation=False, classify_types=True, type_image_bgr=None, type_box_map=None):
    """
    Run the trained hybrid CNN+ViT self-supervised anomaly detector on ONE image.

    Args:
        image_path      : path to an image file (jpg/png/bmp). Use this OR image_array.
        image_array      : HxWx3 RGB uint8 numpy array (e.g. a frame you already loaded/cropped).
        pcb_class        : one of CFG.CLASSES ("pcb1".."pcb4"). If None, the function tries all
                            4 trained detectors and auto-selects the class whose memory bank the
                            image matches best (lowest normalized anomaly score) -- handy when you
                            don't know which board type a photo shows.
        rotation_tta      : if True, runs rotation test-time augmentation (0/90/180/270 degrees,
                            fused by per-pixel median) for robustness to arbitrary input orientation.
                            Works for full boards, oblique shots, AND cropped/partial regions --
                            preprocessing always letterboxes to a square canvas first, so no
                            aspect-ratio distortion is introduced by a non-square crop.
        visualize         : if True, plots input / heatmap / overlay / flagged-regions and prints a verdict.
        verbose           : if True, prints a short text summary.
        localize          : if True AND the image is flagged anomalous, finds the most anomalous
                            connected region(s) in the anomaly map, circles them, and attaches a
                            rough SHAPE-BASED defect-type hint to each (see analyze_anomaly_regions
                            in Cell 11 -- this is a geometric heuristic, NOT a trained classifier).
        region_percentile : per-image percentile used to binarize the anomaly map for localization.
        max_regions       : max number of flagged regions drawn/reported.
        check_orientation : if True, ALSO runs the orientation-aware canonical-alignment pathway
                            (Cell 14) -- estimates the board's rotation and screens for
                            orientation-DEFINED defects (wrong polarity, tombstoning) that the
                            main rotation-INVARIANT pathway above cannot see by design. Requires
                            build_canonical_pathway() + calibrate_canonical_threshold() to have
                            been run for this class first (Cell 14); otherwise prints a note and
                            skips gracefully. Off by default -- purely additive, does not change
                            the main verdict.
        classify_types    : name the DEFECT TYPE of every highlighted region (Part 1.4 classifier; types are added to
                            each region dict as defect_type / type_confidence / type_alternatives). Never changes the verdict.
        type_image_bgr,   : when the inspected image is an ALIGNED copy (Cell 22a), the original photo and a function
        type_box_map        mapping a box in the inspected image's pixels to the original photo's pixels. The type is then
                            read on the original photo, where the classifier learned. None = type on the inspected image.

    Returns:
        dict with keys: pcb_class, anomaly_score, threshold, is_anomaly, anomaly_map, regions,
        orientation (None unless check_orientation=True and the canonical pathway is available;
        otherwise a dict with estimated_angle_deg, canonical_score, canonical_threshold, orientation_flag)
    """
    assert (image_path is not None) != (image_array is not None), \
        "Provide exactly one of image_path or image_array."

    if image_path is not None:
        img_bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(image_path)
        img_rgb_full = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    else:
        img_rgb_full = image_array

    tensor = load_image_from_array(img_rgb_full, **prep_for(pcb_class))

    def run_detector(cls):
        det = detectors[cls]
        if rotation_tta:
            amap, score = det.score_image_tensor_tta(tensor)
            thr = det.threshold_tta if getattr(det, "threshold_tta", None) is not None else det.threshold
        else:
            amap, score = det.score_image_tensor(tensor)
            thr = det.threshold
        margin = _class_match_margin(det, score, thr, use_tta=rotation_tta)
        return amap, score, thr, margin

    if pcb_class is not None:
        chosen_class = pcb_class
        anomaly_map, score, thr, _ = run_detector(pcb_class)
    else:
        best = None
        for cls in CFG.CLASSES:
            amap, score, thr, margin = run_detector(cls)
            if best is None or margin < best[4]:
                best = (cls, amap, score, thr, margin)
        chosen_class, anomaly_map, score, thr, _ = best

    det = detectors[chosen_class]
    is_anomaly = bool(score > thr)
    _rp = det.region_pixels(anomaly_map.shape) if hasattr(det, "region_pixels") else None
    if _rp is not None:                        # regions only on the board, and only where the photo showed it
        anomaly_map = np.where(_rp, anomaly_map, float(np.min(anomaly_map)))

    regions = []
    letterbox = prep_for(pcb_class)["letterbox"]
    if localize and is_anomaly:
        # FIXED (reporting only): regions are searched inside the PHOTO area of the letterboxed map -- the mirror padding of
        # letterbox_square used to echo an edge defect as a phantom region outside the photo.
        regions = regions_in_photo(anomaly_map, img_rgb_full.shape[:2], letterbox, region_percentile, max_regions)
    if regions and classify_types and "type_regions" in globals():
        # Part 1.4: one defect type per highlighted region. Boxes are found in THIS image; for an aligned copy the caller
        # passes the original photo + a box mapping, so the type is read where the classifier learned.
        boxes = {id(r): map_box_to_photo(r["bbox"], anomaly_map.shape[:2], img_rgb_full.shape[:2], letterbox) for r in regions}
        target = cv2.cvtColor(np.ascontiguousarray(img_rgb_full), cv2.COLOR_RGB2BGR)
        if type_image_bgr is not None and type_box_map is not None:
            mapped = {k: type_box_map(b) for k, b in boxes.items()}
            if all(v is not None for v in mapped.values()):
                boxes, target = mapped, type_image_bgr
        type_regions(regions, target, chosen_class, lambda r: boxes[id(r)])

    orientation_result = None
    if check_orientation:
        if getattr(det, "canonical_cnn_knn", None) is None:
            if verbose:
                print("Orientation check requested but the canonical pathway isn't built for '" + chosen_class
                      + "' yet -- run det.build_canonical_pathway(...) and det.calibrate_canonical_threshold(...) "
                      + "first (Cell 14).")
        else:
            est_angle, canon_score, _ = det.estimate_canonical_rotation(tensor)
            canon_thr = det.canonical_threshold
            orientation_flag = bool(canon_thr is not None and canon_score > canon_thr)
            orientation_result = {
                "estimated_angle_deg": est_angle,
                "canonical_score": canon_score,
                "canonical_threshold": canon_thr,
                "orientation_flag": orientation_flag,
            }
            if verbose:
                thr_str = str(round(canon_thr, 4)) if canon_thr is not None else "not calibrated"
                print("Orientation check -- estimated board rotation:", est_angle, "deg from canonical")
                print("  Canonical-alignment score:", round(canon_score, 4), "| threshold:", thr_str)
                print("  ->", ("POSSIBLE ORIENTATION-DEPENDENT DEFECT (wrong polarity / tombstoning / "
                                "upside-down component)") if orientation_flag else "orientation-sensitive check: OK")

    if verbose:
        auto_note = " (auto-detected)" if pcb_class is None else ""
        mode_note = " [TTA]" if rotation_tta else " [single-pass]"
        print("PCB class:", chosen_class + auto_note)
        print("Anomaly score:", round(score, 4), " | threshold" + mode_note + ":", round(thr, 4))
        print("Verdict:", "ANOMALY DETECTED" if is_anomaly else "NORMAL")
        if any(r.get("defect_type") for r in regions) and not visualize:
            print("Region types:", ", ".join("#%d %s" % (i, region_type_text(r)) for i, r in enumerate(regions, 1)))

    if visualize:
        visualize_result(img_rgb_full, anomaly_map, score, thr, title=chosen_class,
                          regions=regions if regions else None, letterbox=letterbox)

    return {
        "pcb_class": chosen_class,
        "anomaly_score": score,
        "threshold": thr,
        "is_anomaly": is_anomaly,
        "anomaly_map": anomaly_map,
        "regions": regions,
        "orientation": orientation_result,
    }

In [ ]:
# ============================================================
# CELL 14 -- Secondary contribution: orientation-aware canonical-alignment pathway (opt-in)
# ============================================================
doc_section("CELL 14", "Secondary contribution: orientation-aware canonical-alignment pathway (opt-in)")
# WHY THIS EXISTS: the main pathway (Cells 6-13) is deliberately rotation-INVARIANT --
# its memory banks are built from many rotated views of every training image (Cell 5's
# random_photometric_geometric_augment, angle = random.uniform(0, 360)), so a component
# rotated 180 degrees is treated as just as 'normal' as the correct orientation. That is
# exactly the right design for catching scratches, missing parts, solder defects, etc.
# regardless of camera angle -- but it also means the main pathway is STRUCTURALLY
# INCAPABLE of ever flagging a defect that IS a wrong orientation: wrong polarity (a
# component installed backwards) or tombstoning (a component standing up on one end)
# can look, to a rotation-invariant matcher, just like a valid rotated view of a normal
# board. This is a known, named trade-off in the anomaly-detection literature -- and it
# was flagged as an open gap in this project's own architecture-planning discussion.
#
# This cell closes that gap with a SECOND, purely self-supervised pathway that does the
# opposite on purpose:
#   1. build_canonical_pathway() builds a reference bank from ONLY the 'clean'
#      (non-rotation-augmented) view of each training image -- reusing features already
#      computed during Cell 9's det.fit(), no new model, no labels, no gradient training.
#   2. estimate_canonical_rotation() searches candidate rotation angles (coarse-to-fine)
#      for the one that makes a test image look MOST normal against this canonical bank
#      -- i.e. it estimates the board's true orientation purely by nearest-neighbor
#      alignment, with no trained regressor.
#   3. The image, rotated to that estimated angle, is then re-scored against the SAME
#      canonical bank. A genuinely normal board matches well at its true orientation. A
#      wrong-polarity or tombstoned component does NOT match well here, even though the
#      main rotation-invariant pathway would score the very same image as normal.
#
# This is entirely OPT-IN and additive: it does not touch the main pathway's banks,
# thresholds, or verdicts. Use it via test_single_image(..., check_orientation=True).
# It is also noticeably more expensive per image (~15 forward passes for the angle
# search vs. 1-4 for the main pathway), so calibration below is capped to a bounded
# number of images per class -- see calibrate_canonical_threshold's max_calib_images.
#
# GATED behind CFG.BUILD_CANONICAL_PATHWAY (default False) -- a real Kaggle run's
# papermill timings showed this cell alone taking 47 minutes (20% of a 3h45m total run)
# for a feature that's already off by default at inference. Set CFG.BUILD_CANONICAL_PATHWAY
# = True in Cell 2 (then re-run this cell) if you specifically need wrong-polarity /
# tombstoning detection and have the time budget for it.
# ============================================================
if not CFG.BUILD_CANONICAL_PATHWAY:
    print("Skipping the canonical-alignment pathway (CFG.BUILD_CANONICAL_PATHWAY=False).")
    print("This is the orientation-aware pathway used by check_orientation=True in test_single_image/")
    print("inspect_pcb -- it detects wrong-polarity/tombstoning defects the main pathway can't see by")
    print("design, but costs ~45-50 minutes to build+calibrate across all 4 classes. Set")
    print("CFG.BUILD_CANONICAL_PATHWAY = True in Cell 2 and re-run this cell if you want it.")
else:
    for cls in CFG.CLASSES:
        det = detectors[cls]
        fl = FILE_LISTS[cls]
        train_paths = fl["train_normal"][:CFG.MAX_TRAIN_IMAGES_PER_CLASS]

        det.build_canonical_pathway(train_paths)

        calib_images = fl.get("calib_images")
        calib_labels = fl.get("calib_labels")
        if calib_images:
            det.calibrate_canonical_threshold(calib_images, calib_labels)
            print("[" + cls + "] canonical-pathway threshold:", round(det.canonical_threshold, 4))
        else:
            print("[" + cls + "] WARNING: no stored calibration split found (re-run Cell 9 first) "
                  "-- skipping canonical threshold calibration.")

    print("\nOrientation-aware canonical pathway built for all classes.")
    print("Use test_single_image(..., check_orientation=True) or test_folder(..., check_orientation=True) to enable it.")

    # ---- quick demo ----
    demo_cls = "pcb1"
    demo_norm = [p for p, l in zip(FILE_LISTS[demo_cls]["eval_images"], FILE_LISTS[demo_cls]["eval_labels"]) if l == 0]
    if demo_norm:
        print("\n--- Orientation check on a known-normal image ---")
        _ = test_single_image(image_path=demo_norm[0], pcb_class=demo_cls, rotation_tta=True,
                               check_orientation=True, visualize=False)

In [ ]:
# ============================================================
# CELL 15 -- inspect_pcb(): practical quick-screen + severity + report wrapper
# ============================================================
doc_section("CELL 15", "inspect_pcb(): practical quick-screen + severity + report wrapper")
# A real inspection tool needs three things a research demo doesn't: (1) it should be
# FAST for the common case, (2) it should give a human a clear verdict, not a raw
# z-score, and (3) it should leave a record. This cell wraps everything already built
# into ONE practical entry point, mirroring how a real QC line actually works: a cheap
# fast screen first, escalating to the full (more expensive) rotation-TTA + region
# localization pass only when that screen is anomalous or borderline -- most boards on
# any real line are normal, so most images never pay the expensive path at all.
# ============================================================
def _severity_label(det, score, use_tta):
    """Maps a raw anomaly score to normal/mild/moderate/severe using this class's own
    calibration score distribution -- a human-readable rating instead of a bare number.
    Falls back to a simple margin-based rule if there aren't enough labeled calibration
    anomaly examples to build reliable percentiles."""
    thr = det.threshold_tta if use_tta else det.threshold
    if thr is None or score <= thr:
        return "normal"
    stats_key = "calib_scores_tta" if use_tta else "calib_scores"
    labels_key = "calib_labels_tta" if use_tta else "calib_labels"
    calib_scores = np.array(det.score_stats.get(stats_key, []))
    calib_labels = np.array(det.score_stats.get(labels_key, []))
    anomaly_scores = calib_scores[calib_labels == 1] if len(calib_scores) == len(calib_labels) and len(calib_scores) > 0 else np.array([])
    if len(anomaly_scores) >= 3:
        p33, p66 = np.percentile(anomaly_scores, [33, 66])
        if score < p33:
            return "mild"
        elif score < p66:
            return "moderate"
        return "severe"
    normal_scores = calib_scores[calib_labels == 0] if len(calib_scores) == len(calib_labels) and len(calib_scores) > 0 else calib_scores
    spread = float(normal_scores.std()) if len(normal_scores) > 1 else max(abs(thr), 1e-3)
    margin = (score - thr) / max(spread, 1e-6)
    if margin < 1.0:
        return "mild"
    elif margin < 3.0:
        return "moderate"
    return "severe"


def build_inspection_report(result):
    """Formats a clean, human-readable multi-line inspection report from an
    inspect_pcb() (or test_single_image()) result dict."""
    lines = []
    lines.append("=" * 52)
    lines.append("PCB INSPECTION REPORT")
    lines.append("=" * 52)
    lines.append("PCB class:        " + str(result.get("pcb_class")))
    lines.append("Verdict:          " + ("ANOMALY DETECTED" if result.get("is_anomaly") else "NORMAL"))
    if "severity" in result:
        lines.append("Severity:         " + str(result["severity"]))
    lines.append("Anomaly score:    " + str(round(result.get("anomaly_score", 0.0), 4)))
    lines.append("Threshold:        " + str(round(result.get("threshold", 0.0), 4)))
    if "mode" in result:
        lines.append("Inspection mode:  " + str(result["mode"]))
    regions = result.get("regions") or []
    if regions:
        critical = [r for r in regions if r.get("priority") == "CRITICAL"]
        major = [r for r in regions if r.get("priority") == "MAJOR"]
        minor = [r for r in regions if r.get("priority") == "MINOR"]

        if critical or major:
            lines.append("-" * 52)
            lines.append("MAJOR ISSUES FOUND (priority and shape hint are heuristics; 'type' is the trained region")
            lines.append("classifier of Part 1.4 -- treat both as 'needs a human/AOI check', not a confirmed diagnosis):")
            for r in critical + major:
                i = regions.index(r) + 1                                # same number as in the figure
                cx, cy = r["center"]
                tag = "[CRITICAL] " if r["priority"] == "CRITICAL" else "[MAJOR] "
                lines.append("  " + tag + "#" + str(i) + ": center=(" + str(round(cx)) + "," + str(round(cy)) + " px)"
                              + "  area=" + str(round(r["area_frac"] * 100, 2)) + "%"
                              + "  score=" + str(round(r["mean_score"], 3))
                              + "  -> " + r["shape_hint"])
                if r.get("defect_type"):
                    lines.append("      type: " + region_type_text(r, alternatives=True))
        if minor:
            lines.append("-" * 52)
            lines.append("  (+ " + str(len(minor)) + " minor issue(s) not detailed here -- e.g. possible "
                          "bend/warp/lighting artifact -- see result['regions'] for full detail)")
            typed_minor = ["#%d %s" % (regions.index(r) + 1, region_type_text(r)) for r in minor if r.get("defect_type")]
            if typed_minor:
                lines.append("    minor region types: " + ", ".join(typed_minor))
        if not critical and not major and not minor:
            lines.append("-" * 52)
            lines.append("No specific region localized (image-level anomaly only).")
    orientation = result.get("orientation")
    if orientation:
        lines.append("-" * 52)
        lines.append("Orientation check:")
        lines.append("  Estimated rotation: " + str(orientation["estimated_angle_deg"]) + " deg")
        lines.append("  Orientation flag:   " + str(orientation["orientation_flag"]))
    lines.append("=" * 52)
    return "\n".join(lines)


def inspect_pcb(image_path=None, image_array=None, pcb_class=None,
                 check_orientation=False, save_dir=None, verbose=True, visualize=True, rotation_tta=True,
                 quick_screen=True, classify_types=True, type_image_bgr=None, type_box_map=None):
    """Practical, single-call PCB inspection.

    Runs a cheap single-pass screen first. If the score is CONFIDENTLY below this
    class's threshold (with a margin derived from that class's own calibration-score
    spread, not a hardcoded constant), returns immediately -- no TTA, no region search,
    no orientation check. Only escalates to the full rotation-TTA + region-localization
    pass (test_single_image) when the quick screen is anomalous or too close to the
    threshold to be confident. This is the same fast-screen-then-closer-look pattern a
    real inspection line uses, and since most boards on any real line ARE normal, most
    images never pay the expensive path.

    Args:
        image_path / image_array : exactly one of these, same as test_single_image.
        pcb_class                : force a class, or None to auto-detect during the quick screen.
        check_orientation         : passed through to the thorough pass only (the quick
                                    screen never runs it -- see CFG.CANONICAL_ANGLE_* cost notes).
        save_dir                  : if given, saves the annotated image (PNG) and a plain-text
                                    report to this directory -- a basic audit trail.
        verbose                   : print the report.
        visualize                 : show (and, if save_dir given, save) the annotated result.
        classify_types            : name the defect type of every highlighted region (Part 1.4 classifier).
        type_image_bgr/type_box_map : original photo + box mapping when an ALIGNED copy is inspected (see test_single_image).

    Returns:
        dict: everything test_single_image returns, plus 'severity', 'mode', and 'report_text'.
    """
    assert (image_path is not None) != (image_array is not None), \
        "Provide exactly one of image_path or image_array."

    if image_path is not None:
        img_bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(image_path)
        img_rgb_full = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    else:
        img_rgb_full = image_array

    tensor = load_image_from_array(img_rgb_full, **prep_for(pcb_class))
    candidate_classes = [pcb_class] if pcb_class is not None else CFG.CLASSES

    quick_results = {}
    for cls in candidate_classes:
        det_c = detectors[cls]
        amap_c, score_c = det_c.score_image_tensor(tensor)
        thr_c = det_c.threshold
        margin_c = _class_match_margin(det_c, score_c, thr_c, use_tta=False)
        quick_results[cls] = (amap_c, score_c, thr_c, margin_c)

    chosen_class = min(quick_results, key=lambda c: quick_results[c][3])
    quick_amap, quick_score, quick_thr, _ = quick_results[chosen_class]
    det = detectors[chosen_class]

    calib_scores = np.array(det.score_stats.get("calib_scores", []))
    buffer = 0.5 * float(calib_scores.std()) if len(calib_scores) > 1 else 0.1
    confidently_normal = quick_screen and quick_thr is not None and quick_score < (quick_thr - buffer)

    if confidently_normal:
        mode = "quick screen (single-pass, no TTA/localization -- confidently normal)"
        result = {
            "pcb_class": chosen_class,
            "anomaly_score": quick_score,
            "threshold": quick_thr,
            "is_anomaly": False,
            "anomaly_map": quick_amap,
            "regions": [],
            "orientation": None,
        }
        used_tta = False
    else:
        mode = ("thorough recheck (" + ("rotation-TTA" if rotation_tta else "single-pass") + " + region localization"
                + (" + orientation check" if check_orientation else "") + ")")
        result = test_single_image(image_array=img_rgb_full, pcb_class=chosen_class,
                                    rotation_tta=rotation_tta, visualize=False, verbose=False,
                                    check_orientation=check_orientation, classify_types=classify_types,
                                    type_image_bgr=type_image_bgr, type_box_map=type_box_map)
        # FIXED: this was always True, so with rotation_tta=False (how Cell 22a runs every single-pass system) severity
        # compared a SINGLE-PASS score with the TTA threshold -- "ANOMALY DETECTED" could come with severity "normal".
        used_tta = bool(rotation_tta)

    result["mode"] = mode
    result["severity"] = _severity_label(det, result["anomaly_score"], use_tta=used_tta)
    report_text = build_inspection_report(result)
    result["report_text"] = report_text

    if verbose:
        print(report_text)

    save_img_path = None
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        base = os.path.splitext(os.path.basename(image_path))[0] if image_path else "image"
        save_img_path = os.path.join(save_dir, base + "_annotated.png")
        report_path = os.path.join(save_dir, base + "_report.txt")
        with open(report_path, "w") as f:
            f.write(report_text)
        if verbose:
            print("\nSaved report to", report_path)

    if visualize and result["anomaly_map"] is not None:
        visualize_result(img_rgb_full, result["anomaly_map"], result["anomaly_score"],
                          result["threshold"], title=chosen_class,
                          regions=result["regions"] if result["regions"] else None,
                          save_path=save_img_path, letterbox=prep_for(pcb_class)["letterbox"])

    return result


# ---- quick demo ----
demo_cls = "pcb1"
demo_anom = [p for p, l in zip(FILE_LISTS[demo_cls]["eval_images"], FILE_LISTS[demo_cls]["eval_labels"]) if l == 1]
demo_norm2 = [p for p, l in zip(FILE_LISTS[demo_cls]["eval_images"], FILE_LISTS[demo_cls]["eval_labels"]) if l == 0]
if demo_norm2:
    print("--- inspect_pcb on a known-pcb1 normal image (expect the fast quick-screen path) ---")
    _ = inspect_pcb(image_path=demo_norm2[0], pcb_class=demo_cls, visualize=False)
if demo_anom:
    print()
    print("--- inspect_pcb on a known-pcb1 anomalous image (expect escalation to the thorough path) ---")
    _ = inspect_pcb(image_path=demo_anom[0], pcb_class=demo_cls, visualize=False)
if demo_norm2:
    print()
    print("--- same normal image again, pcb_class=None (auto-detect should now correctly land on pcb1) ---")
    auto_result = inspect_pcb(image_path=demo_norm2[0], pcb_class=None, visualize=False, verbose=False)
    print("Auto-detected class:", auto_result["pcb_class"], "| expected: pcb1")

# Example: save results to disk for a basic audit trail
# result = inspect_pcb(image_path="/kaggle/working/my_pcb_photo.jpg",
#                       save_dir="/kaggle/working/inspection_reports")

## Part 1.5 -- Hyperparameter sensitivity (KNN_K, TOPK_SCORE_FRAC)
Cheap: no refitting. Each branch is queried once per calibration image at the largest K; every other K is a
slice of the sorted distances and every TOPK_SCORE_FRAC a re-aggregation of the same map. These are
**calibration-split** numbers -- a sensitivity analysis, not a held-out result.

In [ ]:
# ============================================================
# CELL 15b -- Hyperparameter sweep: KNN_K and TOPK_SCORE_FRAC (calibration-set signal)
# ============================================================
doc_section("CELL 15b", "Hyperparameter sweep: KNN_K and TOPK_SCORE_FRAC (calibration-set signal)")
def _sweep_branch_dists(knn, patches, max_k):
    dists, idxs = knn.kneighbors(patches, n_neighbors=max_k)
    return dists, idxs[:, 0]   # nearest-neighbor index is the same regardless of k --
                                # it's what density reweighting looks up, and reweighting
                                # only ever considers the single nearest match


def _sweep_fused_map_at_k(det, cnn_dists, cnn_nn_idx, vit_dists, vit_nn_idx, k, h, w):
    cnn_raw = cnn_dists[:, :k].mean(axis=1)
    vit_raw = vit_dists[:, :k].mean(axis=1)
    if CFG.USE_DENSITY_REWEIGHT and det.cnn_bank_density is not None:
        cnn_raw = det._apply_density_reweight(cnn_raw, det.cnn_bank_density[cnn_nn_idx])
    if CFG.USE_DENSITY_REWEIGHT and det.vit_bank_density is not None:
        vit_raw = det._apply_density_reweight(vit_raw, det.vit_bank_density[vit_nn_idx])
    return det._fused_map(cnn_raw.reshape(h, w), vit_raw.reshape(h, w))


sweep_results = {}
max_k = max(CFG.KNN_K_SWEEP)

for cls in CFG.CLASSES:
    det = detectors[cls]
    fl = FILE_LISTS[cls]
    calib_images = fl.get("calib_images", [])
    calib_labels = fl.get("calib_labels", [])
    if not calib_images:
        print("[" + cls + "] no stored calibration split -- skipping sweep for this class.")
        continue

    scores_by_k = {k: [] for k in CFG.KNN_K_SWEEP}
    scores_by_frac = {frac: [] for frac in CFG.TOPK_SCORE_FRAC_SWEEP}

    for path in tqdm(calib_images, desc="[" + cls + "] sweeping hyperparameters", leave=False):
        t = load_image(path)
        cnn_flat, vit_flat = det._extract_branch_grids(t)
        cnn_dists, cnn_nn_idx = _sweep_branch_dists(det.cnn_knn, cnn_flat, max_k)
        vit_dists, vit_nn_idx = _sweep_branch_dists(det.vit_knn, vit_flat, max_k)
        h, w = det.grid_hw

        for k in CFG.KNN_K_SWEEP:
            fused = _sweep_fused_map_at_k(det, cnn_dists, cnn_nn_idx, vit_dists, vit_nn_idx, k, h, w)
            scores_by_k[k].append(det._topk_mean(fused, frac=CFG.TOPK_SCORE_FRAC))

        fused_default_k = _sweep_fused_map_at_k(det, cnn_dists, cnn_nn_idx, vit_dists, vit_nn_idx, CFG.KNN_K, h, w)
        for frac in CFG.TOPK_SCORE_FRAC_SWEEP:
            scores_by_frac[frac].append(det._topk_mean(fused_default_k, frac=frac))

    k_aurocs = {k: roc_auc_score(calib_labels, scores_by_k[k]) for k in CFG.KNN_K_SWEEP}
    frac_aurocs = {frac: roc_auc_score(calib_labels, scores_by_frac[frac]) for frac in CFG.TOPK_SCORE_FRAC_SWEEP}
    sweep_results[cls] = {"k": k_aurocs, "frac": frac_aurocs}

    best_k = max(k_aurocs, key=k_aurocs.get)
    best_frac = max(frac_aurocs, key=frac_aurocs.get)
    print("[" + cls + "] KNN_K sweep (calib AUROC):", {k: round(float(v), 4) for k, v in k_aurocs.items()}, "-> best:", best_k)
    print("[" + cls + "] TOPK_FRAC sweep (calib AUROC):", {round(f, 4): round(float(v), 4) for f, v in frac_aurocs.items()},
          "-> best:", best_frac)

print()
print("NOTE: these AUROCs are on the CALIBRATION split (the same images alpha/threshold")
print("were tuned on), so treat this as a directional signal for which CFG.KNN_K /")
print("CFG.TOPK_SCORE_FRAC value to try next, not a held-out result to cite directly.")
print("Cell 10's held-out eval AUROC is what actually changes if you adopt a new value")
print("and re-run Cells 2, 7-15 (Cell 2 to set the new CFG value, 7-15 to rebuild/recalibrate).")

## Part 1.6 -- Ablations: does the fusion help, and what does invariance cost?

In [ ]:
# ============================================================
# CELL 15c -- ABLATIONS: (a) which branch carries the signal? (b) what does invariance cost?
# ============================================================
doc_section("CELL 15c", "ABLATIONS: (a) which branch carries the signal? (b) what does invariance cost?")
# (a) BRANCH ABLATION (no refitting): every held-out eval image is scored by the CNN branch
#     alone, the ViT branch alone, and the per-class calibrated fusion actually used, then
#     compared with PAIRED bootstrap CIs. This directly tests the core architectural claim
#     "fusing a CNN and a ViT beats either backbone alone".
# (b) INVARIANCE ABLATION: the SAME architecture refit with NO rotation / perspective /
#     lighting / crop augmentation in the memory bank (CLAHE + letterboxing kept). On
#     VisA's aligned, evenly lit test images invariance can only COST sensitivity -- this
#     measures how much. Part 3's robustness benchmark measures what it buys back.
ABLATION_EVAL = {}     # cls -> {"labels": ..., <variant>: per-image scores in eval-split order}
ABLATION_BOOT = {}     # (cls or "MEAN", variant) -> (point_auroc, bootstrap_replicates)
detectors_noaug = {}
ablation_df = None
ABLATION_VARIANTS = ["Fused (ours, single-pass)", "Fused, legacy F1-rule alpha", "Fused, fixed alpha 0.5",
                     "CNN branch only", "ViT branch only", "Ours w/o invariance aug"]

if CFG.RUN_ABLATION:
    for cls in CFG.CLASSES:
        det = detectors[cls]
        fl = FILE_LISTS[cls]
        # alternative fusion weights, scored from the SAME branch maps (no extra forward passes)
        alt_alphas = {"Fused, legacy F1-rule alpha": getattr(det, "alpha_legacy_f1", None),
                      "Fused, fixed alpha 0.5": 0.5}
        alpha_deployed = det.alpha
        cnn_s, vit_s, fused_s = [], [], []
        alt_s = {k: [] for k, a in alt_alphas.items() if a is not None}
        for path in tqdm(fl["eval_images"], desc="[" + cls + "] branch ablation", leave=False):
            cnn_map, vit_map = det._score_grid_branches(load_image(path))
            cnn_s.append(det._topk_mean(cnn_map))
            vit_s.append(det._topk_mean(vit_map))
            fused_s.append(det._topk_mean(det._fused_map(cnn_map, vit_map)))
            try:
                for k in alt_s:
                    det.alpha = alt_alphas[k]
                    alt_s[k].append(det._topk_mean(det._fused_map(cnn_map, vit_map)))
            finally:
                det.alpha = alpha_deployed          # never leave the detector with a different alpha
        ABLATION_EVAL[cls] = {"labels": np.asarray(fl["eval_labels"], dtype=int),
                              "CNN branch only": np.asarray(cnn_s),
                              "ViT branch only": np.asarray(vit_s),
                              "Fused (ours, single-pass)": np.asarray(fused_s)}
        for k, v in alt_s.items():
            ABLATION_EVAL[cls][k] = np.asarray(v)
        # consistency check: the fused score here must equal Cell 10's single-pass score
        if cls in OUR_EVAL_SCORES:
            gap = float(np.abs(ABLATION_EVAL[cls]["Fused (ours, single-pass)"]
                               - OUR_EVAL_SCORES[cls]["Ours (single-pass)"]).max())
            if gap > 1e-4:
                print("[" + cls + "] WARNING: fused ablation score differs from Cell 10 by", gap)

    for cls in CFG.CLASSES:
        fl = FILE_LISTS[cls]
        train_paths = fl["train_normal"][:CFG.MAX_TRAIN_IMAGES_PER_CLASS]
        det_na = PCBAnomalyDetector(cls)
        det_na.fit(train_paths, augs_per_image=0, verbose=False)
        det_na.calibrate_threshold(fl["calib_images"], fl["calib_labels"])
        detectors_noaug[cls] = det_na
        scores = [det_na.score_image_tensor(load_image(p))[1]
                  for p in tqdm(fl["eval_images"], desc="[" + cls + "] no-aug variant", leave=False)]
        ABLATION_EVAL[cls]["Ours w/o invariance aug"] = np.asarray(scores)
        print("[" + cls + "] no-aug variant fitted: bank", det_na.cnn_bank.shape[0], "pts | alpha",
              round(det_na.alpha, 2), "| threshold", round(det_na.threshold, 4))

    ABLATION_VARIANTS = [v for v in ABLATION_VARIANTS if all(v in ABLATION_EVAL[c] for c in CFG.CLASSES)]
    rows = []
    for i, cls in enumerate(CFG.CLASSES):
        pb = PairedBootstrap(ABLATION_EVAL[cls]["labels"], seed=SEED + 10 + i)
        row = {"pcb_class": cls, "alpha deployed (AUROC rule)": round(detectors[cls].alpha, 2),
               "alpha legacy (F1 rule)": getattr(detectors[cls], "alpha_legacy_f1", float("nan"))}
        for v in ABLATION_VARIANTS:
            ABLATION_BOOT[(cls, v)] = pb.auroc(ABLATION_EVAL[cls][v])
            row[v] = fmt_ci(*ABLATION_BOOT[(cls, v)])
        rows.append(row)
    mean_row = {"pcb_class": "MEAN", "alpha deployed (AUROC rule)": "", "alpha legacy (F1 rule)": ""}
    for v in ABLATION_VARIANTS:
        pts = [ABLATION_BOOT[(c, v)][0] for c in CFG.CLASSES]
        reps = np.mean([ABLATION_BOOT[(c, v)][1] for c in CFG.CLASSES], axis=0)
        ABLATION_BOOT[("MEAN", v)] = (float(np.mean(pts)), reps)
        mean_row[v] = fmt_ci(*ABLATION_BOOT[("MEAN", v)])
    rows.append(mean_row)
    ablation_df = pd.DataFrame(rows)

    print("\nABLATION -- held-out image AUROC [95% paired-bootstrap CI], same images for every variant")
    print(ablation_df.to_string(index=False))
    print()
    ref = "Fused (ours, single-pass)"
    for v in ABLATION_VARIANTS[1:]:
        d, lo, hi, verdict = diff_verdict(ABLATION_BOOT[("MEAN", ref)][1], ABLATION_BOOT[("MEAN", v)][1])
        print("Fused - " + v + ": " + str(round(d, 3)) + " [" + str(round(lo, 3)) + ", " + str(round(hi, 3))
              + "]  -> fused vs '" + v + "': " + verdict + " (mean over classes)")
    print("\n(alpha: 1 = CNN only, 0 = ViT only. The deployed alpha is chosen on the CALIBRATION split by the")
    print(" AUROC of the deployed score; the legacy F1 rule is shown for comparison on the same held-out images.)")
    print("How to read this: if 'Fused' is not significantly higher than the better single branch,")
    print("the fusion is not (yet) earning its complexity on this data. If 'w/o invariance aug' is")
    print("significantly higher than 'Fused', invariance costs accuracy on aligned images -- check")
    print("Part 3 (robustness benchmark) for whether it pays that cost back under rotation/lighting shift.")
else:
    print("CFG.RUN_ABLATION is False -- skipping ablations.")

## Part 1.65 -- Model improvement study (pre-registered, selected on calibration only)

In [ ]:
# ============================================================
# CELL 15c2 -- MODEL IMPROVEMENT STUDY: close the same-backbone gap, selected on the CALIBRATION split only
# ============================================================
doc_section("CELL 15c2", "MODEL IMPROVEMENT STUDY: close the same-backbone gap, selected on the CALIBRATION split only")
# Evidence (real run): our CNN-only branch and PatchCore use the SAME backbone and layers (WideResNet-50, layer2+3) and
# the SAME 3x3 neighbourhood aggregation, yet scored 0.789 vs 0.905 held-out -- the gap is in what we do AFTER the
# backbone. Two documented differences from PatchCore, both testable inside the detector:
#   (1) we L2-normalize CNN patch features, discarding their magnitude; PatchCore keeps it;
#   (2) our density re-weighting AMPLIFIES distances to patches matched in dense memory regions -- an unusual rule.
# PRE-REGISTERED RULE: every variant is fit on training images and scored on the CALIBRATION split; the highest mean
# calibration AUROC wins (ties -> the simpler variant listed first). Held-out eval images are scored only after the
# choice is fixed, and every variant's held-out number is reported, so nothing is hidden.
# Deliberately NOT candidates: switching CLAHE off (it would raise raw-input AUROCs only by letting the model read the
# lighting confound measured in Cell 15e), and any change chosen by looking at held-out results.
_D = bool(CFG.USE_DENSITY_REWEIGHT)
FEATURE_VARIANTS = [
    ("v0: current (no invariance aug)", {"cnn_l2": True, "density_reweight": _D}),
    ("v1: keep CNN feature magnitude", {"cnn_l2": False, "density_reweight": _D}),
    ("v2: no density re-weighting", {"cnn_l2": True, "density_reweight": False}),
    ("v3: both", {"cnn_l2": False, "density_reweight": False}),
]
IMPROVE = {}                 # variant -> {"cfg", "dets", "calib_auroc", "eval": {cls: scores}}
improvement_df = None
detectors_v2 = {}
V2_NAME = "Ours v2 (selected on calibration)"

if CFG.RUN_IMPROVEMENT_STUDY:
    for vname, vcfg in FEATURE_VARIANTS:
        reuse = (vcfg == FEATURE_VARIANTS[0][1] and CFG.RUN_ABLATION and len(detectors_noaug) == len(CFG.CLASSES))
        dets = {}
        for cls in CFG.CLASSES:
            if reuse:                                 # identical to Cell 15c's no-augmentation model -- reuse it
                dets[cls] = detectors_noaug[cls]
                continue
            fl = FILE_LISTS[cls]
            d = PCBAnomalyDetector(cls)
            d.feat_cfg = dict(vcfg)
            d.fit(fl["train_normal"][:CFG.MAX_TRAIN_IMAGES_PER_CLASS], augs_per_image=0, verbose=False)
            d.calibrate_threshold(fl["calib_images"], fl["calib_labels"])
            dets[cls] = d
        cal = float(np.mean([dets[c].score_stats["alpha_search_auroc"] for c in CFG.CLASSES]))
        IMPROVE[vname] = {"cfg": dict(vcfg), "dets": dets, "calib_auroc": cal}
        print("%-36s mean CALIBRATION AUROC %.4f%s" % (vname, cal, "  (reused Cell 15c model)" if reuse else ""))

    # ---- selection: calibration only, ties -> earlier (simpler) variant ----
    best_name = max(IMPROVE, key=lambda k: (round(IMPROVE[k]["calib_auroc"], 4), -[n for n, _ in FEATURE_VARIANTS].index(k)))
    OURS_FEATURE_CFG.clear(); OURS_FEATURE_CFG.update(IMPROVE[best_name]["cfg"])   # inherited by Cell 15e's canonical pipeline
    detectors_v2 = IMPROVE[best_name]["dets"]
    print("\nSELECTED (on calibration):", best_name, "->", OURS_FEATURE_CFG)

    # ---- only now: held-out scores for every variant (reported in full) ----
    rows = []
    for vname, v in IMPROVE.items():
        v["eval"] = {c: np.asarray([v["dets"][c].score_image_tensor(load_image(p))[1] for p in FILE_LISTS[c]["eval_images"]])
                     for c in CFG.CLASSES}
        reps, pts = [], []
        for i, c in enumerate(CFG.CLASSES):
            pt, rp = PairedBootstrap(FILE_LISTS[c]["eval_labels"], seed=SEED + 100 + i).auroc(v["eval"][c])
            pts.append(pt); reps.append(rp)
        rows.append({"variant": vname, "selected": "<- selected" if vname == best_name else "",
                     "calibration AUROC (selection)": round(v["calib_auroc"], 4),
                     "held-out AUROC [95% CI]": fmt_ci(float(np.mean(pts)), np.mean(reps, axis=0)),
                     **{c: round(p, 3) for c, p in zip(CFG.CLASSES, pts)}})
    improvement_df = pd.DataFrame(rows)
    print("\nIMPROVEMENT STUDY -- selection on calibration, held-out reported for every variant:")
    print(improvement_df.to_string(index=False))
    print("Cell 18 compares '" + V2_NAME + "' with every baseline on identical held-out images (paired CIs).")
else:
    print("CFG.RUN_IMPROVEMENT_STUDY is False -- skipping.")

## Part 1.7 -- Robustness test sets
Deterministic rotated / relit / cropped copies of held-out images, written once and read by every model.

In [ ]:
# ============================================================
# CELL 15d -- Build the robustness test sets (the experiment the thesis claim depends on)
# ============================================================
doc_section("CELL 15d", "Build the robustness test sets (the experiment the thesis claim depends on)")
# The project's hard requirements are angle-, lighting- and crop-independence. VisA's test
# images are aligned and evenly lit, so the standard benchmark can never show that benefit --
# only its cost. This cell makes deterministic, shifted copies of a class-balanced subset of
# the HELD-OUT eval split (never used for fitting or calibration):
#   identity      re-saved at the same resolution as every other condition (the control)
#   rot90/rot180  exact lossless rotations          rot15  arbitrary angle, expanded canvas
#   dark          dim / under-exposed capture       uneven_light  strong illumination gradient
#   crop          centered partial view (anomalies kept ONLY if the ground-truth mask shows
#                 >= CFG.ROBUSTNESS_CROP_MIN_DEFECT_FRAC of the defect survives the crop)
# Every condition keeps all board content (or is mask-verified), so labels stay truthful.
# Files are written as PNG (lossless): our model and every baseline read IDENTICAL pixels.
# With CFG.ROBUSTNESS_CANONICALIZE, every condition also gets a "<condition>+canon" copy: the
# same shifted image after golden-reference canonicalization (Cell 7c). Every model scores both,
# so the benefit of canonicalization is measured per model, on identical images.
ROBUST_SETS = {}   # cls -> condition -> list of (png_path, label, stem)
_CAMERA_ROTATIONS = {"rot15": 15.0}   # arbitrary-angle rotations are rendered at CAMERA resolution


def simulate_capture(full_bgr, small_bgr, cond):
    """A camera photographing a rotated board produces a SHARP image. Rendering an arbitrary-angle rotation by bilinear
    resampling at 512 px instead added blur real captures do not have: in a real run every model's normal boards scored
    far higher at rot15 than at the lossless rot90 (ours: median z -0.5 -> +4.7..+13.6, all four classes). So rotate at
    full resolution with cubic interpolation, then downscale by the SAME factor as every other condition (INTER_AREA) --
    a camera capture followed by our usual resizing. Lossless / pixel-wise conditions are unchanged.
    Returns (image, validity mask or None): the mask marks the expanded canvas's PADDING as not real, so
    canonicalization fills it from the reference like any unseen region (see canonicalize's input_valid)."""
    if cond in _CAMERA_ROTATIONS:
        f = small_bgr.shape[1] / float(full_bgr.shape[1])
        rot = rotate_bound(full_bgr, _CAMERA_ROTATIONS[cond], interpolation=cv2.INTER_CUBIC)
        real = rotate_bound(np.full(full_bgr.shape[:2], 255, np.uint8), _CAMERA_ROTATIONS[cond],
                            border=cv2.BORDER_CONSTANT, interpolation=cv2.INTER_NEAREST)
        if f < 0.999:
            rot = cv2.resize(rot, None, fx=f, fy=f, interpolation=cv2.INTER_AREA)
            real = cv2.resize(real, (rot.shape[1], rot.shape[0]), interpolation=cv2.INTER_AREA)
        return rot, real > 250                       # pixels even partly mixed with padding count as unseen
    return apply_perturbation(small_bgr, cond), None
ROBUST_CONDS = list(CFG.ROBUSTNESS_CONDITIONS) + (
    [c + "+canon" for c in CFG.ROBUSTNESS_CONDITIONS] if CFG.ROBUSTNESS_CANONICALIZE else [])
CANON_REG = []     # one row per canonicalized image: registration diagnostics
canon_reg_summary_df = native_orientation_df = None
# rotation that registration must recover for each condition (sign convention verified on synthetic boards)
_EXPECTED_ANGLE = {"identity": 0.0, "rot90": -90.0, "rot180": 180.0, "rot15": 15.0,
                   "dark": 0.0, "uneven_light": 0.0, "crop": 0.0}

if CFG.RUN_ROBUSTNESS:
    assert not {"calib", "eval"} & set(CFG.ROBUSTNESS_CONDITIONS), "condition names clash with split prefixes"
    if os.path.isdir(CFG.ROBUSTNESS_DIR):
        shutil.rmtree(CFG.ROBUSTNESS_DIR)          # never mix in stale images from a previous run
    for cls in CFG.CLASSES:
        fl = FILE_LISTS[cls]
        half = CFG.ROBUSTNESS_MAX_IMAGES_PER_CLASS // 2
        idx0 = [i for i, l in enumerate(fl["eval_labels"]) if l == 0][:half]
        idx1 = [i for i, l in enumerate(fl["eval_labels"]) if l == 1][:half]
        subset = [(fl["eval_images"][i], int(fl["eval_labels"][i])) for i in idx0 + idx1]
        ROBUST_SETS[cls] = {cond: [] for cond in ROBUST_CONDS}
        canon_ref = load_canon_reference(cls, CFG.ROBUSTNESS_LONG_SIDE) if CFG.ROBUSTNESS_CANONICALIZE else None
        for cond in ROBUST_CONDS:
            for sub in ("good", "bad"):
                os.makedirs(os.path.join(CFG.ROBUSTNESS_DIR, cls, cond, sub), exist_ok=True)

        crop_excluded = []
        for path, label in tqdm(subset, desc="[" + cls + "] building robustness sets", leave=False):
            full = cv2.imread(path, cv2.IMREAD_COLOR)
            if full is None:
                continue
            img = downscale_long_side(full, CFG.ROBUSTNESS_LONG_SIDE)
            stem = os.path.splitext(os.path.basename(path))[0]
            crop_ok = True
            if "crop" in CFG.ROBUSTNESS_CONDITIONS and label == 1:
                crop_ok, why = defect_survives_crop(path, CFG.ROBUSTNESS_CROP_FRAC,
                                                    CFG.ROBUSTNESS_CROP_MIN_DEFECT_FRAC)
                if not crop_ok:
                    crop_excluded.append((stem, why))
            for cond in CFG.ROBUSTNESS_CONDITIONS:
                if cond == "crop" and not crop_ok:
                    continue
                sub = "bad" if label else "good"
                pert, pert_valid = simulate_capture(full, img, cond)
                out = os.path.join(CFG.ROBUSTNESS_DIR, cls, cond, sub, stem + ".png")
                cv2.imwrite(out, pert)
                ROBUST_SETS[cls][cond].append((out, label, stem))
                if canon_ref is not None:
                    canon, info = canonicalize(pert, canon_ref, input_valid=pert_valid)
                    out_c = os.path.join(CFG.ROBUSTNESS_DIR, cls, cond + "+canon", sub, stem + ".png")
                    cv2.imwrite(out_c, canon)
                    save_seen(out_c, info)
                    ROBUST_SETS[cls][cond + "+canon"].append((out_c, label, stem))
                    err = abs(((info["angle_deg"] - _EXPECTED_ANGLE.get(cond, 0.0)) + 180.0) % 360.0 - 180.0)
                    CANON_REG.append({"pcb_class": cls, "condition": cond, "stem": stem, "label": label,
                                      "registered": info["registered"], "inliers": info["inliers"],
                                      "angle_deg": info["angle_deg"], "scale": info["scale"], "angle_error_deg": err,
                                      "hypothesis": info.get("hypothesis", ""), "alignment_score": info.get("ecc_cc", float("nan")),
                                      "lighting_dev": info.get("lighting_dev", float("nan")),
                                      "photometric": bool(info.get("photometric", False)), "saturation_excluded": bool(info.get("saturation_excluded", False)),
                                      "valid_frac": info.get("valid_frac", float("nan"))})

        counts = {c: str(sum(1 for _, l, _ in v if l == 0)) + "N/" + str(sum(1 for _, l, _ in v if l == 1)) + "A"
                  for c, v in ROBUST_SETS[cls].items()}
        print("[" + cls + "] robustness sets (normal/anomalous):", counts)
        if crop_excluded:
            print("   'crop': excluded", len(crop_excluded), "anomalous image(s) whose defect would be cut off"
                  " or had no ground-truth mask, e.g.", crop_excluded[:2])
    if CANON_REG:
        reg = pd.DataFrame(CANON_REG)
        # CONSISTENCY, not absolute pose: in a real run pcb1 TEST boards genuinely sat 180 deg from every training
        # board, so 'error vs. the reference pose' falsely flagged correct registrations. Each shifted copy must
        # register to ITS OWN board's identity pose plus the known shift.
        ident = reg[reg["condition"] == "identity"].set_index(["pcb_class", "stem", "label"])[["angle_deg", "scale"]]
        j = reg.join(ident, on=["pcb_class", "stem", "label"], rsuffix="_identity")
        expected = j["condition"].map(_EXPECTED_ANGLE).fillna(0.0)
        reg["angle_error_deg"] = (((j["angle_deg"] - j["angle_deg_identity"] - expected) + 180.0) % 360.0 - 180.0).abs()
        reg["misregistered"] = reg["registered"] & ((reg["angle_error_deg"] > 5.0) |
                                                    ((j["scale"] / j["scale_identity"] - 1.0).abs() > 0.15))
        CANON_REG[:] = reg.to_dict("records")
        native_orientation_df = (reg[reg["condition"] == "identity"].assign(t=lambda d: d["angle_deg"].abs() > 90)
                                 .pivot_table(index="pcb_class", columns="label", values="t", aggfunc="mean") * 100).round(1)
        native_orientation_df.columns = ["% " + ("defective" if c == 1 else "normal") + " boards at 180 deg"
                                         for c in native_orientation_df.columns]
        canon_reg_summary_df = reg.groupby("condition", sort=False).agg(
            images=("stem", "size"), registered_pct=("registered", lambda x: round(100.0 * x.mean(), 1)),
            median_inliers=("inliers", "median"), median_angle_error_deg=("angle_error_deg", "median"),
            misregistered=("misregistered", "sum"),
            turned_180_pct=("hypothesis", lambda x: round(100.0 * (x == "keypoints+180").mean(), 1)),
            lighting_corrected_pct=("photometric", lambda x: round(100.0 * x.mean(), 1)),
            clipped_pixels_excluded_pct=("saturation_excluded", lambda x: round(100.0 * x.mean(), 1)),
            median_lighting_dev=("lighting_dev", "median"),
            median_valid_frac=("valid_frac", "median")).reset_index()
        print("\nCANONICALIZATION -- registration to each class's golden reference (a TRAINING image):")
        print(canon_reg_summary_df.to_string(index=False))
        print("Failed registrations are passed through with tone correction only (they stay in the benchmark,")
        print("so a failure costs accuracy instead of being hidden). 'misregistered' = registered but >5 deg or")
        print(">15% scale away from the known transform -- these are the dangerous cases; expect ~0.")
        print("'turned_180_pct' = the 180-degree hypothesis won (near-symmetric boards); 'lighting_corrected_pct'")
        print("should be ~0 for identity/rotation/crop and high for dark/uneven_light (the lighting gate).")
        print("misregistered per class (consistency check):", reg.groupby("pcb_class")["misregistered"].sum().to_dict())
        print("\nNATIVE ORIENTATION of held-out boards vs the class's training reference:")
        print(native_orientation_df.to_string())
        print("A large normal-vs-defective gap is a DATASET CONFOUND -- quantified in Cell 15e.")
else:
    print("CFG.RUN_ROBUSTNESS is False -- skipping the robustness benchmark.")

## Part 1.75 -- Board vs background surface: only the board is inspected

In [ ]:
# ============================================================
# CELL 15d2 -- BOARD vs BACKGROUND SURFACE: only the board is inspected
# ============================================================
doc_section("CELL 15d2", "BOARD vs BACKGROUND SURFACE: only the board is inspected")
# Every aligned image lies in its class's reference frame, so ONE board mask per class -- found automatically on the
# golden reference (texture + colour against the frame border; build_board_mask, Cell 7c) -- says which pixels are board.
# Every aligned system (ours, PatchCore, EfficientAD) then scores ONLY board pixels that were actually in the photo: the
# background surface, and any part of the board outside a cropped photo, can never raise or hide an alarm.
# GROWTH RULE (pre-registered, CALIBRATION data only): thin, low-contrast parts -- e.g. pcb3's pin header on a dark-green
# surface in a real run -- can fall outside the segmented board, and a defect there could never be found. So each class's
# CALIBRATION defect masks are aligned with the same transform as their photos, and the board mask is grown LOCALLY to
# cover them (convex hull of board + defects near it). A first version grew a UNIFORM margin instead; that added surface
# on every side (pcb1: 35% -> 64% of the frame to reach its last 1.6% of defect pixels) and cost accuracy everywhere.
# Held-out defects are only REPORTED. CHECK THE PICTURES: the outline must enclose the whole board.
# If a mask is wrong, set CFG.BOARD_MASK_MODE = "full" in Cell 2 to inspect the whole frame.


def _aligned_defect_mask(path, ref):
    """Ground-truth defect mask of a defective photo, aligned with the photo's own transform; None if unavailable."""
    full = cv2.imread(path, cv2.IMREAD_COLOR)
    mp = find_defect_mask(path)
    gt = cv2.imread(mp, cv2.IMREAD_GRAYSCALE) if mp else None
    if full is None or gt is None or not (gt > 0).any():
        return None
    small = downscale_long_side(full, CFG.ROBUSTNESS_LONG_SIDE)
    gt = cv2.resize((gt > 0).astype(np.uint8), (small.shape[1], small.shape[0]), interpolation=cv2.INTER_NEAREST)
    _, info = canonicalize(small, ref)
    if not info.get("registered") or info.get("M") is None:
        return None
    h, w = ref["board_src"].shape[:2]
    return cv2.warpAffine(gt, info["M"], (w, h), flags=cv2.INTER_NEAREST) > 0


def _coverage(masks, bm):
    masks = [m for m in masks if m is not None and m.any()]
    return (sum(int((m & bm).sum()) for m in masks) / float(sum(int(m.sum()) for m in masks))) if masks else float("nan")


rows = []
for cls in CFG.CLASSES:
    ref = load_canon_reference(cls, CFG.ROBUSTNESS_LONG_SIDE)
    defective = lambda split: [p for p, l in zip(FILE_LISTS[cls][split + "_images"], FILE_LISTS[cls][split + "_labels"]) if l == 1]
    cal = [_aligned_defect_mask(p, ref) for p in defective("calib")]
    default_mask = ref["board_mask"]
    if CFG.BOARD_MASK_MODE == "auto":
        m, used, note = board_with_calibration_defects(default_mask, cal, CFG.BOARD_DEFECT_MAX_DIST)
        ref["board_mask"], ref["board_note"] = m, "board %.0f%% of the frame; %s" % (100 * m.mean(), note)   # cached: used everywhere
    ev = [_aligned_defect_mask(p, ref) for p in defective("eval")]
    evm = [e for e in ev if e is not None and e.any()]
    rows.append({"pcb_class": cls, "board % of frame, default": round(100 * float(default_mask.mean()), 1),
                 "board % of frame, final": round(100 * float(ref["board_mask"].mean()), 1), "note": ref["board_note"],
                 "calib defect pixels inside, default margin": round(_coverage(cal, default_mask), 3),
                 "calib defect pixels inside, chosen": round(_coverage(cal, ref["board_mask"]), 3),
                 "HELD-OUT defect pixels inside, default (report only)": round(_coverage(evm, default_mask), 3),
                 "HELD-OUT defect pixels inside, final (report only)": round(_coverage(evm, ref["board_mask"]), 3),
                 "held-out defects mostly outside": int(sum((e & ref["board_mask"]).sum() < 0.5 * e.sum() for e in evm))})
board_mask_df = pd.DataFrame(rows)
print(board_mask_df.to_string(index=False))
fig, ax = plt.subplots(1, len(CFG.CLASSES), figsize=(4.2 * len(CFG.CLASSES), 3.4))
for a, cls in zip(np.atleast_1d(ax), CFG.CLASSES):
    ref = load_canon_reference(cls, CFG.ROBUSTNESS_LONG_SIDE)
    m = ref["board_mask"]
    rgb = cv2.cvtColor(ref["board_src"], cv2.COLOR_BGR2RGB).copy()
    rgb[~m] = (rgb[~m] * 0.3).astype(np.uint8)
    cnts, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(rgb, cnts, -1, (0, 255, 0), 2)
    a.imshow(rgb); a.set_title(cls + ": " + ref["board_note"], fontsize=7); a.axis("off")
plt.tight_layout(); plt.show()

## Part 1.8 -- Canonical pipeline: our model trained on aligned boards
Every image this detector sees -- training, calibration, evaluation, robustness -- passes through the same golden-reference canonicalization (train/test consistency).

In [ ]:
# ============================================================
# CELL 15e -- CANONICAL PIPELINE for our model: fit, calibrate AND evaluate on canonicalized images
# ============================================================
doc_section("CELL 15e", "CANONICAL PIPELINE for our model: fit, calibrate AND evaluate on canonicalized images")
# Evidence (real Kaggle run): canonicalizing only at TEST time cut our model's clean AUROC from 0.848 to
# 0.694 (identity vs identity+canon) while PatchCore and EfficientAD were unaffected. Every variant of
# ours -- CNN-only, ViT-only, no-augmentation -- dropped alike, so the cause is a train/test MISMATCH:
# raw training images vs canonicalized test images, to which our CLAHE-normalized, patch-level scoring
# is far more sensitive than the baselines. The standard remedy is to apply the SAME preprocessing at
# training time. Here every image this detector ever sees -- training normals, calibration images,
# held-out eval images and Part 3's "+canon" sets -- passes through the same canonicalize() call against
# the same golden reference. No invariance augmentation: canonicalization replaces it.
detectors_canon = {}
CONFOUND = {}               # (cls, "orientation" | "lighting") -> (AUROC, bootstrap replicates); no defect information
confound_df = None
CANON_EVAL_SCORES = {}
CANON_PATHS = {}            # cls -> {"calib": [...], "eval": [...]} canonicalized copies; Cell 16 hands them to the
                            # baselines so canonicalized-input systems are calibrated on canonicalized images too      # cls -> held-out scores in eval-split order (Cell 18 compares them pairwise)
CANON_PIPE_REG = []         # registration diagnostics for every image canonicalized in this cell
canon_pipeline_df = None


def _canonicalize_split(cls, paths, labels, split, ref):
    """Canonicalize a list of images into CFG.CANON_PIPELINE_DIR/<cls>/<split>/<good|bad>/ (PNG, lossless).
    Returns (output paths -- None where the image could not be read, registered flags)."""
    outs, ok = [], []
    for p, l in tqdm(list(zip(paths, labels)), desc="[" + cls + "] canonicalizing " + split, leave=False):
        img = cv2.imread(p, cv2.IMREAD_COLOR)
        if img is None:
            outs.append(None)
            ok.append(False)
            continue
        canon, info = canonicalize(downscale_long_side(img, CFG.ROBUSTNESS_LONG_SIDE), ref)
        out_dir = os.path.join(CFG.CANON_PIPELINE_DIR, cls, split, "bad" if l else "good")
        os.makedirs(out_dir, exist_ok=True)
        q = os.path.join(out_dir, os.path.splitext(os.path.basename(p))[0] + ".png")
        cv2.imwrite(q, canon)
        save_seen(q, info)
        outs.append(q)
        ok.append(bool(info["registered"]))
        CANON_PIPE_REG.append({"pcb_class": cls, "split": split, "file": os.path.basename(p), "label": int(l),
                               **{k: info.get(k) for k in ["registered", "inliers", "angle_deg", "scale",
                                                           "hypothesis", "lighting_dev", "photometric"]}})
    return outs, ok


if CFG.RUN_CANON_PIPELINE:
    if os.path.isdir(CFG.CANON_PIPELINE_DIR):
        shutil.rmtree(CFG.CANON_PIPELINE_DIR)              # never mix in images from a previous run
    for cls in CFG.CLASSES:
        fl = FILE_LISTS[cls]
        ref = load_canon_reference(cls, CFG.ROBUSTNESS_LONG_SIDE)   # same reference as Part 3's "+canon" sets
        train_paths = fl["train_normal"][:CFG.MAX_TRAIN_IMAGES_PER_CLASS]
        tr, tr_ok = _canonicalize_split(cls, train_paths, [0] * len(train_paths), "train", ref)
        train_c = [q for q, good in zip(tr, tr_ok) if q is not None and good]   # learn ONLY from registered boards
        cal, _ = _canonicalize_split(cls, fl["calib_images"], fl["calib_labels"], "calib", ref)
        ev, _ = _canonicalize_split(cls, fl["eval_images"], fl["eval_labels"], "eval", ref)
        # an unreadable calibration/eval image keeps its original file so every split stays complete
        cal = [q if q is not None else p for q, p in zip(cal, fl["calib_images"])]
        ev = [q if q is not None else p for q, p in zip(ev, fl["eval_images"])]
        CANON_PATHS[cls] = {"train": train_c, "calib": cal, "eval": ev}
        need = max(5, int(np.ceil(0.5 * len(train_paths))))     # registration must work for most boards
        if len(train_c) < need:
            print("[" + cls + "] only", len(train_c), "of", len(train_paths), "training boards registered (need", need,
                  ") -- skipping the canonical pipeline for this class.")
            continue

        det = PCBAnomalyDetector(cls)
        det.feat_cfg["board_only"] = bool(CFG.BOARD_ONLY)      # inspect only the board (Cell 15d2)
        det.fit(train_c, augs_per_image=0, verbose=False)
        det.calibrate_threshold(cal, fl["calib_labels"])
        detectors_canon[cls] = det
        CANON_EVAL_SCORES[cls] = np.asarray(
            [det.score_path(q)[1] for q in tqdm(ev, desc="[" + cls + "] scoring eval", leave=False)],
            dtype=np.float64)
        auc = roc_auc_score(fl["eval_labels"], CANON_EVAL_SCORES[cls])
        print("[" + cls + "] canonical pipeline: trained on " + str(len(train_c)) + "/" + str(len(train_paths))
              + " registered boards | alpha " + str(round(det.alpha, 2)) + " | held-out AUROC " + str(round(auc, 4))
              + " (raw-input model: " + str(round(results[cls]["single_pass"]["auroc"], 4)) + ")")

    if CANON_PIPE_REG:
        reg = pd.DataFrame(CANON_PIPE_REG)
        canon_pipeline_df = reg.groupby(["pcb_class", "split"]).agg(
            images=("file", "size"),
            registered_pct=("registered", lambda x: round(100.0 * x.mean(), 1)),
            turned_180_pct=("hypothesis", lambda x: round(100.0 * (x == "keypoints+180").mean(), 1)),
            lighting_corrected_pct=("photometric", lambda x: round(100.0 * x.mean(), 1)),
            median_abs_angle_deg=("angle_deg", lambda x: round(float(np.nanmedian(np.abs(x))), 2))).reset_index()
        print("\nCANONICAL PIPELINE -- registration of every image the detector saw:")
        print(canon_pipeline_df.to_string(index=False))
        # CAPTURE-CONDITION CONFOUND CHECK: can a score with NO defect information -- the board's orientation, or
        # its lighting deviation from the reference -- separate defective from normal held-out boards? In a real run:
        # orientation alone reached AUROC 0.77 on pcb1 (95% of defective vs 32% of normal pcb1 test boards sat 180
        # deg from every training board); lighting alone 0.96 on pcb2, ~0.78 on pcb1/pcb3 and 0.22 (inverted) on
        # pcb4. Where a cue is strong, raw-input AUROCs partly measure HOW the images were taken; detectors that
        # normalize lighting/orientation (ours via CLAHE, any canonicalized system) cannot use that cue.
        rows = []
        for i, cls in enumerate(CFG.CLASSES):
            ev = reg[(reg["pcb_class"] == cls) & (reg["split"] == "eval")]
            if ev.empty or ev["label"].nunique() < 2:
                continue
            pb = PairedBootstrap(ev["label"].values, seed=SEED + 4000 + i)
            row = {"pcb_class": cls}
            for kind, score in [("orientation", ev["angle_deg"].abs().values),
                                ("lighting", ev["lighting_dev"].astype(float).values)]:
                CONFOUND[(cls, kind)] = pb.auroc(score)
                lo, hi = ci95(CONFOUND[(cls, kind)][1])
                row[kind + "-only AUROC"] = fmt_ci(*CONFOUND[(cls, kind)])
                row[kind + " confound?"] = "YES" if (lo > 0.5 or hi < 0.5) else "no"
            rows.append(row)
        confound_df = pd.DataFrame(rows)
        print("\nCAPTURE-CONDITION CONFOUND CHECK -- held-out AUROC of scores with NO defect information:")
        print(confound_df.to_string(index=False))
        print("'YES' = the cue alone separates defective from normal boards (95% CI excludes 0.5, either direction).")
else:
    print("CFG.RUN_CANON_PIPELINE is False -- skipping the canonical pipeline.")

## Part 1.85 -- Canonical-pipeline study (final, pre-registered): letterboxing x CLAHE

In [ ]:
# ============================================================
# CELL 15e2 -- CANONICAL-PIPELINE STUDY (final, PRE-REGISTERED): letterboxing x CLAHE, selected on calibration only
# ============================================================
doc_section("CELL 15e2", "CANONICAL-PIPELINE STUDY (final, PRE-REGISTERED): letterboxing x CLAHE, selected on calibration only")
# Evidence (real run, same pipeline for every detector): our canonical pipeline trailed PatchCore's in EVERY shift
# condition by 0.01-0.04 AUROC -- a systematic gap, largest for dark (-0.031) and uneven light (-0.030). Two documented
# preprocessing differences fit that pattern:
#   letterbox -- we pad each board to a square with MIRRORED content (~28% of the input); PatchCore uses the full frame,
#                so more of its patch grid lands on the board and it sees no artificial mirror seams.
#   CLAHE     -- added for lighting invariance, but in the canonical pipeline alignment already corrects lighting; CLAHE
#                may now be redundant and amplify noise (the dark / uneven-light gap).
# PRE-REGISTERED RULE (fixed before any result is seen): the four variants below are fit on canonicalized training boards,
# the highest mean CALIBRATION AUROC wins (ties -> the earlier, simpler-to-justify variant), held-out boards are scored
# only afterwards, and every variant's held-out result is reported.
# PRE-REGISTERED GUARD: switching CLAHE off could let a detector read residual lighting differences -- the capture
# confound of Cell 15e. If canonicalized CALIBRATION boards still carry a significant lighting cue in any class, the
# no-CLAHE variants are ineligible. (Calibration labels only; held-out labels are never consulted for this decision.)
# This is the LAST tuning round on this dataset: further changes should be tested on a different PCB dataset.
CANON_VARIANTS = [
    ("c0: letterbox + CLAHE (current)", {"letterbox": True, "normalize_light": True}),
    ("c1: full frame + CLAHE", {"letterbox": False, "normalize_light": True}),
    ("c2: letterbox, no CLAHE", {"letterbox": True, "normalize_light": False}),
    ("c3: full frame, no CLAHE", {"letterbox": False, "normalize_light": False}),
]
CANON_STUDY = {}
canon_study_df = residual_light_df = None
canon_study_pick = CANON_VARIANTS[0][0]


def residual_lighting_dev(img_bgr, ref):
    """Low-frequency brightness deviation of an (already canonicalized) board from its golden reference."""
    h, w = ref["img"].shape[:2]
    if img_bgr.shape[:2] != (h, w):
        img_bgr = cv2.resize(img_bgr, (w, h), interpolation=cv2.INTER_AREA)
    sigma = CFG.CANON_ILLUM_SIGMA_FRAC * max(h, w)
    m = np.ones((h, w), np.float32)
    ls = _lowpass_masked(_gray(img_bgr).astype(np.float32), m, sigma)
    lr = _lowpass_masked(_gray(ref["img"]).astype(np.float32), m, sigma)
    return float(np.median(np.abs(np.log(np.maximum(lr, 1.0) / np.maximum(ls, 1.0)))))


if CFG.RUN_CANON_STUDY and detectors_canon and CANON_PATHS:
    # ---- the guard (calibration split decides; the eval split is shown for the thesis, never used) ----
    rows, cue_left = [], False
    for i, cls in enumerate([c for c in CFG.CLASSES if c in CANON_PATHS]):
        ref = load_canon_reference(cls, CFG.ROBUSTNESS_LONG_SIDE)
        row = {"pcb_class": cls}
        for split, seed_off in (("calib", 5000), ("eval", 6000)):
            devs = np.asarray([residual_lighting_dev(cv2.imread(p, cv2.IMREAD_COLOR), ref) for p in CANON_PATHS[cls][split]])
            pt, reps = PairedBootstrap(FILE_LISTS[cls][split + "_labels"], seed=SEED + seed_off + i).auroc(devs)
            lo, hi = ci95(reps)
            row["lighting-only AUROC after canonicalization (" + split + ")"] = fmt_ci(pt, reps)
            if split == "calib" and (lo > 0.5 or hi < 0.5):
                cue_left = True
                row["cue left (calib)?"] = "YES"
        row.setdefault("cue left (calib)?", "no")
        rows.append(row)
    residual_light_df = pd.DataFrame(rows)
    print("RESIDUAL LIGHTING CUE after canonicalization (raw-image cue was up to 0.96 on pcb2):")
    print(residual_light_df.to_string(index=False))
    eligible = [(n, p) for n, p in CANON_VARIANTS if p["normalize_light"] or not cue_left]
    if cue_left:
        print("-> a cue is left in the calibration boards: no-CLAHE variants are INELIGIBLE (pre-registered guard).")

    # ---- fit + calibrate every eligible variant on the SAME canonicalized boards ----
    for name, prep in eligible:
        if prep == CANON_VARIANTS[0][1]:
            dets = dict(detectors_canon)                       # c0 is the existing canonical pipeline -- reuse it
        else:
            dets = {}
            for cls in [c for c in CFG.CLASSES if c in detectors_canon]:
                d = PCBAnomalyDetector(cls)
                d.feat_cfg.update(prep)
                d.feat_cfg["board_only"] = bool(CFG.BOARD_ONLY)
                d.fit(CANON_PATHS[cls]["train"], augs_per_image=0, verbose=False)
                d.calibrate_threshold(CANON_PATHS[cls]["calib"], FILE_LISTS[cls]["calib_labels"])
                dets[cls] = d
        CANON_STUDY[name] = {"prep": prep, "dets": dets,
                             "calib_auroc": float(np.mean([d.score_stats["alpha_search_auroc"] for d in dets.values()]))}
        print("%-34s mean CALIBRATION AUROC %.4f" % (name, CANON_STUDY[name]["calib_auroc"]))

    order = [n for n, _ in CANON_VARIANTS]
    canon_study_pick = max(CANON_STUDY, key=lambda k: (round(CANON_STUDY[k]["calib_auroc"], 4), -order.index(k)))
    print("\nSELECTED (on calibration):", canon_study_pick)

    # ---- only now: held-out scores for every variant, reported in full ----
    rows = []
    for name, v in CANON_STUDY.items():
        v["eval"] = {c: np.asarray([d.score_path(p)[1] for p in CANON_PATHS[c]["eval"]])
                     for c, d in v["dets"].items()}
        pts, reps = [], []
        for i, c in enumerate(v["eval"]):
            pt, rp = PairedBootstrap(FILE_LISTS[c]["eval_labels"], seed=SEED + 100 + CFG.CLASSES.index(c)).auroc(v["eval"][c])
            pts.append(pt); reps.append(rp)
        rows.append({"variant": name, "selected": "<- selected" if name == canon_study_pick else "",
                     "calibration AUROC (selection)": round(v["calib_auroc"], 4),
                     "held-out AUROC [95% CI]": fmt_ci(float(np.mean(pts)), np.mean(reps, axis=0)),
                     **{c: round(p, 3) for c, p in zip(v["eval"], pts)}})
    canon_study_df = pd.DataFrame(rows)
    print("\nCANONICAL-PIPELINE STUDY -- selection on calibration, held-out reported for every variant:")
    print(canon_study_df.to_string(index=False))

    # ---- the winner BECOMES our canonical pipeline for everything downstream (benchmark, robustness, tool) ----
    if canon_study_pick != CANON_VARIANTS[0][0]:
        detectors_canon.clear()
        detectors_canon.update(CANON_STUDY[canon_study_pick]["dets"])
        CANON_EVAL_SCORES.update(CANON_STUDY[canon_study_pick]["eval"])
        print("Our canonical pipeline now uses:", CANON_STUDY[canon_study_pick]["prep"])
else:
    print("Canonical-pipeline study skipped (CFG.RUN_CANON_STUDY off, or no canonical pipeline in this run).")

## Part 1.9 -- Pipeline-consistent augmentation study (pre-registered, selected on calibration shift sets)

In [ ]:
# ============================================================
# CELL 15e3 -- PIPELINE-CONSISTENT AUGMENTATION STUDY (pre-registered, selected on CALIBRATION shift sets)
# ============================================================
doc_section("CELL 15e3", "PIPELINE-CONSISTENT AUGMENTATION STUDY (pre-registered, selected on CALIBRATION shift sets)")
# Diagnosis (real run, per condition, every detector in the canonical pipeline): ours beat PatchCore and EfficientAD on
# clean boards, all rotations and darkness, but LOST on crop (-0.034 / -0.061) and uneven light (-0.031 / -0.014). In
# exactly those two, NORMAL boards started to look anomalous to our model (median score up by as much as 1.9 / 3.1 SD of
# its clean normal scores; EfficientAD's barely moved). Both shifts make alignment REPAIR the image: a crop leaves 37% of
# the frame to be filled from the reference behind a long blended seam; uneven light (0.35x..1.25x) crushes and saturates
# parts of the board before the lighting correction. Our memory bank was built from boards that needed almost no repair,
# so repair artifacts are novel to it.
# CANDIDATE FIX: add ONE repaired copy of every training board -- alternately a random lighting field (random direction,
# strength and gamma; NOT the test's fixed left-to-right ramp) or a random partial view (random size and position; NOT the
# test's centred 80% crop) -- canonicalized exactly like a test board.
# PRE-REGISTERED RULE: both variants are scored on clean + six shifted versions of CALIBRATION boards (never held-out ones);
# the higher mean calibration AUROC over those 7 conditions wins (tie -> no augmentation). Held-out boards are scored only
# afterwards and both variants' results are reported.
# FAIRNESS: if augmentation wins, the augmented set becomes the training set of EVERY canonical-pipeline detector -- ours,
# PatchCore and EfficientAD (which keeps the same number of training steps) -- a pipeline improvement for all.
# DISCLOSURE: the weakness was FOUND on held-out results; the CHOICE is made on calibration data only.
AUG_STUDY = {}
aug_study_df = None
aug_study_pick = "a0: no augmentation (current)"
CANON_TRAIN_BASE_N = {}        # cls -> training-set size before augmentation (Cell 17 keeps EfficientAD's step budget)
CALIB_SHIFT_SETS = {}          # cls -> cond -> [(canonical png, label)]
AUG_TRAIN = {}                 # cls -> augmented canonical training images


def _random_lighting(img, rng):
    h, w = img.shape[:2]
    ang = rng.uniform(0, 2 * np.pi)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    t = np.cos(ang) * (xx / w - 0.5) + np.sin(ang) * (yy / h - 0.5)
    t = (t - t.min()) / (t.max() - t.min() + 1e-6)
    lo, hi = rng.uniform(0.45, 0.85), rng.uniform(1.0, 1.3)
    x = np.power(img.astype(np.float32) / 255.0, rng.uniform(0.8, 1.25))
    return np.clip(255.0 * x * (lo + (hi - lo) * t)[..., None], 0, 255).astype(np.uint8)


def _random_partial_view(img, rng):
    h, w = img.shape[:2]
    ch, cw = int(h * rng.uniform(0.7, 0.95)), int(w * rng.uniform(0.7, 0.95))
    y0, x0 = int(rng.integers(0, h - ch + 1)), int(rng.integers(0, w - cw + 1))
    return img[y0:y0 + ch, x0:x0 + cw]


def _cal_auroc(dets, cls):
    """Mean calibration AUROC over clean + every shifted calibration set (each needs both labels)."""
    d = dets[cls]
    sets = [(CANON_PATHS[cls]["calib"], FILE_LISTS[cls]["calib_labels"])]
    sets += [([p for p, _ in items], [l for _, l in items]) for items in CALIB_SHIFT_SETS[cls].values()]
    vals = []
    for paths, labels in sets:
        if len(set(labels)) == 2:
            s = np.asarray([d.score_path(p)[1] for p in paths])
            vals.append(roc_auc_score(labels, s))
    return vals


if CFG.RUN_AUG_STUDY and detectors_canon and CANON_PATHS:
    shift_conds = [c for c in CFG.ROBUSTNESS_CONDITIONS if c != "identity"]
    for i, cls in enumerate([c for c in CFG.CLASSES if c in detectors_canon]):
        ref = load_canon_reference(cls, CFG.ROBUSTNESS_LONG_SIDE)
        rng = np.random.default_rng(SEED + 8000 + i)
        # ---- one repaired copy per training board ----
        out_dir = os.path.join(CFG.CANON_PIPELINE_DIR, cls, "train_aug")
        os.makedirs(out_dir, exist_ok=True)
        AUG_TRAIN[cls] = []
        for j, p in enumerate(FILE_LISTS[cls]["train_normal"][:CFG.MAX_TRAIN_IMAGES_PER_CLASS]):
            full = cv2.imread(p, cv2.IMREAD_COLOR)
            if full is None:
                continue
            img = downscale_long_side(full, CFG.ROBUSTNESS_LONG_SIDE)
            view = _random_lighting(img, rng) if j % 2 == 0 else _random_partial_view(img, rng)
            canon, info = canonicalize(view, ref)
            if info["registered"]:
                q = os.path.join(out_dir, os.path.splitext(os.path.basename(p))[0] + "_aug.png")
                cv2.imwrite(q, canon)
                save_seen(q, info)
                AUG_TRAIN[cls].append(q)
        # ---- six shifted versions of CALIBRATION boards (selection data; held-out boards are never touched) ----
        cal = list(zip(FILE_LISTS[cls]["calib_images"], FILE_LISTS[cls]["calib_labels"]))
        pick = []
        for lab in (0, 1):
            idx = [k for k, (_, l) in enumerate(cal) if l == lab]
            rng.shuffle(idx)
            pick += [cal[k] for k in idx[:CFG.ROBUSTNESS_MAX_IMAGES_PER_CLASS // 2]]
        CALIB_SHIFT_SETS[cls] = {c: [] for c in shift_conds}
        for path, label in pick:
            full = cv2.imread(path, cv2.IMREAD_COLOR)
            if full is None:
                continue
            img = downscale_long_side(full, CFG.ROBUSTNESS_LONG_SIDE)
            stem = os.path.splitext(os.path.basename(path))[0]
            crop_ok = True
            if "crop" in shift_conds and label == 1:
                crop_ok, _ = defect_survives_crop(path, CFG.ROBUSTNESS_CROP_FRAC, CFG.ROBUSTNESS_CROP_MIN_DEFECT_FRAC)
            for cond in shift_conds:
                if cond == "crop" and not crop_ok:
                    continue
                pert, pert_valid = simulate_capture(full, img, cond)
                canon, info_s = canonicalize(pert, ref, input_valid=pert_valid)
                q = os.path.join(CFG.CANON_PIPELINE_DIR, cls, "calib_shift", cond, stem + ".png")
                os.makedirs(os.path.dirname(q), exist_ok=True)
                cv2.imwrite(q, canon)
                save_seen(q, info_s)
                CALIB_SHIFT_SETS[cls][cond].append((q, int(label)))
        print("[%s] %d augmented training copies | calibration shift sets: %s" % (
            cls, len(AUG_TRAIN[cls]), {c: len(v) for c, v in CALIB_SHIFT_SETS[cls].items()}))

    # ---- the two variants: a0 = the installed canonical pipeline, a1 = + one repaired copy per training board ----
    a1 = {}
    for cls in detectors_canon:
        d = PCBAnomalyDetector(cls)
        d.feat_cfg = dict(detectors_canon[cls].feat_cfg)
        d.fit(CANON_PATHS[cls]["train"] + AUG_TRAIN[cls], augs_per_image=0, verbose=False)
        d.calibrate_threshold(CANON_PATHS[cls]["calib"], FILE_LISTS[cls]["calib_labels"])
        a1[cls] = d
    variants = [("a0: no augmentation (current)", dict(detectors_canon)), ("a1: + one repaired copy per training board", a1)]
    for name, dets in variants:
        per_cls = {c: _cal_auroc(dets, c) for c in dets}
        AUG_STUDY[name] = {"dets": dets, "calib_auroc": float(np.mean([np.mean(v) for v in per_cls.values()]))}
        print("%-44s mean CALIBRATION AUROC over clean + shifts: %.4f" % (name, AUG_STUDY[name]["calib_auroc"]))
    aug_study_pick = max(AUG_STUDY, key=lambda k: (round(AUG_STUDY[k]["calib_auroc"], 4), -[n for n, _ in variants].index(k)))
    print("SELECTED (on calibration):", aug_study_pick)

    # ---- only now: held-out (clean, canonicalized) scores for both variants ----
    rows = []
    for name, v in AUG_STUDY.items():
        v["eval"] = {c: np.asarray([d.score_path(p)[1] for p in CANON_PATHS[c]["eval"]]) for c, d in v["dets"].items()}
        pts, reps = [], []
        for c in v["eval"]:
            pt, rp = PairedBootstrap(FILE_LISTS[c]["eval_labels"], seed=SEED + 100 + CFG.CLASSES.index(c)).auroc(v["eval"][c])
            pts.append(pt); reps.append(rp)
        rows.append({"variant": name, "selected": "<- selected" if name == aug_study_pick else "",
                     "calibration AUROC, clean + 6 shifts (selection)": round(v["calib_auroc"], 4),
                     "held-out clean AUROC [95% CI]": fmt_ci(float(np.mean(pts)), np.mean(reps, axis=0))})
    aug_study_df = pd.DataFrame(rows)
    print("\nAUGMENTATION STUDY -- selection on calibration (clean + shifts), held-out reported for both:")
    print(aug_study_df.to_string(index=False))

    # ---- the winner becomes the pipeline: our detectors AND every canonical-pipeline baseline's training set ----
    if aug_study_pick != variants[0][0]:
        detectors_canon.update(a1)
        CANON_EVAL_SCORES.update(AUG_STUDY[aug_study_pick]["eval"])
        for cls in a1:
            CANON_TRAIN_BASE_N[cls] = len(CANON_PATHS[cls]["train"])
            CANON_PATHS[cls]["train"] = CANON_PATHS[cls]["train"] + AUG_TRAIN[cls]
        print("Installed: every canonical-pipeline detector now trains on the augmented set (EfficientAD: same step budget).")
    for cls in CALIB_SHIFT_SETS:           # disk: the calibration shift sets were only needed for the choice above
        sdir = os.path.join(CFG.CANON_PIPELINE_DIR, cls, "calib_shift")
        shutil.rmtree(sdir, ignore_errors=True)
        shutil.rmtree(os.path.join(CFG.SEEN_DIR, os.path.abspath(sdir).lstrip(os.sep)), ignore_errors=True)
else:
    print("Augmentation study skipped (CFG.RUN_AUG_STUDY off, or no canonical pipeline in this run).")

## Documentation 4 -- Our final model: settings, thresholds, score distributions

In [ ]:
# ============================================================
# CELL D4 -- OUR FINAL MODEL: settings, memory banks, thresholds and score distributions
# ============================================================
doc_section("CELL D4", "OUR FINAL MODEL: settings, memory banks, thresholds and score distributions")
# The deployed system: our hybrid detector inside the canonical pipeline (aligned boards, board-only inspection), with the
# design choices selected on CALIBRATION data in Cells 15c2 / 15e2 / 15e3. Two operating points per class:
#   empirical threshold -- <= 10% false alarms on calibration normals (higher recall)
#   PAC threshold       -- held-out false alarms <= 10% with 90% confidence (distribution-free guarantee)
final_model_df = None
OURS_SYS = "Ours (canonical pipeline)"
with doc_step("final model settings"):
    rows = []
    for cls, d in detectors_canon.items():
        ss = d.score_stats
        try:
            pac = float(pac_threshold(ss["calib_scores"], ss["calib_labels"]))
        except Exception:
            pac = float("nan")
        rows.append({"pcb_class": cls, "input": "aligned, " + ("letterboxed" if d.prep()["letterbox"] else "full frame"),
                     "CLAHE": bool(d.prep()["normalize_light"]), "board-only inspection": bool(d.feat_cfg.get("board_only")),
                     "CNN features L2-normalized": bool(d.feat_cfg.get("cnn_l2", True)),
                     "aligned training boards": len(CANON_PATHS[cls]["train"]),
                     "CNN bank": str(tuple(getattr(d, "cnn_bank", np.zeros(0)).shape)),
                     "ViT bank": str(tuple(getattr(d, "vit_bank", np.zeros(0)).shape)),
                     "alpha (CNN weight)": round(float(d.alpha), 2), "beta (global-evidence correction)": round(float(getattr(d, "beta_global", 0.0)), 2),
                     "calibration AUROC": round(float(ss.get("alpha_search_auroc", float("nan"))), 4),
                     "empirical threshold": round(float(d.threshold), 4), "PAC threshold": round(pac, 4)})
    final_model_df = pd.DataFrame(rows)
    print(final_model_df.to_string(index=False))
    save_table(final_model_df, "final_model_settings", "Final model per class: settings, memory banks, fusion weight, thresholds")

with doc_step("score distributions and thresholds"):
    n = len(detectors_canon)
    fig, ax = plt.subplots(2, n, figsize=(5 * n, 8), squeeze=False)
    for j, (cls, d) in enumerate(detectors_canon.items()):
        ss = d.score_stats
        pac = float(pac_threshold(ss["calib_scores"], ss["calib_labels"]))
        for r, (y, s, what) in enumerate([(np.asarray(ss["calib_labels"]), np.asarray(ss["calib_scores"]), "CALIBRATION (sets thresholds)"),
                                          (np.asarray(FILE_LISTS[cls]["eval_labels"]), np.asarray(CANON_EVAL_SCORES[cls]), "HELD-OUT (report only)")]):
            a = ax[r, j]
            bins = np.linspace(min(s.min(), d.threshold), max(s.max(), pac), 30)
            a.hist(s[y == 0], bins=bins, alpha=0.6, color="tab:green", label="normal")
            a.hist(s[y == 1], bins=bins, alpha=0.6, color="tab:red", label="defective")
            a.axvline(d.threshold, ls="--", c="k", label="empirical threshold")
            a.axvline(pac, ls=":", c="purple", lw=2, label="PAC threshold")
            a.set_title("%s: %s" % (cls, what), fontsize=9); a.set_xlabel("image anomaly score"); a.legend(fontsize=7)
    fig.suptitle("Our final model: anomaly-score distributions and the two operating points")
    caption("Image anomaly scores of normal and defective boards with both thresholds. Top: calibration boards (used to set "
            "the thresholds). Bottom: held-out boards (never used for any choice).")
    plt.tight_layout(); plt.show()

## Documentation 5 -- Ablation of the final pipeline

In [ ]:
# ============================================================
# CELL D5 -- ABLATION OF THE FINAL PIPELINE: what each component contributes (documentation only)
# ============================================================
doc_section("CELL D5", "ABLATION OF THE FINAL PIPELINE: what each component contributes (documentation only)")
# Each variant removes or changes ONE component of the deployed pipeline and is scored on the same held-out boards with
# paired bootstrap CIs. Nothing here is used to choose anything -- the design was fixed on calibration data before this cell.
#   board-only inspection off -> whole aligned frame scored (recalibrated)      CLAHE off -> refit and recalibrated
#   CNN only / ViT only        -> fusion weight forced to 1 / 0 (same banks)    letterbox / augmentation -> from Cells 15e2 / 15e3
#   no alignment at all        -> our raw-input model (Cell 9)
import copy, gc
ablation_df = None
ABLATION = {}
if CFG.RUN_FINAL_ABLATION and detectors_canon:
    ABLATION["final pipeline (as deployed)"] = {c: np.asarray(CANON_EVAL_SCORES[c]) for c in detectors_canon}
    with doc_step("ablation: board-only inspection off"):
        s = {}
        for c, d in detectors_canon.items():
            v = copy.deepcopy(d)
            v.feat_cfg = dict(d.feat_cfg); v.feat_cfg["board_only"] = False
            v.calibrate_threshold(CANON_PATHS[c]["calib"], FILE_LISTS[c]["calib_labels"])
            s[c] = np.asarray([v.score_path(p)[1] for p in CANON_PATHS[c]["eval"]])
            del v; gc.collect()
        ABLATION["without board-only inspection (whole frame scored)"] = s
    with doc_step("ablation: CLAHE off"):
        s = {}
        for c, d in detectors_canon.items():
            v = PCBAnomalyDetector(c)
            v.feat_cfg = dict(d.feat_cfg); v.feat_cfg["normalize_light"] = False
            v.fit(CANON_PATHS[c]["train"], augs_per_image=0, verbose=False)
            v.calibrate_threshold(CANON_PATHS[c]["calib"], FILE_LISTS[c]["calib_labels"])
            s[c] = np.asarray([v.score_path(p)[1] for p in CANON_PATHS[c]["eval"]])
            del v; gc.collect()
        ABLATION["without CLAHE"] = s
    with doc_step("ablation: single branches"):
        for name, a in [("CNN branch only (alpha = 1)", 1.0), ("ViT branch only (alpha = 0)", 0.0),
                        ("without global-evidence correction (beta = 0)", "beta0")]:
            s = {}
            for c, d in detectors_canon.items():
                v = copy.copy(d); v.beta_global = 0.0 if a == "beta0" else getattr(d, "beta_global", 0.0); v.alpha = d.alpha if a == "beta0" else a                  # scoring only: same banks, nothing refit
                s[c] = np.asarray([v.score_path(p)[1] for p in CANON_PATHS[c]["eval"]])
            ABLATION[name] = s
    with doc_step("ablation: variants from the design studies"):
        for name, v in CANON_STUDY.items():
            if name != canon_study_pick and "eval" in v:
                ABLATION["design study: " + name] = v["eval"]
        for name, v in AUG_STUDY.items():
            if name != aug_study_pick and "eval" in v:
                ABLATION["design study: " + name] = v["eval"]
        ABLATION["no alignment at all (raw-input model)"] = {c: np.asarray(OUR_EVAL_SCORES[c]["Ours (single-pass)"]) for c in CFG.CLASSES}
    with doc_step("ablation table"):
        def _mean_auc(scores):
            pts, reps = [], []
            for i, c in enumerate(CFG.CLASSES):
                pt, rp = PairedBootstrap(FILE_LISTS[c]["eval_labels"], seed=SEED + 100 + i).auroc(np.asarray(scores[c], float))
                pts.append(pt); reps.append(rp)
            return float(np.mean(pts)), np.mean(reps, axis=0), pts
        base = _mean_auc(ABLATION["final pipeline (as deployed)"])
        rows = []
        for name, sc in ABLATION.items():
            if not all(c in sc for c in CFG.CLASSES):
                continue
            m = _mean_auc(sc)
            dd, lo, hi, verdict = diff_verdict(m[1], base[1])
            rows.append({"variant": name, "held-out AUROC [95% CI]": fmt_ci(m[0], m[1]),
                         **{c: round(p, 3) for c, p in zip(CFG.CLASSES, m[2])},
                         "minus final": "--" if name.startswith("final") else "%+.3f [%.3f, %.3f]" % (dd, lo, hi),
                         "verdict vs final": "--" if name.startswith("final") else verdict, "_auc": m[0], "_lo": ci95(m[1])[0], "_hi": ci95(m[1])[1]})
        ablation_df = pd.DataFrame(rows)
        print(ablation_df.drop(columns=["_auc", "_lo", "_hi"]).to_string(index=False))
        save_table(ablation_df.drop(columns=["_auc", "_lo", "_hi"]), "ablation_final_pipeline",
                   "Ablation of the final pipeline on held-out boards (paired bootstrap vs the deployed pipeline)")
        fig, ax = plt.subplots(figsize=(11, 0.55 * len(ablation_df) + 1.5))
        yy = np.arange(len(ablation_df))[::-1]
        cols = ["tab:red" if v.startswith("final") else "tab:gray" for v in ablation_df["variant"]]
        ax.barh(yy, ablation_df["_auc"], color=cols,
                xerr=[ablation_df["_auc"] - ablation_df["_lo"], ablation_df["_hi"] - ablation_df["_auc"]], capsize=3)
        ax.set_yticks(yy); ax.set_yticklabels(ablation_df["variant"], fontsize=8); ax.set_xlim(0.5, 1.0)
        ax.axvline(base[0], ls="--", c="tab:red"); ax.set_xlabel("held-out AUROC (95% CI)")
        ax.set_title("Ablation: each component of the final pipeline (red = deployed pipeline)")
        caption("Ablation study: held-out AUROC when one component of the final pipeline is removed or changed "
                "(paired bootstrap 95% CIs; documentation only, no choice was based on it).")
        plt.tight_layout(); plt.show()
else:
    print("Final-pipeline ablation skipped (CFG.RUN_FINAL_ABLATION off or no canonical pipeline).")

with doc_step("free memory: detectors of design variants that were NOT selected"):
    freed = 0
    for study, pick in ((IMPROVE, globals().get("best_name")), (CANON_STUDY, canon_study_pick), (AUG_STUDY, aug_study_pick)):
        for name, v in study.items():
            if name != pick and v.get("dets") is not None:
                v["dets"] = None                                 # their held-out scores are kept for the tables
                freed += 1
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    log("freed the detectors of %d non-selected design variants (their scores are kept)" % freed)

## Part 2 -- Baseline comparison (Anomalib: PatchCore, PatchCore at matched bank size, PaDiM, EfficientAD)

- **PatchCore** -- the method family ours extends (frozen backbone + memory bank + k-NN), Anomalib's reference
  implementation at its default settings (15,359-point bank).
- **PatchCore (matched bank)** -- same, with its coreset sized to exactly our `MAX_BANK_SIZE`, so the comparison
  isolates the pipeline (backbones, fusion, augmentation) from memory budget.
- **PaDiM** -- per-location Gaussian model; a different family.
- **EfficientAD** -- student-teacher distillation; a very different family, fast at inference.

Fairness: identical train / calibration / eval roles, thresholds chosen by the identical rule on the calibration
split, raw (unclamped) scores, and paired bootstrap tests on identical images. If a cell here errors, it is most
likely an Anomalib version difference -- every model runs in its own try/except and the traceback pinpoints it.

In [ ]:
# ============================================================
# CELL 16 -- Baseline setup: install Anomalib, build LEAK-FREE fit / predict folders
# ============================================================
doc_section("CELL 16", "Baseline setup: install Anomalib, build LEAK-FREE fit / predict folders")
# PROTOCOL -- identical data roles for our model and every baseline:
#   fit        train/good = the same normal training images our memory bank used
#   calibrate  Anomalib's validation set = OUR calibration split. The FIT folder's test set IS
#              the calibration split and val_split_mode="same_as_test", so anything Anomalib
#              fits on validation data (score normalization, EfficientAD's map-normalization
#              quantiles) sees only calibration images -- exactly what our own alpha/threshold
#              calibration sees.
#   evaluate   a SEPARATE predict-only folder: held-out eval split (+ calibration images, used
#              only to pick each baseline's decision threshold the same way ours is picked,
#              + the robustness copies from Cell 15d). Every file is prefixed with its role
#              ("calib__", "eval__", "<condition>__") so predictions route back unambiguously.
# WHY: the previous version used the eval split itself as Anomalib's validation set (to stop
# Anomalib from silently holding back half the test images). That fixed coverage, but meant
# score normalization / EfficientAD's quantiles were fit on the very images being scored.
ANOMALIB_AVAILABLE = False
ANOMALIB_VERSION = "not installed"
BASELINE_ROOTS = {}   # cls -> ({"fit":..., "pred_main":..., "pred_full":...}, n_train_images)

if CFG.RUN_BASELINES:
    # Anomalib was installed in Cell 1, before numpy/torch were imported (see Cell 1 for why).
    try:
        from anomalib.data import Folder
        from anomalib.engine import Engine
        import anomalib
        ANOMALIB_AVAILABLE = True
        ANOMALIB_VERSION = getattr(anomalib, "__version__", "unknown")
        print("Anomalib", ANOMALIB_VERSION, "imported successfully -- recorded in the Excel 'Protocol' sheet.")
    except Exception as e:
        print("Anomalib import failed (" + str(e) + ") -- baseline comparison will be skipped.")
        print("Install status from Cell 1:", ANOMALIB_INSTALL_MODE)
        print("Our own model's results are entirely unaffected by this.")
else:
    print("CFG.RUN_BASELINES is False -- skipping the Anomalib baseline comparison.")


def _link(src, dest):
    try:
        os.symlink(os.path.abspath(src), dest)
    except OSError:
        shutil.copy(src, dest)


def build_baseline_folders(cls):
    fl = FILE_LISTS[cls]
    base = os.path.join(CFG.ANOMALIB_ROOT, cls)
    if os.path.isdir(base):
        shutil.rmtree(base)       # stale links from an earlier run / different split would contaminate results
    train_paths = fl["train_normal"][:CFG.MAX_TRAIN_IMAGES_PER_CLASS]
    roots = {name: os.path.join(base, name) for name in ("fit", "pred_main", "pred_full")}
    for r in roots.values():
        for sub in ("train/good", "test/good", "test/bad"):
            os.makedirs(os.path.join(r, sub), exist_ok=True)
        for p in train_paths:
            _link(p, os.path.join(r, "train", "good", os.path.basename(p)))
    for p, l in zip(fl["calib_images"], fl["calib_labels"]):
        _link(p, os.path.join(roots["fit"], "test", "bad" if l else "good", os.path.basename(p)))
    for rname in ("pred_main", "pred_full"):
        r = roots[rname]
        for p, l in zip(fl["calib_images"], fl["calib_labels"]):
            _link(p, os.path.join(r, "test", "bad" if l else "good", "calib__" + os.path.basename(p)))
        for p, l in zip(fl["eval_images"], fl["eval_labels"]):
            _link(p, os.path.join(r, "test", "bad" if l else "good", "eval__" + os.path.basename(p)))
    if CFG.RUN_ROBUSTNESS and cls in ROBUST_SETS:
        for cond, items in ROBUST_SETS[cls].items():
            for p, l, stem in items:
                _link(p, os.path.join(roots["pred_full"], "test", "bad" if l else "good",
                                      cond + "__" + os.path.basename(p)))
    if cls in CANON_PATHS:     # CANONICAL-PIPELINE folders: canonicalized train / calibration (validation) / eval / +canon
        for name in ("fit_canon", "pred_canon"):
            roots[name] = os.path.join(base, name)
            for sub in ("train/good", "test/good", "test/bad"):
                os.makedirs(os.path.join(roots[name], sub), exist_ok=True)
            for p in CANON_PATHS[cls]["train"]:
                _link(p, os.path.join(roots[name], "train", "good", os.path.basename(p)))
        for p, l in zip(CANON_PATHS[cls]["calib"], fl["calib_labels"]):
            _link(p, os.path.join(roots["fit_canon"], "test", "bad" if l else "good", os.path.basename(p)))
            _link(p, os.path.join(roots["pred_canon"], "test", "bad" if l else "good", "calib__" + os.path.basename(p)))
        for p, l in zip(CANON_PATHS[cls]["eval"], fl["eval_labels"]):
            _link(p, os.path.join(roots["pred_canon"], "test", "bad" if l else "good", "eval__" + os.path.basename(p)))
        if CFG.RUN_ROBUSTNESS and cls in ROBUST_SETS:
            for cond, items in ROBUST_SETS[cls].items():
                if cond.endswith("+canon"):
                    for p, l, stem in items:
                        _link(p, os.path.join(roots["pred_canon"], "test", "bad" if l else "good", cond + "__" + os.path.basename(p)))
    if cls in CANON_PATHS:     # canonicalized calibration + held-out images (Cell 15e): fair thresholds for the
        for split, paths, labels in [("calibcanon", CANON_PATHS[cls]["calib"], fl["calib_labels"]),   # canonicalized-
                                     ("evalcanon", CANON_PATHS[cls]["eval"], fl["eval_labels"])]:     # input systems
            for p, l in zip(paths, labels):
                _link(p, os.path.join(roots["pred_full"], "test", "bad" if l else "good", split + "__" + os.path.basename(p)))
    return roots, len(train_paths)


if ANOMALIB_AVAILABLE:
    for cls in CFG.CLASSES:
        BASELINE_ROOTS[cls] = build_baseline_folders(cls)
        roots, n_train = BASELINE_ROOTS[cls]
        n = {k: len(os.listdir(os.path.join(roots[k], "test", "good"))) + len(os.listdir(os.path.join(roots[k], "test", "bad")))
             for k in roots}
        print(cls, "| train/good:", n_train, "| fit/test (= calib split):", n["fit"],
              "| predict main:", n["pred_main"], "| predict full (+robustness):", n["pred_full"])

In [ ]:
# ============================================================
# CELL 17 -- Train baselines on the fit folder, predict on the held-out predict folder
# ============================================================
doc_section("CELL 17", "Train baselines on the fit folder, predict on the held-out predict folder")
import gc

baseline_results = {}   # model -> cls -> dict (see route_predictions) or None if it failed
RAW_MODE = CFG.BASELINE_RAW_SCORES


def extract_anomalib_predictions(predictions):
    """(basenames, scores, labels) from engine.predict() output, tolerant of the attribute-
    vs dict-style batch APIs different Anomalib versions use."""
    filenames, scores, labels = [], [], []
    for batch in predictions:
        try:
            paths = batch.image_path if hasattr(batch, "image_path") else batch["image_path"]
            pred_scores = batch.pred_score if hasattr(batch, "pred_score") else batch["pred_score"]
            gt_labels = batch.gt_label if hasattr(batch, "gt_label") else batch["label"]
        except Exception as e:
            print("  Could not parse a prediction batch (" + str(e) + ")")
            continue
        if pred_scores is None or gt_labels is None:
            continue
        for p, s, l in zip(list(paths), list(pred_scores), list(gt_labels)):
            filenames.append(os.path.basename(str(p)))
            scores.append(float(s.item()) if hasattr(s, "item") else float(s))
            labels.append(int(l.item()) if hasattr(l, "item") else int(l))
    return filenames, scores, labels


DEPLOY_MODELS = {}   # (model name, cls) -> {"engine", "model", "map_stats", "raw"} for CFG.DEPLOY_BASELINES


def _np(x):
    if x is None:
        return None
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    return np.asarray(x, dtype=np.float32)


def iter_prediction_maps(predictions):
    """Yields (basename, label, score, anomaly map HxW) for every predicted image that carries a map."""
    for batch in predictions:
        get = (lambda k: batch.get(k)) if isinstance(batch, dict) else (lambda k: getattr(batch, k, None))
        paths, maps, scores = get("image_path"), _np(get("anomaly_map")), get("pred_score")
        labels = get("gt_label") if get("gt_label") is not None else get("label")
        if paths is None or maps is None:
            continue
        for i, p in enumerate(list(paths)):
            m = maps[i]
            while m.ndim > 2:
                m = m[0]
            s = scores[i] if scores is not None else float("nan")
            l = labels[i] if labels is not None else -1
            yield (os.path.basename(str(p)), int(l.item() if hasattr(l, "item") else l),
                   float(s.item() if hasattr(s, "item") else s), m)


def calib_normal_map_stats(predictions):
    """Mean / std of anomaly-map pixels over the CALIBRATION normal images (prefix 'calib__', label 0)."""
    vals = [m.ravel() for f, l, _, m in iter_prediction_maps(predictions) if f.startswith("calib__") and l == 0]
    if not vals:
        return None
    v = np.concatenate(vals)
    return {"mean": float(v.mean()), "std": float(v.std() + 1e-6), "n_images": len(vals)}


def _predict_engine():
    """A fresh, light engine used only to predict with a KEPT model (it holds nothing from training)."""
    return Engine(default_root_dir=os.path.join(CFG.ANOMALIB_ROOT, "_predict_tmp"),
                  accelerator="gpu" if DEVICE.type == "cuda" else "cpu", devices=1, logger=False, enable_progress_bar=False)


def _release_engine(engine, model=None):
    """Lets Python free what a Lightning engine holds after fit/predict: its stored predictions (every image and anomaly
    map -- 0.6-1.3 GB per model) and the model's link back to the trainer (optimizer state). Keeping 24 engines with their
    predictions exhausted the machine's RAM in a real run (the kernel died on the last baseline)."""
    try:
        engine.trainer.predict_loop._predictions = []
    except Exception:
        pass
    if model is not None:
        try:
            model._trainer = None
        except Exception:
            pass


def deploy_predict(model_name, cls, datamodule):
    """Predict with a KEPT baseline model through a fresh engine; nothing is retained afterwards except the result."""
    d = DEPLOY_MODELS[(model_name, cls)]
    eng = _predict_engine()
    try:
        return eng.predict(model=d["model"], datamodule=datamodule)
    finally:
        _release_engine(eng, d["model"])
        del eng
        shutil.rmtree(os.path.join(CFG.ANOMALIB_ROOT, "_predict_tmp"), ignore_errors=True)
        gc.collect()


def masked_map_score(amap, region):
    """A baseline's image score inside the inspected region: the max of its anomaly map there."""
    r = cv2.resize(np.asarray(region, np.uint8), (amap.shape[1], amap.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
    return float(amap[r].max()) if r.any() else float(amap.max())


def inspected_region(cls, seen=None):
    """Board (CFG.BOARD_ONLY) AND seen area (eroded like our patch grid), in the aligned frame; None = whole frame."""
    region = board_mask_for(cls).copy() if CFG.BOARD_ONLY else None
    if seen is not None:
        s = cv2.erode(np.asarray(seen, np.uint8), np.ones((25, 25), np.uint8)) > 0
        region = s if region is None else (region & s if (region & s).sum() > 0.02 * region.size else region)
    return region


def _is_aligned_file(path):
    rp = os.path.realpath(str(path))
    return (rp.startswith(os.path.realpath(CFG.CANON_PIPELINE_DIR)) or "+canon" + os.sep in rp or
            os.sep + "sweep" + os.sep in rp)


def board_masked_scores(predictions, cls):
    """basename -> inspected-region score for every ALIGNED image in these predictions (raw images keep their score)."""
    out = {}
    for batch in predictions:
        get = (lambda k: batch.get(k)) if isinstance(batch, dict) else (lambda k: getattr(batch, k, None))
        paths, maps = get("image_path"), _np(get("anomaly_map"))
        if paths is None or maps is None:
            continue
        for i, p in enumerate(list(paths)):
            if _is_aligned_file(p):
                m = maps[i]
                while m.ndim > 2:
                    m = m[0]
                reg = inspected_region(cls, read_seen(os.path.realpath(str(p))))
                if reg is not None:
                    out[os.path.basename(str(p))] = masked_map_score(m, reg)
    return out


def _is_canon_pipe(name):
    return name.endswith("(canonical pipeline)")


def _base_name(name):
    return name.replace(" (canonical pipeline)", "")


def build_baseline_model(name, n_train, raw):
    from anomalib.models import Patchcore, Padim, EfficientAd
    name = _base_name(name)                    # a canonical-pipeline variant is the same model on aligned boards
    if name == "PatchCore":
        ctor, kw = Patchcore, dict(backbone="wide_resnet50_2", layers=["layer2", "layer3"], coreset_sampling_ratio=0.1)
    elif name == "PatchCore (matched bank)":
        # Anomalib's PatchCore sees n_train UN-augmented images at its own 256 px input, i.e.
        # CFG.ANOMALIB_PATCHES_PER_IMAGE (32x32) patches each -- NOT our augmented 42x42 pool.
        # (The earlier formula assumed our pool and produced a 482-point bank, not 5,000.)
        ratio = min(1.0, CFG.MAX_BANK_SIZE / float(n_train * CFG.ANOMALIB_PATCHES_PER_IMAGE))
        ctor, kw = Patchcore, dict(backbone="wide_resnet50_2", layers=["layer2", "layer3"], coreset_sampling_ratio=ratio)
        print("  matched bank: ratio", round(ratio, 5), "-> expected coreset ~",
              int(round(ratio * n_train * CFG.ANOMALIB_PATCHES_PER_IMAGE)), "points (ours:", CFG.MAX_BANK_SIZE, ")"
              " -- confirm against the 'Selecting Coreset Indices: N/N' line below")
    elif name == "PaDiM":
        ctor, kw = Padim, dict(backbone="wide_resnet50_2", layers=["layer2", "layer3"])
    elif name == "EfficientAD":
        ctor, kw = EfficientAd, {}
    elif name == "STFPM":                       # student-teacher feature pyramid matching
        from anomalib.models import Stfpm
        ctor, kw = Stfpm, dict(backbone="resnet18", layers=["layer1", "layer2", "layer3"])
    elif name == "Reverse Distillation":        # one-class embedding, reverse knowledge distillation
        from anomalib.models import ReverseDistillation
        ctor, kw = ReverseDistillation, dict(backbone="wide_resnet50_2", layers=["layer1", "layer2", "layer3"])
    else:
        raise ValueError("Unknown baseline: " + name)
    optional = {"visualizer": False}            # skip writing a visualization per predicted image
    if raw:
        # RAW scores: Anomalib's default post-processor rescales scores with min/max learned on
        # the validation set and CLAMPS them to [0, 1]. Under a distribution shift (Part 3) many
        # scores exceed the calibration range and collapse into ties at 1.0, which drags AUROC
        # toward 0.5 for reasons that have nothing to do with the model. Raw scores avoid that.
        optional.update({"post_processor": False, "evaluator": False})
    model, dropped = construct_with_fallback(ctor, kw, optional, name)
    return model, (raw and "post_processor" not in dropped)


def folder_datamodule(cls, root, model_name):
    base = dict(name=cls, root=root, normal_dir="train/good", abnormal_dir="test/bad", normal_test_dir="test/good")
    if _base_name(model_name) == "EfficientAD":
        base["train_batch_size"] = 1            # EfficientAD requirement (verified in a real run)
    optional = {"task": "classification", "val_split_mode": "same_as_test", "val_split_ratio": 0.0}
    dm, _ = construct_with_fallback(Folder, base, optional, "Folder")
    return dm


def _seed_baseline(model_name, cls, offset=0):
    """The same seed for the same (model, class) in every run. Unseeded, identical baseline code gave different numbers
    run to run in practice (PatchCore raw 0.897 vs 0.905) while our detector repeated exactly."""
    s = SEED + 1009 * CFG.CLASSES.index(cls) + 17 * CFG.BASELINE_MODEL_NAMES.index(model_name) + 100003 * offset
    try:
        from lightning.pytorch import seed_everything
        seed_everything(s, workers=True)
    except Exception:
        import random as _random
        _random.seed(s)
        np.random.seed(s)
        try:
            torch.manual_seed(s)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(s)
        except Exception:
            pass
    if CFG.DETERMINISTIC_BASELINES:
        try:     # seeds fix the random draws, not the GPU's choice of non-deterministic conv kernels
            torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
        except Exception:
            pass
    return s


def run_baseline(model_name, cls, raw, seed_offset=0, deploy=True):
    _seed_baseline(model_name, cls, seed_offset)
    roots, n_train = BASELINE_ROOTS[cls]
    fit_dm = folder_datamodule(cls, roots["fit_canon" if _is_canon_pipe(model_name) else "fit"], model_name)
    model, raw_effective = build_baseline_model(model_name, n_train, raw)
    epochs = CFG.EFFICIENTAD_EPOCHS if _base_name(model_name) == "EfficientAD" else CFG.BASELINE_EPOCHS.get(_base_name(model_name), 1)
    if _is_canon_pipe(model_name) and CANON_TRAIN_BASE_N.get(cls):    # augmented training set (Cell 15e3):
        epochs = max(1, int(round(epochs * CANON_TRAIN_BASE_N[cls] / len(CANON_PATHS[cls]["train"]))))   # same steps
    run_dir = os.path.join(CFG.ANOMALIB_ROOT, "_runs", "%s_%s_s%d" % ("".join(ch if ch.isalnum() else "_" for ch in model_name), cls, seed_offset))
    engine = Engine(
        default_root_dir=run_dir,                # checkpoints land here and are deleted right after this run (disk:
                                                 # 48 kept checkpoints filled Kaggle's 20 GB in a real run)
        accelerator="gpu" if DEVICE.type == "cuda" else "cpu",
        devices=1,
        max_epochs=epochs,
        # validate once, at the end: EfficientAD's normalization quantiles only matter for the
        # final model, and validating after every epoch cost ~13 s x 19 extra epochs per class.
        check_val_every_n_epoch=epochs,
        logger=False,
        enable_progress_bar=False,
        # NOTE: do NOT pass enable_checkpointing=False -- it conflicts with the ModelCheckpoint
        # callback Anomalib attaches internally (verified error in a real run).
    )
    engine.fit(model=model, datamodule=fit_dm)
    use_full = CFG.RUN_ROBUSTNESS and model_name in CFG.ROBUSTNESS_BASELINES
    pred_root = roots["pred_canon"] if _is_canon_pipe(model_name) else roots["pred_full" if use_full else "pred_main"]
    pred_dm = folder_datamodule(cls, pred_root, model_name)
    predictions = engine.predict(model=model, datamodule=pred_dm)
    fnames, scores, labels = extract_anomalib_predictions(predictions)
    _ms = board_masked_scores(predictions, cls)     # aligned images: only the board, only what the photo showed
    scores = [_ms.get(os.path.basename(str(f)), s) for f, s in zip(fnames, scores)]
    if deploy and model_name in CFG.DEPLOY_BASELINES:
        # keep the trained model for the inspection tool (Cell 22a), with the pixel statistics of its anomaly maps on
        # CALIBRATION normals, so its maps can be put on the z-scale our region / priority rules expect
        DEPLOY_MODELS[(model_name, cls)] = {"engine": None, "model": model,     # the MODEL only -- see _release_engine
                                           "map_stats": calib_normal_map_stats(predictions), "raw": raw_effective}
        _release_engine(engine, model)
        del engine
    else:
        _release_engine(engine)
        del engine, model
    del predictions
    shutil.rmtree(run_dir, ignore_errors=True)                       # the checkpoint: the model itself is in memory
    shutil.rmtree(os.path.join(os.getcwd(), "results"), ignore_errors=True)   # Anomalib's default folder, if it was used
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    if len(scores) == 0:
        raise RuntimeError("no parseable predictions (pred_score/gt_label missing from engine.predict output)")
    return fnames, scores, labels, raw_effective


def route_predictions(cls, fnames, scores, labels):
    """Routes prefixed predictions back to our canonical calib / eval / robustness orders,
    keyed by (label, file stem) -- NaN wherever a prediction is missing."""
    buckets = defaultdict(dict)
    for f, s, l in zip(fnames, scores, labels):
        split, stem = parse_prefixed_name(f)
        if split is not None:
            buckets[split][(int(l), stem)] = float(s)
    fl = FILE_LISTS[cls]
    key = lambda p, l: (int(l), os.path.splitext(os.path.basename(p))[0])
    out = {
        "calib_scores": align_scores(buckets["calib"], [key(p, l) for p, l in zip(fl["calib_images"], fl["calib_labels"])]),
        "calib_labels": np.asarray(fl["calib_labels"], dtype=int),
        "eval_scores": align_scores(buckets["eval"], [key(p, l) for p, l in zip(fl["eval_images"], fl["eval_labels"])]),
        "eval_labels": np.asarray(fl["eval_labels"], dtype=int),
        "robust": {},
    }
    if CFG.RUN_ROBUSTNESS and cls in ROBUST_SETS:
        for cond, items in ROBUST_SETS[cls].items():
            if cond in buckets:
                out["robust"][cond] = align_scores(buckets[cond], [(l, stem) for _, l, stem in items])
    for split, part in [("calibcanon", "calib"), ("evalcanon", "eval")]:
        if split in buckets:
            out[part + "_scores_canon"] = align_scores(buckets[split], [key(p, l) for p, l in zip(fl[part + "_images"], fl[part + "_labels"])])
    return out


if ANOMALIB_AVAILABLE:
    for model_name in CFG.BASELINE_MODEL_NAMES:
        baseline_results[model_name] = {}
        for cls in CFG.CLASSES:
            print("=== " + model_name + " on " + cls + " ===")
            if _is_canon_pipe(model_name) and "fit_canon" not in BASELINE_ROOTS.get(cls, ({}, 0))[0]:
                print("  skipped -- needs the canonical pipeline (Cell 15e, CFG.RUN_CANON_PIPELINE)")
                baseline_results[model_name][cls] = None
                continue
            t0 = time.time()
            res = None
            try:
                res = run_baseline(model_name, cls, RAW_MODE)
            except Exception as e:
                if RAW_MODE:
                    print("  raw-score mode failed (" + str(e)[:250] + ")")
                    print("  -> switching ALL remaining baseline runs to Anomalib's default (normalized) scores")
                    RAW_MODE = False
                    try:
                        res = run_baseline(model_name, cls, False)
                    except Exception as e2:
                        print("  FAILED:", str(e2)[:300])
                else:
                    print("  FAILED:", str(e)[:300])
            if res is None:
                baseline_results[model_name][cls] = None
                continue
            fnames, scores, labels, raw_eff = res
            r = route_predictions(cls, fnames, scores, labels)
            r["score_mode"] = "raw" if raw_eff else "normalized (clamped to [0,1])"
            cov_eval = int((~np.isnan(r["eval_scores"])).sum())
            cov_cal = int((~np.isnan(r["calib_scores"])).sum())
            r["auroc"] = (roc_auc_score(r["eval_labels"], r["eval_scores"])
                          if cov_eval == len(r["eval_scores"]) else float("nan"))
            r["threshold"] = (find_best_threshold_constrained_fpr(r["calib_scores"], r["calib_labels"])[0]
                              if cov_cal == len(r["calib_scores"]) else float("nan"))
            if "calib_scores_canon" in r and not np.isnan(r["calib_scores_canon"]).any():
                # canonicalized-input use is calibrated on CANONICALIZED calibration images -- exactly as our canonical
                # pipeline is. (A real run applied raw-calibrated thresholds to canonicalized boards, which skewed the
                # false-alarm tie-break between statistically tied systems in our favour.)
                r["threshold_canon"] = find_best_threshold_constrained_fpr(r["calib_scores_canon"], r["calib_labels"])[0]
            baseline_results[model_name][cls] = r
            rob = {c: int((~np.isnan(v)).sum()) for c, v in r["robust"].items()}
            print("  AUROC (held-out eval):", round(r["auroc"], 4),
                  "| coverage eval", str(cov_eval) + "/" + str(len(r["eval_scores"])),
                  "calib", str(cov_cal) + "/" + str(len(r["calib_scores"])),
                  "| scores:", r["score_mode"], "| " + str(round(time.time() - t0)) + " s")
            if rob:
                print("  robustness predictions per condition:", rob)
            if cov_eval < len(r["eval_scores"]) or cov_cal < len(r["calib_scores"]):
                print("  WARNING: incomplete coverage -- this model/class is excluded from paired comparisons.")
else:
    print("Skipping baseline training (Anomalib unavailable or CFG.RUN_BASELINES=False).")
    for model_name in CFG.BASELINE_MODEL_NAMES:
        baseline_results[model_name] = {cls: None for cls in CFG.CLASSES}

# ---- SELF-TEST of every kept model NOW (about 4 h in) instead of at its first real use hours later ----
# Kept models predict through a fresh engine (deploy_predict). If that fails for a model, it is removed from
# DEPLOY_MODELS here -- every later consumer (sweeps, pixel metrics, inspection tool, demo) then simply skips it.
DEPLOY_SELFTEST = {}
for m in sorted({mm for mm, _ in DEPLOY_MODELS}):
    c = next((cc for cc in CFG.CLASSES if (m, cc) in DEPLOY_MODELS), None)
    try:
        aligned = _is_canon_pipe(m) and c in CANON_PATHS
        imgs, labs = (CANON_PATHS[c]["eval"], FILE_LISTS[c]["eval_labels"]) if aligned else (FILE_LISTS[c]["eval_images"], FILE_LISTS[c]["eval_labels"])
        pick = [next(p for p, l in zip(imgs, labs) if l == 0), next(p for p, l in zip(imgs, labs) if l == 1)]
        root = os.path.join(CFG.ANOMALIB_ROOT, "_selftest", "".join(ch if ch.isalnum() else "_" for ch in m))
        shutil.rmtree(root, ignore_errors=True)
        for sub, p in (("train/good", pick[0]), ("test/good", pick[0]), ("test/bad", pick[1])):
            os.makedirs(os.path.join(root, sub), exist_ok=True)
            _link(p, os.path.join(root, sub, os.path.basename(p)))
        got = extract_anomalib_predictions(deploy_predict(m, c, folder_datamodule(c, root, m)))[0]
        DEPLOY_SELFTEST[m] = len(got) == 2
        shutil.rmtree(root, ignore_errors=True)
    except Exception as e:
        DEPLOY_SELFTEST[m] = False
        print("  kept-model prediction FAILED for %s: %r" % (m, e))
for m, ok in DEPLOY_SELFTEST.items():
    if not ok:
        for c in CFG.CLASSES:
            DEPLOY_MODELS.pop((m, c), None)
print("Kept-model self-test:", {m: ("OK" if ok else "FAILED -> removed from sweeps / inspection") for m, ok in DEPLOY_SELFTEST.items()})

In [ ]:
# ============================================================
# CELL 18 -- Main comparison on the SAME held-out images for every model:
doc_section("CELL 18", "Main comparison on the SAME held-out images for every model:")
#            AUROC with paired-bootstrap 95% CIs, significance-based verdicts, and
#            thresholded accuracy at thresholds chosen on the calibration split
# ============================================================
MODEL_EVAL = {}     # cls -> {"labels": ..., model: per-image scores in eval-split order}
MODEL_THRESH = {}   # (model, cls) -> decision threshold chosen on the calibration split
for cls in CFG.CLASSES:
    MODEL_EVAL[cls] = {"labels": OUR_EVAL_SCORES[cls]["labels"],
                       "Ours (single-pass)": OUR_EVAL_SCORES[cls]["Ours (single-pass)"],
                       "Ours (TTA)": OUR_EVAL_SCORES[cls]["Ours (TTA)"]}
    MODEL_THRESH[("Ours (single-pass)", cls)] = detectors[cls].threshold
    MODEL_THRESH[("Ours (TTA)", cls)] = detectors[cls].threshold_tta
    if CFG.RUN_ABLATION and cls in detectors_noaug:
        MODEL_EVAL[cls]["Ours w/o invariance aug"] = ABLATION_EVAL[cls]["Ours w/o invariance aug"]
        MODEL_THRESH[("Ours w/o invariance aug", cls)] = detectors_noaug[cls].threshold
    if CFG.RUN_IMPROVEMENT_STUDY and cls in detectors_v2:
        MODEL_EVAL[cls][V2_NAME] = IMPROVE[best_name]["eval"][cls]
        MODEL_THRESH[(V2_NAME, cls)] = detectors_v2[cls].threshold
    if CFG.RUN_CANON_PIPELINE and cls in detectors_canon:
        # same held-out boards, seen through canonicalization (paired comparison remains valid: same samples)
        MODEL_EVAL[cls]["Ours (canonical pipeline)"] = CANON_EVAL_SCORES[cls]
        MODEL_THRESH[("Ours (canonical pipeline)", cls)] = detectors_canon[cls].threshold
    for m in CFG.BASELINE_MODEL_NAMES:
        r = baseline_results.get(m, {}).get(cls)
        if r is not None and not np.isnan(r["eval_scores"]).any():
            MODEL_EVAL[cls][m] = r["eval_scores"]
            MODEL_THRESH[(m, cls)] = r["threshold"]
        if r is not None and "threshold_canon" in r and not np.isnan(r.get("eval_scores_canon", np.array([np.nan]))).any():
            # the DECONFOUNDED benchmark: the same held-out boards, canonicalized (orientation/lighting cues removed)
            MODEL_EVAL[cls][m + " (canonicalized input)"] = r["eval_scores_canon"]
            MODEL_THRESH[(m + " (canonicalized input)", cls)] = r["threshold_canon"]

candidates = ["Ours (single-pass)", "Ours (TTA)", "Ours w/o invariance aug", V2_NAME, "Ours (canonical pipeline)"] + CFG.BASELINE_MODEL_NAMES + [m + " (canonicalized input)" for m in CFG.BASELINE_MODEL_NAMES]
MODEL_ORDER = [m for m in candidates if all(m in MODEL_EVAL[c] for c in CFG.CLASSES)]
skipped = [m for m in candidates if m not in MODEL_ORDER and any(m in MODEL_EVAL[c] for c in CFG.CLASSES)]
if skipped:
    print("Excluded (missing/incomplete for at least one class, so no fair mean):", skipped)

BOOT = {}   # (model, cls or "MEAN") -> (point, replicates)
for i, cls in enumerate(CFG.CLASSES):
    pb = PairedBootstrap(MODEL_EVAL[cls]["labels"], seed=SEED + 100 + i)
    for m in MODEL_ORDER:
        BOOT[(m, cls)] = pb.auroc(MODEL_EVAL[cls][m])
for m in MODEL_ORDER:
    BOOT[(m, "MEAN")] = (float(np.mean([BOOT[(m, c)][0] for c in CFG.CLASSES])),
                         np.mean([BOOT[(m, c)][1] for c in CFG.CLASSES], axis=0))

ROWS = CFG.CLASSES + ["MEAN"]
comparison_df = pd.DataFrame({m: [BOOT[(m, r)][0] for r in ROWS] for m in MODEL_ORDER}, index=ROWS)
comparison_ci_df = pd.DataFrame({m: [fmt_ci(*BOOT[(m, r)]) for r in ROWS] for m in MODEL_ORDER}, index=ROWS)

print("=" * 100)
print("MODEL COMPARISON -- held-out image AUROC [95% paired-bootstrap CI]; identical images for every model")
print("n per class:", {c: len(MODEL_EVAL[c]["labels"]) for c in CFG.CLASSES})
print("=" * 100)
print(comparison_ci_df.to_string())
for m in CFG.BASELINE_MODEL_NAMES:
    modes = {baseline_results[m][c]["score_mode"] for c in CFG.CLASSES if baseline_results.get(m, {}).get(c)}
    if modes:
        print("  " + m + " scores:", ", ".join(sorted(modes)))

# ---- paired differences vs our main model ----
REF = "Ours (single-pass)"
paired_rows = []
for m in MODEL_ORDER:
    if m == REF:
        continue
    for r in ROWS:
        d, lo, hi, verdict = diff_verdict(BOOT[(REF, r)][1], BOOT[(m, r)][1])
        paired_rows.append({"comparison": REF + " minus " + m, "pcb_class": r, "mean_diff": round(d, 4),
                            "ci_low": round(lo, 4), "ci_high": round(hi, 4), "ours_is": verdict})
paired_df = pd.DataFrame(paired_rows)
print("\nPAIRED DIFFERENCES (" + REF + " minus each model); a difference is only called when its 95% CI excludes 0:")
print(paired_df[paired_df["pcb_class"] == "MEAN"].to_string(index=False))

# ---- DECONFOUNDED benchmark: our canonical pipeline vs every model that also sees canonicalized boards ----
REF_C = "Ours (canonical pipeline)"
paired_canon_df = None
if REF_C in MODEL_ORDER:
    rows_c = []
    for m in MODEL_ORDER:
        if m != REF_C and ("(canonicalized input)" in m or "(canonical pipeline)" in m):
            d, lo, hi, verdict = diff_verdict(BOOT[(REF_C, "MEAN")][1], BOOT[(m, "MEAN")][1])
            rows_c.append({"comparison": REF_C + " minus " + m, "mean_diff": round(d, 4), "ci_low": round(lo, 4),
                           "ci_high": round(hi, 4), "ours_is": verdict})
    paired_canon_df = pd.DataFrame(rows_c)
    if len(paired_canon_df):
        print("\nCANONICALIZED BENCHMARK -- full held-out split, every model on CANONICALIZED boards (orientation removed,")
        print("lighting cue reduced but not removed -- see Cell 15e2's residual-cue table), paired 95% CIs:")
        print(paired_canon_df.to_string(index=False))

# ---- thresholded metrics at TWO operating points; both thresholds come from the calibration split ----
#   empirical : max recall s.t. <= MAX_FALSE_POSITIVE_RATE false alarms ON the calibration normals (the rule the
#               detector ships with). With ~50 calibration normals this only promises ~12% held-out false alarms
#               ON AVERAGE -- every model exceeded 20% in a real run.
#   pac       : max recall s.t. held-out false alarms <= MAX_FALSE_POSITIVE_RATE with PAC_CONFIDENCE probability
#               (distribution-free order-statistic bound -- pac_threshold, Cell 7b). Report both: the pair shows the
#               REAL recall-vs-false-alarm trade-off, which the empirical rule alone overstates.
MODEL_CALIB = {}   # (model, cls) -> (calibration scores, labels)
for cls in CFG.CLASSES:
    ss = detectors[cls].score_stats
    MODEL_CALIB[("Ours (single-pass)", cls)] = (ss["calib_scores"], ss["calib_labels"])
    if "calib_scores_tta" in ss:
        MODEL_CALIB[("Ours (TTA)", cls)] = (ss["calib_scores_tta"], ss.get("calib_labels_tta", ss["calib_labels"]))
    if cls in detectors_noaug:
        sn = detectors_noaug[cls].score_stats
        MODEL_CALIB[("Ours w/o invariance aug", cls)] = (sn["calib_scores"], sn["calib_labels"])
    if cls in detectors_v2:
        s2 = detectors_v2[cls].score_stats
        MODEL_CALIB[(V2_NAME, cls)] = (s2["calib_scores"], s2["calib_labels"])
    if cls in detectors_canon:
        sc = detectors_canon[cls].score_stats
        MODEL_CALIB[("Ours (canonical pipeline)", cls)] = (sc["calib_scores"], sc["calib_labels"])
    for m in CFG.BASELINE_MODEL_NAMES:
        r = baseline_results.get(m, {}).get(cls)
        if r is not None and not np.isnan(r["calib_scores"]).any():
            MODEL_CALIB[(m, cls)] = (r["calib_scores"], r["calib_labels"])
        if r is not None and "threshold_canon" in r:
            MODEL_CALIB[(m + " (canonicalized input)", cls)] = (r["calib_scores_canon"], r["calib_labels"])

thr_rows = []
for op in ["empirical", "pac"]:
    for m in MODEL_ORDER:
        for cls in CFG.CLASSES:
            if op == "empirical":
                thr = MODEL_THRESH.get((m, cls))
            else:
                cal = MODEL_CALIB.get((m, cls))
                thr = pac_threshold(*cal) if cal is not None else None
            if thr is None or (isinstance(thr, float) and math.isnan(thr)):
                continue
            y = MODEL_EVAL[cls]["labels"]
            pred = (MODEL_EVAL[cls][m] > thr).astype(int)
            tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
            tn = int(((pred == 0) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
            prec = tp / (tp + fp) if tp + fp else float("nan")
            rec = tp / (tp + fn) if tp + fn else float("nan")
            f1 = 2 * prec * rec / (prec + rec) if (tp and not math.isnan(prec + rec)) else 0.0
            thr_rows.append({"operating_point": op, "model": m, "pcb_class": cls, "threshold": round(float(thr), 5),
                             "FPR": round(fp / (fp + tn), 3) if fp + tn else float("nan"), "recall": round(rec, 3),
                             "precision": round(prec, 3), "F1": round(f1, 3), "accuracy": round((tp + tn) / len(y), 3)})
thresholded_df = pd.DataFrame(thr_rows)
if len(thresholded_df):
    tgt = CFG.MAX_FALSE_POSITIVE_RATE
    for op, title in [("empirical", "EMPIRICAL operating point (<= " + str(tgt) + " false alarms on the CALIBRATION split)"),
                      ("pac", "PAC operating point (held-out false alarms <= " + str(tgt) + " with "
                       + str(int(CFG.PAC_CONFIDENCE * 100)) + "% confidence)")]:
        sub = thresholded_df[thresholded_df["operating_point"] == op]
        agg = sub.groupby("model", sort=False)[["FPR", "recall", "precision", "F1", "accuracy"]].mean().round(3)
        agg["FPR <= target?"] = np.where(agg["FPR"] <= tgt, "yes", "NO")
        print("\n" + title + ", held-out, mean over classes:")
        print(agg.to_string())
    print("\nRead the two tables together: the recall a model keeps at the PAC point is what it really delivers")
    print("at the stated false-alarm target. The empirical point's recall is bought with more false alarms than stated.")

# ---- chart with 95% CI error bars ----
fig, ax = plt.subplots(figsize=(12, 5))
n_m = len(MODEL_ORDER)
width = 0.8 / max(n_m, 1)
x = np.arange(len(CFG.CLASSES))
for j, m in enumerate(MODEL_ORDER):
    pts = np.array([BOOT[(m, c)][0] for c in CFG.CLASSES])
    cis = np.array([ci95(BOOT[(m, c)][1]) for c in CFG.CLASSES])
    yerr = np.vstack([pts - cis[:, 0], cis[:, 1] - pts])
    ax.bar(x + (j - (n_m - 1) / 2) * width, pts, width, yerr=yerr, capsize=2, label=m)
ax.set_xticks(x); ax.set_xticklabels(CFG.CLASSES)
ax.set_ylim(0.4, 1.0); ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_ylabel("Image-level AUROC (95% CI)")
ax.set_title("Held-out AUROC by model and PCB class -- identical images for every model")
ax.legend(loc="lower right", fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig("/kaggle/working/model_comparison_chart.png", dpi=130, bbox_inches="tight")
plt.show()

## Documentation 6 -- ROC / precision-recall curves, operating points, confusion matrices

In [ ]:
# ============================================================
# CELL D6 -- ROC & PRECISION-RECALL CURVES, OPERATING POINTS, CONFUSION MATRICES
# ============================================================
doc_section("CELL D6", "ROC & PRECISION-RECALL CURVES, OPERATING POINTS, CONFUSION MATRICES")
# Standard benchmark (held-out boards). Aligned systems are scored on the aligned (canonicalized) held-out boards; raw
# systems on the photos as captured. AUROC is threshold-free; the confusion matrices use OUR calibrated thresholds.
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score, confusion_matrix
roc_table_df = None
KEY_MODELS = [m for m in ["Ours (canonical pipeline)", "PatchCore (canonical pipeline)", "EfficientAD (canonical pipeline)",
                          "STFPM (canonical pipeline)", "Reverse Distillation (canonical pipeline)", "PatchCore", "EfficientAD",
                          "PaDiM", "Ours (single-pass)"] if m in MODEL_ORDER]
with doc_step("ROC and precision-recall curves"):
    n = len(CFG.CLASSES)
    fig, ax = plt.subplots(2, n, figsize=(5.2 * n, 9.5), squeeze=False)
    rows = []
    for j, cls in enumerate(CFG.CLASSES):
        y = np.asarray(FILE_LISTS[cls]["eval_labels"])
        for m in KEY_MODELS:
            s = np.asarray(MODEL_EVAL[cls][m], float)
            fpr, tpr, _ = roc_curve(y, s); prec, rec, _ = precision_recall_curve(y, s)
            auc, ap = roc_auc_score(y, s), average_precision_score(y, s)
            lw, ls = (3.0, "-") if m == OURS_SYS else (1.3, "--" if "(canonical pipeline)" not in m else "-")
            ax[0, j].plot(fpr, tpr, lw=lw, ls=ls, label="%s (%.3f)" % (m, auc))
            ax[1, j].plot(rec, prec, lw=lw, ls=ls, label="%s (AP %.3f)" % (m, ap))
            thr = MODEL_THRESH.get((m, cls))
            flag = s > thr if thr is not None else None
            rows.append({"model": m, "pcb_class": cls, "AUROC": round(auc, 4), "average precision": round(ap, 4),
                         "recall at calibrated threshold": round(float(flag[y == 1].mean()), 3) if flag is not None else None,
                         "false alarms at calibrated threshold": round(float(flag[y == 0].mean()), 3) if flag is not None else None,
                         "best possible recall at <=10% false alarms": round(float(tpr[fpr <= 0.10].max()) if (fpr <= 0.10).any() else 0.0, 3)})
        ax[0, j].plot([0, 1], [0, 1], c="gray", ls=":"); ax[0, j].set_title(cls + ": ROC (AUROC)"); ax[0, j].set_xlabel("false-positive rate"); ax[0, j].set_ylabel("true-positive rate")
        ax[1, j].set_title(cls + ": precision-recall (AP)"); ax[1, j].set_xlabel("recall"); ax[1, j].set_ylabel("precision")
        for a in ax[:, j]:
            a.legend(fontsize=6, loc="lower right" if a is ax[0, j] else "lower left")
    fig.suptitle("Held-out ROC and precision-recall curves (thick red-labelled line: our final system)")
    caption("ROC (top) and precision-recall (bottom) curves of every key system on the held-out boards; legend values are "
            "AUROC and average precision.")
    plt.tight_layout(); plt.show()
    roc_table_df = pd.DataFrame(rows)
    print(roc_table_df.to_string(index=False))
    save_table(roc_table_df, "roc_pr_operating_points", "AUROC, average precision and operating-point recall / false alarms per model and class")

with doc_step("confusion matrices of the final system"):
    n = len(CFG.CLASSES)
    fig, ax = plt.subplots(2, n, figsize=(4.2 * n, 7.6), squeeze=False)
    for j, cls in enumerate(CFG.CLASSES):
        y = np.asarray(FILE_LISTS[cls]["eval_labels"]); s = np.asarray(MODEL_EVAL[cls][OURS_SYS], float)
        cs, cl = MODEL_CALIB[(OURS_SYS, cls)]
        for r, (thr, name) in enumerate([(MODEL_THRESH[(OURS_SYS, cls)], "empirical"), (pac_threshold(cs, cl), "PAC (<=10% FA guaranteed)")]):
            cm = confusion_matrix(y, (s > thr).astype(int), labels=[0, 1])
            a = ax[r, j]; a.imshow(cm, cmap="Blues")
            for (i2, k2), v in np.ndenumerate(cm):
                a.text(k2, i2, str(v), ha="center", va="center", fontsize=13, color="black")
            a.set_xticks([0, 1]); a.set_xticklabels(["pred PASS", "pred DEFECT"]); a.set_yticks([0, 1]); a.set_yticklabels(["normal", "defective"])
            a.set_title("%s -- %s\nrecall %.0f%%, false alarms %.0f%%" % (cls, name, 100 * cm[1, 1] / max(1, cm[1].sum()),
                                                                          100 * cm[0, 1] / max(1, cm[0].sum())), fontsize=9)
    fig.suptitle("Our final system on held-out boards: confusion matrices at both operating points")
    caption("Confusion matrices of the final system on held-out boards at the empirical threshold (top) and the PAC "
            "threshold with a guaranteed false-alarm bound (bottom).")
    plt.tight_layout(); plt.show()

## Part 2.5 -- Ensemble: our model + PatchCore
Blend weight tuned on the calibration split; judged on every held-out image against both of its parts.

In [ ]:
# ============================================================
# CELL 19a -- Ensemble (ours + a baseline): blend weight tuned on the CALIBRATION split,
doc_section("CELL 19a", "Ensemble (ours + a baseline): blend weight tuned on the CALIBRATION split,")
#             evaluated on the FULL held-out eval split with paired bootstrap CIs
# ============================================================
# Both models are now scored on the calibration split (Cell 17 predicts it too), so the blend
# weight is chosen there and the ensemble is judged on all held-out images -- no need to carve
# the eval set in half (the previous version tested on ~50 images/class).
# Caveat worth stating: our own calibration scores are "in-sample" for our alpha/threshold
# choice, which could nudge the tuned weight slightly toward our model.
ENSEMBLE_WITH = "PatchCore"      # or "EfficientAD" / "PatchCore (matched bank)"
ENS_NAME = "Ensemble (ours + " + ENSEMBLE_WITH + ")"
ensemble_df = None
ENS_BOOT = {}

avail = [c for c in CFG.CLASSES
         if baseline_results.get(ENSEMBLE_WITH, {}).get(c) is not None
         and not np.isnan(baseline_results[ENSEMBLE_WITH][c]["calib_scores"]).any()
         and ENSEMBLE_WITH in MODEL_EVAL[c]]
if len(avail) < len(CFG.CLASSES):
    print("Ensemble skipped for classes without complete " + ENSEMBLE_WITH + " predictions:",
          [c for c in CFG.CLASSES if c not in avail])

rows = []
for i, cls in enumerate(avail):
    det = detectors[cls]
    ours_cal = np.asarray(det.score_stats["calib_scores"], dtype=np.float64)
    lab_cal = np.asarray(det.score_stats["calib_labels"], dtype=int)
    assert list(lab_cal) == list(FILE_LISTS[cls]["calib_labels"]), "calibration order mismatch"
    base_cal = baseline_results[ENSEMBLE_WITH][cls]["calib_scores"]
    mu_o, sd_o = ours_cal.mean(), ours_cal.std() + 1e-9
    mu_b, sd_b = base_cal.mean(), base_cal.std() + 1e-9
    z_o, z_b = (ours_cal - mu_o) / sd_o, (base_cal - mu_b) / sd_b
    grid = np.linspace(0.0, 1.0, CFG.ENSEMBLE_ALPHA_GRID_STEPS)
    # best calibration AUROC; ties broken toward an even blend
    alpha = max(grid, key=lambda a: (roc_auc_score(lab_cal, a * z_o + (1 - a) * z_b), -abs(a - 0.5)))

    ens = alpha * (MODEL_EVAL[cls]["Ours (single-pass)"] - mu_o) / sd_o + (1 - alpha) * (MODEL_EVAL[cls][ENSEMBLE_WITH] - mu_b) / sd_b
    pb = PairedBootstrap(MODEL_EVAL[cls]["labels"], seed=SEED + 100 + CFG.CLASSES.index(cls))  # same resamples as Cell 18
    ENS_BOOT[(ENS_NAME, cls)] = pb.auroc(ens)
    rows.append({"pcb_class": cls, "alpha (1=ours only)": round(float(alpha), 2),
                 ENS_NAME: fmt_ci(*ENS_BOOT[(ENS_NAME, cls)]),
                 "Ours (single-pass)": fmt_ci(*BOOT[("Ours (single-pass)", cls)]),
                 ENSEMBLE_WITH: fmt_ci(*BOOT[(ENSEMBLE_WITH, cls)])})

if rows:
    ENS_BOOT[(ENS_NAME, "MEAN")] = (float(np.mean([ENS_BOOT[(ENS_NAME, c)][0] for c in avail])),
                                    np.mean([ENS_BOOT[(ENS_NAME, c)][1] for c in avail], axis=0))
    ours_mean = np.mean([BOOT[("Ours (single-pass)", c)][1] for c in avail], axis=0)
    base_mean = np.mean([BOOT[(ENSEMBLE_WITH, c)][1] for c in avail], axis=0)
    rows.append({"pcb_class": "MEAN", "alpha (1=ours only)": "",
                 ENS_NAME: fmt_ci(*ENS_BOOT[(ENS_NAME, "MEAN")]),
                 "Ours (single-pass)": fmt_ci(float(np.mean([BOOT[("Ours (single-pass)", c)][0] for c in avail])), ours_mean),
                 ENSEMBLE_WITH: fmt_ci(float(np.mean([BOOT[(ENSEMBLE_WITH, c)][0] for c in avail])), base_mean)})
    ensemble_df = pd.DataFrame(rows)
    print("ENSEMBLE -- held-out AUROC [95% paired-bootstrap CI] (blend weight tuned on calibration split)")
    print(ensemble_df.to_string(index=False))
    ens_mean = ENS_BOOT[(ENS_NAME, "MEAN")][1]
    for other_name, other in [("Ours (single-pass)", ours_mean), (ENSEMBLE_WITH, base_mean)]:
        d, lo, hi, verdict = diff_verdict(ens_mean, other)
        print("Ensemble minus " + other_name + ": " + str(round(d, 3)) + " [" + str(round(lo, 3)) + ", "
              + str(round(hi, 3)) + "] -> ensemble vs " + other_name + ": " + verdict)
    print("\nOnly claim the ensemble helps if it is significantly higher than the BETTER of the two parts.")
else:
    print("No classes available for the ensemble.")

In [ ]:
# ============================================================
# CELL 19 -- Per-image inventory: EVERY test image, every model (for the Excel workbook)
# ============================================================
doc_section("CELL 19", "Per-image inventory: EVERY test image, every model (for the Excel workbook)")
# Each row is tagged with its split. Calibration rows were used to pick thresholds (for every
# model alike), so cite accuracy from EVAL rows only -- the Excel 'Summary' sheet does that.
# Baseline verdicts use each baseline's own calibration-derived threshold (same rule as ours),
# replacing the earlier median-of-scores placeholder that forced ~50% positives.
all_results_by_class = {}

if CFG.SCORE_ALL_TEST_IMAGES:
    for cls in CFG.CLASSES:
        fl = FILE_LISTS[cls]
        det = detectors[cls]
        calib_set = set(fl["calib_images"])
        lookups = {}
        for m in CFG.BASELINE_MODEL_NAMES:
            r = baseline_results.get(m, {}).get(cls)
            if r is None:
                continue
            lk = {}
            lk.update({p: s for p, s in zip(fl["calib_images"], r["calib_scores"])})
            lk.update({p: s for p, s in zip(fl["eval_images"], r["eval_scores"])})
            lookups[m] = (lk, r["threshold"])

        rows = []
        for path, label in tqdm(list(zip(fl["test_images"], fl["test_labels"])), desc="[" + cls + "] scoring all test images", leave=False):
            amap, score = det.score_image_tensor(load_image(path))
            is_anom = bool(score > det.threshold)
            regions = analyze_anomaly_regions(amap, top_k=CFG.MAX_REGIONS) if is_anom else []
            row = {
                "filename": os.path.basename(path), "pcb_class": cls,
                "true_label": "anomaly" if label == 1 else "normal",
                "split": "calibration" if path in calib_set else "eval",
                "our_score": round(float(score), 4), "our_threshold": round(float(det.threshold), 4),
                "our_verdict": "anomaly" if is_anom else "normal", "our_correct": is_anom == bool(label),
                "top_shape_hint": regions[0]["shape_hint"] if regions else "",
                "top_priority": regions[0]["priority"] if regions else "",
                "num_regions": len(regions),
            }
            for m in CFG.BASELINE_MODEL_NAMES:
                s, thr = (lookups[m][0].get(path, float("nan")), lookups[m][1]) if m in lookups else (float("nan"), float("nan"))
                ok = not (math.isnan(s) or math.isnan(thr))
                row[m + "_score"] = round(float(s), 4) if not math.isnan(s) else None
                row[m + "_verdict"] = ("anomaly" if s > thr else "normal") if ok else None
                row[m + "_correct"] = ((s > thr) == bool(label)) if ok else None
            rows.append(row)
        df = pd.DataFrame(rows)
        all_results_by_class[cls] = df
        ev = df[df["split"] == "eval"]
        print("[" + cls + "] scored", len(df), "images | our held-out (eval-row) accuracy:",
              round(float(ev["our_correct"].mean()), 3), "on", len(ev), "images")
else:
    print("CFG.SCORE_ALL_TEST_IMAGES is False -- skipping the full-dataset scoring pass.")

## Part 3 -- Robustness benchmark: angle, lighting and crop
This is where the thesis's defining requirement is tested. On aligned, evenly lit images (Part 2) invariance can
only cost accuracy; here every model faces the same rotated, dimmed, unevenly lit and cropped boards.

In [ ]:
# ============================================================
# CELL 20 -- ROBUSTNESS BENCHMARK: raw shifts vs. shifts + canonicalization, every model
# ============================================================
doc_section("CELL 20", "ROBUSTNESS BENCHMARK: raw shifts vs. shifts + canonicalization, every model")
# Every model scores the SAME images for every condition. Reported:
#   * headline: clean AUROC vs. average AUROC over the six shifts, raw and canonicalized
#   * AUROC per condition, with the per-branch (CNN-only / ViT-only) diagnostics of our model --
#     which backbone carries rotation robustness and which lighting robustness?
#   * canonicalization gain per model (paired: same images, same bootstrap resamples)
#   * false-alarm rate on NORMAL boards and recall at each model's calibrated threshold -- a
#     shift that makes every good board 'defective' breaks an inspection line even if AUROC holds.
# "+" marks baselines whose scores are Anomalib-normalized (clamped) rather than raw.
ROBUST_SCORES = {}
robust_summary_df = robust_auroc_df = robust_drop_df = robust_fpr_df = robust_recall_df = None
robust_paired_df = canon_gain_df = None
robust_auroc_long_df = robust_scores_long_df = None      # per-class / per-image exports (Excel)

if CFG.RUN_ROBUSTNESS and ROBUST_SETS:
    BASE_CONDS = list(CFG.ROBUSTNESS_CONDITIONS)
    SHIFTS = [c for c in BASE_CONDS if c != "identity"]
    is_canon = lambda c: c.endswith("+canon")
    use_noaug = CFG.RUN_ABLATION and bool(detectors_noaug)
    thr_of = {"Ours (single-pass)": lambda c: detectors[c].threshold,
              "Ours (TTA)": lambda c: detectors[c].threshold_tta,
              "Ours w/o invariance aug": lambda c: detectors_noaug[c].threshold,
              "Ours (canonical pipeline)": lambda c: detectors_canon[c].threshold}

    for cls in CFG.CLASSES:
        det = detectors[cls]
        ROBUST_SCORES[cls] = {}
        for cond in ROBUST_CONDS:
            items = ROBUST_SETS[cls][cond]
            e = {"labels": np.asarray([l for _, l, _ in items], dtype=int)}
            cols = defaultdict(list)
            run_tta = CFG.ROBUSTNESS_INCLUDE_TTA and not is_canon(cond)   # canonicalized images are upright
            for p, _, _ in tqdm(items, desc="[" + cls + "] " + cond, leave=False):
                t = load_image(p)
                cnn_map, vit_map = det._score_grid_branches(t)          # one pass -> fused AND both branches
                cols["Ours (single-pass)"].append(det._topk_mean(det._fused_map(cnn_map, vit_map)))
                cols["Ours: CNN branch only"].append(det._topk_mean(cnn_map))
                cols["Ours: ViT branch only"].append(det._topk_mean(vit_map))
                if run_tta:
                    cols["Ours (TTA)"].append(det.score_image_tensor_tta(t)[1])
                if use_noaug:
                    cols["Ours w/o invariance aug"].append(detectors_noaug[cls].score_image_tensor(t)[1])
                if is_canon(cond) and cls in detectors_canon:      # its native input: canonicalized images
                    dc = detectors_canon[cls]               # Cell 15e2 may have changed its preprocessing
                    cols["Ours (canonical pipeline)"].append(dc.score_path(p)[1])
            for k, v in cols.items():
                e[k] = np.asarray(v, dtype=np.float64)
            for m in CFG.ROBUSTNESS_BASELINES:
                r = baseline_results.get(m, {}).get(cls)
                if r is not None and cond in r["robust"] and not np.isnan(r["robust"][cond]).any():
                    e[m] = r["robust"][cond]
            ROBUST_SCORES[cls][cond] = e

    MODELS = [m for m in ["Ours (single-pass)", "Ours (TTA)", "Ours w/o invariance aug", "Ours (canonical pipeline)",
                          "Ours: CNN branch only", "Ours: ViT branch only"] + list(CFG.ROBUSTNESS_BASELINES)
              if any(m in ROBUST_SCORES[c][cond] for c in CFG.CLASSES for cond in ROBUST_CONDS)]
    DETECTORS = [m for m in MODELS if not m.startswith("Ours:")]            # models with calibrated thresholds
    tag = {m: m + (" +" if any(baseline_results.get(m, {}).get(c) and baseline_results[m][c]["score_mode"] != "raw"
                               for c in CFG.CLASSES) else "") for m in MODELS}

    # bootstrap: raw and +canon versions of a condition share labels AND seed -> identical resamples (paired)
    RB = {}
    for bi, base in enumerate(BASE_CONDS):
        for cond in [base, base + "+canon"]:
            if cond not in ROBUST_CONDS:
                continue
            valid_cls = []
            for i, cls in enumerate(CFG.CLASSES):
                pb = PairedBootstrap(ROBUST_SCORES[cls][cond]["labels"], seed=SEED + 1000 + 10 * bi + i)
                if not pb.valid:
                    continue
                valid_cls.append(cls)
                for m in MODELS:
                    if m in ROBUST_SCORES[cls][cond]:
                        RB[(m, cond, cls)] = pb.auroc(ROBUST_SCORES[cls][cond][m])
            if len(valid_cls) < len(CFG.CLASSES) and not is_canon(cond):
                print("note: '" + base + "' averages over", valid_cls, "only -- the other classes have no",
                      "mask-verified anomalies for it (raw and +canon use the same classes)")
            for m in MODELS:
                if valid_cls and all((m, cond, c) in RB for c in valid_cls):
                    RB[(m, cond, "MEAN")] = (float(np.mean([RB[(m, cond, c)][0] for c in valid_cls])),
                                             np.mean([RB[(m, cond, c)][1] for c in valid_cls], axis=0))

    def _avg(m, conds):
        keys = [(m, c, "MEAN") for c in conds]
        if not conds or not all(k in RB for k in keys):
            return None
        return float(np.mean([RB[k][0] for k in keys])), np.mean([RB[k][1] for k in keys], axis=0)

    # ---- exports: per-class AUROC and per-image scores (needed to localize any problem to a class) ----
    robust_auroc_long_df = pd.DataFrame([
        {"model": m, "condition": c, "pcb_class": k, "auroc": round(v[0], 4),
         "ci_low": round(ci95(v[1])[0], 4), "ci_high": round(ci95(v[1])[1], 4)} for (m, c, k), v in RB.items()])
    robust_scores_long_df = pd.DataFrame([
        {"pcb_class": cls, "condition": cond, "stem": stem, "label": int(lab), "model": m, "score": float(e[m][i])}
        for cls in CFG.CLASSES for cond, e in ROBUST_SCORES[cls].items()
        for i, (_, lab, stem) in enumerate(ROBUST_SETS[cls][cond]) for m in MODELS if m in e])

    # ---- 1. headline: clean / average / WORST shift, raw and canonicalized ----
    canon_shifts = [c + "+canon" for c in SHIFTS if c + "+canon" in ROBUST_CONDS]

    def _worst(m, conds):
        vals = [(RB[(m, c, "MEAN")][0], c) for c in conds if (m, c, "MEAN") in RB]
        return min(vals) if vals and len(vals) == len(conds) else None

    HEAD = {}
    rows = []
    for m in MODELS:
        HEAD[(m, "clean")] = _avg(m, ["identity"])
        HEAD[(m, "clean_canon")] = _avg(m, ["identity+canon"]) if canon_shifts else None
        HEAD[(m, "raw")] = _avg(m, SHIFTS)
        HEAD[(m, "canon")] = _avg(m, canon_shifts)
        w_raw, w_can = _worst(m, SHIFTS), (_worst(m, canon_shifts) if canon_shifts else None)
        rows.append({"model": tag[m],
                     "clean, raw": fmt_ci(*HEAD[(m, "clean")]) if HEAD[(m, "clean")] else "n/a",
                     "6-shift avg, raw": fmt_ci(*HEAD[(m, "raw")]) if HEAD[(m, "raw")] else "n/a",
                     "worst shift, raw": (str(round(w_raw[0], 3)) + " (" + w_raw[1] + ")") if w_raw else "n/a",
                     "clean, +canon": fmt_ci(*HEAD[(m, "clean_canon")]) if HEAD[(m, "clean_canon")] else "n/a",
                     "6-shift avg, +canon": fmt_ci(*HEAD[(m, "canon")]) if HEAD[(m, "canon")] else "n/a",
                     "worst shift, +canon": (str(round(w_can[0], 3)) + " (" + w_can[1].replace("+canon", "") + ")")
                                            if w_can else "n/a"})
    robust_summary_df = pd.DataFrame(rows)
    print("ROBUSTNESS HEADLINE -- AUROC [95% CI], mean over classes; shifts =", ", ".join(SHIFTS))
    print(robust_summary_df.to_string(index=False))

    # ---- 2. per-condition AUROC ----
    robust_auroc_df = pd.DataFrame({c: [fmt_ci(*RB[(m, c, "MEAN")]) if (m, c, "MEAN") in RB else "n/a" for m in MODELS]
                                    for c in ROBUST_CONDS}, index=[tag[m] for m in MODELS])
    print("\nAUROC PER CONDITION (raw):")
    print(robust_auroc_df[[c for c in ROBUST_CONDS if not is_canon(c)]].to_string())
    if canon_shifts:
        print("\nAUROC PER CONDITION (+canon: after golden-reference canonicalization):")
        print(robust_auroc_df[[c for c in ROBUST_CONDS if is_canon(c)]].to_string())
    robust_drop_df = pd.DataFrame({c: [round(RB[(m, c, "MEAN")][0] - RB[(m, "identity", "MEAN")][0], 3)
                                       if (m, c, "MEAN") in RB and (m, "identity", "MEAN") in RB else float("nan")
                                       for m in MODELS] for c in ROBUST_CONDS if c != "identity"},
                                  index=[tag[m] for m in MODELS])

    # ---- 3. canonicalization gain per model (paired) ----
    if canon_shifts:
        gain_rows = []
        for m in MODELS:
            for base in BASE_CONDS + ["AVG over shifts"]:
                if base == "AVG over shifts":
                    a, b = HEAD[(m, "canon")], HEAD[(m, "raw")]
                else:
                    a, b = RB.get((m, base + "+canon", "MEAN")), RB.get((m, base, "MEAN"))
                if a is None or b is None:
                    continue
                d, lo, hi, verdict = diff_verdict(a[1], b[1])
                gain_rows.append({"model": tag[m], "condition": base, "auroc_raw": round(b[0], 3),
                                  "auroc_canon": round(a[0], 3), "gain": round(d, 3), "ci_low": round(lo, 3),
                                  "ci_high": round(hi, 3), "canonicalization": verdict.replace("significantly HIGHER", "HELPS")
                                  .replace("significantly LOWER", "HURTS").replace("no significant difference", "no sig. effect")})
        canon_gain_df = pd.DataFrame(gain_rows)
        print("\nCANONICALIZATION GAIN (AUROC with minus without, paired):")
        print(canon_gain_df.pivot(index="model", columns="condition", values="gain")
              .reindex(columns=BASE_CONDS + ["AVG over shifts"]).to_string())
        verdicts = canon_gain_df.pivot(index="model", columns="condition", values="canonicalization")
        print(verdicts.reindex(columns=BASE_CONDS + ["AVG over shifts"]).to_string())

    # ---- 4. false alarms / recall at calibrated thresholds ----
    fpr_rows, rec_rows = [], []
    for m in DETECTORS:
        fr, rr = {"model": tag[m]}, {"model": tag[m]}
        for cond in ROBUST_CONDS:
            fprs, recs = [], []
            for cls in CFG.CLASSES:
                e = ROBUST_SCORES[cls][cond]
                if m not in e:
                    continue
                if m in thr_of:
                    thr = thr_of[m](cls)
                else:            # canonicalized boards -> the threshold calibrated on canonicalized calibration images
                    rb = baseline_results[m][cls]
                    thr = rb.get("threshold_canon", rb["threshold"]) if is_canon(cond) else rb["threshold"]
                y, s = e["labels"], e[m]
                if (y == 0).any():
                    fprs.append(float((s[y == 0] > thr).mean()))
                if (y == 1).any():
                    recs.append(float((s[y == 1] > thr).mean()))
            fr[cond] = round(float(np.mean(fprs)), 3) if fprs else float("nan")
            rr[cond] = round(float(np.mean(recs)), 3) if recs else float("nan")
        fpr_rows.append(fr)
        rec_rows.append(rr)
    robust_fpr_df = pd.DataFrame(fpr_rows).set_index("model")
    robust_recall_df = pd.DataFrame(rec_rows).set_index("model")
    print("\nFALSE-ALARM RATE on NORMAL boards at each model's calibrated threshold (compare with 'identity'):")
    print(robust_fpr_df.to_string())
    print("\nRECALL on DEFECTIVE boards at the same thresholds:")
    print(robust_recall_df.to_string())

    # ---- 5. paired verdicts: our detectors vs each baseline ----
    pr = []
    ours = [m for m in DETECTORS if m.startswith("Ours")]
    for o in ours:
        for m in DETECTORS:
            if m.startswith("Ours"):
                continue
            for cond in ROBUST_CONDS + ["AVG raw shifts", "AVG +canon shifts"]:
                if cond == "AVG raw shifts":
                    a, b = HEAD[(o, "raw")], HEAD[(m, "raw")]
                elif cond == "AVG +canon shifts":
                    a, b = HEAD[(o, "canon")], HEAD[(m, "canon")]
                else:
                    a, b = RB.get((o, cond, "MEAN")), RB.get((m, cond, "MEAN"))
                if a is None or b is None:
                    continue
                d, lo, hi, verdict = diff_verdict(a[1], b[1])
                pr.append({"comparison": o + " minus " + m, "condition": cond, "mean_diff": round(d, 3),
                           "ci_low": round(lo, 3), "ci_high": round(hi, 3), "ours_is": verdict})
    robust_paired_df = pd.DataFrame(pr)
    if len(robust_paired_df):
        print("\nPAIRED VERDICTS on the shift averages (full per-condition table in the Excel 'Robust_Paired' sheet):")
        print(robust_paired_df[robust_paired_df["condition"].str.startswith("AVG")].to_string(index=False))

    # ---- charts: AUROC and false-alarm rate, raw vs +canon ----
    chart_models = [m for m in DETECTORS]
    panels = [("raw", [c for c in ROBUST_CONDS if not is_canon(c)])]
    if canon_shifts:
        panels.append(("+canon", [c for c in ROBUST_CONDS if is_canon(c)]))
    fig, axes = plt.subplots(2, len(panels), figsize=(9 * len(panels), 9), squeeze=False)
    width = 0.8 / max(len(chart_models), 1)
    for col, (name, conds) in enumerate(panels):
        xx = np.arange(len(conds))
        for j, m in enumerate(chart_models):
            off = (j - (len(chart_models) - 1) / 2) * width
            pts = np.array([RB[(m, c, "MEAN")][0] if (m, c, "MEAN") in RB else np.nan for c in conds])
            cis = np.array([ci95(RB[(m, c, "MEAN")][1]) if (m, c, "MEAN") in RB else (np.nan, np.nan) for c in conds])
            axes[0, col].bar(xx + off, pts, width, yerr=np.vstack([pts - cis[:, 0], cis[:, 1] - pts]),
                             capsize=2, label=tag[m])
            axes[1, col].bar(xx + off, [robust_fpr_df.loc[tag[m], c] for c in conds], width, label=tag[m])
        labels = [c.replace("+canon", "") for c in conds]
        for row, (title, ylab) in enumerate([("AUROC (95% CI) -- " + name, "AUROC"),
                                             ("False-alarm rate on normal boards -- " + name, "FPR")]):
            ax = axes[row, col]
            ax.set_xticks(xx); ax.set_xticklabels(labels, rotation=20); ax.set_title(title); ax.set_ylabel(ylab)
        axes[0, col].set_ylim(0.3, 1.0); axes[0, col].axhline(0.5, color="gray", linestyle="--", linewidth=1)
        axes[1, col].set_ylim(0, 1.05); axes[1, col].axhline(CFG.MAX_FALSE_POSITIVE_RATE, color="gray", linestyle="--", linewidth=1)
    axes[0, 0].legend(fontsize=7, loc="lower left")
    plt.tight_layout()
    plt.savefig("/kaggle/working/robustness_chart.png", dpi=130, bbox_inches="tight")
    plt.show()
else:
    print("Robustness benchmark skipped (CFG.RUN_ROBUSTNESS is False or no robustness sets were built).")

## Final results -- complete systems, conclusions, deployment recommendation
Ranks complete inspection systems (detector + input preparation) on identical held-out boards; every conclusion is generated from the paired significance tests. Also writes `results_summary.md`.

In [ ]:
# ============================================================
# CELL 20b -- FINAL RESULTS: complete inspection SYSTEMS, conclusions, deployment recommendation
# ============================================================
doc_section("CELL 20b", "FINAL RESULTS: complete inspection SYSTEMS, conclusions, deployment recommendation")
# A SYSTEM = a detector + how its input is prepared. Every system is scored on the same held-out boards
# (Part 3's robustness sets: clean + 6 shifts), mean over classes, with paired 95% bootstrap CIs.
# Conclusions below are GENERATED from the significance tests -- nothing is hard-coded -- so they stay
# correct whatever a run produces. Also written to /kaggle/working/results_summary.md.
SUMMARY_PATH = "/kaggle/working/results_summary.md"
final_summary_df = system_pairwise_df = None
systems = []
CONCLUSIONS = []

SYSTEM_DEFS = [   # (system name, model key in Cell 20, uses canonicalized input?)
    ("Ours -- raw input", "Ours (single-pass)", False),
    ("Ours -- raw input + rotation TTA", "Ours (TTA)", False),
    ("Ours -- canonical pipeline", "Ours (canonical pipeline)", True),
    ("PatchCore -- raw input", "PatchCore", False),
    ("PatchCore -- canonicalized input", "PatchCore", True),
    ("EfficientAD -- raw input", "EfficientAD", False),
    ("EfficientAD -- canonicalized input", "EfficientAD", True),
    ("PatchCore -- canonical pipeline", "PatchCore (canonical pipeline)", True),
    ("EfficientAD -- canonical pipeline", "EfficientAD (canonical pipeline)", True),
    ("STFPM -- canonical pipeline", "STFPM (canonical pipeline)", True),
    ("Reverse Distillation -- canonical pipeline", "Reverse Distillation (canonical pipeline)", True),
]


def _sys_conds(canon):
    return [c + "+canon" for c in SHIFTS] if canon else list(SHIFTS)


def _fmt(v):
    return fmt_ci(*v) if v else "n/a"


if CFG.RUN_ROBUSTNESS and ROBUST_SETS and "RB" in globals():
    systems = []
    for name, m, canon in SYSTEM_DEFS:
        shift_conds = _sys_conds(canon)
        clean_cond = "identity+canon" if canon else "identity"
        if m not in MODELS or not all((m, c, "MEAN") in RB for c in shift_conds + [clean_cond]):
            continue
        clean, shift_avg = RB[(m, clean_cond, "MEAN")], _avg(m, shift_conds)
        worst = min((RB[(m, c, "MEAN")][0], c.replace("+canon", "")) for c in shift_conds)
        fprs = [robust_fpr_df.loc[tag[m], c] for c in shift_conds if tag[m] in robust_fpr_df.index]
        recs = [robust_recall_df.loc[tag[m], c] for c in shift_conds if tag[m] in robust_recall_df.index]
        # standard benchmark (Cell 18, full held-out split) exists only where the model saw that input mode there
        bench_key = m if (not canon or m.endswith("(canonical pipeline)")) else m + " (canonicalized input)"
        bench_key = bench_key if bench_key in MODEL_ORDER else None
        bench = BOOT.get((bench_key, "MEAN")) if bench_key else None
        pac = None
        if bench_key and len(thresholded_df):
            sub = thresholded_df[(thresholded_df.operating_point == "pac") & (thresholded_df.model == bench_key)]
            pac = round(float(sub.recall.mean()), 3) if len(sub) else None
        systems.append({"name": name, "model": m, "canon": canon, "clean": clean, "shift_avg": shift_avg,
                        "worst": worst, "fpr": float(np.nanmean(fprs)) if fprs else float("nan"),
                        "recall": float(np.nanmean(recs)) if recs else float("nan"), "bench": bench, "pac": pac})

    systems.sort(key=lambda s: -s["shift_avg"][0])
if CFG.RUN_ROBUSTNESS and ROBUST_SETS and "RB" in globals() and not systems:
    print("No system has complete robustness results -- nothing to rank.")
elif CFG.RUN_ROBUSTNESS and ROBUST_SETS and "RB" in globals():
    best = systems[0]
    rows = []
    for rank, s in enumerate(systems, start=1):
        if s is best:
            verdict = "BEST"
        else:
            d, lo, hi, v = diff_verdict(best["shift_avg"][1], s["shift_avg"][1])
            verdict = "tied with best" if v == "no significant difference" else "below best (%.3f [%.3f, %.3f])" % (d, lo, hi)
        rows.append({"rank": rank, "system": s["name"],
                     "clean AUROC": _fmt(s["clean"]), "6-shift avg AUROC": _fmt(s["shift_avg"]),
                     "worst shift": "%.3f (%s)" % s["worst"],
                     "false alarms under shift": round(s["fpr"], 3), "recall under shift": round(s["recall"], 3),
                     "benchmark AUROC (full held-out)": _fmt(s["bench"]),
                     "PAC recall @<=10% FA": s["pac"] if s["pac"] is not None else "n/a",
                     "vs best (6-shift avg)": verdict})
        s["verdict"] = verdict
    final_summary_df = pd.DataFrame(rows)
    # pairwise 6-shift-average differences between ALL systems (row minus column; * = 95% CI excludes 0)
    _names = [s["name"] for s in systems]
    system_pairwise_df = pd.DataFrame("--", index=_names, columns=_names)
    for a in systems:
        for b in systems:
            if a is not b:
                d, lo, hi, v = diff_verdict(a["shift_avg"][1], b["shift_avg"][1])
                system_pairwise_df.loc[a["name"], b["name"]] = "%+.3f%s" % (d, "*" if v != "no significant difference" else "")

    # ---------------- generated conclusions ----------------
    tied = [s["name"] for s in systems[1:] if s["verdict"] == "tied with best"]
    CONCLUSIONS.append("Most robust system: **%s** -- average AUROC %s over the six shifts, worst shift %.3f (%s), "
                       "clean AUROC %s%s." % (best["name"], _fmt(best["shift_avg"]), best["worst"][0], best["worst"][1],
                                              _fmt(best["clean"]),
                                              ("; statistically tied with " + ", ".join(tied)) if tied else
                                              "; significantly above every other system"))
    by_name = {s["name"]: s for s in systems}
    for det in ["PatchCore", "EfficientAD"]:
        a, b = by_name.get(det + " -- canonicalized input"), by_name.get(det + " -- raw input")
        if a and b:
            d, lo, hi, v = diff_verdict(a["shift_avg"][1], b["shift_avg"][1])
            CONCLUSIONS.append("Canonicalization changes %s's 6-shift AUROC by %+.3f [%.3f, %.3f] (%s)." % (det, d, lo, hi, v))
    for det in ["PatchCore", "EfficientAD"]:
        a, b = by_name.get(det + " -- canonical pipeline"), by_name.get(det + " -- canonicalized input")
        if a and b:
            d, lo, hi, v = diff_verdict(a["shift_avg"][1], b["shift_avg"][1])
            CONCLUSIONS.append("Training %s on canonicalized boards too (canonical pipeline vs test-only canonicalization): "
                               "6-shift AUROC %+.3f [%.3f, %.3f] (%s)." % (det, d, lo, hi, v))
    if paired_canon_df is not None and len(paired_canon_df):
        CONCLUSIONS.append("Canonicalized standard benchmark (every model on canonicalized held-out boards; orientation removed, "
                           "lighting cue reduced): Ours (canonical "
                           "pipeline) %s; " % _fmt(BOOT[("Ours (canonical pipeline)", "MEAN")]) + "; ".join(
                               "vs %s %+.3f [%.3f, %.3f] (%s)" % (r.comparison.split(" minus ")[1], r.mean_diff, r.ci_low,
                                                                  r.ci_high, r.ours_is) for r in paired_canon_df.itertuples()) + ".")
    a, b = by_name.get("Ours -- canonical pipeline"), by_name.get("Ours -- raw input")
    if a and b:
        d, lo, hi, v = diff_verdict(a["shift_avg"][1], b["shift_avg"][1])
        dc, loc, hic, vc = diff_verdict(a["clean"][1], b["clean"][1])
        CONCLUSIONS.append("Our canonical pipeline vs our raw-input model: 6-shift AUROC %+.3f [%.3f, %.3f] (%s); clean "
                           "AUROC %+.3f [%.3f, %.3f] (%s)." % (d, lo, hi, v, dc, loc, hic, vc))
        if best is not a:
            d2, lo2, hi2, v2 = diff_verdict(a["shift_avg"][1], best["shift_avg"][1])
            CONCLUSIONS.append("Our canonical pipeline vs the most robust system: %+.3f [%.3f, %.3f] (%s)." % (d2, lo2, hi2, v2))
    if "Ours (single-pass)" in MODEL_ORDER:
        cmp_rows = paired_df[(paired_df.pcb_class == "MEAN")]
        worse = cmp_rows[cmp_rows.ours_is == "significantly LOWER"].comparison.str.replace("Ours (single-pass) minus ", "", regex=False).tolist()
        CONCLUSIONS.append("Standard benchmark (aligned, evenly lit VisA): our raw-input model reaches %s held-out AUROC; "
                           "it is significantly below %s." % (_fmt(BOOT[("Ours (single-pass)", "MEAN")]),
                                                              ", ".join(worse) if worse else "no other model"))
    if ablation_df is not None and ("MEAN", "ViT branch only") in ABLATION_BOOT:
        d, lo, hi, v = diff_verdict(ABLATION_BOOT[("MEAN", "Fused (ours, single-pass)")][1], ABLATION_BOOT[("MEAN", "ViT branch only")][1])
        d2, lo2, hi2, v2 = diff_verdict(ABLATION_BOOT[("MEAN", "Fused (ours, single-pass)")][1], ABLATION_BOOT[("MEAN", "CNN branch only")][1])
        CONCLUSIONS.append("CNN+ViT fusion vs ViT alone: %+.3f [%.3f, %.3f] (%s); vs CNN alone: %+.3f [%.3f, %.3f] (%s)." %
                           (d, lo, hi, v, d2, lo2, hi2, v2))
    # Deployment pick: ranking quality alone is not enough on a production line -- among the systems statistically
    # TIED with the most robust one, take the fewest false alarms under shift, then the highest recall.
    contenders = [best] + [s for s in systems[1:] if s["verdict"] == "tied with best"]
    deploy = min(contenders, key=lambda s: (np.nan_to_num(s["fpr"], nan=1.0), -np.nan_to_num(s["recall"], nan=0.0)))
    why = ("" if deploy is best else
           " (statistically tied with %s on AUROC, but with fewer false alarms: %.0f%% vs %.0f%%)" %
           (best["name"], 100 * deploy["fpr"], 100 * best["fpr"]))
    if V2_NAME in MODEL_ORDER:
        parts = []
        for b in [m for m in CFG.BASELINE_MODEL_NAMES if m in MODEL_ORDER]:
            d, lo, hi, v = diff_verdict(BOOT[(V2_NAME, "MEAN")][1], BOOT[(b, "MEAN")][1])
            parts.append("%s %+.3f [%.3f, %.3f] (%s)" % (b, d, lo, hi, v))
        CONCLUSIONS.append("Improvement study (pre-registered, selected on calibration: %s): %s reaches %s held-out AUROC; "
                           "vs %s." % (best_name, V2_NAME, _fmt(BOOT[(V2_NAME, "MEAN")]), "; ".join(parts) or "no baselines"))
    if aug_study_df is not None:
        CONCLUSIONS.append("Pipeline-consistent augmentation study (pre-registered, selected on calibration clean + shift sets): %s; " %
                           aug_study_pick + "held-out clean AUROC: " + "; ".join("%s %s" % (r["variant"].split(":")[0], r["held-out clean AUROC [95% CI]"])
                                                                                for _, r in aug_study_df.iterrows()) +
                           (" -- installed for every canonical-pipeline detector." if aug_study_pick.startswith("a1") else "."))
    if canon_study_df is not None:
        CONCLUSIONS.append("Canonical-pipeline study (pre-registered, selected on calibration): %s; held-out AUROC per variant: %s." %
                           (canon_study_pick, "; ".join("%s %s" % (r["variant"].split(":")[0], r["held-out AUROC [95% CI]"])
                                                        for _, r in canon_study_df.iterrows())))
    flagged = []
    for (c_, kind), (pt, reps) in sorted(CONFOUND.items()):
        lo, hi = ci95(reps)
        if lo > 0.5 or hi < 0.5:
            flagged.append("%s %s-only AUROC %s" % (c_, kind, fmt_ci(pt, reps)))
    if flagged:
        resid = ""
        if residual_light_df is not None and len(residual_light_df):
            col = [c for c in residual_light_df.columns if c.endswith("(eval)")][0]
            resid = (" Canonicalization removes orientation but only REDUCES the lighting cue (held-out lighting-only AUROC "
                     "after canonicalization: " + "; ".join("%s %s" % (r["pcb_class"], r[col]) for _, r in residual_light_df.iterrows()) +
                     "). A detector without lighting equalization (PatchCore, EfficientAD) can be helped by a remaining cue "
                     "(AUROC > 0.5) or misled by an inverted one (AUROC < 0.5); ours equalizes lighting with CLAHE and is "
                     "insensitive to it. Read per-class canonicalized differences in classes with a remaining cue with care.")
        CONCLUSIONS.append("Dataset confound (capture conditions): " + "; ".join(flagged) + ". VisA's defective and normal "
                           "test boards differ in how they were photographed, so raw-input AUROCs partly measure capture "
                           "conditions; the canonicalized comparisons are the fairer ones." + resid)
    CONCLUSIONS.append("Deployment recommendation: run **%s**%s. Under shift it raises false alarms on %.0f%% of good boards "
                       "and catches %.0f%% of defective ones at its calibrated threshold; for a guaranteed <=10%% false-alarm "
                       "rate, expect the PAC recall in the table (where measured)." %
                       (deploy["name"], why, 100 * deploy["fpr"], 100 * deploy["recall"]))
    for s in systems:
        s["deploy"] = s is deploy
    final_summary_df["deployment pick"] = ["<- recommended" if s["deploy"] else "" for s in systems]

    print("=" * 110)
    print("FINAL RESULTS -- complete inspection systems on identical held-out boards (mean over classes, 95% CIs)")
    print("=" * 110)
    print(final_summary_df.to_string(index=False))
    print("\nPAIRWISE 6-shift AUROC differences (row minus column; * = significant):")
    print(system_pairwise_df.to_string())
    print("\nCONCLUSIONS (generated from the paired significance tests):")
    for i, c in enumerate(CONCLUSIONS, start=1):
        print("  %d. %s" % (i, c.replace("**", "")))

    # ---------------- markdown report ----------------
    def _md_table(df):
        cols = list(df.columns)
        out = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
        out += ["| " + " | ".join(str(v) for v in r) + " |" for r in df.itertuples(index=False)]
        return "\n".join(out)

    lines = ["# PCB anomaly detection -- results summary", "",
             "Generated by the thesis evaluation notebook. Data: VisA PCB1-PCB4. All numbers are on held-out images "
             "never used for fitting, calibration or tuning; brackets are paired-bootstrap 95% CIs.", "",
             "## Complete systems under rotation, lighting and crop shifts", "", _md_table(final_summary_df), "",
             "## Pairwise system differences (6-shift AUROC, row minus column; * = 95% CI excludes 0)", "",
             _md_table(system_pairwise_df.reset_index().rename(columns={"index": "system"})), "",
             "## Conclusions", ""] + ["%d. %s" % (i, c) for i, c in enumerate(CONCLUSIONS, start=1)] + [
             "", "## Standard benchmark (held-out AUROC per class)", "", _md_table(comparison_ci_df.reset_index().rename(columns={"index": "class"})), ""]
    if canon_reg_summary_df is not None:
        lines += ["## Canonicalization diagnostics (registration to each class's golden reference)", "",
                  _md_table(canon_reg_summary_df), ""]
    lines += ["## Protocol", "",
              "- Data roles: train (normal only) -> calibration (thresholds, weights, Anomalib validation) -> eval (reported numbers only).",
              "- Thresholds: empirical (<=10% false alarms on calibration) and PAC (held-out <=10% with 90% confidence).",
              "- Robustness sets: deterministic shifts of held-out images; arbitrary-angle rotation rendered at camera resolution "
              "(cubic, then downscaled like every condition); 'crop' keeps anomalies only if the defect mask survives.",
              "- Canonicalization: ORB+RANSAC similarity registration, 180-degree check with ECC refinement, gated lighting correction."]
    with open(SUMMARY_PATH, "w") as f:
        f.write("\n".join(lines) + "\n")
    print("\nWrote", SUMMARY_PATH)
else:
    print("Final system comparison needs the robustness benchmark (CFG.RUN_ROBUSTNESS) -- skipped.")

## Part 3.5 -- Seed robustness: do the top-system comparisons survive re-training?

In [ ]:
# ============================================================
# CELL 20c -- SEED ROBUSTNESS: do the top-system comparisons survive RE-TRAINING with other random seeds?
# ============================================================
doc_section("CELL 20c", "SEED ROBUSTNESS: do the top-system comparisons survive RE-TRAINING with other random seeds?")
# Every other CI in this notebook covers WHICH TEST BOARDS were drawn -- not training randomness. Two real runs that
# differed ONLY in the baselines' training seeds moved EfficientAD's canonical-pipeline benchmark from 0.872 to 0.896
# (one class by 0.045, another baseline's classes by up to 0.08) and turned "ours significantly better than EfficientAD"
# (+0.052) into "no significant difference" (+0.028). Our own model reuses the same random draws every run, so its seed
# sensitivity had never been measured either. Here the three top systems are RE-TRAINED with CFG.SEED_STUDY_EXTRA_SEEDS
# extra seeds on identical data, scored with the SAME paired bootstrap resamples, and a difference is called robust only
# if it is significant in EVERY seed pairing.
import gc, random

SEED_SYSTEMS = ["Ours (canonical pipeline)", "PatchCore (canonical pipeline)", "EfficientAD (canonical pipeline)",
                "Reverse Distillation (canonical pipeline)"]   # FINAL BUILD: the top-ranked system was missing from this study
SEED_RUNS = {}                 # (system, seed index) -> {"bench": {cls: scores}, "robust": {cls: {cond: scores}}}
seed_study_df = seed_pairs_df = None
SHIFT_CANON = [c for c in ROBUST_CONDS if c.endswith("+canon") and c != "identity+canon"]


def _ours_seed_run(k):
    out = {"bench": {}, "robust": {}}
    for i, cls in enumerate([c for c in CFG.CLASSES if c in detectors_canon]):
        s = SEED + 100003 * k + 1009 * i
        random.seed(s); np.random.seed(s)
        try:
            torch.manual_seed(s)
        except Exception:
            pass
        d = PCBAnomalyDetector(cls)
        d.feat_cfg = dict(detectors_canon[cls].feat_cfg)          # the INSTALLED pipeline's exact settings
        d.fit(CANON_PATHS[cls]["train"], augs_per_image=0, verbose=False)
        d.calibrate_threshold(CANON_PATHS[cls]["calib"], FILE_LISTS[cls]["calib_labels"])
        out["bench"][cls] = np.asarray([d.score_path(p)[1] for p in CANON_PATHS[cls]["eval"]])
        out["robust"][cls] = {cond: np.asarray([d.score_path(p)[1] for p, _, _ in ROBUST_SETS[cls][cond]])
                              for cond in SHIFT_CANON if cond in ROBUST_SETS.get(cls, {})}
        del d
        gc.collect()
    return out


def _baseline_seed_run(m, k):
    out = {"bench": {}, "robust": {}}
    for cls in CFG.CLASSES:          # same training code as Cell 17; deploy=False keeps the inspection tool's models
        fnames, scores, labels, _ = run_baseline(m, cls, RAW_MODE, seed_offset=k, deploy=False)
        r = route_predictions(cls, fnames, scores, labels)
        out["bench"][cls] = r["eval_scores"]
        out["robust"][cls] = {cond: r["robust"][cond] for cond in SHIFT_CANON if cond in r["robust"]}
    return out


def _robust_verdict(verdicts):
    hi = sum(v == "significantly HIGHER" for v in verdicts)
    lo = sum(v == "significantly LOWER" for v in verdicts)
    n = len(verdicts)
    if hi == n:
        return "ours significantly HIGHER in every seed pairing"
    if lo == n:
        return "ours significantly LOWER in every seed pairing"
    if hi == 0 and lo == 0:
        return "no significant difference in any seed pairing"
    return "seed-dependent (ours significantly higher in %d, lower in %d of %d pairings)" % (hi, lo, n)


def _md(df):
    cols = list(df.columns)
    return "\n".join(["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)] +
                     ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)])


if CFG.SEED_STUDY_EXTRA_SEEDS > 0 and detectors_canon and ROBUST_SCORES:
    # ---- seed 0 = the main run, already computed ----
    SEED_RUNS[(SEED_SYSTEMS[0], 0)] = {
        "bench": {c: np.asarray(CANON_EVAL_SCORES[c]) for c in detectors_canon},
        "robust": {c: {cond: np.asarray(ROBUST_SCORES[c][cond][SEED_SYSTEMS[0]]) for cond in SHIFT_CANON
                       if SEED_SYSTEMS[0] in ROBUST_SCORES.get(c, {}).get(cond, {})} for c in detectors_canon}}
    for m in SEED_SYSTEMS[1:]:
        if all((baseline_results.get(m) or {}).get(c) for c in CFG.CLASSES):
            SEED_RUNS[(m, 0)] = {"bench": {c: baseline_results[m][c]["eval_scores"] for c in CFG.CLASSES},
                                 "robust": {c: {cond: baseline_results[m][c]["robust"][cond] for cond in SHIFT_CANON
                                                if cond in baseline_results[m][c]["robust"]} for c in CFG.CLASSES}}
    # ---- extra seeds: re-train on identical data ----
    for k in range(1, CFG.SEED_STUDY_EXTRA_SEEDS + 1):
        for sysname in SEED_SYSTEMS:
            if (sysname, 0) not in SEED_RUNS:
                continue
            print("re-training %s with extra seed %d ..." % (sysname, k))
            try:
                SEED_RUNS[(sysname, k)] = _ours_seed_run(k) if sysname.startswith("Ours") else _baseline_seed_run(sysname, k)
            except Exception as e:
                print("  FAILED (excluded from the seed study):", repr(e)[:200])

    # ---- metrics with the SAME paired resamples for every system and seed ----
    PB_B = {c: PairedBootstrap(FILE_LISTS[c]["eval_labels"], seed=SEED + 100 + i) for i, c in enumerate(CFG.CLASSES)}
    PB_R = {}
    for i, c in enumerate(CFG.CLASSES):
        for j, cond in enumerate(SHIFT_CANON):
            if cond in ROBUST_SETS.get(c, {}):
                pb = PairedBootstrap([l for _, l, _ in ROBUST_SETS[c][cond]], seed=SEED + 7000 + 31 * i + j)
                if pb.valid:                 # both classes present (a crop can drop every defect) -- as in Cell 20
                    PB_R[(c, cond)] = pb
    METRIC = {}
    for key, run in SEED_RUNS.items():
        b = [PB_B[c].auroc(run["bench"][c]) for c in CFG.CLASSES
             if c in run["bench"] and not np.isnan(np.asarray(run["bench"][c], float)).any()]
        r = [PB_R[ck].auroc(run["robust"][ck[0]][ck[1]]) for ck in PB_R
             if ck[1] in run["robust"].get(ck[0], {}) and not np.isnan(np.asarray(run["robust"][ck[0]][ck[1]], float)).any()]
        if len(b) == len(CFG.CLASSES) and len(r) == len(PB_R):          # complete runs only -> pairing stays valid
            METRIC[key] = {"bench": (float(np.mean([p for p, _ in b])), np.mean([rp for _, rp in b], axis=0)),
                           "shift": (float(np.mean([p for p, _ in r])), np.mean([rp for _, rp in r], axis=0))}
        else:
            print("incomplete run excluded:", key)
    rows = [{"system": s, "seed": k, "benchmark AUROC": fmt_ci(*METRIC[(s, k)]["bench"]),
             "6-shift avg AUROC": fmt_ci(*METRIC[(s, k)]["shift"])} for (s, k) in sorted(METRIC, key=lambda t: (SEED_SYSTEMS.index(t[0]), t[1]))]
    summ = []
    for s in SEED_SYSTEMS:
        ks = [k for (s2, k) in METRIC if s2 == s]
        if ks:
            bv = [METRIC[(s, k)]["bench"][0] for k in ks]; sv = [METRIC[(s, k)]["shift"][0] for k in ks]
            summ.append({"system": s, "seeds": len(ks), "benchmark mean +- sd": "%.3f +- %.3f" % (np.mean(bv), np.std(bv)),
                         "benchmark range": "%.3f - %.3f" % (min(bv), max(bv)),
                         "6-shift mean +- sd": "%.3f +- %.3f" % (np.mean(sv), np.std(sv)), "6-shift range": "%.3f - %.3f" % (min(sv), max(sv))})
    seed_study_df = pd.DataFrame(summ)
    pairs = []
    ours_ks = [k for (s, k) in METRIC if s == SEED_SYSTEMS[0]]
    for b in SEED_SYSTEMS[1:]:
        b_ks = [k for (s, k) in METRIC if s == b]
        for metric in ("bench", "shift"):
            res = [diff_verdict(METRIC[(SEED_SYSTEMS[0], i)][metric][1], METRIC[(b, j)][metric][1]) for i in ours_ks for j in b_ks]
            if res:
                ds = [r[0] for r in res]
                pairs.append({"comparison": "Ours (canonical pipeline) minus " + b,
                              "metric": "benchmark" if metric == "bench" else "6-shift avg",
                              "mean diff": round(float(np.mean(ds)), 4), "min diff": round(min(ds), 4), "max diff": round(max(ds), 4),
                              "seed pairings": len(res), "ours ahead in": "%d/%d" % (sum(d > 0 for d in ds), len(ds)),
                              "seed-robust verdict": _robust_verdict([r[3] for r in res])})
    seed_pairs_df = pd.DataFrame(pairs)
    print("\nSEED ROBUSTNESS -- every system re-trained with %d seed(s) in total:" % (1 + CFG.SEED_STUDY_EXTRA_SEEDS))
    print(pd.DataFrame(rows).to_string(index=False))
    print("\nAcross seeds:"); print(seed_study_df.to_string(index=False))
    print("\nOur canonical pipeline vs each top baseline, over ALL seed pairings:"); print(seed_pairs_df.to_string(index=False))
    for r in pairs:
        CONCLUSIONS.append("Seed robustness (%d training seeds per system), %s, %s: mean %+.3f (range %+.3f to %+.3f; ours ahead "
                           "in %s seed pairings) -- %s." % (1 + CFG.SEED_STUDY_EXTRA_SEEDS, r["comparison"], r["metric"], r["mean diff"],
                                                           r["min diff"], r["max diff"], r["ours ahead in"], r["seed-robust verdict"]))
    try:
        with open(SUMMARY_PATH, "a") as fh:
            fh.write("\n## Seed robustness (training randomness)\n\nEvery CI above covers which test boards were drawn; this "
                     "section re-trains the top systems with other seeds. A difference is robust only if significant in every "
                     "seed pairing.\n\n" + _md(seed_study_df) + "\n\n" + _md(seed_pairs_df) + "\n")
        print("\nAppended to", SUMMARY_PATH)
    except Exception as e:
        print("could not append to the summary:", repr(e)[:120])
else:
    print("Seed-robustness study skipped (CFG.SEED_STUDY_EXTRA_SEEDS = 0, or no canonical pipeline / robustness scores).")

## Part 3.6 -- Angle, crop & lighting sweeps

In [ ]:
# ============================================================
# CELL 20d -- ANGLE, CROP & LIGHTING SWEEPS: independent of angle and light? defects found in cropped photos?
# ============================================================
doc_section("CELL 20d", "ANGLE, CROP & LIGHTING SWEEPS: independent of angle and light? defects found in cropped photos?")
# The robustness benchmark tests four angles (0/90/180/15 degrees) and one crop (centred 80%). The thesis aims are broader:
# the system must work at ANY angle, and must find defects in a photo that shows only PART of the board. So: every
# CFG.SWEEP_ANGLES (default: every 30 degrees, 0-330) and crops keeping CFG.SWEEP_CROPS of each side (default 90/80/70/60%),
# on the same held-out boards as the robustness benchmark, with the same canonicalization and calibrated thresholds.
# Arbitrary angles are rendered at camera resolution (cubic, then downscaled; canvas padding marked unseen -- see Cell 15d);
# multiples of 90 degrees are lossless. A defective board stays in a crop setting only if >= the robustness benchmark's
# fraction of its ground-truth defect is still in the photo, so every label stays true. Thresholds and models are reused,
# never refit: nothing here is tuned.
import shutil

SWEEP = {}                 # (kind, value) -> cls -> {"labels": [...], "<system>": np.array of scores}
sweep_df = sweep_summary_df = sweep_reg_df = sweep_align_df = None
SWEEP_SYSTEMS = ["Ours (canonical pipeline)", "Ours (raw input)"]


def _label(kind, val):
    return (("%d deg" % val) if kind == "angle" else ("%d%% of each side" % round(100 * val)) if kind == "crop" else
            (val + " half") if kind == "half" else str(val))


def _render_lighting(img, name):
    """The same board under a different light (geometry unchanged, so every label stays true)."""
    x = img.astype(np.float32)
    h, w = x.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    if name == "bright x1.6":
        x = x * 1.6
    elif name == "dim x0.5":
        x = x * 0.5
    elif name == "very dim x0.3":
        x = x * 0.3
    elif name.startswith("gamma"):
        x = 255.0 * np.power(x / 255.0, float(name.split()[1]))
    elif name == "gradient left-right":
        x = x * (0.35 + 0.95 * xx / max(1.0, w - 1.0))[..., None]
    elif name == "gradient top-bottom":
        x = x * (1.3 - 0.95 * yy / max(1.0, h - 1.0))[..., None]
    elif name == "vignette":
        r2 = ((xx - w / 2.0) / (w / 2.0)) ** 2 + ((yy - h / 2.0) / (h / 2.0)) ** 2
        x = x * np.clip(1.0 - 0.45 * r2, 0.3, 1.0)[..., None]
    elif name == "warm light":
        x = x * np.array([0.78, 0.95, 1.2], np.float32)        # BGR: less blue, more red
    elif name == "cool light":
        x = x * np.array([1.2, 1.0, 0.8], np.float32)
    elif name == "low contrast":
        x = 128.0 + (x - 128.0) * 0.5
    elif name == "shadow band":
        d = (xx - 0.6 * yy) / float(max(h, w))
        x = x * (1.0 - 0.6 * np.exp(-((d - 0.15) / 0.12) ** 2))[..., None]
    elif name == "glare spot":
        x = x + 170.0 * np.exp(-(((xx - 0.65 * w) / (0.12 * w)) ** 2 + ((yy - 0.35 * h) / (0.12 * h)) ** 2))[..., None]
    elif name != "normal":
        raise ValueError("unknown lighting condition: " + name)
    return np.clip(x, 0, 255).astype(np.uint8)


def _half_box(shape, which):
    H, W = shape[:2]
    return {"left": (0, 0, H, W // 2), "right": (0, W - W // 2, H, W // 2),
            "top": (0, 0, H // 2, W), "bottom": (H - H // 2, 0, H // 2, W)}[which]


def _defect_inside(path, box, small_shape):
    """Fraction of the ground-truth defect inside a box given in small-image coordinates."""
    mp = find_defect_mask(path)
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE) if mp else None
    if m is None or not (m > 0).any():
        return 0.0
    sy, sx = m.shape[0] / float(small_shape[0]), m.shape[1] / float(small_shape[1])
    y0, x0, h, w = box
    inside = (m[int(y0 * sy):int((y0 + h) * sy), int(x0 * sx):int((x0 + w) * sx)] > 0).sum()
    return float(inside) / float((m > 0).sum())


def _placement_err(M0, M1, box):
    """Max distance (px, aligned frame) between where the crop's corners land when the PARTIAL photo is aligned and where
    the same board points land when the FULL photo is aligned -- 'registered' alone does not prove the right place."""
    if M0 is None or M1 is None:
        return float("inf")
    y0, x0, ch, cw = box
    pts = np.array([[x0, y0], [x0 + cw - 1, y0], [x0, y0 + ch - 1], [x0 + cw - 1, y0 + ch - 1]], np.float64)
    a = pts @ np.asarray(M0)[:, :2].T + np.asarray(M0)[:, 2]
    b = (pts - [x0, y0]) @ np.asarray(M1)[:, :2].T + np.asarray(M1)[:, 2]
    return float(np.abs(a - b).max())


def _render_rotation(full, small, theta):
    """(image, validity mask or None) for a rotation by theta degrees counter-clockwise."""
    theta = theta % 360
    if theta == 0:
        return small, None
    lossless = {90: cv2.ROTATE_90_COUNTERCLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_CLOCKWISE}
    if theta in lossless:
        return cv2.rotate(small, lossless[theta]), None
    f = small.shape[1] / float(full.shape[1])
    rot = rotate_bound(full, float(theta), interpolation=cv2.INTER_CUBIC)
    real = rotate_bound(np.full(full.shape[:2], 255, np.uint8), float(theta), border=cv2.BORDER_CONSTANT,
                        interpolation=cv2.INTER_NEAREST)
    if f < 0.999:
        rot = cv2.resize(rot, None, fx=f, fy=f, interpolation=cv2.INTER_AREA)
        real = cv2.resize(real, (rot.shape[1], rot.shape[0]), interpolation=cv2.INTER_AREA)
    return rot, real > 250


def _baseline_scores(m, cls, items):
    """Scores from the KEPT (not re-trained) Anomalib model for many canonical images in ONE predict call."""
    d = DEPLOY_MODELS[(m, cls)]
    root = os.path.join(CFG.ANOMALIB_ROOT, "_sweep", cls, re.sub(r"[^A-Za-z0-9]+", "_", m))
    if os.path.isdir(root):
        shutil.rmtree(root)
    for sub in ("train/good", "test/good", "test/bad"):
        os.makedirs(os.path.join(root, sub), exist_ok=True)
    tp = FILE_LISTS[cls]["train_normal"][0]
    _link(tp, os.path.join(root, "train", "good", os.path.basename(tp)))
    for k, (p, l) in enumerate(items):
        _link(p, os.path.join(root, "test", "bad" if l else "good", "%06d__%s" % (k, os.path.basename(p))))
    preds = deploy_predict(m, cls, folder_datamodule(cls, root, m))
    fnames, scores, _ = extract_anomalib_predictions(preds)
    ms = board_masked_scores(preds, cls)                    # same inspected region as every other aligned score
    scores = [ms.get(os.path.basename(str(f)), s) for f, s in zip(fnames, scores)]
    out = np.full(len(items), np.nan)
    for fn, s in zip(fnames, scores):
        mm = re.match(r"^(\d{6})__", os.path.basename(str(fn)))
        if mm:
            out[int(mm.group(1))] = float(s)
    return out


if CFG.RUN_SWEEPS and detectors_canon and ROBUST_SETS:
    import re
    SWEEP_SYSTEMS += [m for m in CFG.DEPLOY_BASELINES if m.endswith("(canonical pipeline)")
                      and all((m, c) in DEPLOY_MODELS for c in detectors_canon)]
    settings = ([("angle", a) for a in CFG.SWEEP_ANGLES] + [("crop", c) for c in CFG.SWEEP_CROPS] +
                [("half", h) for h in CFG.SWEEP_HALVES] + [("light", l) for l in CFG.SWEEP_LIGHTS])
    reg_rows = []
    for ci, cls in enumerate([c for c in CFG.CLASSES if c in detectors_canon and c in ROBUST_SETS]):
        ref = load_canon_reference(cls, CFG.ROBUSTNESS_LONG_SIDE)
        orig = {(int(l), os.path.splitext(os.path.basename(p))[0]): p
                for p, l in zip(FILE_LISTS[cls]["eval_images"], FILE_LISTS[cls]["eval_labels"])}
        boards = []
        for lab in (0, 1):                                   # balanced subset of the robustness benchmark's boards
            boards += [(orig[(int(l), s)], int(l), s) for _, l, s in ROBUST_SETS[cls]["identity"]
                       if int(l) == lab and (int(l), s) in orig][:CFG.SWEEP_IMAGES_PER_CLASS // 2]
        dc, dr = detectors_canon[cls], detectors[cls]
        files = {}                                           # (kind, value) -> [(canonical png, label)]
        for path, lab, stem in tqdm(boards, desc="[" + cls + "] angle & crop sweep", leave=False):
            full = cv2.imread(path, cv2.IMREAD_COLOR)
            if full is None:
                continue
            small = downscale_long_side(full, CFG.ROBUSTNESS_LONG_SIDE)
            info0 = canonicalize(small, ref)[1]
            ang0 = info0["angle_deg"]
            for kind, val in settings:
                if kind == "angle":
                    pert, valid = _render_rotation(full, small, val)
                elif kind == "crop":
                    if lab == 1 and not defect_survives_crop(path, val, CFG.ROBUSTNESS_CROP_MIN_DEFECT_FRAC)[0]:
                        continue
                    y0, x0, ch, cw = center_crop_box(small.shape[0], small.shape[1], val)
                    pert, valid = small[y0:y0 + ch, x0:x0 + cw], None
                elif kind == "light":                        # the whole board, under a different light
                    pert, valid = _render_lighting(small, val), None
                else:                                        # half of the board in the photo
                    y0, x0, ch, cw = _half_box(small.shape, val)
                    if lab == 1 and _defect_inside(path, (y0, x0, ch, cw), small.shape) < CFG.ROBUSTNESS_CROP_MIN_DEFECT_FRAC:
                        continue
                    pert, valid = small[y0:y0 + ch, x0:x0 + cw], None
                canon, info = canonicalize(pert, ref, input_valid=valid)
                q = os.path.join(CFG.ROBUSTNESS_DIR, "sweep", cls, "%s_%s" % (kind, val), "bad" if lab else "good", stem + ".png")
                os.makedirs(os.path.dirname(q), exist_ok=True)
                cv2.imwrite(q, canon)
                save_seen(q, info)
                files.setdefault((kind, val), []).append((q, lab))
                e = SWEEP.setdefault((kind, val), {}).setdefault(cls, {"labels": []})
                e["labels"].append(lab)
                e.setdefault(SWEEP_SYSTEMS[0], []).append(dc.score_image_tensor(
                    load_image_from_array(cv2.cvtColor(canon, cv2.COLOR_BGR2RGB), **dc.prep()), seen=info.get("valid_mask"))[1])
                e.setdefault(SWEEP_SYSTEMS[1], []).append(dr.score_image_tensor(
                    load_image_from_array(cv2.cvtColor(pert, cv2.COLOR_BGR2RGB)))[1])
                err = (abs(((info["angle_deg"] - ang0 - val) + 180.0) % 360.0 - 180.0) if info["registered"] else float("nan")) if kind == "angle" else float("nan")
                place = (_placement_err(info0.get("M"), info.get("M"), (y0, x0, ch, cw)) if info["registered"] and info0.get("registered")
                         else float("inf")) if kind in ("crop", "half") else float("nan")
                reg_rows.append({"pcb_class": cls, "setting": _label(kind, val), "stem": stem, "label": lab,
                                 "registered": bool(info["registered"]), "angle_error_deg": err, "placement_error_px": place,
                                 "inliers": info.get("inliers", float("nan")), "align_score": info.get("ecc_cc", float("nan")),
                                 "retried": bool(info.get("retry"))})
        for m in SWEEP_SYSTEMS[2:]:                          # kept baseline models: one predict call per class
            items = [it for k in settings for it in files.get(k, [])]
            try:
                sc = _baseline_scores(m, cls, items) if items else np.array([])
            except Exception as e:                   # one baseline failing must not end the run: it drops out of the sweep
                print("  sweep: %s failed on %s (%r) -- excluded" % (m, cls, e))
                sc = np.full(len(items), np.nan)
            pos = 0
            for k in settings:
                n = len(files.get(k, []))
                SWEEP[k][cls][m] = sc[pos:pos + n]
                pos += n
        for junk in (os.path.join(CFG.ROBUSTNESS_DIR, "sweep", cls), os.path.join(CFG.ANOMALIB_ROOT, "_sweep", cls),
                     os.path.join(CFG.SEEN_DIR, os.path.abspath(os.path.join(CFG.ROBUSTNESS_DIR, "sweep", cls)).lstrip(os.sep))):
            shutil.rmtree(junk, ignore_errors=True)          # disk: this class's sweep images are scored and no longer needed

    # ---- metrics: per class AUROC with paired bootstrap, averaged over classes ----
    THR = lambda sysname, cls: (detectors_canon[cls].threshold if sysname == SWEEP_SYSTEMS[0] else
                                detectors[cls].threshold if sysname == SWEEP_SYSTEMS[1] else baseline_results[sysname][cls]["threshold"])
    rows, REPS = [], {}
    for si, (kind, val) in enumerate(settings):
        for sysname in SWEEP_SYSTEMS:
            pts, reps, fp, nn_, tp, np_ = [], [], 0, 0, 0, 0
            for ci, cls in enumerate(CFG.CLASSES):
                e = SWEEP.get((kind, val), {}).get(cls)
                if not e or sysname not in e or len(e[sysname]) != len(e["labels"]):
                    continue
                y, s = np.asarray(e["labels"]), np.asarray(e[sysname], float)
                pb = PairedBootstrap(y, seed=SEED + 9000 + 37 * ci + si)
                if pb.valid and not np.isnan(s).any():
                    pt, rp = pb.auroc(s); pts.append(pt); reps.append(rp)
                    flag = s > THR(sysname, cls)
                    fp += int(flag[y == 0].sum()); nn_ += int((y == 0).sum()); tp += int(flag[y == 1].sum()); np_ += int((y == 1).sum())
            if pts:
                REPS[(kind, val, sysname)] = (float(np.mean(pts)), np.mean(reps, axis=0))
                rows.append({"sweep": kind, "setting": _label(kind, val),
                             "system": sysname, "AUROC [95% CI]": fmt_ci(*REPS[(kind, val, sysname)]),
                             "false alarms": round(fp / max(1, nn_), 3), "defects caught (recall)": round(tp / max(1, np_), 3)})
    sweep_df = pd.DataFrame(rows)
    reg = pd.DataFrame(reg_rows)
    sweep_align_df = reg.copy()
    # Could a DEPLOYED system tell that a partial photo was aligned to the WRONG place? (Placement is checked here against
    # the same board's full photo, which a deployed system does not have.) Report how well its own signals separate them.
    part = reg[reg["setting"].str.contains("half|each side") & reg["registered"]]
    if len(part) and part["placement_error_px"].gt(10).any() and part["placement_error_px"].le(10).any():
        bad = part["placement_error_px"].gt(10).astype(int).values
        for sig in ("inliers", "align_score"):
            v = part[sig].astype(float).values
            if np.isfinite(v).all():
                print("wrong placements among aligned partial photos: %d of %d | %s separates them with AUROC %.3f "
                      "(low %s -> wrong place)" % (bad.sum(), len(bad), sig, roc_auc_score(bad, -v), sig))
    sweep_reg_df = reg.groupby("setting", sort=False).agg(boards=("registered", "size"), registered_pct=("registered", lambda x: round(100 * x.mean(), 1)),
                                            median_angle_error_deg=("angle_error_deg", "median"),
                                            worst_angle_error_deg=("angle_error_deg", "max"),
                                            correctly_placed_pct=("placement_error_px", lambda x: round(100 * float((x <= 10.0).mean()), 1) if x.notna().any() else float("nan")),
                                            median_placement_error_px=("placement_error_px", lambda x: float(np.median(x[np.isfinite(x)])) if np.isfinite(x).any() else float("nan"))).reset_index()

    # ---- summaries: angle independence (vs 0 degrees) and crops ----
    summ = []
    for sysname in SWEEP_SYSTEMS:
        ang = [(a, REPS[("angle", a, sysname)]) for a in CFG.SWEEP_ANGLES if ("angle", a, sysname) in REPS]
        if not ang or ("angle", 0, sysname) not in REPS:
            continue
        base = REPS[("angle", 0, sysname)]
        drops = [(a, diff_verdict(r[1], base[1])) for a, r in ang if a != 0]
        sig = [a for a, d in drops if d[3] == "significantly LOWER"]
        crops = {c: REPS[("crop", c, sysname)][0] for c in CFG.SWEEP_CROPS if ("crop", c, sysname) in REPS}
        halves = {h: REPS[("half", h, sysname)][0] for h in CFG.SWEEP_HALVES if ("half", h, sysname) in REPS}
        lights = [(l, REPS[("light", l, sysname)]) for l in CFG.SWEEP_LIGHTS if ("light", l, sysname) in REPS]
        summ.append({"system": sysname, "AUROC at 0 deg": round(base[0], 3),
                     "AUROC range over all angles": "%.3f - %.3f" % (min(r[0] for _, r in ang), max(r[0] for _, r in ang)),
                     "angles significantly worse than 0 deg": ", ".join(str(a) for a in sig) or "none",
                     "mean AUROC over all angles": round(float(np.mean([r[0] for _, r in ang])), 3),
                     **{"crop %d%%" % round(100 * c): round(v, 3) for c, v in crops.items()},
                     **{"%s half" % h: round(v, 3) for h, v in halves.items()},
                     **({"AUROC range over lighting": "%.3f - %.3f" % (min(r[0] for _, r in lights), max(r[0] for _, r in lights)),
                         "lighting conditions significantly worse than normal light": ", ".join(l for l, r in lights
                             if diff_verdict(r[1], base[1])[3] == "significantly LOWER") or "none",
                         "mean AUROC over lighting": round(float(np.mean([r[0] for _, r in lights])), 3)} if lights else {})})
    sweep_summary_df = pd.DataFrame(summ)
    print("ANGLE, CROP & LIGHTING SWEEPS -- registration accuracy per setting:"); print(sweep_reg_df.to_string(index=False))
    print("\nSummary:"); print(sweep_summary_df.to_string(index=False))
    save_table(sweep_summary_df, "sweep_summary", "Angle, crop and lighting sweeps: AUROC summary per system")
    save_table(sweep_reg_df, "sweep_registration", "Alignment accuracy per angle / crop / lighting setting")
    save_table(sweep_df, "sweep_all_settings", "AUROC, false alarms and recall for every system at every sweep setting")

    # ---- paired comparisons: ours vs each other system, averaged over all angles and over all crops ----
    ours = SWEEP_SYSTEMS[0]
    for other in SWEEP_SYSTEMS[1:]:
        for kind, vals in (("angle", CFG.SWEEP_ANGLES), ("crop", CFG.SWEEP_CROPS), ("half", CFG.SWEEP_HALVES), ("light", CFG.SWEEP_LIGHTS)):
            ks = [v for v in vals if (kind, v, ours) in REPS and (kind, v, other) in REPS]
            if ks:
                a = np.mean([REPS[(kind, v, ours)][1] for v in ks], axis=0); b = np.mean([REPS[(kind, v, other)][1] for v in ks], axis=0)
                d, lo, hi, v = diff_verdict(a, b)
                CONCLUSIONS.append("%s sweep (%d settings), %s minus %s: %+.3f [%.3f, %.3f] (%s)." % (
                    {"angle": "Angle", "crop": "Crop", "half": "Half-board", "light": "Lighting"}[kind], len(ks), ours, other, d, lo, hi, v))
    for r in summ:
        CONCLUSIONS.append("Angle sweep, %s: AUROC %s over %d angles (0 deg: %.3f); angles significantly worse than 0 deg: %s." % (
            r["system"], r["AUROC range over all angles"], len(CFG.SWEEP_ANGLES), r["AUROC at 0 deg"], r["angles significantly worse than 0 deg"]))
        if "AUROC range over lighting" in r:
            CONCLUSIONS.append("Lighting sweep, %s: AUROC %s over %d lighting conditions (normal light: %.3f); conditions "
                               "significantly worse than normal light: %s." % (r["system"], r["AUROC range over lighting"],
                               len(CFG.SWEEP_LIGHTS), r["AUROC at 0 deg"], r["lighting conditions significantly worse than normal light"]))

    # ---- figure: AUROC vs angle and vs crop ----
    fig, ax = plt.subplots(1, 3, figsize=(22, 5.2), gridspec_kw={"width_ratios": [1.3, 0.8, 1.5]})
    for sysname in SWEEP_SYSTEMS:
        lw = 3.0 if sysname == SWEEP_SYSTEMS[0] else 1.4
        for k, (kind, vals) in enumerate((("angle", CFG.SWEEP_ANGLES), ("crop", CFG.SWEEP_CROPS), ("light", ["normal"] + list(CFG.SWEEP_LIGHTS)))):
            key = lambda v: ("angle", 0, sysname) if (kind == "light" and v == "normal") else (kind, v, sysname)
            pts = [(i2, v, REPS[key(v)]) for i2, v in enumerate(vals) if key(v) in REPS]
            if pts:
                xs = [v if kind == "angle" else (100 * v if kind == "crop" else i2) for i2, v, _ in pts]
                ax[k].plot(xs, [r[0] for _, _, r in pts], marker="o", lw=lw, label=sysname)
                ax[k].fill_between(xs, [ci95(r[1])[0] for _, _, r in pts], [ci95(r[1])[1] for _, _, r in pts], alpha=0.10)
    ax[0].set_xlabel("rotation (degrees, counter-clockwise)"); ax[0].set_xticks(CFG.SWEEP_ANGLES); ax[0].set_title("AUROC vs rotation angle")
    ax[1].set_xlabel("% of each side kept in the photo"); ax[1].invert_xaxis(); ax[1].set_title("AUROC vs crop")
    ax[2].set_xticks(range(1 + len(CFG.SWEEP_LIGHTS))); ax[2].set_xticklabels(["normal"] + list(CFG.SWEEP_LIGHTS), rotation=55, ha="right", fontsize=8)
    ax[2].set_title("AUROC vs lighting condition")
    for a in ax:
        a.set_ylabel("AUROC (95% CI)"); a.axhline(0.5, ls="--", c="gray"); a.set_ylim(0.3, 1.0); a.legend(fontsize=7)
    caption("Held-out AUROC of every aligned system and our raw-input model at every rotation angle (left), crop size "
            "(middle) and lighting condition (right); thick line: our final system. Shaded: paired-bootstrap 95% CIs.")
    plt.tight_layout(); plt.show()
    try:
        with open(SUMMARY_PATH, "a") as fh:
            fh.write("\n## Angle, crop & lighting sweeps\n\n" + _md(sweep_summary_df) + "\n\nRegistration accuracy per angle:\n\n" + _md(sweep_reg_df) +
                     "\n\n" + "\n".join("- " + c for c in CONCLUSIONS if c.startswith(("Angle sweep", "Crop sweep", "Half-board sweep", "Lighting sweep"))) + "\n")
    except Exception as e:
        print("could not append to the summary:", repr(e)[:120])
else:
    print("Angle & crop sweeps skipped (CFG.RUN_SWEEPS off, or no canonical pipeline / robustness sets).")

## Part 3.7 -- Pixel-level inspection engine & defect localization metrics

In [ ]:
# ============================================================
# CELL 20e -- PIXEL-LEVEL INSPECTION ENGINE & LOCALIZATION METRICS: WHERE exactly is the defect?
# ============================================================
doc_section("CELL 20e", "PIXEL-LEVEL INSPECTION ENGINE & LOCALIZATION METRICS: WHERE exactly is the defect?")
# Image-level AUROC says WHETHER a board is defective; this cell measures WHERE. Every system's anomaly map is brought into
# the aligned frame, restricted to the inspected region (the board AND what the photo showed), and compared pixel by pixel
# with VisA's ground-truth defect masks (aligned with each photo's own transform).
# PIXEL THRESHOLD (calibration data only): a pixel is HIGHLIGHTED when it exceeds the strongest pixel of
# CFG.PIXEL_NORMAL_QUANTILE (90%) of CALIBRATION normal boards -- so about 10% of normal boards show any highlight, matching
# the 10% false-alarm design of the image-level threshold.
# Every detection is also mapped back to PIXEL COORDINATES OF THE ORIGINAL PHOTO through the inverse alignment transform.
PIX_THR = {}
pixel_metrics_df = None
OURS_SYS = "Ours (canonical pipeline)"
PIX_SYSTEMS = [OURS_SYS] + [m for m in CFG.DEPLOY_BASELINES if m.endswith("(canonical pipeline)")
                            and all((m, c) in DEPLOY_MODELS for c in detectors_canon)]
_is_aligned_png = lambda p: os.path.realpath(p).startswith(os.path.realpath(CFG.CANON_PIPELINE_DIR))


def _model_map_to_frame(amap, H, W, letterbox):
    """Our anomaly map (CFG.IMG_SIZE square model input) -> the aligned frame (H x W)."""
    amap = np.asarray(amap, np.float32)
    if letterbox:
        side = max(H, W)
        s = amap.shape[0] / float(side)
        top, left = (side - H) // 2, (side - W) // 2
        amap = amap[int(round(top * s)):int(round((top + H) * s)), int(round(left * s)):int(round((left + W) * s))]
    return cv2.resize(amap, (W, H), interpolation=cv2.INTER_LINEAR)


def ours_map(cls, canon_bgr, seen):
    """(anomaly map in the aligned frame, image score) of our final model for an aligned image."""
    d = detectors_canon[cls]
    t = load_image_from_array(cv2.cvtColor(canon_bgr, cv2.COLOR_BGR2RGB), **d.prep())
    amap, score = d.score_image_tensor(t, seen=seen)
    return _model_map_to_frame(amap, canon_bgr.shape[0], canon_bgr.shape[1], d.prep()["letterbox"]), float(score)


def baseline_maps(m, cls, items):
    """[(anomaly map in the aligned frame, inspected-region score)] of a KEPT baseline for aligned PNGs -- one predict call."""
    d = DEPLOY_MODELS[(m, cls)]
    root = os.path.join(CFG.ANOMALIB_ROOT, "_pixel", cls, _slug(m))
    if os.path.isdir(root):
        shutil.rmtree(root)
    for sub in ("train/good", "test/good", "test/bad"):
        os.makedirs(os.path.join(root, sub), exist_ok=True)
    tp = FILE_LISTS[cls]["train_normal"][0]
    _link(tp, os.path.join(root, "train", "good", os.path.basename(tp)))
    for k, (p, l) in enumerate(items):
        _link(p, os.path.join(root, "test", "bad" if l else "good", "%06d__%s" % (k, os.path.basename(p))))
    for sub in ("good", "bad"):                              # Folder wants both test folders non-empty
        if not os.listdir(os.path.join(root, "test", sub)):
            _link(items[0][0], os.path.join(root, "test", sub, "dup__" + os.path.basename(items[0][0])))
    maps = [None] * len(items)
    for batch in deploy_predict(m, cls, folder_datamodule(cls, root, m)):
        get = (lambda k: batch.get(k)) if isinstance(batch, dict) else (lambda k: getattr(batch, k, None))
        paths, mp = get("image_path"), _np(get("anomaly_map"))
        if paths is None or mp is None:
            continue
        for i, p in enumerate(list(paths)):
            mm = re.match(r"^(\d{6})__", os.path.basename(str(p)))
            if mm:
                a = mp[i]
                while a.ndim > 2:
                    a = a[0]
                maps[int(mm.group(1))] = a
    out = []
    for (p, _), a in zip(items, maps):
        img = cv2.imread(p, cv2.IMREAD_COLOR)
        if a is None or img is None:
            out.append((None, float("nan")))
            continue
        region = inspected_region(cls, read_seen(p))
        score = masked_map_score(a, region) if region is not None else float(a.max())
        out.append((cv2.resize(a.astype(np.float32), (img.shape[1], img.shape[0]), interpolation=cv2.INTER_LINEAR), score))
    return out


def identify_board(full_bgr, pcb_class=None, input_valid=None):
    """Board type = the class whose golden reference the photo registers to best. Returns (cls, small, aligned, info, tried)."""
    small = downscale_long_side(full_bgr, CFG.ROBUSTNESS_LONG_SIDE)
    v = None if input_valid is None else cv2.resize(np.asarray(input_valid, np.uint8), (small.shape[1], small.shape[0]),
                                                    interpolation=cv2.INTER_NEAREST) > 0
    best, tried = None, {}
    for c in ([pcb_class] if pcb_class else [c for c in CFG.CLASSES if c in detectors_canon]):
        canon, info = canonicalize(small, load_canon_reference(c, CFG.ROBUSTNESS_LONG_SIDE), input_valid=v)
        tried[c] = int(info["inliers"])
        if best is None or info["inliers"] > best[2]["inliers"]:
            best = (c, canon, info)
    return best[0], small, best[1], best[2], tried


def inspect_board(image, pcb_class=None, system=OURS_SYS, input_valid=None, classify_types=False):
    """Full pixel-level inspection of one photo (path or BGR array) with one system. Returns a dict (see print_inspection).
    classify_types=True: every highlighted region also gets its defect type (Part 1.4), typed on the ORIGINAL photo with the
    region's original-photo box -- where the classifier learned (the aligned frame can turn a board by ~180 deg)."""
    full = cv2.imread(image, cv2.IMREAD_COLOR) if isinstance(image, str) else image
    if full is None:
        raise FileNotFoundError(str(image))
    cls, small, canon, info, tried = identify_board(full, pcb_class, input_valid)
    f = small.shape[1] / float(full.shape[1])
    H, W = canon.shape[:2]
    seen = info.get("valid_mask")
    region = inspected_region(cls, seen)
    region = np.ones((H, W), bool) if region is None else region
    if system == OURS_SYS:
        amap, score = ours_map(cls, canon, seen)
        d = detectors_canon[cls]; thr = float(d.threshold); cs, cl = d.score_stats["calib_scores"], d.score_stats["calib_labels"]
    else:
        tmp = os.path.join(CFG.ANOMALIB_ROOT, "_inspect", _slug(system), "query.png")
        os.makedirs(os.path.dirname(tmp), exist_ok=True)
        cv2.imwrite(tmp, canon); save_seen(tmp, info)
        amap, score = baseline_maps(system, cls, [(tmp, 0)])[0]
        r = baseline_results[system][cls]; thr = float(r["threshold"]); cs, cl = r["calib_scores"], r["calib_labels"]
    pac = float(pac_threshold(cs, cl))
    pthr = PIX_THR.get((system, cls))
    hot = ((amap > pthr) & region) if pthr is not None else np.zeros((H, W), bool)
    Minv = cv2.invertAffineTransform(np.asarray(info["M"], np.float64)) if info.get("M") is not None else None

    def to_orig(x, y):                                       # aligned frame -> original photo pixels
        if Minv is not None:
            x, y = Minv[0, 0] * x + Minv[0, 1] * y + Minv[0, 2], Minv[1, 0] * x + Minv[1, 1] * y + Minv[1, 2]
        return int(round(x / f)), int(round(y / f))

    inside = np.where(region, amap, -np.inf)
    py, px = np.unravel_index(int(np.argmax(inside)), amap.shape)
    verdict = bool(score > thr)
    regions = []
    if verdict:
        floor = float(amap[region].min()) if region.any() else float(amap.min())
        for k, g in enumerate(analyze_anomaly_regions(np.where(region, amap, floor), top_k=CFG.MAX_REGIONS), 1):
            x, y, w, h = [int(v) for v in g["bbox"]]
            box = np.zeros((H, W), bool); box[max(0, y):y + h, max(0, x):x + w] = True
            box &= region
            vals = amap[box] if box.any() else np.array([float(g["mean_score"])])
            by, bx = np.unravel_index(int(np.argmax(np.where(box, amap, -np.inf))), amap.shape)
            corners = [to_orig(cx, cy) for cx, cy in [(x, y), (x + w, y), (x, y + h), (x + w, y + h)]]
            hb = hot & box                                    # the region's pixels above the calibrated pixel threshold
            if hb.any():                                      # their tight box, in original-photo pixels (Part 1.4 typing)
                ys_, xs_ = np.nonzero(hb)
                tc = [to_orig(cx, cy) for cx, cy in [(xs_.min(), ys_.min()), (xs_.max() + 1, ys_.min()),
                                                     (xs_.min(), ys_.max() + 1), (xs_.max() + 1, ys_.max() + 1)]]
                tight = (min(c[0] for c in tc), min(c[1] for c in tc), max(c[0] for c in tc), max(c[1] for c in tc))
            else:
                tight = None
            regions.append({"region": k, "priority": g.get("priority", "?"), "defect-shape hint": g.get("shape_hint", "?"),
                            "bbox in original photo (x0,y0,x1,y1)": (min(c[0] for c in corners), min(c[1] for c in corners),
                                                                    max(c[0] for c in corners), max(c[1] for c in corners)),
                            "bbox highlighted in original photo (x0,y0,x1,y1)": tight,
                            "centre in original photo (x,y)": to_orig(*g["center"]),
                            "peak pixel in original photo (x,y)": to_orig(bx, by),
                            "peak anomaly": round(float(vals.max()), 3), "mean anomaly": round(float(vals.mean()), 3),
                            "highlighted pixels in region": int((hot & box).sum()),
                            "region area (% of inspected board)": round(100.0 * box.sum() / max(1, region.sum()), 3),
                            "bbox aligned (x,y,w,h)": (x, y, w, h)})
    # FINAL BUILD bug fix: priorities came from a shape-only heuristic that leans towards CRITICAL, so a region in which
    # NO pixel exceeds the calibrated pixel threshold was still reported as CRITICAL (2 demo boards in the real run).
    for r in regions:
        if r["highlighted pixels in region"] == 0:
            r["priority"] = "SUSPECT"
            r["defect-shape hint"] = "diffuse evidence, no pixel above the calibrated pixel threshold -- " + str(r["defect-shape hint"])
    if classify_types and regions and "type_regions" in globals():
        # FIXED: typed on the ORIGINAL photo (the classifier's training domain), not on the aligned copy -- aligning turns
        # many PCB1 photos by ~180 deg, and a real run typed PCB1 regions right only 49% of the time on the aligned copy
        # the box read is DEFECT_TYPE_BOX: the whole region box, or the tight box around its highlighted pixels (Cell 20e2
        # selects it on calibration defects)
        type_regions(regions, full, cls, typing_box)
    return {"system": system, "pcb_class": cls, "registration_tried": tried, "info": info, "full": full, "small": small,
            "canon": canon, "amap": amap, "region": region, "hot": hot, "score": float(score), "threshold": thr, "pac_threshold": pac,
            "verdict": verdict, "verdict_pac": bool(score > pac), "pixel_threshold": pthr, "regions": regions, "to_orig": to_orig,
            "max_pixel": {"value": round(float(amap[py, px]), 3), "aligned (x,y)": (int(px), int(py)), "original photo (x,y)": to_orig(px, py)},
            "inspected % of board": round(100.0 * float((board_mask_for(cls) & region).sum()) / max(1, int(board_mask_for(cls).sum())), 1)}


DECISION_TEXT = {"DEFECTIVE": "DEFECTIVE -- auto-reject (score above the strict PAC threshold)",
                 "REVIEW": "REVIEW -- send to a human check (above the normal threshold, below the strict one)",
                 "PASS": "PASS -- accept (below both calibrated thresholds)"}


def three_level_decision(res):
    """FINAL BUILD -- the deployment policy of the thesis: PASS / REVIEW / DEFECTIVE from the two calibrated thresholds."""
    if res.get("verdict_pac"):
        return "DEFECTIVE"
    return "REVIEW" if res.get("verdict") else "PASS"


def print_inspection(res, truth=None, gt=None, true_type=None):
    i = res["info"]
    print("-" * 110)
    print("SYSTEM: %s | BOARD TYPE: %s (keypoint inliers per class: %s)" % (res["system"], res["pcb_class"], res["registration_tried"]))
    print("ALIGNMENT: %s | rotation undone %.2f deg | scale %.3f | %d inliers | lighting corrected: %s | retry: %s" % (
        "OK" if i["registered"] else "FAILED", i["angle_deg"], i["scale"], i["inliers"], i["photometric"], i.get("retry", "-")))
    print("INSPECTED: %.1f%% of the board (background surface ignored)" % res["inspected % of board"])
    print("IMAGE SCORE: %.4f | empirical threshold %.4f -> %s | PAC threshold %.4f -> %s%s" % (
        res["score"], res["threshold"], "DEFECTIVE" if res["verdict"] else "PASS", res["pac_threshold"],
        "DEFECTIVE" if res["verdict_pac"] else "PASS",
        "" if truth is None else " | ground truth: %s" % ("DEFECTIVE" if truth else "NORMAL")))
    print("DECISION: " + DECISION_TEXT[three_level_decision(res)])
    print("MOST ANOMALOUS PIXEL: value %.3f at original-photo pixel %s (aligned %s) | highlighted pixels: %d (pixel threshold %s)" % (
        res["max_pixel"]["value"], res["max_pixel"]["original photo (x,y)"], res["max_pixel"]["aligned (x,y)"], int(res["hot"].sum()),
        "%.3f" % res["pixel_threshold"] if res["pixel_threshold"] is not None else "n/a"))
    if gt is not None and gt.any():
        g = gt & res["region"]; inter = int((res["hot"] & g).sum()); union = int((res["hot"] | g).sum())
        print("GROUND TRUTH: defect %d px | highlighted inside defect %d px (%.0f%% of the defect) | IoU %.3f" % (
            int(g.sum()), inter, 100.0 * inter / max(1, int(g.sum())), inter / max(1, union)))
    if res["regions"]:
        print(pd.DataFrame(res["regions"]).drop(columns=["bbox aligned (x,y,w,h)"]).to_string(index=False))
    # FINAL BUILD -- readable defect report: one line per region (original-photo pixels)
    if not res["regions"]:
        print("DEFECT REPORT: no defect region (board %s)" % ("passed" if not res["verdict"] else "flagged, but no region inside the inspected board"))
    else:
        print("DEFECT REPORT: %d region%s, most anomalous first (coordinates in ORIGINAL photo pixels; anomaly in sd above normal):" % (
            len(res["regions"]), "" if len(res["regions"]) == 1 else "s"))
        for r in res["regions"]:
            typ = region_type_text(r, alternatives=True) if (r.get("defect_type") and "region_type_text" in globals()) else "not classified"
            print("  #%s %-8s | type: %s | shape hint: %s" % (r.get("region"), r.get("priority"), typ, r.get("defect-shape hint")))
            print("      box %s | centre %s | peak pixel %s | peak %.2f, mean %.2f | %s highlighted px | %.1f%% of the inspected board" % (
                r.get("bbox in original photo (x0,y0,x1,y1)"), r.get("centre in original photo (x,y)"), r.get("peak pixel in original photo (x,y)"),
                float(r.get("peak anomaly", float("nan"))), float(r.get("mean anomaly", float("nan"))), r.get("highlighted pixels in region"),
                float(r.get("region area (% of inspected board)", float("nan")))))
    if true_type:                                                 # a VisA test image: compare with its label
        top = res["regions"][0].get("defect_type") if res["regions"] else None
        if not res["verdict"]:
            outcome = "board PASSED (missed defect) -- no region to type"
        elif not res["regions"]:
            outcome = "flagged, but no region inside the inspected board -- nothing to type"
        else:
            outcome = "top region typed: %s -> %s" % (top or "-", "CORRECT" if top in true_type else
                                                      ("WRONG" if top else "no type (no classifier for this class)"))
        print("TRUE DEFECT TYPE (VisA label): %s | %s" % ("+".join(true_type), outcome))


def plot_inspection(res, gt=None, title=""):
    rgb = lambda x: cv2.cvtColor(x, cv2.COLOR_BGR2RGB)
    canon, amap, region, hot = res["canon"], res["amap"], res["region"], res["hot"]
    n = 5 if gt is not None else 4
    fig, ax = plt.subplots(1, n, figsize=(5.2 * n, 4.6))
    orig = rgb(res["full"]).copy()
    lw = max(2, orig.shape[1] // 250)
    for r in res["regions"]:
        x0, y0, x1, y1 = r["bbox in original photo (x0,y0,x1,y1)"]
        col = (255, 0, 0) if r["priority"] == "CRITICAL" else (255, 140, 0) if r["priority"] == "MAJOR" else (255, 220, 0)
        cv2.rectangle(orig, (x0, y0), (x1, y1), col, lw)
        tb = typing_box(r) if (r.get("defect_type") and "typing_box" in globals()) else None
        if tb and tuple(tb) != (x0, y0, x1, y1):              # the box the type was read from (thin white)
            cv2.rectangle(orig, (int(tb[0]), int(tb[1])), (int(tb[2]), int(tb[3])), (255, 255, 255), max(1, lw // 2))
        label = "#%d %s" % (r["region"], r["priority"]) + ((" " + r["defect_type"]) if r.get("defect_type") else "")
        put_label(orig, label, x0, y0 - 6, lw * 0.45, col, lw)
    ax[0].imshow(orig); ax[0].set_title("original photo: detected regions (original pixels)", fontsize=9)
    al = rgb(canon).copy(); al[~region] = (al[~region] * 0.35).astype(np.uint8)
    cnts, _ = cv2.findContours(region.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(al, cnts, -1, (0, 255, 0), 2)
    ax[1].imshow(al); ax[1].set_title("aligned; green = inspected board region", fontsize=9)
    vmax = 2.0 * res["pixel_threshold"] if res["pixel_threshold"] else float(np.percentile(amap[region], 99.9))
    ax[2].imshow(rgb(canon)); hm = ax[2].imshow(np.where(region, amap, np.nan), cmap="jet", alpha=0.55, vmin=float(np.percentile(amap[region], 5)), vmax=vmax)
    plt.colorbar(hm, ax=ax[2], fraction=0.046, pad=0.02).set_label("anomaly")
    ax[2].set_title("anomaly heatmap (score %.3f, threshold %.3f)" % (res["score"], res["threshold"]), fontsize=9)
    hl = rgb(canon).copy(); hl[hot] = (0.35 * hl[hot] + 0.65 * np.array([255, 0, 0])).astype(np.uint8)
    for r in res["regions"]:
        x, y, w, h = r["bbox aligned (x,y,w,h)"]
        cv2.rectangle(hl, (x, y), (x + w, y + h), (255, 255, 0), 2)
        if r.get("defect_type"):
            put_label(hl, "#%d %s" % (r["region"], r["defect_type"]), x, y - 4, 0.4, (255, 255, 0), 1)
    ax[3].imshow(hl); ax[3].set_title("highlighted pixels (red) & regions: %s" % three_level_decision(res), fontsize=9)
    if gt is not None:
        g = gt & region
        ov = rgb(canon).copy()
        ov[hot & g] = (0, 200, 0); ov[g & ~hot] = (0, 90, 255); ov[hot & ~g] = (255, 0, 0)
        inter = int((hot & g).sum()); union = int((hot | g).sum())
        ax[4].imshow(ov); ax[4].set_title("vs ground truth: green hit, blue missed, red extra (IoU %.2f)" % (inter / max(1, union)), fontsize=9)
    for a in ax:
        a.axis("off")
    fig.suptitle(title or "%s -- %s" % (res["pcb_class"], res["system"]))
    plt.tight_layout(); plt.show()


def plot_system_comparison(image, cls, gt=None, title=""):
    """Heatmaps of every aligned system for the same photo, side by side."""
    res_all = [inspect_board(image, cls, system=s) for s in PIX_SYSTEMS]
    fig, ax = plt.subplots(1, len(res_all), figsize=(4.8 * len(res_all), 4.2), squeeze=False)
    for a, r in zip(ax[0], res_all):
        a.imshow(cv2.cvtColor(r["canon"], cv2.COLOR_BGR2RGB))
        a.imshow(np.where(r["region"], r["amap"], np.nan), cmap="jet", alpha=0.55,
                 vmin=float(np.percentile(r["amap"][r["region"]], 5)), vmax=float(np.percentile(r["amap"][r["region"]], 99.9)))
        if gt is not None:
            cnts, _ = cv2.findContours((gt & r["region"]).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            vis = np.zeros((*r["amap"].shape, 4)); cv2.drawContours(vis, cnts, -1, (1, 1, 1, 1), 2); a.imshow(vis)
        a.set_title("%s\n%s (score %.3f / thr %.3f)" % (r["system"], "DEFECTIVE" if r["verdict"] else "PASS", r["score"], r["threshold"]), fontsize=8)
        a.axis("off")
    fig.suptitle(title or "Same board, every aligned system (white outline: ground-truth defect)")
    plt.tight_layout(); plt.show()
    return res_all


with doc_step("pixel thresholds from CALIBRATION normal boards"):
    rows = []
    for cls in detectors_canon:
        normals = [p for p, l in zip(CANON_PATHS[cls]["calib"], FILE_LISTS[cls]["calib_labels"]) if l == 0 and _is_aligned_png(p)]
        normals = normals[:CFG.PIXEL_CALIB_IMAGES]
        maxima = {s: [] for s in PIX_SYSTEMS}
        for p in normals:
            seen = read_seen(p); region = inspected_region(cls, seen)
            m, _ = ours_map(cls, cv2.imread(p, cv2.IMREAD_COLOR), seen)
            maxima[OURS_SYS].append(float(m[region].max()) if region is not None and region.any() else float(m.max()))
        for s in PIX_SYSTEMS[1:]:
            for p, (m, _) in zip(normals, baseline_maps(s, cls, [(p, 0) for p in normals])):
                if m is not None:
                    region = inspected_region(cls, read_seen(p))
                    maxima[s].append(float(m[region].max()) if region is not None and region.any() else float(m.max()))
        for s, v in maxima.items():
            if v:
                PIX_THR[(s, cls)] = float(np.quantile(v, CFG.PIXEL_NORMAL_QUANTILE))
                rows.append({"system": s, "pcb_class": cls, "calibration normal boards": len(v), "pixel threshold": round(PIX_THR[(s, cls)], 4)})
    pix_thr_df = pd.DataFrame(rows); print(pix_thr_df.to_string(index=False))
    save_table(pix_thr_df, "pixel_thresholds", "Pixel thresholds per system and class (from calibration normal boards only)")

with doc_step("pixel-level localization metrics on held-out boards"):
    rows, rng = [], np.random.default_rng(SEED + 7)
    for cls in detectors_canon:
        ref = load_canon_reference(cls, CFG.ROBUSTNESS_LONG_SIDE)
        items = [(p, int(l), o) for p, l, o in zip(CANON_PATHS[cls]["eval"], FILE_LISTS[cls]["eval_labels"], FILE_LISTS[cls]["eval_images"])
                 if _is_aligned_png(p)]
        gts = [(_aligned_defect_mask(o, ref) if l == 1 else None) for p, l, o in items]
        maps = {OURS_SYS: [ours_map(cls, cv2.imread(p, cv2.IMREAD_COLOR), read_seen(p))[0] for p, _, _ in items]}
        for s in PIX_SYSTEMS[1:]:
            maps[s] = [m for m, _ in baseline_maps(s, cls, [(p, l) for p, l, _ in items])]
        for s, mlist in maps.items():
            thr = PIX_THR.get((s, cls)); ys, vs, hits, ious, fpb = [], [], [], [], []
            for (p, l, _), m, gt in zip(items, mlist, gts):
                if m is None:
                    continue
                region = inspected_region(cls, read_seen(p))
                region = np.ones(m.shape, bool) if region is None else region
                g = np.zeros(m.shape, bool) if gt is None else (gt & region)
                if l == 1 and not g.any():
                    continue
                pos = np.flatnonzero(g.ravel()); neg = np.flatnonzero((region & ~g).ravel())
                if len(neg) > CFG.PIXEL_NEG_SAMPLES:
                    neg = rng.choice(neg, CFG.PIXEL_NEG_SAMPLES, replace=False)
                ys.append(np.r_[np.ones(len(pos)), np.zeros(len(neg))]); vs.append(np.r_[m.ravel()[pos], m.ravel()[neg]])
                if thr is not None:
                    hot = (m > thr) & region
                    if l == 1:
                        inter, union = int((hot & g).sum()), int((hot | g).sum())
                        hits.append(inter > 0); ious.append(inter / max(1, union))
                    else:
                        fpb.append(bool(hot.any()))
            if ys:
                y, v = np.concatenate(ys), np.concatenate(vs)
                rows.append({"system": s, "pcb_class": cls, "pixel AUROC (inspected region)": round(float(roc_auc_score(y, v)), 4),
                             "defects localized (highlight touches the defect)": round(float(np.mean(hits)), 3) if hits else None,
                             "mean IoU (highlight vs defect)": round(float(np.mean(ious)), 3) if ious else None,
                             "normal boards with any highlight": round(float(np.mean(fpb)), 3) if fpb else None})
    pixel_metrics_df = pd.DataFrame(rows)
    if len(pixel_metrics_df):
        mean = pixel_metrics_df.groupby("system", sort=False).mean(numeric_only=True).round(4).reset_index(); mean.insert(1, "pcb_class", "MEAN")
        pixel_metrics_df = pd.concat([pixel_metrics_df, mean], ignore_index=True)
    print(pixel_metrics_df.to_string(index=False))
    save_table(pixel_metrics_df, "pixel_level_localization", "Pixel-level localization on held-out boards vs ground-truth defect masks")
    pv = pixel_metrics_df[pixel_metrics_df.pcb_class != "MEAN"].pivot(index="pcb_class", columns="system", values="pixel AUROC (inspected region)")
    fig, ax = plt.subplots(figsize=(12, 4.2)); pv[[s for s in PIX_SYSTEMS if s in pv.columns]].plot.bar(ax=ax, rot=0)
    ax.set_ylim(0.5, 1.0); ax.set_ylabel("pixel AUROC"); ax.set_title("Pixel-level defect localization on held-out boards (inside the inspected board region)")
    ax.legend(fontsize=8)
    caption("Pixel-level localization: AUROC of each system's anomaly map against VisA's ground-truth defect masks, "
            "inside the inspected board region, on held-out boards.")
    plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# CELL 20e2 -- DEFECT TYPES ON THE FINAL PIPELINE'S REGIONS: what a user of the deployed system actually gets
# ============================================================
doc_section("CELL 20e2", "DEFECT TYPES ON THE FINAL PIPELINE'S REGIONS: what a user of the deployed system actually gets")
# The deployed inspection (inspect_board: Cells 20e / 22b / 22c) works on the ALIGNED board, inspects the board only and has
# a calibrated pixel threshold. This cell runs exactly that function on labelled defective boards and types its regions.
# 1. WHICH BOX TO TYPE -- selected on CALIBRATION defects (CFG.DEFECT_TYPE_BOX = "auto"), never on held-out ones:
#      region       the whole region box (a real run: median 19% of the photo, while defects cover 0.2-0.9%)
#      highlighted  the tight box around the region's pixels above the calibrated pixel threshold (region box if none)
#    The classifier was trained on these calibration images, so each one is typed by an OUT-OF-FOLD classifier (same C as
#    the deployed one, trained without that image) -- otherwise the box resembling the training boxes would win by
#    memory. Rule (pre-registered): the box with the higher out-of-fold accuracy on regions that lie on a defect; ties keep
#    "region". Our final model's regions are used for the choice; the chosen box is then used everywhere.
# 2. HELD-OUT EVALUATION -- our final model and Cell 20b's deployment pick, every labelled held-out defective board:
#      on a defect  the region box covers >= CFG.REGION_HIT_FRAC of a true defect blob (the same set for both boxes)
#      typed right  the region's type is one of the image's true types
#      END TO END   flagged, top region on the defect, and typed right with the SELECTED box
#    The other box is reported for transparency only, with a paired comparison; nothing is chosen on held-out data.
type_final_df = type_final_paired_df = type_box_selection_df = None
TYPE_FINAL = {}                               # system -> list of per-image records


def _pick_inspect_name(name):
    """Cell 20b system name -> inspect_board system name ('X -- canonical pipeline' -> 'X (canonical pipeline)')."""
    if not name or "canonical pipeline" not in name:
        return None
    s = OURS_SYS if name.startswith("Ours") else name.split(" -- ")[0] + " (canonical pipeline)"
    return s if s in PIX_SYSTEMS else None


def _inspect_boxes(path, cls, system):
    """One inspection WITHOUT typing -> (result, region boxes, highlighted boxes) in original-photo pixels."""
    res = inspect_board(path, cls, system=system, classify_types=False)
    rb = [r["bbox in original photo (x0,y0,x1,y1)"] for r in res["regions"]]
    hb = [r.get("bbox highlighted in original photo (x0,y0,x1,y1)") or b for r, b in zip(res["regions"], rb)]
    return res, rb, hb


def _box_pct(b, hw):
    return 100.0 * max(0, b[2] - b[0]) * max(0, b[3] - b[1]) / float(hw[0] * hw[1])


def _fit_fixed_C(regions, C):
    """The Cell 12c classifier with a FIXED C (no inner CV) -- for the out-of-fold models of the box selection."""
    X, y, g, w = _region_xy(regions)
    if len(y) == 0:
        return None
    if len(set(y.tolist())) < 2:
        return {"kind": "constant", "label": str(y[0])}
    m = _defect_lr(C).fit(X, y, logisticregression__sample_weight=w)
    return {"kind": "lr", "model": m, "C": C, "classes": [str(c) for c in m.classes_]}


def _e2e(r, mode):
    """1 if the board is flagged, its top region lies on the defect and is typed right with box `mode`."""
    top = r["regions"][0] if r["regions"] else None
    return float(r["flagged"] and top is not None and top["hit"] and top["ok_" + mode])


_pick = next((s["name"] for s in (globals().get("systems") or []) if s.get("deploy")), None)
TYPE_FINAL_SYSTEMS = list(dict.fromkeys([OURS_SYS] + [s for s in [_pick_inspect_name(_pick)] if s]))
_ready = (getattr(CFG, "RUN_DEFECT_TYPE_FINAL_EVAL", True) and globals().get("DEFECT_CLASSIFIERS")
          and globals().get("DEFECT_TRUE") and globals().get("DEFECT_REGIONS"))

if not _ready:
    print("Skipped: needs the Part 1.4 classifiers (Cell 12c) and CFG.RUN_DEFECT_TYPE_FINAL_EVAL = True.")
else:
    if CFG.DEFECT_TYPE_BOX == "auto":
        with doc_step("select the typing box on CALIBRATION defects (out-of-fold classifiers, our final model's regions)"):
            per_cls, srows = {}, []
            for cls, full_model in DEFECT_CLASSIFIERS.items():
                if cls not in detectors_canon:
                    continue
                cal = sorted({r["path"] for r in DEFECT_REGIONS[cls] if r["split"] == "calib"
                              and (cls, os.path.basename(r["path"])) in DEFECT_TRUE})
                random.Random(SEED + 53).shuffle(cal)
                fold = {p: i % 3 for i, p in enumerate(cal)}
                C = full_model.get("C", 1.0)
                models = {k: _fit_fixed_C([r for r in DEFECT_REGIONS[cls] if r["split"] == "calib" and r["hit"]
                                           and fold.get(r["path"]) != k], C) for k in range(3)}
                vals = []
                for p in tqdm(cal, desc="[%s] calibration boxes" % cls, leave=False):
                    m = models[fold[p]]
                    if m is None:
                        continue
                    res, rb, hb = _inspect_boxes(p, cls, OURS_SYS)
                    if not res["regions"]:
                        continue
                    types, blobs = DEFECT_TRUE[(cls, os.path.basename(p))], defect_blobs(p, res["full"].shape[:2])
                    tr = classify_region_boxes(res["full"], cls, rb, model=m)
                    th = classify_region_boxes(res["full"], cls, hb, model=m)
                    on = [k for k, b in enumerate(rb) if blobs and box_hits_defect(b, blobs)]
                    if on:                                             # (correct with region box, with tight box, regions)
                        vals.append((sum(tr[k]["defect_type"] in types for k in on), sum(th[k]["defect_type"] in types for k in on), len(on)))
                per_cls[cls] = vals
                if vals:
                    srows.append({"pcb_class": cls, "calibration boards with a region on the defect": len(vals),
                                  "regions on a defect": int(sum(v[2] for v in vals)),
                                  "out-of-fold accuracy, region box": round(sum(v[0] for v in vals) / sum(v[2] for v in vals), 3),
                                  "out-of-fold accuracy, highlighted box": round(sum(v[1] for v in vals) / sum(v[2] for v in vals), 3),
                                  "highlighted minus region [95% CI]": "", "SELECTED": ""})
            allv = [v for vals in per_cls.values() for v in vals]
            if allv:
                acc_r = sum(v[0] for v in allv) / sum(v[2] for v in allv)
                acc_h = sum(v[1] for v in allv) / sum(v[2] for v in allv)
                d, ci = _boot_ratio({c: [(v[1] - v[0], v[2]) for v in vals] for c, vals in per_cls.items() if vals}, SEED + 59)
                DEFECT_TYPE_BOX = "highlighted" if acc_h > acc_r else "region"
                srows.append({"pcb_class": "all", "calibration boards with a region on the defect": len(allv),
                              "regions on a defect": int(sum(v[2] for v in allv)),
                              "out-of-fold accuracy, region box": round(acc_r, 3), "out-of-fold accuracy, highlighted box": round(acc_h, 3),
                              "highlighted minus region [95% CI]": _pct(d, ci, diff=True), "SELECTED": DEFECT_TYPE_BOX})
            type_box_selection_df = pd.DataFrame(srows)
            print("\nTYPING BOX -- SELECTED ON CALIBRATION DEFECTS (out-of-fold classifiers; held-out data not used)")
            print(type_box_selection_df.to_string(index=False))
            save_table(type_box_selection_df, "defect_type_box_selection",
                       "Which region box the classifier reads: out-of-fold accuracy on calibration defects (selection data)")
            log("Typing box selected on calibration: " + DEFECT_TYPE_BOX)
            pk = os.path.join(CFG.ARTIFACT_DIR, "defect_type_classifiers.pkl")
            if os.path.exists(pk):                                     # keep the saved pipeline consistent
                with open(pk, "rb") as fh:
                    pack = pickle.load(fh)
                pack["type_box"] = DEFECT_TYPE_BOX
                with open(pk, "wb") as fh:
                    pickle.dump(pack, fh)
    else:
        log("Typing box fixed by CFG.DEFECT_TYPE_BOX: " + DEFECT_TYPE_BOX)

    with doc_step("inspect every labelled held-out defective board with the final pipeline (" + ", ".join(TYPE_FINAL_SYSTEMS) + ")"):
        for system in TYPE_FINAL_SYSTEMS:
            recs = []
            for cls in DEFECT_CLASSIFIERS:
                if cls not in detectors_canon:
                    continue
                paths = [p for p, l in zip(FILE_LISTS[cls]["eval_images"], FILE_LISTS[cls]["eval_labels"])
                         if l == 1 and (cls, os.path.basename(p)) in DEFECT_TRUE]
                for p in tqdm(paths, desc="[%s] %s" % (cls, system), leave=False):
                    types = DEFECT_TRUE[(cls, os.path.basename(p))]
                    try:
                        res, rb, hb = _inspect_boxes(p, cls, system)
                    except Exception as e:
                        log("[%s] %s: inspection failed (%r) -- counted as missed" % (cls, os.path.basename(p), e), "WARN")
                        recs.append({"cls": cls, "path": p, "flagged": False, "regions": []})
                        continue
                    hw = res["full"].shape[:2]
                    blobs = defect_blobs(p, hw)
                    tr = classify_region_boxes(res["full"], cls, rb)
                    th = classify_region_boxes(res["full"], cls, hb)
                    regs = [{"hit": bool(blobs) and box_hits_defect(b, blobs),
                             "hit_tight": bool(blobs) and box_hits_defect(h, blobs),
                             "ok_region": a["defect_type"] in types, "ok_highlighted": t["defect_type"] in types,
                             "pct_region": _box_pct(b, hw), "pct_highlighted": _box_pct(h, hw)}
                            for b, h, a, t in zip(rb, hb, tr, th)]
                    recs.append({"cls": cls, "path": p, "flagged": bool(res["verdict"]), "regions": regs})
            TYPE_FINAL[system] = recs
            log("%s: %d held-out defective boards inspected" % (system, len(recs)))

    with doc_step("score the final pipeline's regions and compare with the Cell 9 detector"):
        sel, other = DEFECT_TYPE_BOX, ("highlighted" if DEFECT_TYPE_BOX == "region" else "region")
        raw_e2e = {r["path"]: float(r["flagged"] and r["hit"] and r["ok"]) for r in globals().get("DR", []) if r["rank"] == 0}
        rows, prow = [], []
        for system, recs in TYPE_FINAL.items():
            for c in sorted({r["cls"] for r in recs}) + ["all"]:
                rr = [r for r in recs if c in ("all", r["cls"])]
                regs = [g for r in rr for g in r["regions"]]
                acc = {}
                for mode in (sel, other):
                    per = {}
                    for r in rr:
                        on = [g for g in r["regions"] if g["hit"]]
                        if on:
                            per.setdefault(r["cls"], []).append((sum(g["ok_" + mode] for g in on), len(on)))
                    acc[mode] = _boot_ratio(per, SEED + 31)
                e2e, e2e_ci = _boot_ratio({k: [(_e2e(r, sel), 1) for r in rr if r["cls"] == k] for k in {r["cls"] for r in rr}}, SEED + 31)
                rows.append({"system": system, "pcb_class": c, "held-out defective boards": len(rr),
                             "flagged": round(float(np.mean([r["flagged"] for r in rr])), 3),
                             "regions on a defect (region box)": round(float(np.mean([g["hit"] for g in regs])), 3) if regs else None,
                             "regions on a defect (highlighted box)": round(float(np.mean([g["hit_tight"] for g in regs])), 3) if regs else None,
                             "median box % of photo (region / highlighted)": ("%.2f / %.2f" % (np.median([g["pct_region"] for g in regs]),
                                                                              np.median([g["pct_highlighted"] for g in regs]))) if regs else None,
                             "typing accuracy, SELECTED box (%s) [95%% CI]" % sel: _pct(*acc[sel]),
                             "typing accuracy, other box (%s, transparency) [95%% CI]" % other: _pct(*acc[other]),
                             "END TO END: flagged, top region on the defect, typed right [95% CI]": _pct(e2e, e2e_ci)})
            both = [r for r in recs if r["path"] in raw_e2e]
            if both:
                d, ci = _boot_ratio({k: [(_e2e(r, sel) - raw_e2e[r["path"]], 1) for r in both if r["cls"] == k]
                                     for k in {r["cls"] for r in both}}, SEED + 37)
                prow.append({"system": system, "comparison": "END TO END minus Cell 9 detector", "boards or regions": len(both),
                             "difference [95% CI]": _pct(d, ci, diff=True),
                             "verdict": "final pipeline better" if ci[0] > 0 else "final pipeline worse" if ci[1] < 0 else "no significant difference"})
            per = {}
            for r in recs:
                on = [g for g in r["regions"] if g["hit"]]
                if on:
                    per.setdefault(r["cls"], []).append((sum(g["ok_highlighted"] - g["ok_region"] for g in on), len(on)))
            d, ci = _boot_ratio(per, SEED + 43)
            prow.append({"system": system, "comparison": "typing: highlighted box minus region box (held-out, transparency)",
                         "boards or regions": sum(v[1] for vals in per.values() for v in vals), "difference [95% CI]": _pct(d, ci, diff=True),
                         "verdict": "highlighted better" if ci[0] > 0 else "highlighted worse" if ci[1] < 0 else "no significant difference"})
        type_final_df, type_final_paired_df = pd.DataFrame(rows), pd.DataFrame(prow)
        print("\nREGION TYPES WITH THE FINAL PIPELINE'S REGIONS (held-out defective boards; typing box selected on calibration: %s)" % sel)
        print(type_final_df.to_string(index=False))
        print("\nPAIRED COMPARISONS (same held-out boards / regions)")
        print(type_final_paired_df.to_string(index=False))
        save_table(type_final_df, "defect_type_final_pipeline",
                   "Region defect types with the final pipeline's regions: on-defect share, box size, typing accuracy, end to end")
        save_table(type_final_paired_df, "defect_type_final_vs_raw",
                   "Region typing: final pipeline vs the Cell 9 detector (end to end) and highlighted vs region box (paired CIs)")

    if TYPE_FINAL:
        with doc_step("figure: end-to-end region typing, Cell 9 detector vs the final pipeline"):
            labels, vals, errs = [], [], []
            if globals().get("DR"):
                raw = {}
                for r in DR:
                    if r["rank"] == 0:
                        raw.setdefault(r["cls"], []).append((float(r["flagged"] and r["hit"] and r["ok"]), 1))
                p, ci = _boot_ratio(raw, SEED + 31)
                labels.append("Cell 9 raw-input detector"); vals.append(p); errs.append(ci)
            for system, recs in TYPE_FINAL.items():
                p, ci = _boot_ratio({k: [(_e2e(r, DEFECT_TYPE_BOX), 1) for r in recs if r["cls"] == k] for k in {r["cls"] for r in recs}},
                                    SEED + 31)
                labels.append("%s (%s box)" % (system, DEFECT_TYPE_BOX)); vals.append(p); errs.append(ci)
            fig, ax = plt.subplots(figsize=(9, 0.7 * len(labels) + 1.6))
            ax.barh(range(len(labels)), vals, color=["#9aa5b1"] + ["#2a7ab9"] * (len(labels) - 1), capsize=3,
                    xerr=[[v - e[0] for v, e in zip(vals, errs)], [e[1] - v for v, e in zip(vals, errs)]])
            ax.set_yticks(range(len(labels)))
            ax.set_yticklabels(labels)
            ax.invert_yaxis()
            ax.set_xlim(0, 1)
            ax.set_xlabel("end to end: flagged, top region on the defect, typed right (95% CI)")
            ax.set_title("End-to-end defect typing: whose regions?")
            caption("End-to-end region defect typing on held-out defective boards: the Cell 9 raw-input detector's regions vs "
                    "the final pipeline's regions (typing box selected on calibration)")
            plt.tight_layout()
            plt.show()

    with doc_step("append the defect-type results to results_summary.md"):
        def _md_simple(df):
            cols = [str(c) for c in df.columns]
            rows = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
            rows += ["| " + " | ".join("" if (isinstance(v, float) and np.isnan(v)) else str(v) for v in r) + " |"
                     for r in df.itertuples(index=False)]
            return "\n".join(rows)
        parts = ["\n## Defect type of every highlighted region (Part 1.4)\n",
                 "Supervised region classifier on the FROZEN self-supervised CNN+ViT features (the detector is unchanged), one type "
                 "per highlighted region, trained on calibration regions only. A = exact (ground-truth) regions; B = the Cell 9 "
                 "raw-input detector's regions that lie on a defect. Final pipeline = the deployed inspect_board regions, read with "
                 "the box selected on calibration: **%s**." % DEFECT_TYPE_BOX]
        for title, df in [("Classifier on held-out regions", globals().get("defect_results_df")),
                          ("Trained classifier vs the majority type (paired)", globals().get("defect_paired_df")),
                          ("Typing box, selected on calibration (out-of-fold)", type_box_selection_df),
                          ("Final pipeline's regions, all classes (held-out)",
                           type_final_df[type_final_df["pcb_class"] == "all"] if type_final_df is not None else None),
                          ("Paired comparisons", type_final_paired_df)]:
            if isinstance(df, pd.DataFrame) and len(df):
                parts += ["\n### " + title + "\n", _md_simple(df)]
        with open(globals().get("SUMMARY_PATH", "/kaggle/working/results_summary.md"), "a") as fh:
            fh.write("\n".join(parts) + "\n")
        print("Appended the defect-type results to results_summary.md")

## Part 3.8 -- Rotation & lighting galleries

In [ ]:
# ============================================================
# CELL 20f -- ROTATION & LIGHTING GALLERIES: the same defective board at every angle and under every light
# ============================================================
doc_section("CELL 20f", "ROTATION & LIGHTING GALLERIES: the same defective board at every angle and under every light")
# Visual evidence for the sweeps of Cell 20d: one held-out defective board per class (random, fixed seed) is rotated to every
# angle in CFG.SWEEP_ANGLES and lit by every condition in CFG.SWEEP_LIGHTS, then inspected by our final system. Top row:
# what the camera captured; bottom row: our heatmap on the aligned board with the highlighted defect pixels.
gallery_df = None
with doc_step("rotation & lighting galleries"):
    rng = np.random.default_rng(SEED + 20)
    rows = []
    for cls in detectors_canon:
        bad = [o for o, l in zip(FILE_LISTS[cls]["eval_images"], FILE_LISTS[cls]["eval_labels"]) if l == 1]
        path = bad[int(rng.integers(len(bad)))]
        full = cv2.imread(path, cv2.IMREAD_COLOR); small = downscale_long_side(full, CFG.ROBUSTNESS_LONG_SIDE)
        for kind, settings in (("rotation", CFG.SWEEP_ANGLES), ("lighting", ["normal"] + list(CFG.SWEEP_LIGHTS))):
            ncol = 8
            nblk = int(np.ceil(len(settings) / float(ncol)))
            fig, ax = plt.subplots(2 * nblk, ncol, figsize=(2.6 * ncol, 2.5 * 2 * nblk), squeeze=False)
            for a in ax.ravel():
                a.axis("off")
            for k, sval in enumerate(settings):
                if kind == "rotation":
                    img, valid = _render_rotation(full, small, sval)
                else:
                    img, valid = _render_lighting(small, sval), None
                res = inspect_board(img, cls, input_valid=valid)
                r0, c0 = 2 * (k // ncol), k % ncol
                ax[r0, c0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                ax[r0, c0].set_title(("%d deg" % sval) if kind == "rotation" else sval, fontsize=8)
                hl = cv2.cvtColor(res["canon"], cv2.COLOR_BGR2RGB).copy()
                hl[res["hot"]] = (0.3 * hl[res["hot"]] + 0.7 * np.array([255, 0, 0])).astype(np.uint8)
                ax[r0 + 1, c0].imshow(hl)
                ax[r0 + 1, c0].imshow(np.where(res["region"], res["amap"], np.nan), cmap="jet", alpha=0.35)
                ax[r0 + 1, c0].set_title("%s (%.2f/%.2f)" % ("DEFECTIVE" if res["verdict"] else "PASS", res["score"], res["threshold"]),
                                         fontsize=8, color="darkred" if res["verdict"] else "darkgreen")
                rows.append({"pcb_class": cls, "file": os.path.basename(path), "sweep": kind,
                             "setting": ("%d deg" % sval) if kind == "rotation" else sval, "aligned": res["info"]["registered"],
                             "rotation undone (deg)": round(res["info"]["angle_deg"], 2), "score": round(res["score"], 4),
                             "threshold": round(res["threshold"], 4), "verdict": "DEFECTIVE" if res["verdict"] else "PASS",
                             "highlighted pixels": int(res["hot"].sum())})
            fig.suptitle("%s -- one DEFECTIVE board, every %s (top: captured photo; bottom: our heatmap + highlighted pixels)" % (cls, kind))
            caption("%s: the same held-out defective board under every %s setting -- the captured photo and our final "
                    "system's verdict, heatmap and highlighted defect pixels." % (cls, kind))
            plt.tight_layout(); plt.show()
    gallery_df = pd.DataFrame(rows)
    print(gallery_df.to_string(index=False))
    save_table(gallery_df, "rotation_lighting_gallery", "Verdicts for one defective board per class at every angle and lighting condition")
    print("\nVerdict consistency (share of settings flagged DEFECTIVE, per class and sweep):")
    print((gallery_df.assign(flag=gallery_df.verdict.eq("DEFECTIVE")).groupby(["pcb_class", "sweep"])["flag"].mean() * 100).round(1).to_string())

In [ ]:
# ============================================================
# CELL 21 -- Export everything to one multi-sheet Excel workbook
# ============================================================
doc_section("CELL 21", "Export everything to one multi-sheet Excel workbook")
def _safe_sheet(writer, df, name, index=False):
    if df is not None and len(df):
        df.to_excel(writer, sheet_name=name[:31], index=index)
        return name[:31]
    return None


def _score_modes(m):
    modes = sorted({baseline_results[m][c]["score_mode"] for c in CFG.CLASSES
                    if baseline_results.get(m, {}).get(c)})
    return ", ".join(modes) if modes else "not run"


written = []
with pd.ExcelWriter(CFG.EXCEL_OUTPUT_PATH, engine="openpyxl") as writer:
    # Summary: held-out (eval-split) numbers only -- the sheet to cite
    summary_rows = []
    for cls in CFG.CLASSES + ["MEAN"]:
        row = {"pcb_class": cls,
               "n_eval_images": (len(MODEL_EVAL[cls]["labels"]) if cls in MODEL_EVAL
                                 else sum(len(MODEL_EVAL[c]["labels"]) for c in CFG.CLASSES))}
        for m in MODEL_ORDER:
            row[m + " AUROC [95% CI]"] = fmt_ci(*BOOT[(m, cls)])
        summary_rows.append(row)
    written.append(_safe_sheet(writer, pd.DataFrame(summary_rows), "Summary"))
    written.append(_safe_sheet(writer, comparison_df.round(4), "AUROC_Point", index=True))
    written.append(_safe_sheet(writer, comparison_ci_df, "AUROC_CI", index=True))
    written.append(_safe_sheet(writer, paired_df, "Paired_vs_Ours"))
    written.append(_safe_sheet(writer, paired_canon_df, "Paired_Deconfounded"))
    written.append(_safe_sheet(writer, thresholded_df, "Thresholded_Metrics"))
    written.append(_safe_sheet(writer, ablation_df, "Ablation"))
    written.append(_safe_sheet(writer, ensemble_df, "Ensemble"))
    written.append(_safe_sheet(writer, final_summary_df, "Final_Summary"))
    if CONCLUSIONS:
        written.append(_safe_sheet(writer, pd.DataFrame({"conclusion": [c.replace("**", "") for c in CONCLUSIONS]}), "Conclusions"))
    written.append(_safe_sheet(writer, system_pairwise_df, "System_Pairwise", index=True))
    written.append(_safe_sheet(writer, robust_summary_df, "Robust_Summary"))
    written.append(_safe_sheet(writer, robust_auroc_long_df, "Robust_AUROC_per_class"))
    written.append(_safe_sheet(writer, robust_scores_long_df, "Robust_Per_Image"))
    written.append(_safe_sheet(writer, canon_pipeline_df, "Canon_Pipeline"))
    written.append(_safe_sheet(writer, confound_df, "Confound_Check"))
    written.append(_safe_sheet(writer, improvement_df, "Improvement_Study"))
    written.append(_safe_sheet(writer, canon_study_df, "Canon_Pipeline_Study"))
    written.append(_safe_sheet(writer, aug_study_df, "Aug_Study"))
    written.append(_safe_sheet(writer, seed_study_df, "Seed_Robustness"))
    written.append(_safe_sheet(writer, sweep_summary_df, "Sweep_Summary"))
    written.append(_safe_sheet(writer, board_mask_df, "Board_Masks"))
    written.append(_safe_sheet(writer, sweep_align_df, "Sweep_Alignment_Per_Image"))
    written.append(_safe_sheet(writer, sweep_df, "Angle_Crop_Sweep"))
    written.append(_safe_sheet(writer, sweep_reg_df, "Sweep_Registration"))
    written.append(_safe_sheet(writer, seed_pairs_df, "Seed_Pairs"))
    written.append(_safe_sheet(writer, residual_light_df, "Residual_Lighting_Cue"))
    written.append(_safe_sheet(writer, native_orientation_df, "Native_Orientation", index=True))
    if CANON_PIPE_REG:
        written.append(_safe_sheet(writer, pd.DataFrame(CANON_PIPE_REG), "Canon_Pipeline_Per_Image"))
    written.append(_safe_sheet(writer, robust_auroc_df, "Robust_AUROC", index=True))
    written.append(_safe_sheet(writer, robust_drop_df, "Robust_Drop_vs_Identity", index=True))
    written.append(_safe_sheet(writer, robust_fpr_df, "Robust_FalseAlarm", index=True))
    written.append(_safe_sheet(writer, robust_recall_df, "Robust_Recall", index=True))
    written.append(_safe_sheet(writer, robust_paired_df, "Robust_Paired"))
    written.append(_safe_sheet(writer, canon_gain_df, "Canon_Gain"))
    written.append(_safe_sheet(writer, canon_reg_summary_df, "Canon_Registration"))
    if CANON_REG:
        written.append(_safe_sheet(writer, pd.DataFrame(CANON_REG), "Canon_Reg_Per_Image"))
    for cls in CFG.CLASSES:
        if cls in all_results_by_class:
            written.append(_safe_sheet(writer, all_results_by_class[cls], cls.upper()))

    protocol = [
        ("Splits", "train = normal training images (memory banks / baseline fitting); calibration = half of "
                   "the labeled test split (thresholds, blend weights, Anomalib validation); eval = the other "
                   "half, used ONLY for reported metrics. Robustness sets are copies of eval images."),
        ("Split sizes", "; ".join(c + ": train " + str(min(len(FILE_LISTS[c]["train_normal"]), CFG.MAX_TRAIN_IMAGES_PER_CLASS))
                                  + ", calib " + str(len(FILE_LISTS[c]["calib_images"]))
                                  + ", eval " + str(len(FILE_LISTS[c]["eval_images"])) for c in CFG.CLASSES)),
        ("Thresholds", "Every model: max recall subject to FPR <= " + str(CFG.MAX_FALSE_POSITIVE_RATE)
                       + " on the calibration split."),
        ("Confidence intervals", "Stratified paired bootstrap, " + str(CFG.BOOTSTRAP_N) + " replicates; "
                                 "differences called significant only if the 95% CI excludes 0."),
        ("Our model", "Hybrid " + CFG.CNN_BACKBONE + " + " + CFG.VIT_BACKBONE + ", bank " + str(CFG.MAX_BANK_SIZE)
                      + " pts/branch, KNN_K " + str(CFG.KNN_K) + ", TOPK_SCORE_FRAC " + str(CFG.TOPK_SCORE_FRAC)
                      + ", " + str(CFG.AUGS_PER_IMAGE) + " invariance augmentations/image, IMG_SIZE " + str(CFG.IMG_SIZE)),
        ("Baseline score modes", "; ".join(m + ": " + _score_modes(m) for m in CFG.BASELINE_MODEL_NAMES)),
        ("Fusion weight alpha", "per class, on the calibration split: maximize AUROC of the deployed score "
                                "(top-k mean of the fused map), ties -> 0.5. The legacy F1 rule is reported in 'Ablation'. "
                                + "; ".join(c + ": " + str(round(detectors[c].alpha, 2)) for c in CFG.CLASSES)),
        ("Canonicalization", ("ORB " + str(CFG.CANON_ORB_FEATURES) + " features, ratio test " + str(CFG.CANON_RATIO_TEST)
                              + ", RANSAC similarity (min " + str(CFG.CANON_MIN_INLIERS) + " inliers), illumination sigma "
                              + str(CFG.CANON_ILLUM_SIGMA_FRAC) + " x size, reference = first training image per class")
                             if CFG.ROBUSTNESS_CANONICALIZE else "off"),
        ("Golden references", "; ".join(c + ": " + os.path.basename(_CANON_REF_CACHE[(c, CFG.ROBUSTNESS_LONG_SIDE)]["path"])
                                         for c in CFG.CLASSES if (c, CFG.ROBUSTNESS_LONG_SIDE) in _CANON_REF_CACHE)
                              + " (median-brightness training board per class)"),
        ("Canonical pipeline", ("our model fit + calibrated on canonicalized images (no invariance augmentation); "
                                "only registered training boards used") if CFG.RUN_CANON_PIPELINE else "off"),
        ("Canonicalization v2", "180-degree hypothesis check (phase correlation + ECC on illumination-normalized structure), "
                                "lighting gate " + str(CFG.CANON_PHOTOMETRIC_GATE) + ", feather " + str(CFG.CANON_FEATHER_PX) + " px"),
        ("Anomalib version", str(ANOMALIB_VERSION) + " (pin with pip install anomalib==<version> to reproduce)"),
        ("Environment", "; ".join(k + " " + v for k, v in VERSIONS.items()) + " | Anomalib install: " + ANOMALIB_INSTALL_MODE),
        ("Split seeds", "per class: SEED + 7919 x (class index + 1), SEED = " + str(SEED)),
        ("Operating points", "empirical: <= " + str(CFG.MAX_FALSE_POSITIVE_RATE) + " false alarms on calibration; PAC: held-out <= "
                             + str(CFG.MAX_FALSE_POSITIVE_RATE) + " with " + str(int(CFG.PAC_CONFIDENCE * 100))
                             + "% confidence (Thresholded_Metrics sheet, column operating_point)"),
        ("Robustness conditions", ", ".join(CFG.ROBUSTNESS_CONDITIONS) + " (long side " + str(CFG.ROBUSTNESS_LONG_SIDE)
                                  + " px, PNG, crop frac " + str(CFG.ROBUSTNESS_CROP_FRAC) + ")"),
    ]
    written.append(_safe_sheet(writer, pd.DataFrame(protocol, columns=["item", "detail"]), "Protocol"))

print("Saved", CFG.EXCEL_OUTPUT_PATH)
print("Sheets:", ", ".join(w for w in written if w))
print("\nStart with 'Final_Summary' + 'Conclusions' (also in results_summary.md). Then 'Summary' / 'AUROC_CI' /")
print("'Paired_vs_Ours' for the standard benchmark, 'Robust_*' / 'Canon_*' for the invariance claim.")
print("'Ablation' backs the architecture claim; the per-class sheets are the complete per-image inventory.")

## Part 4 -- Try it on a single image
Set `MY_IMAGE_PATH` to any PCB photo (a dataset image or your own upload) and run the cell: input, heatmap,
overlay and circled regions colour-coded by priority (red = CRITICAL, orange = MAJOR, yellow = MINOR), plus a
report with verdict, severity and a shape-based description of each region.

## Part 4 -- Inspection tool: any evaluated system, including the recommended one

In [ ]:
# ============================================================
# CELL 22a -- DEPLOYABLE INSPECTION: run ANY evaluated system, including the one Cell 20b recommends
# ============================================================
doc_section("CELL 22a", "DEPLOYABLE INSPECTION: run ANY evaluated system, including the one Cell 20b recommends")
# The evaluation may recommend a baseline detector on canonicalized input (e.g. EfficientAD). This turns any
# system into the full workflow: identify the board type by registration -> canonicalize -> detect -> put the
# anomaly map on a z-scale using the detector's OWN calibration-normal statistics -> our region analysis,
# defect-type hints and CRITICAL/MAJOR/MINOR priorities -> report + figure.
# Single-image prediction reuses the exact Folder -> engine.predict path Cell 17 ran on Kaggle.
from contextlib import contextmanager, nullcontext

SINGLE_ROOT = os.path.join(CFG.ANOMALIB_ROOT, "_single_image")


@contextmanager
def _using_detectors(dets):
    """inspect_pcb() looks detectors up in the global `detectors` dict; swap it in temporarily, always restore."""
    global detectors
    saved = detectors
    detectors = dets
    try:
        yield
    finally:
        detectors = saved


@contextmanager
def _operating_point(dets, operating_point, use_tta):
    """Our detectors carry the empirical thresholds; for the 'pac' operating point swap in the PAC thresholds (from each
    detector's own calibration scores) for the duration of the call, then restore them."""
    saved = {c: (d.threshold, getattr(d, "threshold_tta", None)) for c, d in dets.items()}
    try:
        if operating_point == "pac":
            for d in dets.values():
                ss = d.score_stats
                d.threshold = pac_threshold(ss["calib_scores"], ss["calib_labels"])
                if use_tta and "calib_scores_tta" in ss:
                    d.threshold_tta = pac_threshold(ss["calib_scores_tta"], ss.get("calib_labels_tta", ss["calib_labels"]))
        yield
    finally:
        for c, (t, tt) in saved.items():
            dets[c].threshold, dets[c].threshold_tta = t, tt


def _print_inspected_area(info, cls):
    board = board_mask_for(cls) if CFG.BOARD_ONLY else None
    seen = info.get("valid_mask")
    if board is not None and seen is not None and seen.shape == board.shape:
        frac = float((board & seen).sum()) / max(1, int(board.sum()))
    else:
        frac = info.get("valid_frac", 1.0)
    print("Inspected: %.0f%% of the board%s%s" % (100 * frac, " (background surface ignored)" if board is not None else "",
          "" if frac > 0.97 else " -- the rest was not in the photo and is NOT inspected"))


def system_parts(system_name):
    """'EfficientAD -- canonicalized input' -> ('EfficientAD', True); 'Ours -- canonical pipeline' -> ('Ours', True)."""
    det, _, mode = system_name.partition(" -- ")
    return det.strip(), ("canonical" in mode)


def identify_and_canonicalize(img_bgr, pcb_class=None, classes=None):
    """Board type = the class whose golden reference it registers to best. Returns (class, canonical BGR, info)."""
    small = downscale_long_side(img_bgr, CFG.ROBUSTNESS_LONG_SIDE)
    best = None
    for c in ([pcb_class] if pcb_class else (classes or CFG.CLASSES)):
        canon, info = canonicalize(small, load_canon_reference(c, CFG.ROBUSTNESS_LONG_SIDE))
        if best is None or info["inliers"] > best[2]["inliers"]:
            best = (c, canon, info)
    return best


def baseline_predict_single(model_name, cls, img_bgr, region=None):
    """(raw image score, anomaly map resized to the image) from a kept Anomalib model."""
    d = DEPLOY_MODELS[(model_name, cls)]
    root = os.path.join(SINGLE_ROOT, cls)
    if os.path.isdir(root):
        shutil.rmtree(root)
    for sub in ("train/good", "test/good", "test/bad"):
        os.makedirs(os.path.join(root, sub), exist_ok=True)
    tp = FILE_LISTS[cls]["train_normal"][0]
    _link(tp, os.path.join(root, "train", "good", os.path.basename(tp)))
    for sub in ("good", "bad"):                    # Folder expects both test folders: the same image in each
        cv2.imwrite(os.path.join(root, "test", sub, "query.png"), img_bgr)
    items = list(iter_prediction_maps(deploy_predict(model_name, cls, folder_datamodule(cls, root, model_name))))
    if not items:
        raise RuntimeError(model_name + " returned no anomaly map for the query image")
    _, _, score, amap = items[0]
    if region is not None:                         # same inspected-region score as the benchmark (Cell 17)
        score = masked_map_score(amap, region)
    return score, cv2.resize(amap.astype(np.float32), (img_bgr.shape[1], img_bgr.shape[0]), interpolation=cv2.INTER_LINEAR)


def inspect_with_system(image_path, system_name, pcb_class=None, operating_point="empirical", visualize=True):
    """Full inspection of one photo with the named system (any name in Cell 20b's table).
    operating_point: 'empirical' (higher recall) or 'pac' (held-out false alarms <= CFG.MAX_FALSE_POSITIVE_RATE with
    CFG.PAC_CONFIDENCE probability). Returns a result dict; prints a report and draws a figure."""
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError("Could not read " + image_path)
    det_name, canon = system_parts(system_name)

    if det_name == "Ours":                         # our model: its own report (inspect_pcb, Cell 15)
        # inspect with EXACTLY the rule the benchmark scored this system with: single-pass for the canonical pipeline and
        # the raw model, rotation-TTA only for the TTA system. (inspect_pcb used to escalate to TTA by default, so a real
        # run's gallery showed NORMAL for boards the benchmark had flagged.)
        use_tta = "TTA" in system_name
        dets = detectors_canon if canon else detectors
        if canon:
            cls, canon_bgr, info = identify_and_canonicalize(img, pcb_class, [c for c in CFG.CLASSES if c in detectors_canon])
            print("Board: %s | registration: %s, %.1f deg, %d keypoints%s" % (cls, "OK" if info["registered"] else "FAILED",
                  info["angle_deg"], info["inliers"], " | lighting corrected" if info["photometric"] else ""))
            _print_inspected_area(info, cls)
            kwargs = dict(image_array=cv2.cvtColor(canon_bgr, cv2.COLOR_BGR2RGB), pcb_class=cls,
                          # region types are read on the ORIGINAL photo (boxes mapped back through the alignment)
                          type_image_bgr=img, type_box_map=lambda b, info=info: aligned_box_to_photo(b, info, img.shape[:2]))
        else:
            kwargs = dict(image_path=image_path, pcb_class=pcb_class)
        seen_ctx = dets[cls].seen_context(info.get("valid_mask")) if canon else nullcontext()   # board + seen area only
        with _using_detectors(dets), _operating_point(dets, operating_point, use_tta), seen_ctx:
            # the single-pass quick screen is consistent with single-pass systems only; a TTA system is decided by TTA
            return inspect_pcb(rotation_tta=use_tta, quick_screen=not use_tta, visualize=visualize, verbose=True, **kwargs)

    cls, canon_bgr, info = identify_and_canonicalize(img, pcb_class)
    model_key = det_name + " (canonical pipeline)" if "canonical pipeline" in system_name else det_name
    if (model_key, cls) not in DEPLOY_MODELS:
        raise KeyError(det_name + " for " + cls + " was not kept -- add it to CFG.DEPLOY_BASELINES and re-run Cell 17")
    x = canon_bgr if canon else downscale_long_side(img, CFG.ROBUSTNESS_LONG_SIDE)
    region = inspected_region(cls, info.get("valid_mask")) if canon else None
    score, amap = baseline_predict_single(model_key, cls, x, region=region)
    st = DEPLOY_MODELS[(model_key, cls)]["map_stats"]
    zmap = (amap - st["mean"]) / st["std"] if st else (amap - amap.mean()) / (amap.std() + 1e-6)
    seen = region
    if seen is not None and seen.shape == zmap.shape and not seen.all():
        zmap = np.where(seen, zmap, min(float(zmap.min()), 0.0))    # never report a defect where the photo shows nothing
    r = baseline_results[model_key][cls]
    cal_key, thr_key = ("calib_scores_canon", "threshold_canon") if (canon and "threshold_canon" in r) else ("calib_scores", "threshold")
    thr = pac_threshold(r[cal_key], r["calib_labels"]) if operating_point == "pac" else r[thr_key]
    is_anom = bool(score > thr)
    regions = analyze_anomaly_regions(zmap, top_k=CFG.MAX_REGIONS) if is_anom else []
    if regions and "type_regions" in globals():      # Part 1.4: one defect type per region, read on the ORIGINAL photo
        fy, fx = x.shape[0] / float(zmap.shape[0]), x.shape[1] / float(zmap.shape[1])
        bx = {id(g): (g["bbox"][0] * fx, g["bbox"][1] * fy, (g["bbox"][0] + g["bbox"][2]) * fx, (g["bbox"][1] + g["bbox"][3]) * fy)
              for g in regions}                                        # boxes in the scored image's pixels
        if canon:                                                      # aligned frame -> original photo
            ph = {k: aligned_box_to_photo(b, info, img.shape[:2]) for k, b in bx.items()}
        else:                                                          # downscaled photo -> original photo
            s = img.shape[1] / float(x.shape[1])
            ph = {k: tuple(v * s for v in b) for k, b in bx.items()}
        if all(v is not None for v in ph.values()):
            type_regions(regions, img, cls, lambda g: ph[id(g)])
        else:                                                          # not registered: type on the scored image
            type_regions(regions, x, cls, lambda g: bx[id(g)])
    counts = {p: sum(1 for g in regions if g.get("priority") == p) for p in ("CRITICAL", "MAJOR", "MINOR")}

    print("=" * 78)
    print("PCB INSPECTION REPORT -- " + system_name)
    print("Board: %s | registration: %s, rotation undone %.1f deg, scale %.3f, %d keypoints%s" % (
        cls, "OK" if info["registered"] else "FAILED (inspected without alignment)", info["angle_deg"], info["scale"],
        info["inliers"], " | lighting corrected" if info["photometric"] else ""))
    if canon:
        _print_inspected_area(info, cls)
    print("Verdict: %s  (score %.4f vs threshold %.4f, %s operating point)" % (
        "DEFECTIVE" if is_anom else "PASS", score, thr, operating_point))
    if regions:
        print("Defect regions: %d  (CRITICAL %d, MAJOR %d, MINOR %d)" % (len(regions), counts["CRITICAL"], counts["MAJOR"], counts["MINOR"]))
        for k, g in enumerate(regions, 1):
            print("  #%d [%s] centre (%d, %d) px, %.2f%% of board -> %s%s" % (
                k, g.get("priority", "?"), g["center"][0], g["center"][1], 100 * g["area_frac"], g["shape_hint"],
                ("  | type: " + region_type_text(g, alternatives=True)) if g.get("defect_type") else ""))
        print("Priority and shape hints are heuristics; 'type' is the trained region classifier of Part 1.4.")
    elif is_anom:
        print("No defect region inside the inspected area -- the evidence lies at the edge of the photo; photograph the "
              "whole board to confirm.")
    print("=" * 78)

    if visualize:
        rgb = cv2.cvtColor(x, cv2.COLOR_BGR2RGB)
        if seen is not None and seen.shape == rgb.shape[:2] and not seen.all():
            rgb = rgb.copy()
            rgb[~seen] = (rgb[~seen] * 0.35).astype(np.uint8)          # darkened = not in the photo, not inspected
        fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
        ax[0].imshow(rgb); ax[0].set_title("input (canonicalized)" if canon else "input")
        ax[1].imshow(rgb); ax[1].imshow(np.clip(zmap, 0, 6), cmap="jet", alpha=0.45); ax[1].set_title("anomaly map (z-scale)")
        ax[2].imshow(draw_regions_on_image(rgb, regions) if regions else rgb)
        ax[2].set_title(("DEFECTIVE: %d region(s)" % len(regions)) if is_anom else "PASS")
        for a in ax:
            a.axis("off")
        plt.tight_layout()
        plt.show()
    return {"system": system_name, "pcb_class": cls, "score": score, "threshold": thr, "is_anomalous": is_anom,
            "regions": regions, "priority_counts": counts, "registration": info, "anomaly_map_z": zmap}


print("Inspection tool ready. Systems with kept models:",
      sorted({m for m, _ in DEPLOY_MODELS}) or "none (baselines not run)", "+ our pipelines")

In [ ]:
# ============================================================
# CELL 22 -- DEMO: honest qualitative results with the RECOMMENDED system (or any system / your own photos)
# ============================================================
doc_section("CELL 22", "DEMO: honest qualitative results with the RECOMMENDED system (or any system / your own photos)")
# One example can mislead either way: a real run's demo happened to pick a defective pcb2 board that the recommended
# system MISSED -- consistent with its ~55% recall at the calibrated threshold. So by default this shows ONE EXAMPLE OF
# EACH OUTCOME -- caught defect, missed defect, correctly passed board, false alarm -- drawn at random (fixed seed) from
# the held-out boards, using the same scores and threshold as the benchmark, plus the full outcome tally.
MY_IMAGE_PATHS = []          # leave empty for the gallery; or list your own photo paths, e.g. ["/kaggle/working/my_pcb.jpg"]
MY_PCB_CLASS = None          # "pcb1".."pcb4" if known; None = identify it by registration
DEMO_SYSTEM = "auto"         # "auto" = Cell 20b's deployment pick; or any name from its table, e.g.
                             # "EfficientAD -- canonicalized input", "Ours -- canonical pipeline", "Ours -- raw input"
OPERATING_POINT = "empirical"   # "empirical" (higher recall) or "pac" (guaranteed <= 10% false alarms, 90% confidence)


def _available(name):
    det, canon = system_parts(name)
    if det == "Ours":
        return bool(detectors_canon) if canon else True
    key = det + " (canonical pipeline)" if "canonical pipeline" in name else det
    return all((key, c) in DEPLOY_MODELS for c in CFG.CLASSES)


pick = next((s["name"] for s in systems if s.get("deploy")), None) if systems else None
system = pick if DEMO_SYSTEM == "auto" else DEMO_SYSTEM
for fallback in [system, "Ours -- canonical pipeline", "Ours -- raw input"]:
    if fallback and _available(fallback):
        if fallback != system:
            print(("'%s' is not available in this run" % system) if system else "No ranked system in this run",
                  "-- using '%s'." % fallback)
        system = fallback
        break
print("System:", system, "| operating point:", OPERATING_POINT)
OUTCOMES = ["caught defect (true positive)", "missed defect (false negative)",
            "correctly passed (true negative)", "false alarm (false positive)"]


def outcome_gallery(system, seed=SEED):
    """Pools of held-out boards per outcome for `system`, from the benchmark's identity(+canon) scores."""
    det, canon = system_parts(system)
    if det == "Ours":
        model = "Ours (canonical pipeline)" if canon else ("Ours (TTA)" if "TTA" in system else "Ours (single-pass)")
    else:
        model = det + " (canonical pipeline)" if "canonical pipeline" in system else det
    cond = "identity+canon" if canon else "identity"
    pools = {k: [] for k in OUTCOMES}
    for cls in CFG.CLASSES:
        e = ROBUST_SCORES.get(cls, {}).get(cond) if "ROBUST_SCORES" in globals() else None
        if not e or model not in e:
            continue
        if det == "Ours":
            d = (detectors_canon if canon else detectors)[cls]
            ss = d.score_stats
            tta = model == "Ours (TTA)"            # TTA systems are judged on TTA scores -> TTA calibration scores
            if OPERATING_POINT == "pac":
                thr = (pac_threshold(ss["calib_scores_tta"], ss.get("calib_labels_tta", ss["calib_labels"])) if tta
                       else pac_threshold(ss["calib_scores"], ss["calib_labels"]))
            else:
                thr = d.threshold_tta if tta else d.threshold
        else:
            r = baseline_results[model][cls]
            ck, tk = ("calib_scores_canon", "threshold_canon") if (canon and "threshold_canon" in r) else ("calib_scores", "threshold")
            thr = pac_threshold(r[ck], r["calib_labels"]) if OPERATING_POINT == "pac" else r[tk]
        orig = {(int(l), os.path.splitext(os.path.basename(p))[0]): p
                for p, l in zip(FILE_LISTS[cls]["eval_images"], FILE_LISTS[cls]["eval_labels"])}
        for (_, lab, stem), sc in zip(ROBUST_SETS[cls][cond], e[model]):
            p = orig.get((int(lab), stem))
            if p is not None:
                flagged = bool(sc > thr)
                pools[OUTCOMES[(0 if flagged else 1) if lab == 1 else (3 if flagged else 2)]].append((cls, p))
    return pools


demo_results = []
pools = outcome_gallery(system) if not MY_IMAGE_PATHS else {}
if MY_IMAGE_PATHS:
    for path in MY_IMAGE_PATHS:
        print("\n### " + os.path.basename(path) + " ###")
        demo_results.append(inspect_with_system(path, system, pcb_class=MY_PCB_CLASS, operating_point=OPERATING_POINT))
elif any(pools.values()):
    n = {k: len(v) for k, v in pools.items()}
    tp, fn, tn, fp = (n[k] for k in OUTCOMES)
    print("Outcome tally on %d held-out boards: %d caught, %d missed (recall %.0f%%) | %d passed, %d false alarms (%.0f%%)" % (
        sum(n.values()), tp, fn, 100.0 * tp / max(1, tp + fn), tn, fp, 100.0 * fp / max(1, tn + fp)))
    rng = np.random.default_rng(SEED)
    for k in OUTCOMES:
        if not pools[k]:
            print("\n### " + k + ": none in this run ###")
            continue
        cls, path = pools[k][int(rng.integers(len(pools[k])))]
        print("\n### " + k.upper() + " -- " + cls + "/" + os.path.basename(path) + " ###")
        res = inspect_with_system(path, system, pcb_class=cls, operating_point=OPERATING_POINT)
        demo_results.append(res)
        live = bool(res.get("is_anomalous", res.get("is_anomaly", False))) if isinstance(res, dict) else None
        expect = k.startswith("caught") or k.startswith("false alarm")
        print("SELF-CHECK: benchmark outcome '%s' -> live verdict %s: %s" % (
            k, "DEFECTIVE" if live else "PASS", "consistent" if live == expect else "MISMATCH -- please report"))
        _tt = globals().get("DEFECT_TRUE", {}).get((cls, os.path.basename(path)))
        _regs = [g for g in ((res.get("regions") or []) if isinstance(res, dict) else []) if g.get("defect_type")]
        if _tt or _regs:
            print("REGION TYPES: %s | true type (VisA label): %s" % (
                ", ".join("#%d %s" % (i, region_type_text(g)) for i, g in enumerate(_regs, 1)) or "-",
                "+".join(_tt) if _tt else "- (normal board)"))
else:                                           # no robustness scores in this run: one defective + one good board
    _ev = FILE_LISTS["pcb2"]
    for lab in (1, 0):
        path = next(p for p, l in zip(_ev["eval_images"], _ev["eval_labels"]) if l == lab)
        print("\n### " + ("known DEFECTIVE" if lab else "known GOOD") + " board " + os.path.basename(path) + " ###")
        demo_results.append(inspect_with_system(path, system, pcb_class=MY_PCB_CLASS, operating_point=OPERATING_POINT))
result = demo_results[0] if demo_results else None

## Part 4.1 -- Demo: 2 good + 2 defective boards per PCB, heatmaps, highlighted defects, pixel details

In [ ]:
# ============================================================
# CELL 22b -- DEMO: 2 GOOD + 2 DEFECTIVE BOARDS PER PCB -- verdict, heatmap, highlighted defect, pixel details
# ============================================================
doc_section("CELL 22b", "DEMO: 2 GOOD + 2 DEFECTIVE BOARDS PER PCB -- verdict, heatmap, highlighted defect, pixel details")
# The boards are drawn AT RANDOM (fixed seed) from the held-out set -- not hand-picked -- so the demo shows the system as it
# is, including any mistake. For every board: the original photo with the detected regions drawn in ORIGINAL-PHOTO pixel
# coordinates, the aligned board with the inspected region, the anomaly heatmap, the highlighted defect pixels, and for
# defective boards the pixel-level comparison with VisA's ground truth. Each defective board is then shown under every
# aligned system side by side.
demo_summary_df = demo_pixel_df = None
with doc_step("demo boards"):
    rng = np.random.default_rng(SEED + 22)
    sums, pix = [], []
    for cls in detectors_canon:
        ref = load_canon_reference(cls, CFG.ROBUSTNESS_LONG_SIDE)
        ev = list(zip(FILE_LISTS[cls]["eval_images"], FILE_LISTS[cls]["eval_labels"]))
        good = [p for p, l in ev if l == 0]; bad = [p for p, l in ev if l == 1]
        picks = ([(good[k], 0) for k in rng.choice(len(good), CFG.DEMO_GOOD_PER_CLASS, replace=False)] +
                 [(bad[k], 1) for k in rng.choice(len(bad), CFG.DEMO_BAD_PER_CLASS, replace=False)])
        for p, l in picks:
            res = inspect_board(p, cls, classify_types=True)
            true_type = globals().get("DEFECT_TRUE", {}).get((cls, os.path.basename(p))) if l == 1 else None
            gt = _aligned_defect_mask(p, ref) if l == 1 else None
            log("%s | %s | truth %s -> verdict %s" % (cls, os.path.basename(p), "DEFECTIVE" if l else "NORMAL", "DEFECTIVE" if res["verdict"] else "PASS"))
            print_inspection(res, truth=l, gt=gt, true_type=true_type)
            plot_inspection(res, gt=gt, title="%s -- %s board %s: %s (score %.3f, threshold %.3f)" % (
                cls, "DEFECTIVE" if l else "NORMAL", os.path.basename(p), "DEFECTIVE" if res["verdict"] else "PASS", res["score"], res["threshold"]))
            g = (gt & res["region"]) if gt is not None else None
            inter = int((res["hot"] & g).sum()) if g is not None else None
            sums.append({"pcb_class": cls, "file": os.path.basename(p), "ground truth": "DEFECTIVE" if l else "NORMAL",
                         "verdict": "DEFECTIVE" if res["verdict"] else "PASS", "correct": bool(res["verdict"]) == bool(l),
                         "score": round(res["score"], 4), "threshold": round(res["threshold"], 4),
                         "PAC verdict": "DEFECTIVE" if res["verdict_pac"] else "PASS", "regions": len(res["regions"]),
                         "highlighted pixels": int(res["hot"].sum()), "most anomalous pixel (original x,y)": res["max_pixel"]["original photo (x,y)"],
                         "defect pixels": int(g.sum()) if g is not None else None,
                         "defect pixels highlighted (%)": round(100.0 * inter / max(1, int(g.sum())), 1) if g is not None else None,
                         "IoU": round(inter / max(1, int((res["hot"] | g).sum())), 3) if g is not None else None,
                         "rotation undone (deg)": round(res["info"]["angle_deg"], 2), "inspected % of board": res["inspected % of board"],
                         "decision (3-level)": three_level_decision(res),
                         "true defect type": "+".join(true_type) if true_type else ("-" if l == 0 else "unlabelled"),
                         "region types": ", ".join("#%d %s" % (g["region"], g["defect_type"]) for g in res["regions"] if g.get("defect_type")) or "-",
                         "top region type correct": (res["regions"][0].get("defect_type") in true_type)
                                                    if (true_type and res["regions"] and res["regions"][0].get("defect_type")) else None})
            pix += [dict({"pcb_class": cls, "file": os.path.basename(p)}, **{k: v for k, v in r.items() if k != "bbox aligned (x,y,w,h)"})
                    for r in res["regions"]]
        for p, l in picks:
            if l == 1 and len(PIX_SYSTEMS) > 1:
                plot_system_comparison(p, cls, gt=_aligned_defect_mask(p, ref),
                                       title="%s -- defective board %s under every aligned system" % (cls, os.path.basename(p)))
    demo_summary_df, demo_pixel_df = pd.DataFrame(sums), pd.DataFrame(pix)
    print(demo_summary_df.to_string(index=False))
    save_table(demo_summary_df, "demo_boards_summary", "Demo: 2 normal + 2 defective random held-out boards per class -- verdicts and pixel agreement")
    save_table(demo_pixel_df, "demo_pixel_details", "Demo: every detected region with original-photo pixel coordinates, peak and mean anomaly, priority")
    log("demo: %d of %d boards judged correctly" % (int(demo_summary_df.correct.sum()), len(demo_summary_df)))
    _typed = demo_summary_df["top region type correct"].dropna()
    if len(_typed):
        log("demo: top region typed correctly on %d of %d flagged defective boards" % (int(_typed.sum()), len(_typed)))

## Part 4.2 -- Test any image

In [ ]:
# ============================================================
# CELL 22c -- TEST ANY IMAGE: point MY_IMAGE_PATH at a photo of a PCB
# ============================================================
doc_section("CELL 22c", "TEST ANY IMAGE: point MY_IMAGE_PATH at a photo of a PCB")
# The board type is identified automatically (registration to each class's golden reference); the photo may be rotated,
# unevenly lit or show only part of the board. The report gives the verdict at both operating points, the inspected share
# of the board, the most anomalous pixel and every detected region in ORIGINAL-PHOTO pixel coordinates, and (optionally)
# the same photo under every other aligned system.
MY_IMAGE_PATH = None             # e.g. "/kaggle/input/my-pcb-photos/board.jpg"; None = a held-out example board
MY_PCB_CLASS = None              # "pcb1".."pcb4" if known; None = identify automatically
COMPARE_WITH_OTHER_MODELS = True


def test_image(path, pcb_class=None, compare=True):
    res = inspect_board(path, pcb_class, classify_types=True)
    print_inspection(res, true_type=globals().get("DEFECT_TRUE", {}).get((res["pcb_class"], os.path.basename(str(path)))))
    plot_inspection(res, title="Test image %s -- %s: %s" % (os.path.basename(str(path)), res["pcb_class"], "DEFECTIVE" if res["verdict"] else "PASS"))
    if compare and len(PIX_SYSTEMS) > 1:
        plot_system_comparison(path, res["pcb_class"], title="Test image under every aligned system")
    return res


with doc_step("test a single image"):
    if MY_IMAGE_PATH is None:
        cls0 = [c for c in CFG.CLASSES if c in detectors_canon][0]
        MY_IMAGE_PATH = [o for o, l in zip(FILE_LISTS[cls0]["eval_images"], FILE_LISTS[cls0]["eval_labels"]) if l == 1][0]
        log("No MY_IMAGE_PATH set -- using a held-out defective board as the example: " + MY_IMAGE_PATH)
    my_result = test_image(MY_IMAGE_PATH, MY_PCB_CLASS, COMPARE_WITH_OTHER_MODELS)

## Thesis output index

In [ ]:
# ============================================================
# CELL 99 -- THESIS OUTPUT INDEX: every figure, table and log of this run, organised for writing
# ============================================================
doc_section("CELL 99", "THESIS OUTPUT INDEX: every figure, table and log of this run, organised for writing")
import json, shutil
_close_section(); _DOC["section"] = None
KEY_TABLES = [   # (variable, caption, the cell that produces it)
    ('comparison_df', 'Standard benchmark: held-out AUROC with 95% CIs, every model and class', '18'),
    ('paired_df', 'Paired differences: our raw model vs every model', '18'),
    ('paired_canon_df', 'Canonicalized benchmark: our final system vs every aligned model', '18'),
    ('thresholded_df', 'Operating points: false alarms and recall at the empirical and PAC thresholds', '18'),
    ('final_summary_df', 'System ranking under rotation, lighting and crop shifts', '20b'),
    ('system_pairwise_df', 'Pairwise 6-shift differences between systems', '20b'),
    ('seed_study_df', 'Seed robustness: every top system re-trained with 3 seeds', '20c'),
    ('seed_pairs_df', 'Seed-robust verdicts', '20c'),
    ('robust_summary_df', 'Robustness benchmark per condition', '20'),
    ('board_mask_df', 'Board masks: area and ground-truth coverage', '15d2'),
    ('confound_df', 'Capture-condition confounds of VisA', '15e'),
    ('residual_light_df', 'Lighting cue left after alignment', '15e2'),
    ('canon_study_df', 'Design study: letterbox x CLAHE (selected on calibration)', '15e2'),
    ('aug_study_df', 'Design study: augmentation (selected on calibration)', '15e3'),
    ('improvement_df', 'Design study: feature variants (selected on calibration)', '15c2'),
    ('ablation_df', None, 'D5'),
]
with doc_step("export key result tables"):
    # FIXED: each table is recorded under the cell that produced it. They used to be saved with no section, so the chapter
    # map listed no tables for the seed study, the design studies and the dataset confounds.
    for var, cap, src in KEY_TABLES:
        df = globals().get(var)
        if isinstance(df, pd.DataFrame) and len(df) and cap:
            _DOC["section"] = "CELL " + src
            try:
                save_table(df.reset_index() if var == "system_pairwise_df" else df, var, cap)
            finally:
                _DOC["section"] = None
CHAPTERS = [("Dataset", ["D1"]), ("Method: preprocessing, alignment, board-only inspection", ["D2", "7c", "15d2"]),
            ("Method: the hybrid CNN+ViT detector", ["D3", "8", "9"]), ("Design choices (selected on calibration)", ["15c2", "15e2", "15e3"]),
            ("Final model", ["D4"]), ("Ablation", ["D5", "15c"]), ("Benchmark & comparison with other models", ["18", "D6"]),
            ("Robustness: rotation, lighting, crops", ["20", "20b", "20d", "20f"]), ("Seed robustness", ["20c"]),
            ("Defect localization (pixel level)", ["20e"]), ("Defect type of every highlighted region", ["12b", "12c", "20e2"]), ("Qualitative results & demo", ["22", "22b", "22c"]),
            ("Dataset confounds & limitations", ["15e", "15e2"])]
with doc_step("chapter map"):
    # FIXED: also written to thesis_chapter_map.md, BEFORE the archive is made -- it used to exist only in the notebook log
    _rows = ["# Where each thesis chapter finds its evidence (figures / tables of this run)", "",
             "| chapter | figures | tables |", "|---|---|---|"]
    print("\nWHERE EACH THESIS CHAPTER FINDS ITS EVIDENCE (figures / tables of this run):")
    for chap, labels in CHAPTERS:
        f = [r["figure"] for r in FIGURES if any(str(r["section"]).endswith(" " + lb) for lb in labels)]
        t = [r["table"] for r in TABLES if any(str(r["section"]).endswith(" " + lb) for lb in labels)]
        fs, ts = (", ".join(map(str, f)) or "-"), (", ".join(map(str, t)) or "-")
        print("  %-58s figures %s | tables %s" % (chap, fs, ts))
        _rows.append("| %s | %s | %s |" % (chap, fs, ts))
    with open(os.path.join(DOC_DIR, "thesis_chapter_map.md"), "w") as fh:
        fh.write("\n".join(_rows) + "\n")
with doc_step("write indexes, manifest and archive"):
    steps_df = pd.DataFrame(STEPS)
    steps_df.to_csv(os.path.join(DOC_DIR, "section_timings.csv"), index=False)
    fig_df, tab_df = pd.DataFrame(FIGURES), pd.DataFrame(TABLES)
    fig_df.to_csv(os.path.join(DOC_DIR, "figures_index.csv"), index=False); tab_df.to_csv(os.path.join(DOC_DIR, "tables_index.csv"), index=False)
    with open(os.path.join(DOC_DIR, "figures_index.md"), "w") as fh:
        fh.write("# Figures of this run\n\n")
        for sec, g in fig_df.groupby("section", sort=False):
            fh.write("## %s -- %s\n\n" % (sec, g["section title"].iloc[0]))
            for r in g.itertuples():
                fh.write("- **Figure %d** (`figures/%s`): %s\n" % (r.figure, r.file, r.caption))
            fh.write("\n")
    with pd.ExcelWriter(os.path.join(DOC_DIR, "thesis_tables.xlsx")) as xw:
        for r in tab_df.itertuples():
            pd.read_csv(os.path.join(TAB_DIR, r.file)).to_excel(xw, sheet_name=("T%03d_" % r.table + _slug(r.file[7:-4]))[:31], index=False)
    manifest = {"finished": datetime.datetime.now().isoformat(timespec="seconds"), "total_minutes": round((time.time() - _DOC["t0"]) / 60.0, 1),
                "figures": len(FIGURES), "tables": len(TABLES), "versions": globals().get("VERSIONS", {}),
                "config": {k: v for k, v in vars(CFG).items() if not k.startswith("_") and isinstance(v, (int, float, str, bool, list, tuple, dict))}}
    with open(os.path.join(DOC_DIR, "run_manifest.json"), "w") as fh:
        json.dump(manifest, fh, indent=1, default=str)
    for extra in ("results_summary.md", os.path.basename(CFG.EXCEL_OUTPUT_PATH)):
        src = "/kaggle/working/" + extra
        if os.path.exists(src):
            shutil.copy(src, os.path.join(DOC_DIR, extra))
    print("Disk at the end of the run (%.1f GB free):" % disk_free_gb()); disk_report()
    shutil.make_archive("/kaggle/working/thesis_outputs", "zip", DOC_DIR)
    print(steps_df.to_string(index=False))
    log("Run finished: %d figures, %d tables, %.1f min. Download /kaggle/working/thesis_outputs.zip" % (
        len(FIGURES), len(TABLES), manifest["total_minutes"]))

## Part 10 -- Final system for the live-demo wrapper (added after the code freeze)

Saves the final canonical-pipeline system so that a small wrapper can inspect new photographs without re-fitting. This cell only reads what the notebook built and writes files; every result above is unchanged.

In [ ]:
doc_section("CELL 100", "Save the final system for the live-demo wrapper")
# CELL 100 -- Save the final system for the live-demo wrapper
# =====================================================================================================
# ADDED AFTER THE CODE FREEZE. Writes everything a small inference wrapper needs to inspect a new photograph with
# the FINAL system (Ours -- canonical pipeline) without re-fitting. It only READS objects built above and WRITES
# files: no result of this notebook is changed. Every step is fail-safe -- a problem is logged, never raised.
# Output folder: /kaggle/working/pcb_anomaly_artifacts/final_system/
#   final_detectors.pkl    canonical-pipeline detectors as plain data: memory banks, bank densities, alpha, branch
#                          statistics, thresholds, score_stats (incl. the strict PAC threshold), feat_cfg, beta
#   golden_references.pkl  per design: golden reference image, GROWN board mask, ORB keypoints (as arrays) and
#                          descriptors, ECC images, path
#   pixel_thresholds.json  calibrated pixel-highlight thresholds per (system, design)
#   demo_images.json       held-out image paths and labels per design, for a demo picker
#   config.json            snapshot of CFG
#   manifest.json          files and sizes, library versions, anything skipped
# The defect-type head is already saved by Cells 12c / 20e2 in the parent folder (defect_type_classifiers.pkl).
# Neural networks (the frozen CNN and ViT) are NOT saved: a wrapper rebuilds them from the code, as Cell 6 does.
# =====================================================================================================
import pickle as _pk, json as _js, os as _os, time as _tm, platform as _pf
FINAL_DIR = _os.path.join(CFG.ARTIFACT_DIR, "final_system")
_os.makedirs(FINAL_DIR, exist_ok=True)
_FS = {"created": _tm.strftime("%Y-%m-%d %H:%M:%S"), "system": "Ours (canonical pipeline)", "files": {}, "skipped": {}}


def _fs_plain(v):
    """Detached CPU copy of torch tensors (containers handled recursively); everything else unchanged."""
    try:
        import torch as _torch
        if isinstance(v, _torch.Tensor):
            return v.detach().cpu()
    except ImportError:
        pass
    if isinstance(v, dict):
        return {k: _fs_plain(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        out = [_fs_plain(x) for x in v]
        return tuple(out) if isinstance(v, tuple) else out
    return v


def _fs_is_network(v):
    try:
        import torch as _torch
        return isinstance(v, _torch.nn.Module)
    except ImportError:
        return False


def _fs_state(obj, max_mb=600):
    """Every attribute of obj that pickles (networks excluded); returns (state, skipped)."""
    keep, skipped = {}, []
    for k, v in vars(obj).items():
        if _fs_is_network(v):
            skipped.append("%s (network: rebuilt from code)" % k)
            continue
        try:
            pv = _fs_plain(v)
            size = len(_pk.dumps(pv, protocol=4))
            if size > max_mb * 1e6:
                skipped.append("%s (%.0f MB)" % (k, size / 1e6))
                continue
            keep[k] = pv
        except Exception as e:
            skipped.append("%s (%s)" % (k, type(e).__name__))
    return keep, skipped


def _fs_write(name, obj, as_json=False):
    path = _os.path.join(FINAL_DIR, name)
    if as_json:
        with open(path, "w") as fh:
            _js.dump(obj, fh, indent=1, default=str)
    else:
        with open(path, "wb") as fh:
            _pk.dump(obj, fh, protocol=4)
    _FS["files"][name] = round(_os.path.getsize(path) / 1e6, 2)
    return path


# 1. the final canonical-pipeline detectors
try:
    _dets = {}
    for _cls, _det in globals().get("detectors_canon", {}).items():
        _state, _skip = _fs_state(_det)
        _dets[_cls] = {"class": type(_det).__name__, "state": _state}
        _FS["skipped"]["detector " + _cls] = _skip
    _fs_write("final_detectors.pkl", {"system": _FS["system"], "detectors": _dets})
    log("final system: %d canonical-pipeline detectors saved (%s)" % (len(_dets), ", ".join(sorted(_dets))))
except Exception as _e:
    log("final system: detectors NOT saved (%r)" % (_e,), "WARN")

# 2. golden references with their grown board masks (keypoints stored as arrays: cv2.KeyPoint does not pickle)
try:
    _refs = {}
    for (_cls, _ls), _ref in globals().get("_CANON_REF_CACHE", {}).items():
        _r = {"long_side": _ls}
        for _k, _v in _ref.items():
            if _k == "kp":
                _r["kp_arrays"] = np.array([[p.pt[0], p.pt[1], p.size, p.angle, p.response, p.octave, p.class_id] for p in _v], np.float32)
                continue
            try:
                _pk.dumps(_v, protocol=4)
                _r[_k] = _v
            except Exception as _e:
                _FS["skipped"].setdefault("reference " + _cls, []).append("%s (%s)" % (_k, type(_e).__name__))
        _refs["%s|%d" % (_cls, _ls)] = _r
    _fs_write("golden_references.pkl", _refs)
    log("final system: golden references and grown board masks saved (%s)" % ", ".join(sorted(_refs)))
except Exception as _e:
    log("final system: golden references NOT saved (%r)" % (_e,), "WARN")

# 3. calibrated pixel thresholds, 4. demo images, 5. configuration
for _name, _make in (
        ("pixel_thresholds.json", lambda: [{"system": s, "pcb_class": c, "threshold": float(v)} for (s, c), v in globals().get("PIX_THR", {}).items()]),
        ("demo_images.json", lambda: {c: {"eval_images": list(FILE_LISTS[c]["eval_images"]), "eval_labels": [int(x) for x in FILE_LISTS[c]["eval_labels"]]} for c in CFG.CLASSES}),
        ("config.json", lambda: {k: getattr(CFG, k) for k in dir(CFG) if k.isupper()})):
    try:
        _fs_write(_name, _make(), as_json=True)
    except Exception as _e:
        log("final system: %s NOT saved (%r)" % (_name, _e), "WARN")

# 6. manifest
try:
    _FS["versions"] = {}
    for _mod in ("torch", "timm", "cv2", "numpy", "sklearn", "PIL"):
        try:
            _FS["versions"][_mod] = __import__(_mod).__version__
        except Exception:
            pass
    _FS["python"] = _pf.python_version()
    _FS["type_head"] = "../defect_type_classifiers.pkl (written by Cells 12c / 20e2)"
    _FS["how_to_load"] = ("Run the definition cells (1-8, 11, 12c, 17, 20e) or the wrapper notebook, rebuild each golden "
                          "reference from kp_arrays/des, and restore each detector's state into a PCBAnomalyDetector.")
    _fs_write("manifest.json", _FS, as_json=True)
except Exception as _e:
    log("final system: manifest NOT saved (%r)" % (_e,), "WARN")

print("Final system for the live-demo wrapper -> " + FINAL_DIR)
for _n, _mb in sorted(_FS["files"].items()):
    print("   %-24s %8.2f MB" % (_n, _mb))
print("   total %.1f MB | skipped: %s" % (sum(_FS["files"].values()), {k: v for k, v in _FS["skipped"].items() if v} or "nothing"))
